<a href="https://colab.research.google.com/github/dbhangre/nifty-breach-mlcore-vs-normal/blob/main/1004_Simple_Project_2_Acceptable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
# ============================================================================
# C1_Config — CONFIGURATION AND SHARED LIBRARY
# ============================================================================
#
# WHAT THIS IS
# ------------
# The one place every constant and every shared function lives. It reads no
# data, trains nothing and writes nothing. Run it first; every later script
# adds to the same namespace.
#
# THE STUDY IN SHORT
# ------------------
# NIFTY 50 trades a weekly options cycle. At the first day's close a price band
# is drawn around the index. On each later day we ask: will the expiry close
# land outside the band? Two contestants answer. The GAUSSIAN computes the
# probability from a lognormal assumption and has no free parameters. The ML
# CLASSIFIER learns it from past cycles. The question is whether learning beats
# the closed form on the same inputs.
#
# THE PIPELINE — EXCEL IN, EXCEL OUT
# ----------------------------------
#   C1_Config          (this file)         no file
#   C2_Fetch           NSE             ->  01_daily_raw.xlsx
#   C3_Clean           01              ->  02_daily_clean.xlsx
#   C4_Cycles          02              ->  03_cycles_<align>.xlsx
#   C5_Features        02 + 03         ->  04_features_sigma<k>_win<W>_<align>.xlsx
#   C6_TrainPredict    04              ->  05_results_sigma<k>_win<W>_<align>.xlsx
#
# Every script reads the previous script's workbook from disk. Nothing is
# passed in memory, so any step can be re-run on its own and every intermediate
# stage can be opened and inspected in Excel.
#
# ----------------------------------------------------------------------------
# THREE THINGS THAT CHANGED FROM THE CAPSTONE VERSION — READ THESE
# ----------------------------------------------------------------------------
#
# 1. SHORT CYCLES ARE NOW INCLUDED.
#    A holiday week has four trading days instead of five. Those cycles used to
#    be dropped, which cost 104 of 396 cycles. They are now kept, each row
#    declaring its own `cycle_days`, and days_left is derived from that rather
#    than assumed. A four-day cycle contributes its D2 and D3 decisions and has
#    no D4, which is simply the truth about that week.
#
#    Nothing is imputed to achieve this. Copying the previous day's prices into
#    a missing day would manufacture a zero return and a zero gap, deflating
#    realized_vol on a quarter of all rows — and z_dist DIVIDES by realized_vol.
#    The label would stay real while the feature became fiction, with no way to
#    tell afterwards which rows were touched.
#
# 2. THE BAND STATISTICS ARE ESTIMATED PER STEP.
#    This follows directly from (1) and is the one subtle consequence. sigma is
#    the standard deviation of ln(expiry_close / d1_close), but that move spans
#    three trading steps in a four-day cycle and four in a five-day cycle, so
#    the two cannot be pooled raw. Both statistics are therefore estimated on a
#    PER-STEP basis and scaled back to the cycle in hand:
#
#        mu_step    = rolling mean of  ln(expiry/d1) / steps
#        sigma_step = rolling std  of  ln(expiry/d1) / sqrt(steps)
#
#        band_upper = d1_close * exp(mu_step*C + k*sigma_step*sqrt(C))
#        band_lower = d1_close * exp(mu_step*C - k*sigma_step*sqrt(C))
#
#    where C is that cycle's own step count. On an all-five-day sample this is
#    identical to the old formula, so nothing is given up.
#
# 3. days_left IS NOW A REAL FEATURE.
#    It used to be constant inside a task cell and therefore carried no
#    information — a scaler cancels a per-column constant exactly. With short
#    cycles included it varies across rows, so it is offered to the models. Only
#    the DECISION day's value is included: days_left_D1..D4 differ by fixed
#    constants within a row, so including all four would be exactly collinear.
#
# ----------------------------------------------------------------------------
# WHAT IS DELIBERATELY KEPT
# ----------------------------------------------------------------------------
# The one-cycle lag on the band statistics, the feature whitelist, the
# walk-forward split, the cross-fitted thresholds, the consistency gate and the
# self-tests. Those are what make the numbers trustworthy. The plumbing has been
# simplified repeatedly; the guards never have.
#
# WHAT WAS REMOVED along the way: the cache-version machinery and its filename
# parsers (replaced by RUN_TAG in every filename), the three CYCLE_ALIGN modes
# (replaced by IMPUTE_DAILY, which fixes the calendar one layer earlier), the
# asymmetric-band path, the feature-builder chain and its runtime function
# wrapping, the hand-rolled clip-and-scale pair (now an sklearn Pipeline), the
# parquet checkpoints and the timestamped results folders.
#
# The neural net and the payoff matrix were removed once and then put BACK, so
# do not trust an older comment that says otherwise — both are live here.
# ============================================================================

import os
import math
import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 78)
print("  C1_Config — configuration and shared library")
print("=" * 78)


# ============================================================================
# SECTION 0 — THE CONTROL PANEL
# ============================================================================
# EVERY knob you would change between runs is in this one block. Nothing below
# needs editing to try a different combination; the long explanations for each
# setting stay where they always were, further down, and each of those places
# now points back here instead of holding a second copy of the value.
#
# WHAT CHANGES WHAT, because the three groups are not interchangeable:
#
#   DATA knobs change the ROWS and therefore the LABEL. The band defines a
#   breach, so moving any of these re-scores the Gaussian too. Two runs that
#   differ here are two different studies, not two models.
#       SIGMA_GRID, SIGMA_WINDOW_GRID, RV_WINDOW_DAYS, IMPUTE_DAILY,
#       CYCLE_LENGTHS
#
#   MODEL knobs change only what the learner is given or allowed to do. The
#   label is untouched, so results ARE directly comparable.
#       FEATURE_SET, MODELS, TOP_K, TUNE_ITERS, DROP_CONSTANT_FEATURES
#
#   SCORING knobs change how the same predictions are judged. Nothing is
#   refitted; only the verdict moves.
#       PRIMARY_METRIC, F_BETA, FBETA_MIN_SKILL, GATE_MAX_GAP, GATE_MIN_RATIO,
#       FOLD_PASS_FRACTION, the four PAYOFF_* constants
#
# RUN_TAG carries CAL_TAG, CYCLE_LENGTHS, FEATURE_SET and RV_WINDOW_DAYS into
# every output filename, so those four can be varied without runs overwriting
# each other. SIGMA and the band window are in the filename too. The SCORING
# knobs are NOT in the filename, because they do not change the written
# features — but they do change C6's results workbook, so a scoring change
# overwrites the previous scoring of the same data. Re-run C6, not C5.
# ============================================================================

# ---- DATA: these change the rows and therefore the LABEL -------------------
#
# SIGMA_GRID — band width multipliers, k. EACH VALUE IS A SEPARATE STUDY. The
#   band defines a breach, so a different k gives different outcomes and a
#   different base rate. Report per sigma; never pool them into one leaderboard.
#
# SIGMA_WINDOW_GRID — how many CYCLES of history the band's mu and sigma see.
#
# RV_WINDOW_DAYS — how many TRADING DAYS go into realized_vol. None keeps the
#   old coupling (band window x 4). Measured on this file, two independent
#   tests — forecasting the next 1-4 days' dispersion with no model involved,
#   and scoring the Gaussian's own Brier with the band held fixed:
#
#       days      forecast MSE      Gaussian Brier    std error of the std
#          4         1.21x best         0.0998               +/- 41%
#          5         1.13x best         0.0991               +/- 35%
#          8         1.00x best         0.0969               +/- 27%
#         10         1.00x BEST         0.0966               +/- 24%
#         16         1.03x best         0.0953               +/- 18%
#         20         1.06x best         0.0948               +/- 16%
#     EWMA .94       1.00x best         0.0945 (best)            n/a
#
#   The curve is flat from roughly 8 to 20 days and rises sharply below it, and
#   z_dist DIVIDES by this number. 4 is the WORST of the seven. Run it as the
#   deliberate short-window arm of a comparison, not as the setting to report on
#   its own; RV_TAG keeps its files separate so 4 and 16 can be scored directly
#   against each other.
#
# IMPUTE_DAILY — True lets C2b insert a row for every missing weekday that was a
#   market holiday, carrying the previous day's OHLC and marking is_original = 0,
#   so every weekly cycle has four decision days and the D4 task keeps the rows a
#   holiday week used to cost it. NOMINAL EXPIRY DAYS ARE NEVER FILLED: when the
#   settlement weekday is a holiday the exchange settles early, so the missing day
#   IS the expiry and filling it would fabricate the label. C4 drops those cycles.
#   What makes this safe is that daily_realized_vol computes on is_original rows
#   only — an imputed day has a log return of exactly zero, and zeros shrink a
#   standard deviation that z_dist then divides by. Without that rule this switch
#   is not usable at all.
#
# CYCLE_LENGTHS — "all" keeps holiday-shortened cycles beside full ones (390 on
#   the 2019+ sample: 292 full, 98 short), which is the only thing that makes
#   days_left vary and so usable as a feature. "full_only" uses five-trading-day
#   cycles exclusively: a quarter of the sample discarded, but every row has the
#   same geometry. Under IMPUTE_DAILY=True this switch has little left to do.
SIGMA_GRID = [0.1, 0.25, 0.5] # band multipliers; each is a SEPARATE study
                              # sigma 0.10 alone is near-unanswerable: perfect
                              # foresight earns only +0.06 to +0.14 there, so
                              # there is almost nothing for a model to win.
SIGMA_WINDOW_GRID = [4]       # band mu/sigma window, in CYCLES
RV_WINDOW_DAYS = None         # realized_vol window in DAYS; None = window x 4
                              # None gives 16 days, inside the flat 8-20 region
                              # the table below measures. 4 was the WORST of the
                              # seven tried and z_dist divides by this number.
IMPUTE_DAILY = True           # True fills holiday weekdays; expiry days never
CYCLE_LENGTHS = "full_only"   # "all" keeps short cycles | "full_only" drops them

# ---- MODEL: these change only what the learner sees ------------------------
#
# FEATURE_SET — eight choices. Feature counts are D2 / D3 / D4, cumulative from
#   D1, own side only:
#
#     no macro file needed
#       "full"            z_dist, realized_vol, gap, range per day
#                         + band_width_pct, days_left_dec        10 / 14 / 18
#       "no_rv"           the same without realized_vol           8 / 11 / 14
#       "z_plus_width"    z_dist, gap, range + band_width_pct     7 / 10 / 13
#       "z_only"          z_dist per day, nothing else            2 /  3 /  4
#       "compact"         the path RE-EXPRESSED, not expanded:     6 /  7 /  8
#                         z level / direction / closest approach /
#                         spread, capped at the day index, plus
#                         realized_vol on the current day, the
#                         cumulative gap, the worst range and
#                         band_width_pct
#
#     needs 01c_macro_daily_<cal>.xlsx from C2c_FetchMacro
#       "full_plus_vix"   full, plus the VIX LEVEL per day       12 / 17 / 22
#       -"no_rv_plus_vix"  no_rv, plus the VIX LEVEL per day      10 / 14 / 18
#       -"full_plus_macro" full + VIX per day + usdinr/brent      14 / 19 / 24
#                         THE WIDEST SET. At D4 that is 24 features, which is
#                         7.5 events per variable at 180 breaches and 1.7 at 40.
#                         Expect it to lose to "full" and expect the gate to
#                         refuse it at wide sigmas. It is here as the
#                         everything-in arm, to show where more stops helping.
#       "compact_plus_macro"  "compact" plus VIX, usdinr and brent on
#                         the current day. The ONLY difference from
#                         "compact", so the pair isolates whether
#                         macro adds anything                      9 / 10 / 11
#       -"z_plus_macro"    z + VIX/rv per day + usdinr + brent     6 /  8 / 10
#       "z_only_plus_vix" z + VIX/rv per day, nothing else        4 /  6 /  8
#
#   LEVEL vs RATIO is not interchangeable. The _plus_vix sets use vix_daily
#   because realized_vol is either present (so the ratio would be a second,
#   nonlinear function of a column the model already has) or deliberately absent
#   (so the ratio would smuggle it back in through the denominator). See
#   task_features for the full argument.
#
#   MIND THE EPV COST. At roughly 63 positives in a D4 cell, "full" carries 18
#   features — about 3.5 events per variable, where 10 is the usual minimum for a
#   coefficient to mean anything. The wider sets exist to be COMPARED against
#   z_only, not assumed better for containing more.
#
#   EVIDENCE SO FAR, and its limits. On 386 cycles, out-of-sample AUC over four
#   expanding folds, logistic regression, everything else held equal: z_only
#   matched or beat "full" in 13 of 18 (sigma x task) cells, widest at D4 where
#   the sample is thinnest. The reason is arithmetic — z_dist already CONTAINS
#   the volatility and the horizon, so adding realized_vol back supplies no new
#   information, only parameters. But that test used one metric and one model,
#   while the headline here is F-beta at a fitted cut with four non-linear
#   learners. The switch exists so the comparison can be RUN; let C7's paired
#   bootstrap say whether a gap is real.
#
# MODELS — every name needs an entry in HYPERPARAMS.
# TOP_K — the best k per task are blended, weighted by out-of-fold score.
# TUNE_ITERS — RandomizedSearchCV draws. Small on purpose: at 60-100 positives
#   an aggressive search selects whatever best fits inner-fold noise.
# DROP_CONSTANT_FEATURES — a column with no variation carries no information,
#   occupies a feature slot and inflates the count quoted in the write-up.
#   days_left_dec is the usual case once IMPUTE_DAILY makes every cycle 4 days.
FEATURE_SET = "compact_plus_macro"        # one of the eleven listed above
                              # z_only is the PRE-REGISTERED PRIMARY: it gives
                              # the ML exactly what the Gaussian sees, which is
                              # the study's stated question. compact,
                              # compact_plus_macro and full are the declared
                              # secondaries; run each in turn.
MODELS = ["LogisticRegression", "SVM", "XGBoost", "NeuralNet", "RandomForest"]
TOP_K = 3                     # ensemble size; 1 reports the single best model
TUNE_ITERS = 15               # RandomizedSearchCV draws per model per fold
DROP_CONSTANT_FEATURES = True # drop a column with no variation, per task

# ---- SCORING: these change the verdict, not the fit ------------------------
#
# PRIMARY_METRIC — what gets tuned on, gated on and reported as the headline.
#   All three are always WRITTEN to the sheets; this picks which one decides.
#
#   "payoff"      mean payoff per cycle at p*, the break-even cut implied by the
#                 payoff matrix. This is the STATED objective of the study, so it
#                 is the default. Tuning, deciding and scoring all happen at the
#                 same fixed cut, which is what makes it coherent — there is no
#                 threshold to fit and therefore no extra freedom to overfit.
#
#                 Two honest costs. (1) At a FIXED cut, payoff rewards
#                 calibration as well as ranking: a model whose probabilities sit
#                 on the wrong side of p* scores badly even if it orders cycles
#                 perfectly. For a study that will DECIDE at p* that is correct,
#                 but expect class_weight="balanced" to stop being chosen, since
#                 it deliberately shifts probabilities toward a 50/50 prior.
#                 (2) It is coarse. In a 57-row inner fold mean payoff takes only
#                 about 124 distinct values, so the search sees ties and breaks
#                 them arbitrarily. Workable, but noisier than Brier.
#
#   "fbeta"       F-beta at a fitted cut. Useful, but see the warning under
#                 F_BETA: an arbitrary beta can disagree with the payoff matrix.
#
#   "brier_skill" the proper scoring rule. No threshold, cannot be gamed by
#                 moving a cut, but says nothing about what a mistake costs.
#
# F_BETA — how much more a MISS costs than a FALSE ALARM, in the F-measure
#   sense. The weight is beta SQUARED, so beta = 3 weights recall NINE times
#   precision. It has one consequence that must be handled rather than ignored:
#   the score of the TRIVIAL always-predict-breach model rises sharply with beta.
#   At base rate b that model scores (1 + beta^2) b / (beta^2 b + 1), which is
#   0.50 at b = 1/3 for beta = 1 but 0.83 for beta = 3. Any fixed score floor is
#   therefore meaningless, which is why the gate floor is a fraction of headroom.
#
#   BETA SHOULD AGREE WITH THE PAYOFF MATRIX, and the default here is derived
#   from it rather than chosen. From the four PAYOFF_* constants:
#
#       a false alarm costs   NOBREACH_CORRECT - BREACH_WRONG   =  1.0 - (-1.0) = 2
#       catching a breach is  BREACH_CORRECT - NOBREACH_WRONG   = -1.0 - (-4.0) = 3
#       ratio 3/2 = 1.5  ->  beta = sqrt(1.5) = 1.225
#
#   An earlier run used F_BETA = 1.5, i.e. beta^2 = 2.25 against an implied 1.50 —
#   tuning for one tradeoff while deciding at another. 1.225 makes the two agree.
#   Change the payoffs and this number should be recomputed; C1 prints both the
#   configured beta and the implied one so a mismatch is visible.
#
# MIN_SKILL — the gate floor, as a FRACTION of the room above the trivial
#   score rather than an absolute number of points. The room shrinks as breaches
#   get commoner, so an absolute margin silently gets harsher as sigma narrows:
#
#     base rate   always-breach F3   headroom to 1.0   a flat +0.03 asks for
#        0.10           0.526             0.474          6% of the room
#        0.20           0.714             0.286         10%
#        0.33           0.833             0.167         18%
#        0.47           0.898             0.102         29%   <- near-impossible
#
#   So the gate scores SKILL = (score - trivial) / (perfect - trivial), the same
#   device brier_skill uses, and asks for MIN_SKILL of it. 0.0 would mean "merely
#   beat the no-model predictor", which is the weakest bar worth having.
#
#   The reference points differ by metric, which is exactly why the gate cannot
#   use a raw number. For PAYOFF they also move with the base rate, and the best
#   trivial strategy even SWITCHES:
#
#     base   always-hold   always-exit   best trivial   perfect   room
#     0.10       0.500        -1.000         0.500        0.800   0.300
#     0.20       0.000        -1.000         0.000        0.600   0.600
#     0.33      -0.667        -1.000        -0.667        0.333   1.000
#     0.47      -1.350        -1.000        -1.000        0.060   1.060
#
#   Below about a third, doing nothing is the thing to beat; at 0.47 it is
#   exiting every week. trivial_payoff takes the max of the two.
#
# GATE_MAX_GAP / GATE_MIN_RATIO — overfitting tests, applied on the RAW scale.
#   See SECTION 9 for why they are not on the skill scale.
#
# PAYOFF_* — what each outcome is worth, in arbitrary risk units. Both exit
#   cells are equal here, a FLAT EXIT COST: closing a position costs the spread,
#   the fees and the slippage whatever the index does next. The consequence is
#   that exiting is INSENSITIVE to being right, and all the asymmetry sits on the
#   hold side, +1.0 against -4.0. That 4:1 ratio is the whole content of p*.
#   Change the four numbers and p* follows automatically.
PRIMARY_METRIC = "fbeta"  # "brier_skill" | "payoff" | "fbeta"
                              # brier_skill is the PRE-REGISTERED PRIMARY, on
                              # the ground that it is the only PROPER scoring
                              # rule of the three — it cannot be improved by
                              # moving a threshold. That reason holds whichever
                              # contestant it ends up favouring, which is what
                              # makes it a defensible choice after having tried
                              # several. payoff and fbeta are reported as
                              # secondaries and must not be swapped in as the
                              # headline on the strength of their results.
F_BETA = 1.225                # 1 = plain F1; recall weight is beta SQUARED
MIN_SKILL = 0.10              # gate floor, as a FRACTION of headroom over trivial
GATE_MAX_GAP = 0.20           # |in-fold - out-of-fold| ceiling, RAW scale
                              # (payoff is on a wider scale than F-beta, so
                              #  GATE["payoff"] sets its own gap and ratio)
GATE_MIN_RATIO = 0.60         # out-of-fold / in-fold floor, RAW scale
FOLD_PASS_FRACTION = 0.75     # share of folds that must pass individually

PAYOFF_BREACH_CORRECT = -1.0    # exit, breach happened      (flat exit cost)
PAYOFF_BREACH_WRONG = -1.0      # exit, no breach            (false alarm)
PAYOFF_NOBREACH_CORRECT = 1.0   # hold, no breach            (best outcome)
PAYOFF_NOBREACH_WRONG = -4.0    # hold, breach happened      (worst outcome)

# ---- PLUMBING: rarely touched ----------------------------------------------
ROOT = "/content/drive/MyDrive/LSTM/Project1_Simple"

# Every per-configuration output — 04 features, 05 results, 06 comparison, the
# cumulative 07 table and the 00 run index — is written into ROOT/<this>. The
# raw and cycle files stay in ROOT, because they are shared by every
# configuration and regenerating them per run would be waste.
#
# WHY A SEPARATE FOLDER. A publication table is assembled from many runs, and
# the only thing that distinguishes one run's workbook from another's is the
# filename. Mixed in among the scripts, the daily files and the figures, that
# set is impossible to hand over or audit as a unit. One folder makes the whole
# evidence set a single object: copy it, zip it, upload it, delete it.
#
# Set to "" to put everything back in ROOT.
RESULTS_SUBDIR = "results"

RANDOM_STATE = 42
N_FOLDS = 4                   # walk-forward folds, expanding window
TEST_FRACTION = 0.15          # test block size; validation block matches it
MIN_TRAIN = 20                # a fold needs at least this many training rows
CROSSFIT_FOLDS = 5            # inner folds used to cross-fit the threshold


# ---- VALIDATION, all of it here so a bad combination fails at line one -----
_FEATURE_SETS = ("full", "no_rv", "z_plus_width", "z_only",
                 "compact", "compact_plus_macro",
                 "full_plus_vix", "no_rv_plus_vix", "full_plus_macro",
                 "z_plus_macro", "z_only_plus_vix")
_PRIMARY_METRICS = ("payoff", "fbeta", "brier_skill")

if FEATURE_SET not in _FEATURE_SETS:
    raise ValueError(f"FEATURE_SET must be one of {_FEATURE_SETS}")
if CYCLE_LENGTHS not in ("all", "full_only"):
    raise ValueError('CYCLE_LENGTHS must be "all" or "full_only"')
if not isinstance(IMPUTE_DAILY, bool):
    raise ValueError("IMPUTE_DAILY must be True or False")
if RV_WINDOW_DAYS is not None:
    if not isinstance(RV_WINDOW_DAYS, int) or RV_WINDOW_DAYS < 2:
        raise ValueError("RV_WINDOW_DAYS must be None or an int >= 2 "
                         "(a std needs at least 2 returns)")
if PRIMARY_METRIC not in _PRIMARY_METRICS:
    raise ValueError(f"PRIMARY_METRIC must be one of {_PRIMARY_METRICS}")
if not (np.isfinite(F_BETA) and F_BETA > 0):
    raise ValueError(f"F_BETA must be finite and positive, got {F_BETA!r}")
if not 0.0 <= MIN_SKILL < 1.0:
    raise ValueError("MIN_SKILL is a fraction of headroom; it must be in "
                     f"[0, 1). Got {MIN_SKILL!r}. 0.0 means 'merely beat the "
                     "no-model predictor', which is the weakest bar worth having.")
if not MODELS:
    raise ValueError("MODELS is empty — nothing would be trained")
if int(TOP_K) < 1:
    raise ValueError("TOP_K must be at least 1")
if any(k <= 0 for k in SIGMA_GRID):
    raise ValueError("every SIGMA_GRID entry must be positive")

# A macro FEATURE_SET without the macro workbook silently becomes its non-macro
# parent, so the list is defined once and C5 and C8 both enforce it. Adding a
# macro set in one place and forgetting the other is how a run falls back without
# saying so and reports a comparison that was never made.
MACRO_FEATURE_SETS = ("full_plus_vix", "no_rv_plus_vix", "full_plus_macro",
                      "compact_plus_macro",
                      "z_plus_macro", "z_only_plus_vix")
_NEEDS_MACRO = FEATURE_SET in MACRO_FEATURE_SETS

# What to call the measure in printed output and in written sheets. Never the
# bare string "F1" unless beta really is 1 — a report showing "F1 = 0.88" for a
# number that is actually F1.5 is a misstatement, and this is the single place
# that prevents it.
FBETA_LABEL = "F1" if abs(F_BETA - 1.0) < 1e-12 else f"F{F_BETA:g}"

# Old name, kept so C6 and C8 keep importing successfully after the rename.
# The floor is a fraction of headroom whatever the metric, so one knob serves
# all three rather than one per metric.
FBETA_MIN_SKILL = MIN_SKILL

np.random.seed(RANDOM_STATE)



# ============================================================================
# 2. PATHS
# ----------------------------------------------------------------------------
# One folder, numbered workbooks, no timestamps. A re-run overwrites in place so
# the folder never fills with near-identical copies, and the file you opened
# last is always the current one.
# ============================================================================
def mount_drive(mount_path="/content/drive"):
    """Mount Google Drive when running in Colab; do nothing elsewhere.

    Parameters
    ----------
    mount_path : str
        Colab's standard mount point.

    Returns
    -------
    str
        The mount path, whether or not a mount happened.
    """
    if os.path.ismount(mount_path):
        print("  Drive already mounted")
        return mount_path
    try:
        from google.colab import drive
        drive.mount(mount_path)
        print("  Drive mounted")
    except Exception as e:
        print(f"  Not in Colab, continuing without Drive ({type(e).__name__})")
    return mount_path


mount_drive()

os.makedirs(ROOT, exist_ok=True)   # ROOT is set in SECTION 0

# Everything a single configuration produces lands here; the shared raw and
# cycle files stay in ROOT. RESULTS_SUBDIR = "" collapses the two.
RESULTS_DIR = os.path.join(ROOT, RESULTS_SUBDIR) if RESULTS_SUBDIR else ROOT
os.makedirs(RESULTS_DIR, exist_ok=True)


def rv_window_days(band_window):
    """Trading days in the realized_vol window, for a given band window.

    Parameters
    ----------
    band_window : int
        An entry from SIGMA_WINDOW_GRID, measured in CYCLES.

    Returns
    -------
    int
        Trading days. RV_WINDOW_DAYS when set, otherwise band_window * 4.

    Notes
    -----
    Every script asks this function rather than multiplying by four itself. That
    is the point of it: the daily audit column in C3, the feature in C5, the
    independent recomputation in C5b and the live path all have to agree on the
    same number, and four separate copies of "window * 4" is exactly how they
    would stop agreeing.
    """
    return int(RV_WINDOW_DAYS) if RV_WINDOW_DAYS is not None else int(band_window) * 4


# Every switch that changes the DATA goes in the filename. Two configurations
# must never be able to overwrite one another, and a file found on disk months
# later has to say what produced it. This is what replaces the old
# CACHE_VERSION string.
CAL_TAG = "filled" if IMPUTE_DAILY else "raw"
# The vol window changes the FEATURES, so it goes in the tag. Without it a 5-day
# run would overwrite a 16-day run's files and the two could not be compared —
# the comparison being the only reason to try 5 in the first place.
RV_TAG = "" if RV_WINDOW_DAYS is None else f"_rv{int(RV_WINDOW_DAYS)}d"
RUN_TAG = f"{CAL_TAG}_{CYCLE_LENGTHS}_{FEATURE_SET}{RV_TAG}"
DATA_TAG = f"{CAL_TAG}_{CYCLE_LENGTHS}"          # cycles depend on neither

# The daily files carry CAL_TAG too. Without it a sweep that flips IMPUTE_DAILY
# would overwrite the other setting's daily file, and the run after it would read
# a calendar built under the opposite switch without any sign that it had.
F_RAW      = os.path.join(ROOT, "01_daily_raw.xlsx")
F_FILLED   = os.path.join(ROOT, f"01b_daily_filled_{CAL_TAG}.xlsx")
F_CLEAN    = os.path.join(ROOT, f"02_daily_clean_{CAL_TAG}.xlsx")
F_CYCLES   = os.path.join(ROOT, f"03_cycles_{DATA_TAG}.xlsx")
F_MACRO    = os.path.join(ROOT, f"01c_macro_daily_{CAL_TAG}.xlsx")

# The macro columns C2c writes, and the DERIVED names the models actually see.
#
# NEVER the PRICE levels. usdinr went 69 -> 88 across this sample and Brent
# doubled; a model fitted on either learns the trend, not the relationship. Those
# two are offered only as log changes. VIX is the exception — it is already a
# stationary, dimensionless rate, not a price, so its level is meaningful and is
# offered directly. What the models can see:
#
#   vix_daily_Dn      india_vix / 100 / sqrt(252). The market's forward-looking
#                     volatility forecast, rescaled to a DAILY sigma so it sits on
#                     the same scale as realized_vol_Dn and the two can be
#                     compared or substituted. Used by full_plus_vix and
#                     no_rv_plus_vix.
#   vix_over_rv_Dn    vix_daily_Dn / realized_vol_Dn
#                     The variance risk premium: is the market pricing more
#                     volatility than has recently realised? Dimensionless, so it
#                     means the same thing in 2019 and 2026, and it is the one
#                     macro quantity the Gaussian structurally cannot see — its
#                     sigma is backward-looking by construction.
#   usdinr_ln1_Dn     ln(rate / previous rate). Rupee weakness tracks foreign
#                     outflow, which tracks equity weakness.
#   brent_ln1_Dn      ln(Brent / previous Brent). India is a large net importer.
#
#                     ln5 versions of both are also built and sit in the frame
#                     unused; z_plus_macro now asks for ln1. The 5-day forms are
#                     the quieter signal, so if ln1 adds nothing, try ln5 before
#                     concluding that oil and the rupee carry no information.
MACRO_SOURCE_COLS = ["india_vix", "vix_daily", "usdinr_ln1", "usdinr_ln5",
                     "brent_ln1", "brent_ln5"]

def f_features(sigma, window):
    """Path of the feature workbook for one (sigma, window).

    Parameters
    ----------
    sigma : float
        Band multiplier.
    window : int
        Rolling window in cycles.

    Returns
    -------
    str
        Absolute path. The configuration is IN THE FILENAME, which is what
        replaces the old cache-version string: two configurations cannot
        overwrite each other, and no file can be misread as another.
    """
    return os.path.join(
        RESULTS_DIR,
        f"04_features_sigma{sigma:.2f}_win{int(window)}_{RUN_TAG}.xlsx")


def f_results(sigma, window):
    """Path of the results workbook for one (sigma, window)."""
    return os.path.join(
        RESULTS_DIR,
        f"05_results_sigma{sigma:.2f}_win{int(window)}_{RUN_TAG}.xlsx")


# ---- THE THREE FILES C7 WRITES ---------------------------------------------
# Named here rather than inside C7 so that every path in the project is
# declared in one file, and so C8 or a notebook can read the cumulative table
# without re-deriving its name.
#
# F_COMPARE          one workbook per configuration, as before.
# F_ALL_COMPARISONS  EVERY configuration's rows in one sheet. This is the file
#                    the publication table is built from. C7 merges its own
#                    rows into whatever is already there, replacing any row
#                    with the same (run tag, sigma, window, task, metric,
#                    rule) rather than appending a duplicate, so re-running a
#                    configuration updates it in place.
# F_RUN_INDEX        one row per (configuration, sigma, window) with the
#                    CONFIG FINGERPRINT that produced it. This exists because
#                    a folder of workbooks whose filenames differ only by
#                    feature set gives no way to tell which PRIMARY_METRIC,
#                    payoff matrix or code generation each one came from — and
#                    pooling runs that disagree on those is not a comparison.
F_COMPARE         = os.path.join(RESULTS_DIR, f"06_comparison_{RUN_TAG}.xlsx")
F_ALL_COMPARISONS = os.path.join(RESULTS_DIR, "07_all_comparisons.xlsx")
F_RUN_INDEX       = os.path.join(RESULTS_DIR, "00_run_index.xlsx")


# Keys in a 05_results manifest that MUST agree across every run pooled into
# one table, and what each one changes if it does not.
#
#   primary metric   drives the consistency gate, the hyperparameter search,
#                    top-k ensemble membership and the fitted cut. A brier_skill
#                    column taken from an f1-primary run is the Brier score of
#                    an ensemble that was selected to maximise f1. Different
#                    models, different predictions, not a like-for-like cell.
#   f_beta           changes the fitted cut and therefore every count-based
#                    metric.
#   decision rules   a rename ("f1_optimal" -> "fitted_cut") marks a different
#                    code generation, and the column names in the predictions
#                    sheet move with it.
#   the four payoffs changes p*, the tuning scorer and every payoff figure.
#   economic cut p*  derived from the payoffs; checked separately because a
#                    hand-edited matrix that leaves p* unchanged is harmless
#                    while one that moves it is not.
FINGERPRINT_KEYS = (
    "primary metric",
    "f_beta",
    "decision rules",
    "payoff: exit & breach",
    "payoff: exit & no breach",
    "payoff: hold & no breach",
    "payoff: hold & breach",
    "economic cut p*",
)


def config_fingerprint():
    """The current C1's values for FINGERPRINT_KEYS, as plain strings.

    Returns
    -------
    dict
        Key -> string. Strings, not numbers, because the comparison is against
        values round-tripped through Excel, where 1.225 can come back as
        1.2249999999999999 and -1.0 as -1.

    Notes
    -----
    C7 compares this against each 05 manifest it reads and refuses to pool a
    file that disagrees. That check is the whole reason the function exists: a
    mixed table is worse than no table, because it looks finished.
    """
    def s(v):
        if isinstance(v, float):
            return f"{v:.6g}"
        return str(v).strip()

    return {
        "primary metric": s(PRIMARY_METRIC),
        "f_beta": s(F_BETA),
        "decision rules": s(", ".join(DECISION_RULES)),
        "payoff: exit & breach": s(PAYOFF_BREACH_CORRECT),
        "payoff: exit & no breach": s(PAYOFF_BREACH_WRONG),
        "payoff: hold & no breach": s(PAYOFF_NOBREACH_CORRECT),
        "payoff: hold & breach": s(PAYOFF_NOBREACH_WRONG),
        "economic cut p*": s(round(ECONOMIC_THRESHOLD, 4)),
    }


# Column names in the clean daily file.
DATE, OPEN, HIGH, LOW, CLOSE = "Date", "Open", "High", "Low", "Close"


def read_xlsx(path, sheet=0):
    """Read one sheet, parsing the Date column if there is one.

    Parameters
    ----------
    path : str
        Workbook path.
    sheet : str or int
        Sheet name or position.

    Returns
    -------
    pandas.DataFrame

    Raises
    ------
    FileNotFoundError
        Naming the script that produces the missing file, so the fix is obvious
        from the message alone.
    """
    if not os.path.exists(path):
        made_by = {F_RAW: "C2_Fetch", F_CLEAN: "C3_Clean",
                   F_CYCLES: "C4_Cycles"}.get(path, "the previous script")
        raise FileNotFoundError(f"{path} not found — run {made_by} first.")
    df = pd.read_excel(path, sheet_name=sheet)
    for c in df.columns:
        if "date" in str(c).lower():
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


def write_xlsx(path, sheets):
    """Write a dict of {sheet_name: DataFrame} to one workbook.

    Parameters
    ----------
    path : str
        Destination. Overwritten if it exists.
    sheets : dict
        Sheet name to frame. Insertion order is preserved.

    Notes
    -----
    sheet_name is passed as a KEYWORD deliberately. pandas 3.0 removed the
    positional form, and `to_excel(writer, "Name")` there silently writes
    nothing at all — a data-loss bug with no error message.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        for name, df in sheets.items():
            df.to_excel(xw, sheet_name=str(name)[:31], index=False)
    print(f"  wrote {os.path.basename(path)}  ({len(sheets)} sheet(s))")


# ============================================================================
# 3. EXPIRY AND CYCLE GEOMETRY
# ----------------------------------------------------------------------------
# NSE moved the weekly NIFTY expiry from Thursday to Tuesday on 2025-09-01. Each
# regime has a holiday fallback: if the nominal day did not trade, the previous
# trading day settles the contract.
#
# A cycle runs from one expiry to the next. Its DECISION DAYS are every trading
# day in it except the expiry itself, so
#
#     five trading days -> D1 D2 D3 D4 + expiry   cycle_days = 4
#     four  trading days -> D1 D2 D3    + expiry   cycle_days = 3
#
# and days_left = cycle_days + 1 - n. At D1 that equals cycle_days, which is the
# full horizon the band was drawn for; at the last decision day it equals 1.
# ============================================================================
EXPIRY_CUTOFF = pd.Timestamp("2025-09-01")
MIN_CYCLE_DAYS = 3          # fewer decision days than this and the cycle is dropped


def is_expiry(date, all_dates, last_date=None):
    """Is this date the settlement day of its weekly cycle?

    Parameters
    ----------
    date : pandas.Timestamp
        Date to classify.
    all_dates : set of pandas.Timestamp
        Every trading date in the dataset, normalised.
    last_date : pandas.Timestamp or None
        Final trading date. Defaults to max(all_dates).

    Returns
    -------
    bool

    Notes
    -----
    The holiday fallback promotes a day to expiry when the nominal expiry day is
    ABSENT from the data, on the reasoning that a missing trading day was a
    market holiday. That reasoning holds for the past and fails at the end of the
    file, where days are missing because they have not happened yet. The absence
    test is therefore applied only to dates at or before last_date. Without that
    guard the final row always reads as an expiry and live inference reports "no
    cycle in progress".
    """
    wd = date.weekday()
    expiry_wd, prev_wd = (1, 0) if date >= EXPIRY_CUTOFF else (3, 2)
    if wd == expiry_wd:
        return True

    if last_date is None:
        last_date = max(all_dates) if all_dates else None

    def was_holiday(d):
        d = d.normalize()
        if last_date is not None and d > last_date:
            return False
        return d not in all_dates

    if wd == prev_wd:
        return was_holiday(date + pd.Timedelta(days=(expiry_wd - wd) % 7))

    # Two days early: both the nominal day and its fallback must have been shut.
    if wd == (prev_wd - 1) % 5:
        nominal = date + pd.Timedelta(days=(expiry_wd - wd) % 7)
        fallback = date + pd.Timedelta(days=(prev_wd - wd) % 7)
        return was_holiday(nominal) and was_holiday(fallback)
    return False


def days_left(n, cycle_days):
    """Trading steps from Dn's close to the expiry close.

    Parameters
    ----------
    n : int
        Decision day, 1-based.
    cycle_days : int
        Number of decision days in THIS cycle (4 normally, 3 in a holiday week).

    Returns
    -------
    float
        cycle_days + 1 - n. So 4/3/2/1 in a normal cycle and 3/2/1 in a short
        one, which is why it varies across rows and can be used as a feature.
    """
    return float(cycle_days) + 1.0 - float(n)


# ============================================================================
# 4. BAND STATISTICS AND LABELS
# ----------------------------------------------------------------------------
# Estimated PER STEP so cycles of different lengths can be pooled — see point 2
# of the header. Both statistics are lagged one cycle, which is the single most
# important line in this file: without it a cycle's own outcome helps draw the
# band that labels it, and every score in the study becomes meaningless.
# ============================================================================
MIN_PERIODS_FRAC = 0.75
MIN_PERIODS_ABS = 4


def min_periods(window):
    """Observations required before a rolling statistic is defined.

    Parameters
    ----------
    window : int
        Window length in cycles.

    Returns
    -------
    int
        A real fraction of the window, floored at MIN_PERIODS_ABS. A standard
        deviation from two observations is not an estimate, so the warm-up is
        left undefined rather than filled.
    """
    return int(max(MIN_PERIODS_ABS, math.ceil(window * MIN_PERIODS_FRAC)))


def add_bands_and_labels(cycles, sigma, window):
    """Attach per-step statistics, both band edges and the breach labels.

    Parameters
    ----------
    cycles : pandas.DataFrame
        One row per cycle, oldest first. Needs cycle_days, d1_close and
        expiry_close.
    sigma : float
        Band multiplier k.
    window : int
        Rolling window in cycles.

    Returns
    -------
    pandas.DataFrame
        A copy with mu_step, sigma_step, mu, sigma, band_upper, band_lower,
        band_width_pct, upper_breach and lower_breach.

    Notes
    -----
    A NaN band gives a NaN label, never a 0. Scoring a warm-up cycle as "no
    breach" would invent an easy negative and flatter every model; drop those
    rows with drop_warmup instead.
    """
    c = cycles.sort_values("expiry_date").reset_index(drop=True).copy()
    steps = pd.to_numeric(c["cycle_days"], errors="coerce").astype(float)

    total = np.log(c["expiry_close"] / c["d1_close"]).replace([np.inf, -np.inf], np.nan)
    per_step_mu = total / steps                 # drift per step
    per_step_sd = total / np.sqrt(steps)        # a sqrt-time normalised move

    mp = min_periods(window)
    c["mu_step"] = per_step_mu.rolling(window, min_periods=mp).mean().shift(1)
    c["sigma_step"] = per_step_sd.rolling(window, min_periods=mp).std().shift(1)

    # Scale the per-step statistics back onto this cycle's own horizon.
    c["mu"] = c["mu_step"] * steps
    c["sigma"] = c["sigma_step"] * np.sqrt(steps)

    c["band_upper"] = c["d1_close"] * np.exp(c["mu"] + sigma * c["sigma"])
    c["band_lower"] = c["d1_close"] * np.exp(c["mu"] - sigma * c["sigma"])
    c["band_width_pct"] = np.log(c["band_upper"] / c["band_lower"])

    bad = c["band_upper"].isna() | c["band_lower"].isna() | c["expiry_close"].isna()
    c["upper_breach"] = (c["expiry_close"] > c["band_upper"]).astype(float).mask(bad)
    c["lower_breach"] = (c["expiry_close"] < c["band_lower"]).astype(float).mask(bad)
    return c


def drop_warmup(c, verbose=True):
    """Remove cycles whose rolling statistics are undefined.

    Parameters
    ----------
    c : pandas.DataFrame
        Output of add_bands_and_labels.
    verbose : bool
        Report how many rows went.

    Returns
    -------
    pandas.DataFrame
        Only rows with a finite, strictly positive sigma.
    """
    ok = c["mu"].notna() & c["sigma"].notna() & (c["sigma"] > 0)
    n = int((~ok).sum())
    if verbose and n:
        print(f"  warm-up: dropped {n} cycle(s) with undefined mu/sigma")
    return c[ok].reset_index(drop=True)


# ============================================================================
# 5. FEATURES
# ----------------------------------------------------------------------------
# Per decision day n, three columns:
#
#   z_dist_{side}_Dn = ln(band_{side} / close_Dn) / (realized_vol_Dn * sqrt(days_left_n))
#         How many standard deviations of the REMAINING move the band edge sits
#         away. 1 - Phi(z) is exactly the Gaussian's breach probability, so this
#         is the baseline's own statistic rather than a loose analogue.
#
#   realized_vol_Dn  std of DAILY log returns over rv_window_days(window) trading
#         days, read on d{n}_date and INCLUDING that day's return. No lag: the
#         decision is taken minutes before the close, so ln(C_t / C_t-1) is
#         already known, and the label is the expiry close days away.
#
#   gap_Dn = ln(Open on Dn / Close on D(n-1))
#         The overnight jump into that day. Known at the open, used at the close.
#
# Plus two cycle-level columns:
#
#   band_width_pct = ln(band_upper / band_lower) = 2 * k * sigma
#   days_left_dec  = days_left at THE DECISION DAY
#
# Cumulative from D1, so a D3 task sees D1, D2 and D3:
#
#     D2 ->  6 + 2 =  8 features
#     D3 ->  9 + 2 = 11
#     D4 -> 12 + 2 = 14
#
# ONE SIDE PER TASK, AND WHY band_width_pct IS ALLOWED
# ---------------------------------------------------
# An upper task gets z_dist_upper_Dn and never z_dist_lower_Dn. That restriction
# is what keeps band_width_pct legal, because for any n
#
#     ln(U/C) - ln(L/C) = ln(U/L) = band_width_pct
#
# exactly. A task holding both sides plus the width would have a rank-deficient
# design matrix — infinite VIF, coefficients that are not identified. Holding one
# side breaks the identity and all three coexist safely. Do not re-admit the
# opposite side without also dropping the width.
#
# WHAT THE D1 BLOCK ACTUALLY IS
# -----------------------------
# z_dist_upper_D1 is built from ln(U / d1_close) = mu + k*sigma, so the D1 block
# hands the model the band's own location and width rather than another day of
# price action. Useful, but it should be stated in the write-up rather than left
# for a reader to discover.
# ============================================================================
Z_MIN_VOL = 1e-8          # below this, dividing is meaningless -> NaN, not inf


def daily_realized_vol(daily, window_days):
    """Rolling std of daily log returns, including the current day.

    Parameters
    ----------
    daily : pandas.DataFrame
        The clean daily file; needs DATE and CLOSE, and is_original when the
        calendar has been filled.
    window_days : int
        Trading days in the window. Counted in DAYS THAT ACTUALLY TRADED.

    Returns
    -------
    pandas.Series
        Indexed by normalised date, and defined on EVERY date in `daily`,
        including imputed ones.

    Notes
    -----
    min_periods is the FULL window, not half of it. A standard deviation built
    from part of its intended sample is noisier, z_dist divides by it, and the
    few extra warm-up rows get dropped anyway.

    IMPUTED DAYS ARE EXCLUDED FROM THE WINDOW, AND THIS IS NOT OPTIONAL.
    An imputed day repeats the previous close, so its log return is exactly zero,
    and a zero does not sit quietly inside a standard deviation — it shrinks it.
    Measured on the current file, letting them in deflates this number by more
    than 2 per cent on 30 per cent of rows and by more than 5 per cent on 18 per
    cent, worst case times 0.48. z_dist DIVIDES by it, so on the worst cycle z
    would roughly double and the Gaussian's breach probability would collapse
    toward zero. That corrupts the baseline the whole study is measured against,
    in a direction no output would reveal.

    So the returns are taken across CONSECUTIVE TRADING DAYS only: the window is
    built on is_original rows, and the result is then reindexed onto every date
    and carried forward, so an imputed day reports the volatility last actually
    observed. Nothing new was learnt while the market was shut.

    With no is_original column — an unfilled file, or an older workbook — this
    reduces exactly to the plain rolling std, so the function is safe either way.
    """
    d = daily.sort_values(DATE).reset_index(drop=True)
    dates = pd.to_datetime(d[DATE]).dt.normalize()

    if "is_original" in d.columns:
        real = pd.to_numeric(d["is_original"], errors="coerce").fillna(1).astype(int) == 1
    else:
        real = pd.Series(True, index=d.index)

    obs = d.loc[real, [CLOSE]].copy()
    obs_dates = dates[real]
    r = np.log(obs[CLOSE] / obs[CLOSE].shift(1))
    rv = r.rolling(int(window_days), min_periods=int(window_days)).std()

    out = pd.Series(rv.values, index=pd.to_datetime(obs_dates).values)
    # Reindex onto the full calendar and carry the last observed value forward,
    # so an imputed date is defined without ever having contributed to the window.
    return out.reindex(dates.values).ffill()


def add_features(cycles, daily, window, side_days=4):
    """Attach realized_vol_Dn, gap_Dn and z_dist_{side}_Dn for every day.

    Parameters
    ----------
    cycles : pandas.DataFrame
        Output of add_bands_and_labels; needs d{n}_date, d{n}_close and the
        bands.
    daily : pandas.DataFrame
        The clean daily file.
    window : int
        Band window in CYCLES. The daily volatility window comes from
        rv_window_days(window), which is window * 4 unless
        RV_WINDOW_DAYS overrides it.
    side_days : int
        Highest decision-day index to build, normally 4.

    Returns
    -------
    pandas.DataFrame
        A copy with the feature columns. A day that does not exist in a short
        cycle stays NaN and is dropped later — never zero-filled, because a
        volatility of exactly zero is a confident wrong number, not a missing
        one.
    """
    out = cycles.copy()
    rv = daily_realized_vol(daily, rv_window_days(window))

    d = daily.sort_values(DATE).reset_index(drop=True)
    dates = pd.to_datetime(d[DATE]).dt.normalize()
    open_by_date = pd.Series(pd.to_numeric(d[OPEN], errors="coerce").values, index=dates)
    prev_close_by_date = pd.Series(
        pd.to_numeric(d[CLOSE], errors="coerce").shift(1).values, index=dates)
    high_by_date = pd.Series(pd.to_numeric(d[HIGH], errors="coerce").values,
                             index=dates)
    low_by_date = pd.Series(pd.to_numeric(d[LOW], errors="coerce").values,
                            index=dates)

    for n in range(1, side_days + 1):
        dcol, ccol = f"d{n}_date", f"d{n}_close"
        if dcol not in out.columns:
            continue
        dts = pd.to_datetime(out[dcol], errors="coerce").dt.normalize()

        out[f"realized_vol_D{n}"] = dts.map(rv).astype(float)

        # For n > 1 the previous close is on the same row. For D1 there is no D0,
        # so the previous TRADING day is used — the prior cycle's expiry close,
        # i.e. the weekend gap carried into the new cycle.
        prev = (pd.to_numeric(out[f"d{n-1}_close"], errors="coerce") if n > 1
                else dts.map(prev_close_by_date).astype(float))

        # A REPEATED SLOT needs the previous TRADING day, not the previous slot.
        #
        # This no longer fires under IMPUTE_DAILY, because a filled day has its
        # own calendar date and no two slots share one. It is kept because it
        # costs nothing and because a slot that repeats a date is exactly the
        # situation in which gap_Dn would silently become ln(Open / Close) of one
        # and the same day — a same-day reversal, not an overnight gap, and not
        # comparable with the gap on any other row.
        #
        # WORTH KNOWING about a filled day's gap: its Open is a copy of the
        # previous day's Open while the close it is measured against is the
        # previous day's Close, so gap on a filled row is ln(prevOpen / prevClose)
        # — the previous session's intraday move, read backwards. It is finite and
        # stable but it is not an overnight gap, and roughly 5 per cent of rows
        # carry one. FEATURE_SET="z_only" excludes gap entirely and avoids it.
        if n > 1:
            prev_dts = pd.to_datetime(out[f"d{n-1}_date"],
                                      errors="coerce").dt.normalize()
            repeated = dts.eq(prev_dts) & dts.notna()
            if bool(repeated.any()):
                prev = prev.where(~repeated,
                                  dts.map(prev_close_by_date).astype(float))

        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"gap_D{n}"] = np.log(dts.map(open_by_date).astype(float)
                                      / prev.where(prev > 0))

        # range_Dn = ln(High / Low) on that day.
        #
        # The day's own trading range, in logs. It is a SAME-DAY quantity, known
        # only once the day is over — which is fine here because the decision is
        # taken at that day's close, but it must never be used to predict
        # anything about the day it comes from.
        #
        # It says something realized_vol cannot. realized_vol averages over
        # rv_window_days(window) sessions, so a single violent day is diluted by
        # however many days that is, while range_Dn spikes the same day. A wide
        # range on the decision day is the clearest available signal that today
        # is not like the recent past. The shorter RV_WINDOW_DAYS is, the more
        # the two overlap — at 4 days they are close to measuring the same thing,
        # which is worth remembering when reading a feature-importance table.
        hi = dts.map(high_by_date).astype(float)
        lo = dts.map(low_by_date).astype(float)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"range_D{n}"] = np.log(hi.where(hi > 0) / lo.where(lo > 0))

        # --- macro, read on this decision day's own date ---------------------
        #
        # Only attached when C5 merged the macro columns onto `daily`. Absent, the
        # columns simply are not created, available_features does not offer them,
        # and a macro FEATURE_SET falls back to its non-macro features rather than
        # crashing. That is deliberate: a missing optional source should narrow the
        # feature set, not stop the run.
        if "vix_daily" in d.columns:
            vixd = pd.Series(pd.to_numeric(d["vix_daily"], errors="coerce").values,
                             index=dates)
            rv_n = pd.to_numeric(out[f"realized_vol_D{n}"], errors="coerce")
            vx = dts.map(vixd).astype(float)

            # The raw forward-looking level, already converted to a daily sigma
            # by C5 so it is directly comparable with realized_vol_Dn.
            #
            # Both forms are built because which one is appropriate depends on
            # whether realized_vol is also in the feature set:
            #
            #   realized_vol ABSENT  -> use the level. The ratio has realized_vol
            #                           in its denominator, so adding the ratio to
            #                           a deliberately rv-free set smuggles rv
            #                           back in and the set stops being what its
            #                           name says.
            #   realized_vol PRESENT -> the level is still the cleaner column. A
            #                           linear model cannot recover vix from
            #                           {rv, vix/rv}, because the ratio is a
            #                           nonlinear transform; given {rv, vix} it
            #                           can form whatever combination the data
            #                           supports, including the ratio.
            out[f"vix_daily_D{n}"] = vx
            with np.errstate(divide="ignore", invalid="ignore"):
                out[f"vix_over_rv_D{n}"] = (
                    vx / rv_n.where(rv_n > Z_MIN_VOL)).replace(
                        [np.inf, -np.inf], np.nan)
        for mc in ("usdinr_ln5", "brent_ln5", "usdinr_ln1", "brent_ln1"):
            if mc in d.columns:
                ser = pd.Series(pd.to_numeric(d[mc], errors="coerce").values,
                                index=dates)
                out[f"{mc}_D{n}"] = dts.map(ser).astype(float)

        close_n = pd.to_numeric(out[ccol], errors="coerce")
        dl = pd.to_numeric(out["cycle_days"], errors="coerce") + 1.0 - float(n)
        vol = pd.to_numeric(out[f"realized_vol_D{n}"], errors="coerce")
        denom = vol.where(vol > Z_MIN_VOL) * np.sqrt(dl.where(dl > 0))
        for side in ("upper", "lower"):
            with np.errstate(divide="ignore", invalid="ignore"):
                num = np.log(out[f"band_{side}"] / close_n.where(close_n > 0))
            out[f"z_dist_{side}_D{n}"] = (num / denom).replace(
                [np.inf, -np.inf], np.nan)

    # ---- PATH SUMMARIES, one set per possible decision day ------------------
    #
    # The compact feature sets replace the per-day expansion with summaries of
    # the path SO FAR. Those depend on which day the task sits at, so one set is
    # built per candidate day and task_features picks the matching one — exactly
    # how usdinr_ln1_D{day} already works.
    #
    # WHY THESE FOUR for the z path. At day d there are d values of z_dist, so at
    # most d INDEPENDENT summaries of them exist. Ordered by how soon each earns
    # its place:
    #
    #   z_now    the current distance. Always informative.
    #   z_chg    z_now - z_D1. The direction of travel.
    #   z_min    the CLOSEST the index came to this band at any point. For a
    #            breach question this is the natural path statistic: did it
    #            already nearly breach and pull back?
    #   z_rng    max - min. How much it wandered.
    #
    # At d = 2 the path is two numbers, so z_min and z_rng are EXACTLY
    # recoverable from z_now and z_chg — the design matrix drops to rank 3 of 4
    # and those columns carry zero information. task_features therefore takes
    # only the first d of them, which is why the compact sets are 6/7/8 rather
    # than flat. A flat version was proposed first and was wrong.
    #
    # gap and range get ONE column each at every day, because four columns of
    # each were describing two things: how much the index gapped across the
    # cycle, and how violent its worst single session was.
    #
    # All four days are built regardless of FEATURE_SET. Unused columns cost one
    # Excel column each and keep C5's output independent of the feature choice.
    for day in (2, 3, 4):
        have = [k for k in range(1, day + 1) if f"d{k}_date" in out.columns]
        if len(have) < day:
            continue

        for side in ("upper", "lower"):
            zc = [f"z_dist_{side}_D{k}" for k in have]
            if not all(c in out.columns for c in zc):
                continue
            z = out[zc].apply(pd.to_numeric, errors="coerce")
            out[f"z_chg_{side}_upto_D{day}"] = z[zc[-1]] - z[zc[0]]
            # MIN, not max, for BOTH sides. z_dist_upper is small when the upper
            # band is close; z_dist_lower is negative and more negative when the
            # lower band is close. The minimum is the closest approach in signed
            # terms either way, so one rule serves both and the lower side is not
            # silently inverted.
            out[f"z_min_{side}_upto_D{day}"] = z.min(axis=1)
            out[f"z_rng_{side}_upto_D{day}"] = z.max(axis=1) - z.min(axis=1)

        gc = [f"gap_D{k}" for k in have if f"gap_D{k}" in out.columns]
        if gc:
            # SUM, not mean: the cumulative overnight move across the cycle is
            # what shifts the index relative to a band fixed at D1. A mean would
            # divide that out and make a 4-day cycle look like a 2-day one.
            # min_count keeps a NaN day from silently becoming a zero.
            out[f"gap_cum_upto_D{day}"] = (
                out[gc].apply(pd.to_numeric, errors="coerce").sum(
                    axis=1, min_count=len(gc)))
        rc = [f"range_D{k}" for k in have if f"range_D{k}" in out.columns]
        if rc:
            # MAX, not mean: one violent session is the signal. Averaging it
            # against three quiet days is what realized_vol already does.
            out[f"range_max_upto_D{day}"] = (
                out[rc].apply(pd.to_numeric, errors="coerce").max(axis=1))
    return out


def task_features(day, side):
    """Feature names for one task: cumulative from D1, own side only.

    Parameters
    ----------
    day : int
        Decision day, 2, 3 or 4.
    side : {"upper", "lower"}
        Which edge this task predicts. Required — defaulting it would hand an
        upper model the lower distance, which is exactly what the one-side rule
        exists to prevent.

    Returns
    -------
    list of str
        Column names in a stable order.
    """
    if side not in ("upper", "lower"):
        raise ValueError(f"side must be 'upper' or 'lower', got {side!r}")
    if int(day) not in (2, 3, 4):
        raise ValueError(f"day must be 2, 3 or 4, got {day!r}")
    days = range(1, int(day) + 1)
    if FEATURE_SET == "full":
        per_day = [f"z_dist_{side}_D{{n}}", "realized_vol_D{n}", "gap_D{n}",
                   "range_D{n}"]
        cycle_level = ["band_width_pct", "days_left_dec"]
    elif FEATURE_SET == "full_plus_vix":
        # "full" with the one measurement the Gaussian cannot see.
        #
        # The level, not vix_over_rv: realized_vol_Dn is already a column here, so
        # the ratio would be a second, nonlinear function of a variable the model
        # already has. Given {realized_vol, vix} a model can form the ratio if the
        # ratio is what matters; given {realized_vol, ratio} a linear model cannot
        # recover the level.
        #
        # Cost: 5 columns per day. At D4 that is 22 features. Check the EPV line
        # C6 prints — below 10 events per feature the extra columns are more
        # likely to be fitting noise than finding anything.
        per_day = [f"z_dist_{side}_D{{n}}", "realized_vol_D{n}", "vix_daily_D{n}",
                   "gap_D{n}", "range_D{n}"]
        cycle_level = ["band_width_pct", "days_left_dec"]
    elif FEATURE_SET == "no_rv":
        per_day = [f"z_dist_{side}_D{{n}}", "gap_D{n}", "range_D{n}"]
        cycle_level = ["band_width_pct", "days_left_dec"]
    elif FEATURE_SET == "no_rv_plus_vix":
        # The interesting one: drop the BACKWARD-looking volatility estimate and
        # put the FORWARD-looking one in its place, holding everything else fixed.
        #
        # Against "no_rv" it isolates what VIX adds. Against "full" it asks the
        # sharper question — is the market's implied forecast better than a
        # backward-looking standard deviation over rv_window_days at the same
        # job? That is the comparison this set exists for.
        #
        # vix_daily_Dn and not vix_over_rv_Dn, necessarily: the ratio divides by
        # realized_vol, so using it here would put realized_vol back into the one
        # set defined by its absence.
        per_day = [f"z_dist_{side}_D{{n}}", "vix_daily_D{n}", "gap_D{n}",
                   "range_D{n}"]
        cycle_level = ["band_width_pct", "days_left_dec"]
    elif FEATURE_SET == "full_plus_macro":
        # EVERYTHING AVAILABLE. "full", plus the forward-looking VIX level on
        # every day, plus the two slow macro series on the current decision day.
        #
        # VIX per day because it is a daily volatility measure and changes
        # meaningfully from one session to the next. usdinr and brent on the
        # CURRENT day only because they move on a weekly horizon — their D1 and
        # D2 values are nearly the same number, so repeating them would add
        # collinear columns and spend events for nothing.
        #
        # THE COST, stated plainly: 5 per day + 4 cycle-level, so 24 features at
        # D4. That is 7.5 events per variable at 180 breaches and 1.7 at 40.
        # Below 10 the extra columns are more likely fitting noise than finding
        # anything, so expect this to LOSE to "full" and expect the gate to
        # refuse it at wide sigmas. It is the everything-in arm of the
        # comparison, included to show where adding features stops helping —
        # which is a result worth reporting, not a failure.
        per_day = [f"z_dist_{side}_D{{n}}", "realized_vol_D{n}", "vix_daily_D{n}",
                   "gap_D{n}", "range_D{n}"]
        cycle_level = ["band_width_pct", "days_left_dec",
                       f"usdinr_ln1_D{int(day)}", f"brent_ln1_D{int(day)}"]
    elif FEATURE_SET in ("compact", "compact_plus_macro"):
        # THE PATH RE-EXPRESSED, NOT EXPANDED.
        #
        # Against "full" this changes two things and nothing else:
        #   the z path becomes level / direction / closest approach / spread
        #     instead of d raw positions — the SAME dimension, better axes;
        #   realized_vol, gap and range collapse from d columns each to one.
        #
        # realized_vol is the big saving and it costs nothing: z_dist already
        # equals ln(band/close) / (realized_vol * sqrt(days_left)), so the
        # volatility and the horizon are inside it. Three of its four columns
        # were never earning anything.
        #
        # THE Z BUDGET IS CAPPED AT THE DAY INDEX. d values of z support at most
        # d independent summaries, so D2 gets 2, D3 gets 3, D4 gets 4. At D4 the
        # z path is therefore not compressed at all, only re-expressed; every
        # actual saving comes from the other three families. Counts 6 / 7 / 8,
        # or 9 / 10 / 11 once macro is added, against full's 10 / 14 / 18.
        d = int(day)
        z_stats = [f"z_dist_{side}_D{d}",             # level, always
                   f"z_chg_{side}_upto_D{d}"]         # direction, always
        if d >= 3:
            z_stats.append(f"z_min_{side}_upto_D{d}")     # closest approach
        if d >= 4:
            z_stats.append(f"z_rng_{side}_upto_D{d}")     # how far it wandered
        per_day = []                                   # nothing repeats per day
        cycle_level = z_stats + [
            f"realized_vol_D{d}",                      # current day only
            f"gap_cum_upto_D{d}",                      # summed over the path
            f"range_max_upto_D{d}",                    # worst single session
            "band_width_pct",
        ]
        if FEATURE_SET == "compact_plus_macro":
            # The ONLY difference from "compact", so the pair isolates the
            # question "does VIX, the rupee and oil add anything?". Comparing
            # "full" with "full_plus_macro" cannot answer it — both are too wide
            # at D4 (2.2 and 1.7 events per variable) for the difference to mean
            # much either way.
            cycle_level += [f"vix_daily_D{d}", f"usdinr_ln1_D{d}",
                            f"brent_ln1_D{d}"]
    elif FEATURE_SET == "z_plus_width":
        per_day = [f"z_dist_{side}_D{{n}}", "gap_D{n}", "range_D{n}"]
        cycle_level = ["band_width_pct"]
    elif FEATURE_SET == "z_plus_macro":
        # z_dist and the variance risk premium for EVERY day, but the two slow
        # macro series only for the CURRENT decision day.
        #
        # Why only the current day: usdinr and brent move on a weekly horizon, so
        # their D1 and D2 values are nearly the same number. Repeating them per day
        # would add collinear columns and spend events for nothing. At D4 this set
        # is 4 + 4 + 2 = 10 features against roughly 80 to 190 breaches, which is
        # the most this sample supports once macro is in at all.
        per_day = [f"z_dist_{side}_D{{n}}", "vix_over_rv_D{n}"]
        # ln1, the ONE-day change, on the current decision day.
        #
        # Switched from ln5 by request. Two consequences worth knowing:
        #   - ln1 is noisier. A single day of currency or oil movement carries
        #     less signal about a weekly horizon than a five-day move does.
        #   - ln1 is EXACTLY ZERO whenever the previous calendar row was
        #     forward-filled, because FRED publishes on the US calendar and C2c
        #     carries the last value forward onto NSE dates. ln5 spans a wider
        #     window and so degrades more gracefully. Check the `coverage` sheet
        #     in 01c_macro_daily_<cal>.xlsx: a high forward-fill share means a
        #     pile-up of zeros here, which a tree will happily split on.
        cycle_level = [f"usdinr_ln1_D{int(day)}", f"brent_ln1_D{int(day)}"]
    elif FEATURE_SET == "z_only_plus_vix":
        # The minimal honest test of whether VIX adds anything: z_dist, which is
        # the Gaussian's own statistic, plus the one quantity the Gaussian cannot
        # see. If this does not beat z_only, no larger macro set will.
        per_day = [f"z_dist_{side}_D{{n}}", "vix_over_rv_D{n}"]
        cycle_level = []
    else:                                    # "z_only"
        per_day = [f"z_dist_{side}_D{{n}}"]
        cycle_level = []
    feats = [t.format(n=n) for n in days for t in per_day]
    return feats + cycle_level


def available_features(frame, day, side):
    """The task's features that are actually present on a frame.

    Parameters
    ----------
    frame : pandas.DataFrame
        A feature sheet or an equivalent frame.
    day : int
    side : {"upper", "lower"}

    Returns
    -------
    list of str

    Notes
    -----
    Needed because DROP_CONSTANT_FEATURES lets C5 omit a column that carries no
    variation — days_left_dec at D4, for instance. Anything that rebuilds a
    feature list from the configuration rather than from the sheet has to
    intersect with what the sheet holds, or it will ask for a column that was
    deliberately left out.
    """
    want = task_features(day, side)
    have = set(frame.columns)
    return [c for c in want if c in have]


# The six task cells. upper_D2, upper_D3 and upper_D4 share an IDENTICAL label
# vector and differ only in features, because a breach is a property of the cycle
# rather than of the decision day. That matters for multiple-testing correction:
# there are about two independent families here, not six.
TASKS = [{"name": f"{side}_D{day}", "side": side, "day": day,
          "label": f"{side}_breach"}
         for day in (2, 3, 4) for side in ("upper", "lower")]


def task_rows(features, day):
    """Rows usable by one task: the decision day must exist in that cycle.

    Parameters
    ----------
    features : pandas.DataFrame
        Output of add_features.
    day : int
        Decision day.

    Returns
    -------
    pandas.DataFrame
        A copy carrying days_left_dec for this day. A four-day cycle appears in
        the D2 and D3 tasks and is absent from D4 — not because of a data
        problem, but because that week had no fourth decision day.
    """
    f = features[pd.to_numeric(features["cycle_days"], errors="coerce")
                 >= int(day)].copy()
    f["days_left_dec"] = (pd.to_numeric(f["cycle_days"], errors="coerce")
                          + 1.0 - float(day))
    return f.reset_index(drop=True)


# ============================================================================
# 6. THE GAUSSIAN BASELINE
# ----------------------------------------------------------------------------
# The contestant with no free parameters. At decision day n:
#
#     dist = ln(band / close_Dn)
#     sa   = sigma_step * sqrt(days_left_n)
#     P(upper breach) = 1 - Phi(dist / sa)
#     P(lower breach) =     Phi(dist / sa)
#
# sigma_step is used rather than the cycle-scaled sigma, because days_left is
# already counted in steps. Sanity anchor: at D1, sa = sigma_step*sqrt(C) =
# sigma, so with mu = 0 the answer is 1 - Phi(k) — the band's own quantile.
# Asserted in the self-test.
# ============================================================================
def gaussian_breach_prob(rows, day, side):
    """Analytical P(breach) for one task.

    Parameters
    ----------
    rows : pandas.DataFrame
        Cycle rows carrying band_upper / band_lower, sigma_step, cycle_days and
        d{day}_close.
    day : int
        Decision day.
    side : {"upper", "lower"}
        Band edge.

    Returns
    -------
    numpy.ndarray
        One probability per row; NaN where the inputs are unusable.

    Notes
    -----
    An unusable row returns NaN, never 0.0. A zero here would read as "certain
    no breach" and quietly flatter the baseline on exactly the rows it could not
    handle.
    """
    n = len(rows)
    if n == 0:
        return np.array([], float)

    def num(col):
        return pd.to_numeric(rows[col], errors="coerce").to_numpy(float)

    band = num(f"band_{side}")
    ref = num(f"d{int(day)}_close")
    sig = num("sigma_step")
    dl = num("cycle_days") + 1.0 - float(day)

    with np.errstate(divide="ignore", invalid="ignore"):
        sa = sig * np.sqrt(np.clip(dl, 0.0, None))
        z = np.log(band / ref) / sa
        p = (1.0 - norm_cdf(z)) if side == "upper" else norm_cdf(z)

    ok = (np.isfinite(band) & np.isfinite(ref) & (ref > 0)
          & np.isfinite(sig) & (sig > 0) & (dl > 0) & np.isfinite(p))
    return np.where(ok, p, np.nan)


_erf = np.vectorize(math.erf, otypes=[float])


def norm_cdf(z):
    """Standard normal CDF: vectorised, NaN-safe, empty-safe."""
    z = np.asarray(z, float)
    out = np.full(z.shape, np.nan)
    ok = np.isfinite(z)
    if ok.any():
        out[ok] = 0.5 * (1.0 + _erf(z[ok] / math.sqrt(2.0)))
    return out


# ============================================================================
# 7. METRICS
# ----------------------------------------------------------------------------
# F-beta at a fitted cut is the headline because it is what a trader acts on: a
# decision, taken at a threshold, with misses weighted against false alarms by
# F_BETA. Brier skill is reported alongside because it is threshold-free and
# PROPER, so it cannot be gamed by moving a cut.
#
# Reading them together separates the two explanations for a tie:
#   large reliability  -> the Gaussian is mis-calibrated, so there is structure
#                         left to find and the ML has somewhere to go.
#   low resolution for both -> the inputs are at their limit, and no model
#                         restricted to them can win.
#
# THREE NUMBERS ARE REPORTED FOR EVERY PREDICTION, and they answer different
# questions. Quote the right one:
#
#   fbeta         raw F-beta at the fitted cut. Moves with the base rate ON ITS
#                 OWN, so it is NOT comparable across sigmas.
#   fbeta_trivial what always-predicting-breach would have scored. The bar.
#   fbeta_skill   (fbeta - trivial) / (1 - trivial). Zero means no better than
#                 the constant predictor, one means perfect. THIS is the figure
#                 to compare across sigmas and tasks, and the one the gate uses.
#
# Plain f1 is also returned whatever F_BETA is set to, so a beta-driven gain can
# never be mistaken for a real one.
# ============================================================================
def fbeta_at(y, p, thr, beta=None):
    """F-beta of a probability vector at one threshold.

    Parameters
    ----------
    y : array-like of int
    p : array-like of float
    thr : float
    beta : float or None
        Recall weight. None uses F_BETA. beta = 1 gives exactly F1.

    Notes
    -----
    Written from the confusion counts rather than as a precision/recall product
    so that the degenerate cases stay finite: with no predicted positives and no
    actual positives the denominator vanishes and the score is 0.0, where
    precision x recall would be 0/0.

        F_beta = (1 + b2) * tp / ((1 + b2) * tp + b2 * fn + fp),   b2 = beta^2

    At beta = 1 this reduces to 2tp / (2tp + fn + fp), the familiar F1.
    """
    b2 = float(F_BETA if beta is None else beta) ** 2
    yp = (np.asarray(p, float) >= thr).astype(int)
    yt = np.asarray(y, int)
    tp = int(((yp == 1) & (yt == 1)).sum())
    fp = int(((yp == 1) & (yt == 0)).sum())
    fn = int(((yp == 0) & (yt == 1)).sum())
    den = (1.0 + b2) * tp + b2 * fn + fp
    return 0.0 if den <= 0 else (1.0 + b2) * tp / den


def f1_at(y, p, thr):
    """Plain F1, kept so the reports can show it next to F-beta."""
    return fbeta_at(y, p, thr, beta=1.0)


def trivial_fbeta(base_rate, beta=None):
    """F-beta earned by always predicting breach, at a given base rate.

    This is the score to beat. It needs no model, so any floor below it passes
    a constant predictor, and at beta = 3 it is high enough that the old F1
    floor of 0.55 would have waved one straight through.
    """
    b = float(base_rate)
    b2 = float(F_BETA if beta is None else beta) ** 2
    if not np.isfinite(b) or b <= 0.0:
        return 0.0
    return float((1.0 + b2) * b / (b2 * b + 1.0))


def fbeta_skill(score, base_rate):
    """F-beta expressed as a fraction of the headroom above the trivial model.

    Parameters
    ----------
    score : float
        A raw F-beta value.
    base_rate : float
        Share of positives, which fixes the trivial always-breach score.

    Returns
    -------
    float
        0.0 = no better than always predicting breach. 1.0 = perfect.
        Negative = worse than the trivial model. NaN when undefined.

    Notes
    -----
    This is what makes results COMPARABLE ACROSS SIGMA. A raw F3 of 0.905 is
    near-worthless at a base rate of 0.47 (trivial is 0.898) and excellent at
    0.10 (trivial is 0.526). The same two numbers become 0.07 and 0.80 on this
    scale, which is the comparison the write-up actually needs.
    """
    sc = float(score)
    triv = trivial_fbeta(base_rate)
    room = 1.0 - triv
    if not np.isfinite(sc) or not np.isfinite(triv) or room <= 1e-12:
        return float("nan")
    return float((sc - triv) / room)


def best_threshold(y, p, fallback=None, objective=None):
    """The cut that maximises `objective`, chosen for out-of-sample stability.

    Parameters
    ----------
    y : array-like of int
        True labels.
    p : array-like of float
        Predicted probabilities; non-finite entries are dropped.
    fallback : float or None
        Returned when a threshold cannot be estimated.
    objective : callable or None
        f(y, p, thr) -> float, higher is better. None uses F-beta, which is what
        every caller wanted before the primary metric became configurable.

    Returns
    -------
    (float or None, float)
        (threshold, objective value there). Returns (fallback, 0.0) when the
        label has fewer than two positives or is all-positive, so callers must
        handle None rather than silently receive 0.5.

    Notes
    -----
    Candidates are MIDPOINTS between observed probabilities plus the two outside
    endpoints, so a cut is found even when every probability sits outside a fixed
    grid. Among thresholds tied at the maximum, the midpoint of the WIDEST tied
    run is returned: at these sample sizes the tied region is often broad and its
    edge is the least stable choice.

    BOTH contestants must call this. A different tie-break for ML than for the
    Gaussian would quietly reintroduce an asymmetry into the comparison.
    """
    obj = (lambda yy, qq, tt: fbeta_at(yy, qq, tt)) if objective is None else objective
    yt = np.asarray(y, int)
    pp = np.asarray(p, float)
    ok = np.isfinite(pp)
    yt, pp = yt[ok], pp[ok]
    if len(yt) == 0 or yt.sum() < 2 or yt.sum() == len(yt):
        return fallback, 0.0

    u = np.unique(pp)
    if len(u) == 1:
        return ((fallback if fallback is not None else float(u[0])),
                obj(yt, pp, u[0]))
    cand = np.concatenate([[u[0] - 1e-9], (u[:-1] + u[1:]) / 2.0, [u[-1] + 1e-9]])
    scores = np.array([obj(yt, pp, t) for t in cand])

    best = scores.max()
    tied = np.flatnonzero(scores >= best - 1e-12)
    runs, start = [], tied[0]
    for a, b in zip(tied[:-1], tied[1:]):
        if b != a + 1:
            runs.append((start, a))
            start = b
    runs.append((start, tied[-1]))
    lo, hi = max(runs, key=lambda r: r[1] - r[0])
    return float(cand[(lo + hi) // 2]), float(best)


def best_fbeta_threshold(y, p, fallback=None):
    """The F-beta-optimal cut. Kept as a name because C8 asks for it by name."""
    return best_threshold(y, p, fallback, objective=fbeta_at)


# Old name, kept so every existing call site keeps working. It maximises F-beta,
# which IS F1 when F_BETA = 1.
best_f1_threshold = best_fbeta_threshold


def brier_score(y, p):
    """Mean squared error of a probability forecast. Lower is better."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    return float(np.mean((p[ok] - y[ok]) ** 2)) if ok.any() else np.nan


def brier_skill(y, p):
    """Brier score relative to always predicting the base rate.

    Returns
    -------
    float
        1 is perfect, 0 is no better than the base rate, negative is worse.
        Comparable across tasks with different base rates, which raw Brier is
        not — and the base rate DOES move with sigma, because the band is the
        label.
    """
    y, p = np.asarray(y, float), np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    if not ok.any():
        return np.nan
    y, p = y[ok], p[ok]
    base = float(y.mean())
    ref = base * (1 - base)
    return np.nan if ref < 1e-12 else float(1.0 - brier_score(y, p) / ref)


def brier_parts(y, p, bins=10):
    """Murphy decomposition: Brier = reliability - resolution + uncertainty."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    y, p = y[ok], p[ok]
    if len(y) == 0:
        return dict(brier=np.nan, reliability=np.nan, resolution=np.nan,
                    uncertainty=np.nan)
    ybar, N = float(y.mean()), len(y)
    edges = np.linspace(0, 1, bins + 1)
    edges[-1] += 1e-9
    rel = res = 0.0
    for k in range(bins):
        m = (p >= edges[k]) & (p < edges[k + 1])
        nk = int(m.sum())
        if nk:
            rel += nk / N * (p[m].mean() - y[m].mean()) ** 2
            res += nk / N * (y[m].mean() - ybar) ** 2
    return dict(brier=brier_score(y, p), reliability=float(rel),
                resolution=float(res), uncertainty=float(ybar * (1 - ybar)))


def all_scores(y, prob, thr):
    """Every metric at once for one set of predictions.

    Parameters
    ----------
    y : array-like of int
        True labels.
    prob : array-like of float
        Predicted probabilities.
    thr : float
        Decision cut used for the threshold metrics.

    Returns
    -------
    dict
        fbeta, f1, brier, brier_skill, reliability, resolution, uncertainty,
        base_rate, n, threshold, trivial_fbeta, and `primary` — whichever
        PRIMARY_METRIC names.

        Both fbeta AND plain f1 are returned. The headline follows F_BETA, but
        F1 stays in every sheet so the two can be read side by side and a
        beta-driven gain cannot be mistaken for a real one.
    """
    parts = brier_parts(y, prob)
    fb = fbeta_at(y, prob, thr)
    f1 = f1_at(y, prob, thr)
    bs = brier_skill(y, prob)
    br = float(np.nanmean(np.asarray(y, float)))
    # The payoff is read at p*, NOT at `thr`. p* is the cut the payoff is defined
    # against and it is not fitted, so using the fitted cut here would make the
    # headline depend on a threshold search that the economic rule never uses.
    pay = payoff_at(y, prob)
    return {"primary": {"fbeta": fb, "payoff": pay}.get(PRIMARY_METRIC, bs),
            "fbeta": fb, "fbeta_trivial": trivial_fbeta(br),
            # The sigma-comparable number. Quote THIS across sigmas, never raw
            # F-beta, because raw F-beta moves with the base rate on its own.
            "fbeta_skill": fbeta_skill(fb, br),
            # Payoff at p*, its no-model reference, its ceiling, and the
            # sigma-comparable fraction of the room between them.
            "payoff_at_pstar": pay,
            "payoff_trivial": trivial_payoff(br),
            "payoff_perfect": perfect_payoff(br),
            "primary_skill": skill_of(
                {"fbeta": fb, "payoff": pay}.get(PRIMARY_METRIC, bs), br),
            "f1": f1, "brier": parts["brier"], "brier_skill": bs,
            "reliability": parts["reliability"], "resolution": parts["resolution"],
            "uncertainty": parts["uncertainty"],
            "base_rate": float(np.nanmean(np.asarray(y, float))),
            "n": int(len(y)), "threshold": float(thr)}


# ============================================================================
# 7b. PAYOFF MATRIX AND THE ECONOMIC THRESHOLD
# ----------------------------------------------------------------------------
# F1 asks which cut best balances precision and recall. It does not ask what a
# mistake costs. The economic rule does: it turns a probability into the decision
# with the higher expected value, given what each outcome is worth.
#
# Four outcomes, in arbitrary risk units:
#
#                          breach happened      no breach
#     predicted breach     BREACH_CORRECT       BREACH_WRONG
#     (exit the position)  you exited into      a false alarm: you paid
#                          a real move          to leave for nothing
#
#     predicted no breach  NOBREACH_WRONG       NOBREACH_CORRECT
#     (hold)               held into a breach   held correctly
#
# ⚠️ STATE THIS IN THE WRITE-UP. EXITING COSTS THE SAME WHETHER THE CALL WAS
#    RIGHT OR WRONG — both exit cells are -1.0. That is a FLAT EXIT COST, and it
#    is a cleaner assumption than the one it replaces: closing a position costs
#    the spread, the fees and the slippage, and you pay those on the way out
#    whatever the index does next. The earlier version charged -1.5 for a correct
#    exit, on the reasoning that you exit into a move already under way; that is
#    arguable too, but it bundles a market outcome into what is really a
#    transaction cost.
#
#    THE CONSEQUENCE, said plainly: with the two exit cells equal, EXITING IS
#    INSENSITIVE TO BEING RIGHT. Every bit of asymmetry now sits on the hold
#    side — holding correctly (+1.0) against holding into a breach (-4.0). That
#    4:1 ratio is what the model is being asked to avoid, and it is the whole
#    content of p*.
#
#    Change the four constants and the threshold follows automatically. Nothing
#    else needs editing, and the self-tests below check the ORDERING rather than
#    any particular value.
#
# THE BREAK-EVEN PROBABILITY
# --------------------------
#     EV(exit) = p*BREACH_CORRECT + (1-p)*BREACH_WRONG
#     EV(hold) = p*NOBREACH_WRONG + (1-p)*NOBREACH_CORRECT
#
# Setting them equal gives
#
#     p* = (NOBREACH_CORRECT - BREACH_WRONG)
#          / (BREACH_CORRECT - BREACH_WRONG - NOBREACH_WRONG + NOBREACH_CORRECT)
#
# which at these values is 2.0 / 5.0 = 0.4000.
#
# THE HONEST CAVEAT
# -----------------
# p* is a cut on a CALIBRATED probability. It compares expected values, and an
# expected value computed from a probability that is systematically too high or
# too low is wrong by the same amount. A model tuned for F1 has no reason to be
# calibrated, so the economic rule can score badly for a model that ranks cycles
# perfectly well. Read the `reliability` column beside every economic result: a
# large value means the payoff figure is measuring mis-calibration rather than
# skill, and the fix is to calibrate the probability, not to move the cut.
#
# This is also why both rules are reported rather than one being chosen. F1 says
# how well the ordering separates breaches; the payoff says what acting on it
# would have been worth.
# ============================================================================
# The four PAYOFF_* constants are set in SECTION 0.

# "fitted_cut" was called "f1_optimal" until the threshold started following
# PRIMARY_METRIC — the old name promised F1 and could deliver a payoff-optimal
# cut instead. The rule is now named for what it IS (a cut learned from the
# training block) rather than for one metric it used to use.
DECISION_RULES = ["fitted_cut", "economic"]


def payoff_vector(y, decision):
    """Payoff per cycle for a set of decisions.

    Parameters
    ----------
    y : array-like of int
        Outcomes, 0 or 1.
    decision : array-like of int
        Calls made, 0 or 1.

    Returns
    -------
    ndarray of float
        One payoff per row, drawn from the four constants above.
    """
    y = np.asarray(y, int)
    d = np.asarray(decision, int)
    return np.where((d == 1) & (y == 1), PAYOFF_BREACH_CORRECT,
           np.where((d == 1) & (y == 0), PAYOFF_BREACH_WRONG,
           np.where((d == 0) & (y == 0), PAYOFF_NOBREACH_CORRECT,
                                         PAYOFF_NOBREACH_WRONG)))


def economic_threshold():
    """The break-even P(breach) implied by the payoff matrix.

    Returns
    -------
    float
        The probability above which exiting has the higher expected value,
        clipped to [0, 1]. Falls back to 0.5 if the four constants make the
        denominator vanish, which would mean the two actions have identical
        expected values at every probability.
    """
    num = PAYOFF_NOBREACH_CORRECT - PAYOFF_BREACH_WRONG
    den = (PAYOFF_BREACH_CORRECT - PAYOFF_BREACH_WRONG
           - PAYOFF_NOBREACH_WRONG + PAYOFF_NOBREACH_CORRECT)
    return 0.5 if abs(den) < 1e-12 else float(np.clip(num / den, 0.0, 1.0))


ECONOMIC_THRESHOLD = economic_threshold()


def payoff_at(y, p, thr=None):
    """Mean payoff per cycle from deciding at one threshold.

    Parameters
    ----------
    y : array-like of int
    p : array-like of float
    thr : float or None
        The cut. None uses ECONOMIC_THRESHOLD, which is the right default:
        p* is the decision rule the payoff is defined against, and it is not
        fitted, so there is no threshold freedom to overfit.

    Returns
    -------
    float
        In the payoff matrix's own units. NaN when nothing is scoreable.

    Notes
    -----
    With a FLAT EXIT COST this is an affine function of the confusion counts,

        n * payoff = BC*TP + BW*FP + NC*TN + NW*FN

    i.e. a cost-weighted accuracy. That is what makes it a sound tuning target
    at a fixed cut: well-behaved, bounded, and monotone in the thing you care
    about. It is NOT a proper scoring rule, so never quote it as evidence that
    the probabilities are calibrated — read `reliability` for that.
    """
    yy = np.asarray(y, int)
    pp = np.asarray(p, float)
    cut = ECONOMIC_THRESHOLD if thr is None else float(thr)
    ok = np.isfinite(pp)
    if not ok.any():
        return float("nan")
    return float(payoff_vector(yy[ok], (pp[ok] >= cut).astype(int)).mean())


def trivial_payoff(base_rate):
    """The payoff of the best strategy that uses no model at all.

    Returns
    -------
    float
        max(always hold, always exit) at this base rate.

    Notes
    -----
    WHICH ONE WINS SWITCHES with the base rate, and that is why the gate cannot
    use a fixed number. Always-hold earns NC when quiet and NW when breached, so
    it collapses as breaches get common; always-exit earns a flat BW/BC whatever
    happens. On this payoff matrix the crossover sits near a base rate of 0.33.
    A model has to beat whichever is better on the task in front of it.
    """
    b = float(base_rate)
    if not np.isfinite(b):
        return float("nan")
    hold = b * PAYOFF_NOBREACH_WRONG + (1 - b) * PAYOFF_NOBREACH_CORRECT
    exit_ = b * PAYOFF_BREACH_CORRECT + (1 - b) * PAYOFF_BREACH_WRONG
    return float(max(hold, exit_))


def perfect_payoff(base_rate):
    """The payoff of perfect foresight: exit exactly the weeks that breach."""
    b = float(base_rate)
    if not np.isfinite(b):
        return float("nan")
    return float(b * PAYOFF_BREACH_CORRECT + (1 - b) * PAYOFF_NOBREACH_CORRECT)


def best_primary_threshold(y, p, fallback=None):
    """The cut that maximises whatever PRIMARY_METRIC names.

    Parameters
    ----------
    y : array-like of int
    p : array-like of float
    fallback : float or None

    Returns
    -------
    (float or None, float)

    Notes
    -----
    WHY THIS EXISTS. The "fitted_cut" rule used to fit its threshold with
    best_fbeta_threshold whatever the primary metric was. Under
    PRIMARY_METRIC="payoff" that meant choosing a cut to maximise F-beta and then
    reporting the PAYOFF at it — two unrelated things. On a measured run it made
    ML look far worse than it was: D2_lower showed ML skill 0.108 at the fitted
    cut while its own members scored about 0.35 out of fold, and the economic row
    for the same cell read 0.354. The 0.108 was an artifact of the mismatch.

    Fitting the cut on the primary metric makes the rule answer a real question:
    is a threshold LEARNED from the training block better than the fixed p* the
    payoff matrix implies? Both are now reported, so the answer is visible.

    brier_skill is the exception. It is threshold-free, so there is no cut to
    optimise and no sensible objective to search; it falls back to F-beta, which
    is at least a defined decision rule. That fallback is stated here rather than
    hidden, because the resulting threshold is NOT optimal for Brier and must not
    be described as if it were.
    """
    if PRIMARY_METRIC == "payoff":
        return best_threshold(
            y, p, fallback,
            objective=lambda yy, qq, tt: float(
                payoff_vector(yy, (qq >= tt).astype(int)).mean()))
    return best_threshold(y, p, fallback, objective=fbeta_at)


def implied_beta():
    """The F-beta recall weight the payoff matrix itself implies.

    Returns
    -------
    float
        sqrt(cost of a miss / cost of a false alarm), or NaN when the costs make
        the ratio undefined.

    Notes
    -----
        false alarm  = NOBREACH_CORRECT - BREACH_WRONG     (you exited for nothing)
        catching one = BREACH_CORRECT - NOBREACH_WRONG     (you avoided the drop)

    F-beta is not a linear cost function, so this is a correspondence rather than
    an identity — but it is the right order of magnitude, and a configured F_BETA
    far from it means the fit is being tuned for one tradeoff while the decision
    is taken at another. C1's banner prints both so the gap cannot hide.
    """
    fa = PAYOFF_NOBREACH_CORRECT - PAYOFF_BREACH_WRONG
    miss = PAYOFF_BREACH_CORRECT - PAYOFF_NOBREACH_WRONG
    if not (np.isfinite(fa) and np.isfinite(miss)) or fa <= 0 or miss <= 0:
        return float("nan")
    return float(np.sqrt(miss / fa))

# Printed here as well as in the banner. These four constants are the ones most
# likely to be edited by hand, and a wrong value changes every economic result
# without anything else complaining.
print(f"  economic cut   : p* = {ECONOMIC_THRESHOLD:.4f}   from exit/breach "
      f"{PAYOFF_BREACH_CORRECT}, exit/no-breach {PAYOFF_BREACH_WRONG}, "
      f"hold/no-breach {PAYOFF_NOBREACH_CORRECT}, "
      f"hold/breach {PAYOFF_NOBREACH_WRONG}")


# ============================================================================
# 8. WALK-FORWARD VALIDATION
# ----------------------------------------------------------------------------
# Expanding window, oldest fold first. Test blocks are contiguous, disjoint and
# strictly later than their own validation block, so no cycle can appear in two
# test sets and be double-counted.
# ============================================================================
# N_FOLDS, TEST_FRACTION, MIN_TRAIN and CROSSFIT_FOLDS are set in SECTION 0.


def walk_forward_splits(n, n_folds=N_FOLDS, test_frac=TEST_FRACTION,
                        min_train=MIN_TRAIN):
    """Expanding-window (train, val, test) index splits, oldest first.

    Parameters
    ----------
    n : int
        Rows available.
    n_folds : int
        Maximum folds.
    test_frac : float
        Test block size as a fraction of n; the validation block matches it.
    min_train : int
        Minimum training rows for a fold to be emitted.

    Returns
    -------
    list of (ndarray, ndarray, ndarray)
    """
    test_n = max(5, int(round(n * test_frac)))
    splits = []
    for fold in range(n_folds):
        test_end = n - fold * test_n
        test_start = test_end - test_n
        val_start = test_start - test_n
        if val_start <= min_train:
            break
        tr = np.arange(0, val_start)
        va = np.arange(val_start, test_start)
        te = np.arange(test_start, test_end)
        if len(tr) >= min_train and len(va) >= 3 and len(te) >= 3:
            splits.append((tr, va, te))
    return list(reversed(splits))


def stratified_folds(y, k=CROSSFIT_FOLDS, seed=RANDOM_STATE):
    """Stratified (fit, out) index pairs over positions 0..len(y)-1.

    Parameters
    ----------
    y : array-like of int
        Binary labels for the block being split.
    k : int
        Requested folds, reduced automatically when the minority class is small.
    seed : int
        Shuffle seed.

    Returns
    -------
    list of (ndarray, ndarray)
        Falls back to a single 80/20 split when stratification is impossible.

    Notes
    -----
    Used to CROSS-FIT the decision threshold inside train+val, so every row gets
    an out-of-fold probability from a model that never saw it. This is not
    leakage: train+val lies entirely before the sealed test block, and the inner
    split only affects how well the threshold is estimated. Both contestants fit
    their cut on the same block, which is what keeps the comparison symmetric.
    """
    y = np.asarray(y, int)
    n = len(y)
    pos, neg = int(y.sum()), int((1 - y).sum())
    k = max(2, min(int(k), pos, neg)) if (pos >= 2 and neg >= 2) else 0
    if k < 2:
        cut = max(1, int(round(n * 0.8)))
        return [(np.arange(0, cut), np.arange(cut, n))] if cut < n else []
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    return [(np.asarray(a), np.asarray(b)) for a, b in skf.split(np.zeros(n), y)]


# ============================================================================
# 9. CONSISTENCY GATE
# ----------------------------------------------------------------------------
# A trained model is TRUSTED only if it generalises within the training era. The
# sealed test block is never involved, so the gate cannot leak. Three conditions
# per fold, then a quorum across folds.
#
# TWO SCALES, ON PURPOSE — each test on the scale where it means something.
#
#   FLOOR -> the SKILL scale. "Is this better than always predicting breach?"
#            is only answerable relative to the trivial score, and that score
#            moves with the base rate. See FBETA_MIN_SKILL in SECTION 0 for the
#            table that shows why an absolute margin cannot work.
#
#   RATIO -> the SKILL scale too. A ratio is only meaningful where ZERO MEANS
#            NO SKILL. Raw payoff runs from about -4 to +1, so two negative
#            scores give oof/infold > 1 whenever the model is WORSE out of fold
#            — the test inverts. A ratio on skill is also scale-free, because
#            both terms carry the same divisor and it cancels.
#
#   GAP   -> the RAW scale. A gap is a DIFFERENCE in consistent units, needs no
#            reference point, and staying raw keeps it calibrated per metric.
#
# WHY THE GAP IS NOT ON SKILL. Dividing by the room multiplies the SAMPLING
# NOISE of a per-fold score by the same factor:
#
#     base rate    noise x, beta=1    noise x, beta=3
#        0.10          1.22               2.11
#        0.33          2.00               6.00
#        0.47          2.77               9.83
#
# With four folds and a few hundred rows, a per-fold skill at base 0.47 and
# beta = 3 carries roughly TEN TIMES the noise of the raw score, so a gap or
# ratio test there is largely a test of noise. An earlier version did put all
# three on skill and rejected sound models for it. The floor compares against a
# FIXED reference and is unaffected, which is why it is the one that moves.
# ============================================================================
# The FLOOR is a fraction of headroom for every metric (see skill_of), so it is
# the same knob throughout. The GAP is in the metric's RAW units and therefore
# cannot be: F-beta lives in [0, 1] while payoff spans roughly -4 to +1, so the
# same 0.20 would be a tight gap for one and a loose one for the other. The
# payoff gap is set to 0.20 x the typical room (about 0.6 units) scaled up.
GATE = {"payoff":      dict(gap=0.60, floor=MIN_SKILL, ratio=GATE_MIN_RATIO),
        "fbeta":       dict(gap=GATE_MAX_GAP, floor=MIN_SKILL,
                            ratio=GATE_MIN_RATIO),
        "brier_skill": dict(gap=0.15, floor=0.02, ratio=0.40)}

# GATE_MAX_GAP, GATE_MIN_RATIO and FOLD_PASS_FRACTION are set in SECTION 0.

MAX_GAP = GATE[PRIMARY_METRIC]["gap"]        # |in-fold - out-of-fold| ceiling
MIN_OOF = GATE[PRIMARY_METRIC]["floor"]      # out-of-fold floor
MIN_RATIO = GATE[PRIMARY_METRIC]["ratio"]    # out-of-fold / in-fold floor


def trivial_score(base_rate):
    """What a no-model predictor earns, on the PRIMARY metric's own scale.

    fbeta        -> always predict breach
    payoff       -> the better of always-hold and always-exit
    brier_skill  -> 0.0 by construction (it is already a skill score)
    """
    if PRIMARY_METRIC == "fbeta":
        return trivial_fbeta(base_rate)
    if PRIMARY_METRIC == "payoff":
        return trivial_payoff(base_rate)
    return 0.0


def perfect_score(base_rate):
    """The ceiling on the PRIMARY metric's own scale.

    1.0 for fbeta and brier_skill; for payoff it depends on the base rate,
    because perfect foresight still pays the exit cost on every breach week.
    """
    if PRIMARY_METRIC == "payoff":
        return perfect_payoff(base_rate)
    return 1.0


def skill_of(score, base_rate):
    """Turn a raw primary-metric score into a fraction of the available room.

    Parameters
    ----------
    score : float
        A raw score on whatever scale PRIMARY_METRIC uses.
    base_rate : float
        Share of positives, which fixes both reference points.

    Returns
    -------
    float
        0.0 = no better than the no-model predictor. 1.0 = perfect. Negative =
        worse than doing nothing. NaN when the room is degenerate.

    Notes
    -----
    This is the ONE function the gate uses, so adding a metric means adding it to
    trivial_score and perfect_score and nothing else. It also makes results
    comparable across sigmas: raw F-beta and raw payoff both move with the base
    rate on their own, and this divides that movement out.

    For brier_skill it is the identity, since trivial is 0 and perfect is 1 —
    which is why the old brier floor of 0.02 keeps meaning exactly what it did.
    """
    sc, t, c = float(score), trivial_score(base_rate), perfect_score(base_rate)
    room = c - t
    if not (np.isfinite(sc) and np.isfinite(t) and np.isfinite(c)) or room <= 1e-12:
        return float("nan")
    return float((sc - t) / room)


def effective_floor(base_rate=None):
    """The out-of-fold floor actually applied, given the task's base rate.

    Parameters
    ----------
    base_rate : float or None
        Share of positives in the task. None keeps the static floor, which is
        what the old two-argument gate did.

    Returns
    -------
    float

    Notes
    -----
    For F-beta the gate is applied on the SKILL scale, so MIN_OOF is a required
    fraction of the headroom rather than a raw score. This function maps that
    requirement back onto raw F-beta purely so the banner and the per-task line
    can print a number in the same units the models report:

        raw floor = trivial + MIN_OOF * (1 - trivial)

    which scales with the base rate automatically. At base 0.10 it asks for
    0.573 and at 0.47 for 0.908, and in both cases it is the same 10 per cent of
    whatever room there was.
    """
    if base_rate is None:
        return MIN_OOF
    t, c = trivial_score(base_rate), perfect_score(base_rate)
    if not (np.isfinite(t) and np.isfinite(c)):
        return MIN_OOF
    return float(t + MIN_OOF * (c - t))


def fold_passes(infold, oof, base_rate=None):
    """Does one fold's (in-fold, out-of-fold) pair clear all three conditions?"""
    a, b = float(infold), float(oof)
    if not (np.isfinite(a) and np.isfinite(b)):
        return False

    # THREE TESTS, TWO SCALES.
    #
    #   FLOOR -> SKILL. "Better than no model?" needs the no-model reference.
    #   RATIO -> SKILL. A ratio needs a scale on which ZERO MEANS NOTHING. Raw
    #            payoff fails that: it runs from about -4 to +1, so with two
    #            negative scores oof/infold rises as the model gets WORSE out of
    #            fold. Measured on a real run, LogisticRegression at in-fold
    #            -0.499 and out-of-fold -0.679 scored a "ratio" of 1.36 and
    #            passed, and a model at -0.5 / -2.0 would score 4.0 and sail
    #            through. Four overfitted models passed that way, their
    #            out-of-fold skill only 48-56 per cent of in-fold.
    #   GAP   -> RAW. A gap is a DIFFERENCE in consistent units, so it needs no
    #            reference point, and leaving it raw keeps it calibrated per
    #            metric through GATE.
    #
    # The earlier worry about skill amplifying noise applies to F-beta at a high
    # base rate (room 0.102, so noise x9.8) and NOT to a ratio, which is scale
    # free: both terms are divided by the same room, so the factor cancels. For
    # payoff the room is about 1.06, so the transform barely changes anything.
    if base_rate is not None:
        sk_oof, sk_in = skill_of(b, base_rate), skill_of(a, base_rate)
        if not (np.isfinite(sk_oof) and np.isfinite(sk_in)):
            return False
        floor_ok = sk_oof >= MIN_OOF
        ratio = (sk_oof / sk_in if abs(sk_in) > 1e-6
                 else (1.0 if sk_oof >= 0 else -1.0))
    else:
        floor_ok = b >= effective_floor(base_rate)
        ratio = b / a if abs(a) > 1e-6 else (1.0 if b >= 0 else -1.0)

    return bool(abs(a - b) <= MAX_GAP and floor_ok and ratio >= MIN_RATIO)


def model_passes(pairs, base_rate=None):
    """Apply the gate across every fold of one trained combination.

    Parameters
    ----------
    pairs : iterable of (float, float)
        (in-fold, out-of-fold) score pairs, one per fold.
    base_rate : float or None
        Share of positives in this task. Supplying it raises the floor to the
        trivial-predictor score plus a margin, which matters whenever
        F_BETA > 1. None reproduces the old static behaviour.

    Returns
    -------
    (bool, float)
        (passed, mean out-of-fold score). Passing needs BOTH the per-fold quorum
        and a mean above the floor — a model whose average gap looks fine can
        still fail because too few individual folds passed.
    """
    p = [(a, b) for a, b in pairs if np.isfinite(a) and np.isfinite(b)]
    if not p:
        return False, float("nan")
    mean_oof = float(np.mean([b for _, b in p]))
    quorum = float(np.mean([fold_passes(a, b, base_rate) for a, b in p]))

    # The mean is tested on the same scale the per-fold rule used, but the RAW
    # mean is what gets returned and reported, so the printed number stays
    # directly comparable with the scores above it.
    if base_rate is not None:
        ok_mean = skill_of(mean_oof, base_rate) >= MIN_OOF
    else:
        ok_mean = mean_oof >= effective_floor(base_rate)
    return bool(quorum >= FOLD_PASS_FRACTION and ok_mean), mean_oof


# ============================================================================
# 10. ENSEMBLE
# ----------------------------------------------------------------------------
# The reported probability and the decision must be the SAME quantity. If the
# decision came from a normalised margin while the returned number was a plain
# weighted mean, every calibration curve would describe something that did not
# drive the decision.
# ============================================================================
def pick_top_k(rows, k=TOP_K):
    """Rank one task's gate-passing candidates and keep the best k.

    Parameters
    ----------
    rows : pandas.DataFrame
        Candidate rows for a single task, with oof_score and infold_score.
    k : int
        How many to keep.

    Returns
    -------
    pandas.DataFrame
        Best first. Sorted by out-of-fold score, then by the smallest in-fold
        gap. The TEST score is never a sort key anywhere in selection.
    """
    r = rows.copy()
    r["_gap"] = (r["infold_score"] - r["oof_score"]).abs()
    return (r.sort_values(["oof_score", "_gap"], ascending=[False, True])
             .drop(columns=["_gap"]).head(int(k)))


def ensemble(probs, thresholds, weights=None):
    """Weighted blend of the top-k members for one row.

    Parameters
    ----------
    probs : sequence of float
        One probability per member.
    thresholds : sequence of float
        Each member's own cut; non-finite entries become 0.5.
    weights : sequence of float or None
        Normally each member's out-of-fold score. None weights equally. Floored
        at 0.01 so a zero-scoring member cannot erase the others.

    Returns
    -------
    (int, float, float)
        (decision, probability, effective threshold). The decision is exactly
        int(probability >= effective threshold), which is what keeps the reported
        number and the acted-on number identical.
    """
    p = np.asarray(probs, float)
    t = np.where(np.isfinite(np.asarray(thresholds, float)), thresholds, 0.5)
    w = (np.maximum(np.asarray(weights, float), 0.01)
         if weights is not None and len(weights) == len(p) else np.ones(len(p)))
    w = np.where(np.isfinite(w), w, 0.01)
    total = w.sum() if w.sum() > 0 else 1.0
    prob = float((p * w).sum() / total)
    eff = float((t * w).sum() / total)
    return int(prob >= eff), prob, eff


# ============================================================================
# 11. MODELS
# ----------------------------------------------------------------------------
# Deliberately small search spaces. At roughly 60-100 positives per task cell an
# aggressive search selects whatever best fits inner-fold noise.
#
# Winsorising and scaling live INSIDE an sklearn Pipeline, so they are re-fitted
# on the training rows of every fold and no information about a validation or
# test row can reach the transform. That replaces the hand-rolled clip-and-scale
# pair, which had to be threaded through by hand and saved alongside each model.
# ============================================================================
HYPERPARAMS = {
    "LogisticRegression": {"clf__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
                           "clf__class_weight": ["balanced", None]},
    "SVM":                {"clf__C": [0.1, 0.3, 1.0, 3.0, 10.0],
                           "clf__gamma": ["scale", 0.01, 0.1, 1.0],
                           "clf__class_weight": ["balanced", None]},
    # The net's WIDTH is barely searched on purpose — see make_pipeline. Only the
    # L2 penalty and the learning rate move, because the question is whether a net
    # small enough to be fittable at this sample size can match logistic
    # regression, not whether a wide net can be made to fit the folds.
    "NeuralNet":          {"clf__alpha": [1e-4, 1e-3, 1e-2, 1e-1],
                           "clf__learning_rate_init": [3e-4, 1e-3, 3e-3],
                           "clf__hidden_layer_sizes": [(4,), (4, 2), (8,)]},
    # A forest at this sample size lives or dies on depth and leaf size, so those
    # are the two axes searched hardest. n_estimators barely matters above ~300 for
    # a probability estimate, so it is not searched wide — more trees average away
    # variance but cannot fix a tree that is too deep for 77 events.
    "RandomForest":       {"clf__max_depth": [2, 3, 4, 5],
                           "clf__min_samples_leaf": [5, 10, 15, 20],
                           "clf__max_features": ["sqrt", 0.5, None],
                           "clf__n_estimators": [300, 600],
                           "clf__class_weight": ["balanced_subsample", None]},
    "XGBoost":            {"clf__n_estimators": [200, 400],
                           "clf__max_depth": [2, 3, 4, 6],
                           "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
                           "clf__subsample": [0.7, 0.9, 1.0],
                           "clf__min_child_weight": [1, 3, 5],
                           "clf__reg_lambda": [0.5, 1.0, 5.0]},
}
TUNE_INNER_FOLDS = 3        # TUNE_ITERS is set in SECTION 0


from sklearn.base import BaseEstimator, TransformerMixin


class Winsoriser(BaseEstimator, TransformerMixin):
    """Clip every column to bounds LEARNED FROM THE TRAINING ROWS.

    Parameters
    ----------
    low, high : float
        Percentiles for the bounds.

    Notes
    -----
    This has to be a real transformer rather than a FunctionTransformer wrapping
    a clip. A FunctionTransformer runs its function on whatever block is passed
    to transform(), so the percentiles would be recomputed from the VALIDATION or
    TEST rows and each block would be clipped to its own spread. Two things go
    wrong at once: the test distribution influences its own preprocessing, which
    is leakage, and an extreme test row is never actually clipped because it
    defines the bound it is clipped to. Measured on a deliberately wider test
    block, a value of 10.59 came back as 10.41 when it should have been pulled to
    the training 99th percentile of 1.82.

    Learning the bounds in fit() and reusing them in transform() is what makes
    the pipeline safe to drop inside a cross-validation loop.
    """

    def __init__(self, low=1.0, high=99.0):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        """Record the bounds from the training rows only."""
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.low, axis=0)
        self.hi_ = np.nanpercentile(X, self.high, axis=0)
        return self

    def transform(self, X):
        """Clip to the bounds recorded in fit."""
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)


def make_pipeline(model_name):
    """Winsorise, robust-scale, then classify — as one fitted object.

    Parameters
    ----------
    model_name : str
        One of MODELS.

    Returns
    -------
    sklearn.pipeline.Pipeline
        Steps named "clip", "scale" and "clf", so HYPERPARAMS keys read
        "clf__C".

    Raises
    ------
    ValueError
        On an unknown model name, naming the ones that exist.

    Notes
    -----
    RobustScaler rather than StandardScaler, on purpose. It centres on the median
    and divides by the interquartile range, so a single violent cycle moves the
    scaling far less than it moves a mean and a standard deviation. After
    winsorising the difference is small, but these features are heavy-tailed by
    nature and the robust version is the safer default.

    Scaling is essential for two of the three models and irrelevant for the
    third: L2 logistic regression penalises coefficients, so the units a feature
    happens to be measured in change which coefficients survive, and an RBF
    kernel measures plain Euclidean distance. A boosted tree splits on order
    alone and is unaffected — it costs nothing to scale it and keeps one code
    path.
    """
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import RobustScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC

    if model_name == "LogisticRegression":
        clf = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    elif model_name == "NeuralNet":
        # A DELIBERATELY TINY feed-forward net, via sklearn's MLPClassifier.
        #
        # Width is pinned to a handful of units because of the arithmetic, not
        # taste. At 14 inputs, a single hidden layer of 4 already carries
        # 14*4 + 4 + 4 + 1 = 65 parameters, against roughly 60 to 100 positive
        # cases in a task cell — about one event per parameter. A wider net would
        # not answer a harder question, it would simply fit the folds. It is
        # included to SHOW that, not because it is expected to win.
        #
        # early_stopping holds back 15 per cent of the TRAINING block as an
        # internal validation set and stops when it stops improving. That split is
        # taken from training rows only, so it cannot see the outer validation or
        # test blocks.
        #
        # ONE ASYMMETRY TO DISCLOSE: MLPClassifier has no class_weight, and its
        # fit() takes no sample_weight either. The other three contestants can be
        # told the classes are unbalanced and this one cannot, so at a base rate
        # near 0.21 it is fitting a harder problem. Report that rather than
        # letting a weak result read as a statement about neural networks.
        from sklearn.neural_network import MLPClassifier
        clf = MLPClassifier(hidden_layer_sizes=(4,), activation="relu",
                            solver="adam", max_iter=2000,
                            early_stopping=True, validation_fraction=0.15,
                            n_iter_no_change=20, random_state=RANDOM_STATE)
    elif model_name == "SVM":
        clf = SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
    elif model_name == "XGBoost":
        from xgboost import XGBClassifier
        clf = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE,
                            verbosity=0)
    elif model_name == "RandomForest":
        # A forest, not a single tree, and deliberately shallow.
        #
        # With 77 to 120 events a fully grown forest memorises the training block:
        # every leaf ends up pure, out-of-fold probabilities collapse to 0 or 1,
        # and Brier gets worse the better the in-fold fit looks. So max_depth and
        # min_samples_leaf are searched over SMALL values and the defaults here are
        # the conservative end of each. The gate will report the in-fold/out-of-fold
        # gap, which is the number that exposes this if the search picks badly.
        #
        # bootstrap with oob_score off: the walk-forward already provides the
        # honest estimate, and oob would be computed on rows the fold owns.
        from sklearn.ensemble import RandomForestClassifier
        clf = RandomForestClassifier(
            n_estimators=400, max_depth=3, min_samples_leaf=10,
            max_features="sqrt", bootstrap=True, oob_score=False,
            random_state=RANDOM_STATE, n_jobs=-1)
    else:
        raise ValueError(f"unknown model {model_name!r}; MODELS holds {MODELS}")

    return Pipeline([("clip", Winsoriser()),
                     ("scale", RobustScaler()),
                     ("clf", clf)])


# ============================================================================
# 12. SELF-TESTS
# ----------------------------------------------------------------------------
# Every claim this file makes is asserted here, at load time, on frames whose
# answers are known by hand. A broken assumption should surface now, not as a
# strange number six scripts later.
# ============================================================================
def self_test(verbose=True):
    """Assert every invariant in this file. Raises AssertionError listing all
    failures, so one run reports every problem rather than only the first."""
    fails = []

    def chk(name, ok, detail=""):
        """A hard check: a failure stops the run."""
        if not ok:
            fails.append(f"{name}: {detail}")
        if verbose:
            print(f"    {'OK  ' if ok else 'FAIL'} {name}"
                  + ("" if ok else f"  -- {detail}"))

    def advise(name, ok, detail=""):
        """A soft check: prints WARN and carries on.

        For things that are a judgement call rather than a defect. A deliberate
        choice should not stop a run, but it should not pass silently either.
        """
        if verbose:
            print(f"    {'OK  ' if ok else 'WARN'} {name}"
                  + ("" if ok else f"  -- {detail}"))

    # --- geometry, including the short-cycle case ---------------------------
    chk("days_left in a normal cycle = 4/3/2/1",
        [days_left(n, 4) for n in (1, 2, 3, 4)] == [4.0, 3.0, 2.0, 1.0])
    chk("days_left in a 4-day cycle = 3/2/1",
        [days_left(n, 3) for n in (1, 2, 3)] == [3.0, 2.0, 1.0])

    # --- feature whitelist -------------------------------------------------
    # Counted by hand, per day times days plus the cycle-level tail:
    #   full            4/day + 2  -> 10, 14, 18
    #   full_plus_vix   5/day + 2  -> 12, 17, 22
    #   no_rv           3/day + 2  ->  8, 11, 14
    #   no_rv_plus_vix  4/day + 2  -> 10, 14, 18   (same width as full)
    #   full_plus_macro 5/day + 4  -> 14, 19, 24
    #   compact         (day z stats) + 4   ->  6,  7,  8
    #   compact_plus_macro  the same + 3      ->  9, 10, 11
    #   z_plus_width    3/day + 1  ->  7, 10, 13
    #   z_only          1/day + 0  ->  2,  3,  4
    #   z_plus_macro    2/day + 2  ->  6,  8, 10
    #   z_only_plus_vix 2/day + 0  ->  4,  6,  8
    _expect = {"full": (10, 14, 18), "full_plus_vix": (12, 17, 22),
               "full_plus_macro": (14, 19, 24),
               "compact": (6, 7, 8), "compact_plus_macro": (9, 10, 11),
               "no_rv": (8, 11, 14), "no_rv_plus_vix": (10, 14, 18),
               "z_plus_width": (7, 10, 13), "z_only": (2, 3, 4),
               "z_plus_macro": (6, 8, 10), "z_only_plus_vix": (4, 6, 8),
               }[FEATURE_SET]
    counts = tuple(len(task_features(d, "upper")) for d in (2, 3, 4))
    chk(f"feature counts D2/D3/D4 for '{FEATURE_SET}' = {_expect}",
        counts == _expect, str(counts))
    # The EXPANDED sets carry one z_dist per day. The COMPACT sets carry exactly
    # one — the current day's level — and describe the rest of the path through
    # summaries instead, so they are checked separately below.
    chk("the expanded sets carry one z_dist per day",
        (FEATURE_SET in ("compact", "compact_plus_macro")) or all(
            sum(1 for f in task_features(d, "upper")
                if f.startswith("z_dist")) == d for d in (2, 3, 4)))
    chk("the compact sets carry exactly one z_dist, the current day's",
        (FEATURE_SET not in ("compact", "compact_plus_macro")) or all(
            [f for f in task_features(d, "upper") if f.startswith("z_dist")]
            == [f"z_dist_upper_D{d}"] for d in (2, 3, 4)))

    # THE RANK CAP. d values of z support at most d independent summaries. At
    # d = 2 the path is two numbers, so z_min and z_rng are exactly recoverable
    # from z_now and z_chg — two dead columns and rank 3 of 4. An earlier draft
    # of these sets was flat at 9 features and carried them anyway.
    chk("the number of z summaries never exceeds the number of days",
        (FEATURE_SET not in ("compact", "compact_plus_macro")) or all(
            sum(1 for f in task_features(d, "upper")
                if f.startswith(("z_dist", "z_chg", "z_min", "z_rng"))) == d
            for d in (2, 3, 4)),
        str({d: [f for f in task_features(d, "upper")
                 if f.startswith(("z_dist", "z_chg", "z_min", "z_rng"))]
              for d in (2, 3, 4)}))
    chk("z_rng appears only at D4 and z_min only from D3",
        (FEATURE_SET not in ("compact", "compact_plus_macro")) or (
            not any("z_rng" in f for f in task_features(2, "upper"))
            and not any("z_min" in f for f in task_features(2, "upper"))
            and any("z_min" in f for f in task_features(3, "upper"))
            and not any("z_rng" in f for f in task_features(3, "upper"))
            and any("z_rng" in f for f in task_features(4, "upper"))))
    chk("the compact sets repeat nothing per day",
        (FEATURE_SET not in ("compact", "compact_plus_macro")) or all(
            len(task_features(d, "upper")) == len(set(task_features(d, "upper")))
            and not any(f.endswith(("_D1", "_D2", "_D3"))
                        and "upto" not in f and f != f"z_dist_upper_D{d}"
                        and f != f"realized_vol_D{d}"
                        and not f.startswith(("vix_daily", "usdinr", "brent"))
                        for f in task_features(d, "upper"))
            for d in (2, 3, 4)))
    chk("the macro sets carry vix_over_rv once per day, and the slow series once",
        (FEATURE_SET not in ("z_plus_macro", "z_only_plus_vix")) or all(
            sum(1 for f in task_features(d, "upper")
                if f.startswith("vix_over_rv")) == d for d in (2, 3, 4)))
    # Two sets carry the slow series now. Both use ln1 and both read it on the
    # CURRENT decision day only, so one check covers them — and it would fail if
    # either one started repeating them per day.
    chk("the usdinr/brent sets carry them once, for the current day only",
        (FEATURE_SET not in ("z_plus_macro", "full_plus_macro",
                             "compact_plus_macro")) or all(
            sorted(f for f in task_features(d, "upper")
                   if f.startswith(("usdinr", "brent")))
            == [f"brent_ln1_D{d}", f"usdinr_ln1_D{d}"] for d in (2, 3, 4)))
    # "no per-day range_Dn / gap_Dn column". The compact sets belong here even
    # though they DO see the range and the gap — through range_max_upto_D{d} and
    # gap_cum_upto_D{d}, which are summaries rather than per-day columns.
    _NO_RANGE = ("z_only", "z_plus_macro", "z_only_plus_vix",
                 "compact", "compact_plus_macro")
    chk("range_Dn is offered only by the sets that claim it",
        (any(f.startswith("range_D") for f in task_features(4, "upper"))
         == (FEATURE_SET not in _NO_RANGE)))
    chk("gap_Dn is offered only by the sets that claim it",
        (any(f.startswith("gap_D") for f in task_features(4, "upper"))
         == (FEATURE_SET not in _NO_RANGE)))
    _WITH_RV = ("full", "full_plus_vix", "full_plus_macro",
                "compact", "compact_plus_macro")
    chk("realized_vol appears only in the sets that claim it",
        (any("realized_vol" in f for f in task_features(4, "upper"))
         == (FEATURE_SET in _WITH_RV)))
    # The point of no_rv_plus_vix is that the backward-looking estimate is GONE
    # and the forward-looking one stands in its place. vix_over_rv would divide
    # by realized_vol and put it straight back, so that column must not appear.
    chk("no_rv_plus_vix contains no realized_vol, directly or inside a ratio",
        (FEATURE_SET != "no_rv_plus_vix")
        or not any(("realized_vol" in f) or f.startswith("vix_over_rv")
                   for d in (2, 3, 4) for s in ("upper", "lower")
                   for f in task_features(d, s)))
    # The PER-DAY vix sets. compact_plus_macro also uses the VIX level but reads
    # it once, on the current decision day, so it is checked separately below.
    _WITH_VIX_LEVEL = ("full_plus_vix", "no_rv_plus_vix", "full_plus_macro")
    chk("the _plus_vix sets carry the VIX level once per day and no ratio",
        (FEATURE_SET not in _WITH_VIX_LEVEL)
        or all(sum(1 for f in task_features(d, "upper")
                   if f.startswith("vix_daily_D")) == d
               and not any(f.startswith("vix_over_rv")
                           for f in task_features(d, "upper"))
               for d in (2, 3, 4)))
    chk("compact_plus_macro reads VIX once, on the current day",
        (FEATURE_SET != "compact_plus_macro") or all(
            [f for f in task_features(d, "upper") if f.startswith("vix_")]
            == [f"vix_daily_D{d}"] for d in (2, 3, 4)))
    chk("compact carries no macro column at all",
        (FEATURE_SET != "compact") or not any(
            f.startswith(("vix_", "usdinr", "brent"))
            for d in (2, 3, 4) for f in task_features(d, "upper")),
        "compact and compact_plus_macro must differ ONLY by the three macro "
        "columns, or the pair cannot isolate whether macro helps")
    chk("compact_plus_macro is compact plus exactly three columns",
        FEATURE_SET not in ("compact", "compact_plus_macro") or all(
            len(task_features(d, "upper"))
            == {"compact": 0, "compact_plus_macro": 3}[FEATURE_SET]
            + (d + 4) for d in (2, 3, 4)))
    chk("z_only carries nothing but z_dist",
        (FEATURE_SET != "z_only")
        or all(f.startswith("z_dist") for f in task_features(4, "upper")))
    # Built from the task's OWN list rather than a hard-coded column name. The
    # old version asserted z_dist_upper_D1, which every EXPANDED set asks for at
    # D2 but no COMPACT set does — it reads the current day's level instead. A
    # check that assumes one naming scheme stops testing the moment another
    # arrives, and then fails for the wrong reason.
    _want2 = task_features(2, "upper")
    chk("available_features returns exactly what the frame holds",
        available_features(pd.DataFrame(columns=_want2), 2, "upper") == _want2)
    chk("available_features drops a column the frame is missing",
        available_features(pd.DataFrame(columns=_want2[1:]), 2, "upper")
        == _want2[1:],
        "DROP_CONSTANT_FEATURES lets C5 omit a column, so this must not raise")
    chk("available_features ignores columns the task did not ask for",
        available_features(
            pd.DataFrame(columns=_want2 + ["not_a_feature"]), 2, "upper")
        == _want2)
    chk("an upper task never sees the lower side",
        not any("lower" in f for d in (2, 3, 4) for f in task_features(d, "upper")))
    chk("a lower task never sees the upper side",
        not any("upper" in f for d in (2, 3, 4) for f in task_features(d, "lower")))
    chk("no task sees a later day than its own",
        all(not any(f"_D{later}" in f for later in range(d + 1, 5))
            for d in (2, 3, 4) for s in ("upper", "lower")
            for f in task_features(d, s)))
    chk("side is required, there is no silent default",
        _raises(lambda: task_features(4, None), ValueError))
    chk("norm_dist is gone from the feature set",
        not any("norm_dist" in f for d in (2, 3, 4) for s in ("upper", "lower")
                for f in task_features(d, s)))

    # --- the per-step band reduces to the old formula on a 5-day sample ------
    # This is the check that licenses pooling cycles of different lengths: when
    # every cycle has the same length, the new estimator must agree exactly with
    # the flat one it replaces, or something has been quietly rescaled.
    rng = np.random.default_rng(0)
    n = 40
    d1 = 23000.0 * np.exp(np.cumsum(rng.normal(0, 0.01, n)))
    ret = rng.normal(0.001, 0.02, n)
    five = pd.DataFrame({"cycle_id": np.arange(n),
                         "expiry_date": pd.bdate_range("2024-01-05", periods=n, freq="7D"),
                         "cycle_days": 4, "d1_close": d1,
                         "expiry_close": d1 * np.exp(ret)})
    for k in (1, 2, 3, 4):
        five[f"d{k}_close"] = d1 * (1.0 + 0.001 * k)
    b = add_bands_and_labels(five, 1.0, 4)
    mp = min_periods(4)
    total = np.log(five["expiry_close"] / five["d1_close"])
    old_mu = total.rolling(4, min_periods=mp).mean().shift(1)
    old_sd = total.rolling(4, min_periods=mp).std().shift(1)
    ok = b["mu"].notna()
    chk("per-step mu equals the flat-window mu on an all-5-day sample",
        np.allclose(b.loc[ok, "mu"], old_mu[ok]))
    chk("per-step sigma equals the flat-window sigma on an all-5-day sample",
        np.allclose(b.loc[ok, "sigma"], old_sd[ok]))

    # --- band identity and the one-cycle lag -------------------------------
    chk("band_width_pct == 2*k*sigma",
        np.allclose(b.loc[ok, "band_width_pct"], 2.0 * 1.0 * b.loc[ok, "sigma"]))
    spike = five.copy()
    spike.loc[spike.index[-1], "expiry_close"] = spike["d1_close"].iloc[-1] * 1.5
    bs = add_bands_and_labels(spike, 1.0, 4)
    chk("the one-cycle lag holds: a cycle cannot set its own band",
        np.isclose(bs["sigma"].iloc[-1], b["sigma"].iloc[-1]),
        "the final cycle's own outcome moved its own sigma")
    chk("a NaN band gives a NaN label, never 0",
        bool(b["upper_breach"].iloc[0] != b["upper_breach"].iloc[0]))

    # --- features, on a daily frame whose answers are known by hand ---------
    days = pd.bdate_range("2024-01-01", periods=60)
    # Returns must VARY. A constant-drift series has a standard deviation of
    # exactly zero, which would make realized_vol zero and z_dist correctly NaN —
    # testing the NaN guard rather than the arithmetic it is meant to protect.
    close = 23000.0 * np.exp(np.cumsum(
        np.random.default_rng(7).normal(0.0005, 0.008, 60)))
    daily = pd.DataFrame({DATE: days, OPEN: close * 1.0005, CLOSE: close,
                          HIGH: close * 1.01, LOW: close * 0.99})
    cyc = pd.DataFrame([{
        "cycle_id": 1, "cycle_days": 4,
        "expiry_date": days[54], "expiry_close": float(close[54]),
        **{f"d{k}_date": days[49 + k] for k in (1, 2, 3, 4)},
        **{f"d{k}_close": float(close[49 + k]) for k in (1, 2, 3, 4)},
        "mu": 0.001, "sigma": 0.02, "sigma_step": 0.01,
        "band_upper": float(close[50]) * 1.03,
        "band_lower": float(close[50]) * 0.97,
        "band_width_pct": math.log(1.03 / 0.97)}])
    ft = add_features(cyc, daily, 4)

    # EVERY day, not one example. D3 alone would pass even if the day index were
    # mis-wired for D1, D2 or D4 — and D1 is the one most likely to be wrong,
    # because it is the only day whose gap looks outside the cycle.
    _all_ret = np.log(daily[CLOSE] / daily[CLOSE].shift(1))
    _rv_ok = _gap_ok = _z_ok = _sign_ok = _rg_ok = True
    _rv_detail = _gap_detail = _z_detail = _rg_detail = ""
    for _n in (1, 2, 3, 4):
        _pos = 49 + _n                      # index of d{n}_date in `days`
        _dl = 4 + 1 - _n                    # days_left for a 4-day cycle

        _rv_file = float(ft[f"realized_vol_D{_n}"].iloc[0])
        # The window comes from rv_window_days, not a literal 16. A self-test that
        # hard-codes the default silently stops testing anything the moment the
        # default changes — and then fails loudly for the wrong reason, which is
        # what happened the first time RV_WINDOW_DAYS was set to 5.
        _rvw = rv_window_days(4)
        _rv_want = float(_all_ret.rolling(_rvw, min_periods=_rvw).std().iloc[_pos])
        if not np.isclose(_rv_file, _rv_want):
            _rv_ok = False
            _rv_detail += (f" D{_n}: {_rv_file:.6f} vs {_rv_want:.6f};")

        # For n > 1 the denominator is the previous day's close on the same row.
        # For D1 there is no D0, so it is the previous TRADING day — the prior
        # cycle's expiry close.
        _gap_file = float(ft[f"gap_D{_n}"].iloc[0])
        _gap_want = math.log(float(close[_pos]) * 1.0005 / float(close[_pos - 1]))
        if not np.isclose(_gap_file, _gap_want):
            _gap_ok = False
            _gap_detail += f" D{_n}: {_gap_file:.6f} vs {_gap_want:.6f};"

        _z_file = float(ft[f"z_dist_upper_D{_n}"].iloc[0])
        _z_want = (math.log(float(ft["band_upper"].iloc[0]) / float(close[_pos]))
                   / (_rv_want * math.sqrt(_dl)))
        if not np.isclose(_z_file, _z_want):
            _z_ok = False
            _z_detail += f" D{_n}: {_z_file:.6f} vs {_z_want:.6f};"

        if not float(ft[f"z_dist_lower_D{_n}"].iloc[0]) < 0:
            _sign_ok = False

        _rg_file = float(ft[f"range_D{_n}"].iloc[0])
        _rg_want = math.log(float(close[_pos]) * 1.01 / (float(close[_pos]) * 0.99))
        if not np.isclose(_rg_file, _rg_want):
            _rg_ok = False
            _rg_detail += f" D{_n}: {_rg_file:.6f} vs {_rg_want:.6f};"

    chk(f"realized_vol_Dn is a {rv_window_days(4)}-day std read on d{{n}}_date, "
        f"including that day, for EVERY n", _rv_ok, _rv_detail)
    chk("gap_Dn == ln(open on Dn / previous close) for EVERY n, with D1 using "
        "the previous TRADING day since there is no D0", _gap_ok, _gap_detail)
    chk("z_dist_upper_Dn == ln(U/Cn) / (RVn * sqrt(days_left_n)) for EVERY n, "
        "with days_left 4/3/2/1", _z_ok, _z_detail)
    chk("the lower side keeps its sign on every day", _sign_ok)
    chk("range_Dn == ln(High/Low) on d{n}_date for EVERY n", _rg_ok, _rg_detail)
    chk("range_Dn is always positive, since High >= Low",
        bool(all(float(ft[f"range_D{_n}"].iloc[0]) > 0 for _n in (1, 2, 3, 4))))

    # --- the path summaries, against hand computation -----------------------
    # ft was built from `cyc`, a single 4-day cycle whose closes are known, so
    # every aggregate can be checked against the raw per-day columns it came
    # from. A summary that silently disagrees with its own inputs is the one bug
    # these columns could introduce.
    _agg_ok, _agg_detail = True, ""
    for _d in (2, 3, 4):
        _zc = [f"z_dist_upper_D{_k}" for _k in range(1, _d + 1)]
        _zv = np.array([float(ft[c].iloc[0]) for c in _zc])
        for _nm, _want in (
                (f"z_chg_upper_upto_D{_d}", _zv[-1] - _zv[0]),
                (f"z_min_upper_upto_D{_d}", _zv.min()),
                (f"z_rng_upper_upto_D{_d}", _zv.max() - _zv.min()),
                (f"gap_cum_upto_D{_d}",
                 sum(float(ft[f"gap_D{_k}"].iloc[0]) for _k in range(1, _d + 1))),
                (f"range_max_upto_D{_d}",
                 max(float(ft[f"range_D{_k}"].iloc[0]) for _k in range(1, _d + 1))),
        ):
            if _nm not in ft.columns or not np.isclose(float(ft[_nm].iloc[0]), _want):
                _agg_ok = False
                _got = float(ft[_nm].iloc[0]) if _nm in ft.columns else float("nan")
                _agg_detail += f" {_nm}: {_got:.6f} vs {_want:.6f};"
    chk("every path summary equals its hand computation from the per-day columns",
        _agg_ok, _agg_detail)
    chk("z_rng is never negative, since max >= min",
        all(float(ft[f"z_rng_upper_upto_D{_d}"].iloc[0]) >= -1e-12
            for _d in (2, 3, 4)))
    # The rank cap, demonstrated rather than asserted: at D2 the two shape
    # statistics are exact functions of the two that are always kept.
    _z1, _z2 = (float(ft["z_dist_upper_D1"].iloc[0]),
                float(ft["z_dist_upper_D2"].iloc[0]))
    chk("at D2, z_min and z_rng are recoverable from z_now and z_chg",
        np.isclose(float(ft["z_min_upper_upto_D2"].iloc[0]),
                   _z2 - max(0.0, _z2 - _z1))
        and np.isclose(float(ft["z_rng_upper_upto_D2"].iloc[0]), abs(_z2 - _z1)),
        "if this ever fails the arithmetic changed; if it passes, it is the "
        "reason the compact sets take only d z summaries at day d")
    chk("the lower side's summaries use the lower columns only",
        all(not any("upper" in f for f in task_features(_d, "lower"))
            for _d in (2, 3, 4)))

    zero = cyc.copy()
    zero_daily = daily.copy()
    zero_daily[CLOSE] = 23000.0                 # a flat series -> zero volatility
    _zf = add_features(zero, zero_daily, 4)
    chk("zero volatility gives NaN, never infinity, on every day",
        bool(all(pd.isna(_zf[f"z_dist_upper_D{_n}"].iloc[0]) for _n in (1, 2, 3, 4))))

    # --- short cycles reach the right tasks ---------------------------------
    short = pd.concat([cyc.assign(cycle_id=2, cycle_days=3)], ignore_index=True)
    both = pd.concat([cyc, short], ignore_index=True)
    chk("a 4-day cycle appears in the D3 task", len(task_rows(both, 3)) == 2)
    chk("a 4-day cycle is absent from the D4 task", len(task_rows(both, 4)) == 1)
    chk("days_left_dec varies once short cycles are present",
        task_rows(both, 3)["days_left_dec"].nunique() == 2,
        "it is still constant, so it carries no information")

    # --- the Gaussian's anchor ---------------------------------------------
    k = 1.0
    syn = pd.DataFrame([{"cycle_days": 4, "sigma_step": 0.01,
                         "d1_close": 100.0, "d2_close": 100.0,
                         "d3_close": 100.0, "d4_close": 100.0,
                         "band_upper": 100.0 * math.exp(k * 0.01 * 2.0),
                         "band_lower": 100.0 * math.exp(-k * 0.01 * 2.0)}])
    p1 = gaussian_breach_prob(syn, 1, "upper")[0]
    chk("at D1 the Gaussian returns the band's own quantile, 1 - Phi(k)",
        np.isclose(p1, 1 - 0.5 * (1 + math.erf(k / math.sqrt(2)))),
        f"got {p1:.6f}")
    bad = syn.copy()
    bad.loc[0, "sigma_step"] = np.nan
    chk("an unscoreable row returns NaN, not a confident 0.0",
        bool(np.isnan(gaussian_breach_prob(bad, 2, "upper")[0])))

    # --- thresholds and metrics --------------------------------------------
    y = np.array([0, 0, 1, 0, 1, 1, 0, 1, 0, 0])
    p = np.array([.01, .02, .03, .011, .035, .04, .015, .05, .012, .005])
    t, f = best_fbeta_threshold(y, p)
    chk("a cut is found even when every probability is below 0.05",
        t is not None and f > 0, f"thr={t} {FBETA_LABEL.lower()}={f}")
    chk(f"{FBETA_LABEL} is unchanged by a monotone rescale",
        np.isclose(f, best_fbeta_threshold(y, p * 3.0)[1]))

    # beta = 1 must reproduce the old F1 exactly, element for element. This is
    # the check that proves the generalisation did not change the beta = 1 case.
    _f1_ref = lambda yy, pp, tt: (lambda tp, fp, fn: 0.0 if 2 * tp + fp + fn == 0
                                  else 2.0 * tp / (2 * tp + fp + fn))(
        int((((pp >= tt) == 1) & (yy == 1)).sum()),
        int((((pp >= tt) == 1) & (yy == 0)).sum()),
        int((((pp >= tt) == 0) & (yy == 1)).sum()))
    chk("beta=1 F-beta equals textbook F1 at every candidate cut",
        all(np.isclose(fbeta_at(y, p, tt, beta=1.0), _f1_ref(y, p, tt))
            for tt in np.r_[0.0, np.unique(p), 1.0]))
    chk("f1_at is beta=1 regardless of F_BETA",
        all(np.isclose(f1_at(y, p, tt), _f1_ref(y, p, tt))
            for tt in np.r_[0.0, np.unique(p), 1.0]))
    chk("raising beta never lowers the score of a recall-heavy cut",
        fbeta_at(y, p, 0.0, beta=3.0) >= fbeta_at(y, p, 0.0, beta=1.0))

    # The trivial predictor, by hand. Always predicting breach gives
    # precision = b, recall = 1, so F_beta = (1+b2)b / (b2 b + 1).
    for _b in (0.10, 0.25, 1.0 / 3.0, 0.43):
        _yv = np.zeros(10000, int)
        _yv[:int(round(_b * 10000))] = 1
        # Compare against the vector's OWN base rate, not the requested one.
        # 1/3 cannot be represented exactly in 10000 rows, and an earlier version
        # of this check compared trivial_fbeta(1/3) with the score of a 0.3333
        # vector and failed by 1e-5 while printing two identical-looking numbers.
        _emp = float(_yv.mean())
        _got = fbeta_at(_yv, np.ones(10000), 0.5)
        chk(f"trivial_fbeta matches always-predict-breach at base rate {_emp:.4f}",
            np.isclose(trivial_fbeta(_emp), _got, rtol=0, atol=1e-12),
            f"{trivial_fbeta(_emp):.10f} vs {_got:.10f}")

    chk("the gate floor rises above the trivial score once a base rate is known",
        (PRIMARY_METRIC != "fbeta")
        or effective_floor(0.43) > trivial_fbeta(0.43) - 1e-12)
    chk("a constant predictor cannot pass the gate at any base rate",
        (PRIMARY_METRIC != "fbeta")
        or not any(model_passes([(trivial_fbeta(bb), trivial_fbeta(bb))] * 4,
                                base_rate=bb)[0]
                   for bb in (0.10, 0.20, 1.0 / 3.0, 0.43, 0.50)))

    # SCALE INVARIANCE. This is the property the absolute margin did not have and
    # the reason the gate failed every model at sigma 0.10 while passing weaker
    # ones at sigma 0.50. A model capturing the same FRACTION of the available
    # headroom must get the same verdict whatever the base rate.
    chk("the floor gives the same verdict for the same skill at every base rate",
        (PRIMARY_METRIC != "fbeta")
        or len({model_passes(
            [(trivial_fbeta(bb) + 0.30 * (1 - trivial_fbeta(bb)),
              trivial_fbeta(bb) + 0.25 * (1 - trivial_fbeta(bb)))] * 4,
            base_rate=bb)[0]
            for bb in (0.05, 0.10, 0.20, 1.0 / 3.0, 0.43, 0.47, 0.50)}) == 1,
        "a 25 per cent-of-headroom model is accepted at some base rates and "
        "refused at others, which is exactly the bug this replaced")
    chk("skill is 0 for the trivial model and 1 for a perfect one, at any base rate",
        all(abs(fbeta_skill(trivial_fbeta(bb), bb)) < 1e-12
            and abs(fbeta_skill(1.0, bb) - 1.0) < 1e-12
            for bb in (0.05, 0.10, 1.0 / 3.0, 0.47, 0.50)))
    # DERIVED from F_BETA, never hard-coded. The previous version compared
    # fbeta_skill(0.898, 0.469) against zero, and 0.898 is the trivial score only
    # when beta is exactly 3 — so it failed the moment F_BETA changed, for a
    # reason that had nothing to do with the code it was testing. Exactly the
    # same defect as the old 2.0/4.5 payoff check and the old rolling(16).
    _bt = 0.469
    _triv_here = trivial_fbeta(_bt)
    chk("raw F-beta equal to the trivial score is exactly zero skill",
        abs(fbeta_skill(_triv_here, _bt)) < 1e-12,
        f"at base {_bt} and beta {F_BETA:g} the trivial score is "
        f"{_triv_here:.4f}, skill {fbeta_skill(_triv_here, _bt):.2e}")
    chk("a score a hair above trivial is near-zero skill, not a strong model",
        0.0 < fbeta_skill(_triv_here + 0.007, _bt) < 0.30,
        f"{_triv_here:.3f}+0.007 at beta {F_BETA:g} is "
        f"{fbeta_skill(_triv_here + 0.007, _bt):.3f} skill")
    # --- the payoff metric and its reference points -------------------------
    # Hand-checkable: at base rate b, always-hold earns NC when quiet and NW when
    # breached, always-exit earns BW when quiet and BC when breached, and perfect
    # foresight earns BC on breach weeks and NC on quiet ones.
    for _b in (0.10, 0.20, 1.0 / 3.0, 0.47, 0.60):
        _hold = _b * PAYOFF_NOBREACH_WRONG + (1 - _b) * PAYOFF_NOBREACH_CORRECT
        _exit = _b * PAYOFF_BREACH_CORRECT + (1 - _b) * PAYOFF_BREACH_WRONG
        _perf = _b * PAYOFF_BREACH_CORRECT + (1 - _b) * PAYOFF_NOBREACH_CORRECT
        chk(f"trivial_payoff is the better of hold/exit at base {_b:.2f}",
            np.isclose(trivial_payoff(_b), max(_hold, _exit)),
            f"{trivial_payoff(_b):.4f} vs max({_hold:.4f}, {_exit:.4f})")
        chk(f"perfect_payoff beats the trivial best at base {_b:.2f}",
            perfect_payoff(_b) > trivial_payoff(_b) - 1e-12,
            f"perfect {_perf:.4f} vs trivial {trivial_payoff(_b):.4f}")

    # payoff_at must agree with payoff_vector on the SAME decisions, and must
    # default to p* rather than to 0.5.
    _yv = np.array([1, 0, 1, 0, 1, 0, 0, 1])
    _pv = np.array([.9, .1, .45, .2, .35, .8, .05, .6])
    chk("payoff_at defaults to the economic cut, not to 0.5",
        np.isclose(payoff_at(_yv, _pv),
                   payoff_vector(_yv, (_pv >= ECONOMIC_THRESHOLD).astype(int)).mean()))
    chk("payoff_at honours an explicit threshold",
        np.isclose(payoff_at(_yv, _pv, 0.5),
                   payoff_vector(_yv, (_pv >= 0.5).astype(int)).mean()))
    chk("a payoff can never exceed perfect foresight on the same rows",
        payoff_at(_yv, _pv) <= payoff_vector(_yv, _yv).mean() + 1e-12)

    # The generic skill transform must behave for EVERY metric, not just the
    # configured one — otherwise switching PRIMARY_METRIC later finds the bug.
    chk("skill_of is 0 at the trivial score and 1 at perfect, on this metric",
        all(abs(skill_of(trivial_score(bb), bb)) < 1e-9
            and abs(skill_of(perfect_score(bb), bb) - 1.0) < 1e-9
            for bb in (0.10, 0.20, 1.0 / 3.0, 0.47)),
        f"PRIMARY_METRIC={PRIMARY_METRIC}")
    chk("a no-model predictor cannot pass the gate at any base rate",
        not any(model_passes([(trivial_score(bb), trivial_score(bb))] * 4,
                             base_rate=bb)[0]
                for bb in (0.10, 0.20, 1.0 / 3.0, 0.43, 0.50)))
    chk("the same skill gets the same verdict at every base rate",
        len({model_passes(
            [(trivial_score(bb) + 0.30 * (perfect_score(bb) - trivial_score(bb)),
              trivial_score(bb) + 0.25 * (perfect_score(bb) - trivial_score(bb)))] * 4,
            base_rate=bb)[0]
            for bb in (0.05, 0.10, 0.20, 1.0 / 3.0, 0.43, 0.47, 0.50)}) == 1,
        "a 25 per cent-of-room model is accepted at some base rates and refused "
        "at others, which is the bug the skill transform exists to prevent")

    # The configured beta should agree with the payoff matrix. This WARNS rather
    # than fails, because a deliberate mismatch is a legitimate choice — but an
    # accidental one should not pass unnoticed.
    _ib = implied_beta()
    advise("F_BETA agrees with the payoff matrix",
           (not np.isfinite(_ib)) or abs(F_BETA - _ib) <= 0.15,
           f"F_BETA={F_BETA:g} but the payoffs imply beta={_ib:.3f}. That is a "
           f"legitimate choice, so this does not stop the run — but it means the "
           f"fit is tuned for one tradeoff while the economic decision is taken "
           f"at another. Either set F_BETA={_ib:.3f} or say why in the write-up.")

    # --- the ratio test must not invert on a scale that crosses zero --------
    # This is the bug the fix exists for. On raw payoff, two negative scores make
    # oof/infold RISE as the model gets worse: in-fold -0.499 with out-of-fold
    # -0.679 gave 1.36 and passed. On skill the same pair is 0.302/0.472 = 0.64,
    # which is the honest reading.
    if PRIMARY_METRIC == "payoff":
        _b0 = 0.469
        _worse = (-0.499, -2.000)      # catastrophic: oof far below in-fold
        chk("a model that collapses out of fold cannot pass on the raw ratio trick",
            not fold_passes(_worse[0], _worse[1], base_rate=_b0),
            f"raw ratio would be {_worse[1] / _worse[0]:.2f}, which the old test "
            f"read as excellent")
        chk("the ratio is computed on skill, not on raw payoff",
            (lambda a, b: np.isclose(
                skill_of(b, _b0) / skill_of(a, _b0), 0.6412, atol=5e-3))(
                -0.499, -0.679),
            "in-fold -0.499 / out-of-fold -0.679 must read as a skill ratio of "
            "about 0.64, not as the raw 1.36")

    # A ratio on skill must be SCALE FREE: the same relative shortfall gets the
    # same verdict whatever the base rate, because the room cancels.
    chk("the skill ratio gives the same verdict at every base rate",
        len({fold_passes(
            trivial_score(bb) + 0.80 * (perfect_score(bb) - trivial_score(bb)),
            trivial_score(bb) + 0.40 * (perfect_score(bb) - trivial_score(bb)),
            base_rate=bb)
            for bb in (0.10, 0.20, 1.0 / 3.0, 0.43, 0.47)}) == 1,
        "an out-of-fold score at half the in-fold skill is judged differently at "
        "different base rates, which means the room did not cancel")

    # --- the fitted cut must optimise the metric it is reported on -----------
    _yt = np.array([0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0])
    _pt = np.array([.1, .9, .2, .55, .7, .3, .05, .45, .15, .8, .35, .25])
    _tp, _vp = best_primary_threshold(_yt, _pt, fallback=0.5)
    chk(f"best_primary_threshold returns a usable cut for {PRIMARY_METRIC}",
        _tp is not None and np.isfinite(_vp), f"thr={_tp} value={_vp}")
    if PRIMARY_METRIC == "payoff":
        # No other candidate cut may beat the one it chose, on its own objective.
        _best = max(float(payoff_vector(_yt, (_pt >= t).astype(int)).mean())
                    for t in np.r_[0.0, np.unique(_pt), 1.0])
        chk("no threshold beats the chosen one on payoff",
            float(payoff_vector(_yt, (_pt >= _tp).astype(int)).mean())
            >= _best - 1e-9,
            f"chose {_tp:.3f} worth "
            f"{float(payoff_vector(_yt, (_pt >= _tp).astype(int)).mean()):.4f}, "
            f"best available {_best:.4f}")
        # And it should differ from the F-beta cut, or the fix changed nothing.
        _tf, _ = best_fbeta_threshold(_yt, _pt, fallback=0.5)
        advise("the payoff-optimal cut differs from the F-beta cut",
               not np.isclose(_tp, _tf),
               f"both chose {_tp:.3f} on this toy frame — possible, but on real "
               f"data they should usually differ")
    chk("the fitted-cut rule is no longer named after one metric",
        "f1_optimal" not in DECISION_RULES and "fitted_cut" in DECISION_RULES,
        str(DECISION_RULES))

    chk("fewer than two positives refuses rather than returning 0.5",
        best_f1_threshold(np.array([0, 0, 0, 1]), np.array([.1, .2, .3, .4]))[0] is None)

    yy = np.array([0, 0, 1, 1, 0, 1, 0, 0, 1, 0], float)
    parts = brier_parts(yy, np.clip(yy * .6 + .2, 0, 1), bins=5)
    chk("Brier identity: BS = reliability - resolution + uncertainty",
        abs(parts["reliability"] - parts["resolution"] + parts["uncertainty"]
            - parts["brier"]) < 1e-9)
    chk("Brier skill of a perfect forecast == 1", np.isclose(brier_skill(yy, yy), 1.0))
    chk("Brier skill of the base rate == 0",
        abs(brier_skill(yy, np.full(len(yy), yy.mean()))) < 1e-9)

    # --- every configured model builds and returns probabilities -------------
    _Xs = np.random.default_rng(5).normal(0, 1, (80, 6))
    _ys = (_Xs[:, 0] + 0.5 * _Xs[:, 1] > 0).astype(int)
    for _m in MODELS:
        try:
            _pipe = make_pipeline(_m).fit(_Xs, _ys)
            _pr = _pipe.predict_proba(_Xs)[:, 1]
            _ok = (_pr.shape == (80,) and np.isfinite(_pr).all()
                   and float(_pr.min()) >= 0.0 and float(_pr.max()) <= 1.0)
            _why = ""
        except ImportError as _e:
            # Failing HERE is the point. Discovering a missing library thirty
            # minutes into C6, after several folds have already been fitted, wastes
            # the run; the message names the fix instead of the symptom.
            _ok, _why = False, (f"{_e}. Install it (pip install xgboost) or "
                                f"remove '{_m}' from MODELS.")
        except Exception as _e:
            _ok, _why = False, f"{type(_e).__name__}: {str(_e)[:80]}"
        chk(f"{_m} fits and returns probabilities in [0, 1]", _ok, _why)
    chk("every configured model has a hyperparameter space",
        all(m in HYPERPARAMS for m in MODELS),
        str([m for m in MODELS if m not in HYPERPARAMS]))

    # --- payoff matrix and the economic cut ---------------------------------
    # Solve the indifference equation NUMERICALLY instead of comparing against a
    # number worked out by hand. The old version hard-coded 2.0/4.5 = 0.4444,
    # correct only for the original constants, and failed with "0.4000" the moment
    # the payoffs were changed — reporting a config error where there was none.
    #
    # It stays a genuine independent check: economic_threshold() solves the
    # equation algebraically, this one finds the root by search. Agreement means
    # two different routes reached the same place.
    def _edge(_p):
        """Expected value of exiting minus expected value of holding, at P=_p."""
        return ((_p * PAYOFF_BREACH_CORRECT + (1 - _p) * PAYOFF_BREACH_WRONG)
                - (_p * PAYOFF_NOBREACH_WRONG
                   + (1 - _p) * PAYOFF_NOBREACH_CORRECT))

    _lo, _hi = 0.0, 1.0
    if _edge(_lo) * _edge(_hi) > 0:
        # No sign change in [0, 1]: one action dominates at every probability, so
        # there is no interior break-even point and economic_threshold() should
        # have clipped to a boundary.
        _root = 0.0 if _edge(_lo) > 0 else 1.0
    else:
        for _ in range(200):
            _mid = 0.5 * (_lo + _hi)
            if _edge(_lo) * _edge(_mid) <= 0:
                _hi = _mid
            else:
                _lo = _mid
        _root = 0.5 * (_lo + _hi)

    chk("the economic threshold solves the indifference equation",
        abs(ECONOMIC_THRESHOLD - _root) < 1e-9,
        f"formula {ECONOMIC_THRESHOLD:.6f} vs bisection {_root:.6f}")
    chk("the two actions break even exactly at p*",
        abs((lambda p: (p * PAYOFF_BREACH_CORRECT + (1 - p) * PAYOFF_BREACH_WRONG)
             - (p * PAYOFF_NOBREACH_WRONG + (1 - p) * PAYOFF_NOBREACH_CORRECT))(
            ECONOMIC_THRESHOLD)) < 1e-9,
        "exiting and holding should have equal expected value at the cut")
    chk("exiting is preferred above p* and holding below it",
        (lambda lo, hi: lo < 0 < hi)(
            (ECONOMIC_THRESHOLD - 0.1) * PAYOFF_BREACH_CORRECT
            + (1 - (ECONOMIC_THRESHOLD - 0.1)) * PAYOFF_BREACH_WRONG
            - ((ECONOMIC_THRESHOLD - 0.1) * PAYOFF_NOBREACH_WRONG
               + (1 - (ECONOMIC_THRESHOLD - 0.1)) * PAYOFF_NOBREACH_CORRECT),
            (ECONOMIC_THRESHOLD + 0.1) * PAYOFF_BREACH_CORRECT
            + (1 - (ECONOMIC_THRESHOLD + 0.1)) * PAYOFF_BREACH_WRONG
            - ((ECONOMIC_THRESHOLD + 0.1) * PAYOFF_NOBREACH_WRONG
               + (1 - (ECONOMIC_THRESHOLD + 0.1)) * PAYOFF_NOBREACH_CORRECT)))
    # NOT a distinctness check any more. Under a flat exit cost the two exit
    # cells are deliberately equal, so demanding four distinct values fails on a
    # CORRECT configuration — which is why the previous version of this check had
    # to be commented out to get a run through. What actually matters is that the
    # four values are usable numbers and that they are ORDERED the way the story
    # above claims, because a sign slip would leave every check passing while the
    # threshold pointed the decision the wrong way.
    _cells = (PAYOFF_BREACH_CORRECT, PAYOFF_BREACH_WRONG,
              PAYOFF_NOBREACH_CORRECT, PAYOFF_NOBREACH_WRONG)
    chk("every payoff cell is a finite number",
        all(np.isfinite(v) for v in _cells), str(_cells))
    chk("holding into a breach is the worst outcome available",
        PAYOFF_NOBREACH_WRONG == min(_cells),
        f"worst cell is {min(_cells)}, hold/breach is {PAYOFF_NOBREACH_WRONG}")
    chk("holding through a quiet week is the best outcome available",
        PAYOFF_NOBREACH_CORRECT == max(_cells),
        f"best cell is {max(_cells)}, hold/no-breach is {PAYOFF_NOBREACH_CORRECT}")
    chk("p* lies strictly inside (0, 1), so both actions are reachable",
        0.0 < ECONOMIC_THRESHOLD < 1.0,
        f"{ECONOMIC_THRESHOLD:.4f} — at a boundary one action is never chosen")
    _py = payoff_vector([1, 0, 0, 1], [1, 1, 0, 0])
    chk("payoff_vector maps each of the four outcomes correctly",
        list(_py) == [PAYOFF_BREACH_CORRECT, PAYOFF_BREACH_WRONG,
                      PAYOFF_NOBREACH_CORRECT, PAYOFF_NOBREACH_WRONG],
        str(list(_py)))
    chk("always holding is worse than always exiting at a high base rate",
        payoff_vector(np.ones(10), np.zeros(10)).mean()
        < payoff_vector(np.ones(10), np.ones(10)).mean())

    # --- preprocessing learns its bounds from TRAINING rows only ------------
    tr = np.random.default_rng(0).normal(0, 1, (200, 1))
    te = np.random.default_rng(1).normal(0, 1, (50, 1)) * 5
    w = Winsoriser().fit(tr)
    chk("winsorising clips test rows to the TRAINING bounds",
        float(w.transform(te).max()) <= float(np.nanpercentile(tr, 99)) + 1e-9,
        f"test max came through as {float(w.transform(te).max()):.3f} against a "
        f"training 99th percentile of {float(np.nanpercentile(tr, 99)):.3f}")
    chk("winsorising leaves training rows inside their own bounds",
        float(w.transform(tr).max()) <= float(np.nanpercentile(tr, 99)) + 1e-9)

    # --- ensemble and gate --------------------------------------------------
    dec, prob, eff = ensemble([0.10, 0.80, 0.30], [0.5, 0.2, 0.4])
    chk("the ensemble decision equals int(prob >= effective threshold)",
        dec == int(prob >= eff), f"dec={dec} prob={prob:.3f} eff={eff:.3f}")
    chk("the gate rejects a model that only works in-fold",
        not fold_passes(0.95, 0.20))
    chk("the gate accepts a consistent model",
        fold_passes(0.62, 0.58) if PRIMARY_METRIC == "fbeta" else True)
    chk("a model failing most folds fails overall",
        not model_passes([(0.9, 0.2), (0.9, 0.25), (0.6, 0.58), (0.9, 0.1)])[0])

    # --- walk-forward -------------------------------------------------------
    sp = walk_forward_splits(292)
    te = np.concatenate([t for _, _, t in sp])
    chk("test blocks are disjoint, so no cycle is counted twice",
        len(te) == len(set(te.tolist())))
    chk("no fold puts its test block before its validation block",
        all(v.max() < t.min() for _, v, t in sp))
    chk("training always precedes validation",
        all(tr.max() < v.min() for tr, v, _ in sp))

    if fails:
        raise AssertionError("C1 self-tests FAILED:\n  - " + "\n  - ".join(fails))
    return True


def _raises(fn, exc):
    """True if calling fn() raises exc. Keeps the self-test readable."""
    try:
        fn()
        return False
    except exc:
        return True
    except Exception:
        return False


# ============================================================================
# 13. BANNER
# ============================================================================
print(f"\n  sigmas         : {SIGMA_GRID}   (each one is a separate study)")
print(f"  windows        : {SIGMA_WINDOW_GRID} cycles")
print(f"  models         : {MODELS}")
print(f"                   top-{TOP_K} ensemble, weighted by out-of-fold score")
print(f"  feature set    : {FEATURE_SET}")
print(f"  cycle lengths  : {CYCLE_LENGTHS}"
      + ("   (full and holiday-shortened)" if CYCLE_LENGTHS == "all"
         else "   (five-trading-day cycles only)"))
print(f"  daily calendar : {'FILLED' if IMPUTE_DAILY else 'raw'}"
      + ("   (holidays carry the previous day; realized_vol ignores them)"
         if IMPUTE_DAILY else "   (holiday weeks stay 3 decision days long)"))
print(f"  features       : D2={len(task_features(2, 'upper'))} "
      f"D3={len(task_features(3, 'upper'))} "
      f"D4={len(task_features(4, 'upper'))}")
_IB = implied_beta()
if PRIMARY_METRIC == "payoff":
    print(f"  primary metric : payoff — mean payoff per cycle at p* = "
          f"{ECONOMIC_THRESHOLD:.4f}")
    print(f"                   tuned, gated and reported at the SAME fixed cut, "
          f"so no threshold is fitted")
elif PRIMARY_METRIC == "fbeta":
    print(f"  primary metric : fbeta ({FBETA_LABEL}, beta={F_BETA:g}, recall "
          f"weighted {F_BETA ** 2:g}x precision)")
else:
    print(f"  primary metric : brier_skill — proper, threshold-free")
print(f"  beta setting   : F_BETA={F_BETA:g}, payoff matrix implies "
      f"{_IB:.3f}"
      + ("   OK, they agree" if np.isfinite(_IB) and abs(F_BETA - _IB) <= 0.15
         else "   MISMATCH — tuning one tradeoff, deciding at another"))

# Say which scale each test uses. An earlier banner read "on the SKILL scale"
# for all three, which was wrong once gap and ratio moved back to raw.
print(f"  gate           : skill>={MIN_OOF:.2f} (SKILL scale)   "
      f"gap<={MAX_GAP} ratio>={MIN_RATIO} (RAW scale, {PRIMARY_METRIC} units)")
print("  score to beat  : "
      + "  ".join(f"b={bb:.2f}->{effective_floor(bb):+.3f}"
                  for bb in (0.10, 0.20, 1 / 3, 0.47))
      + f"   ({MIN_OOF:.0%} of the room above no-model)")
if PRIMARY_METRIC == "payoff":
    print("  no-model / perfect : "
          + "  ".join(f"b={bb:.2f}->{trivial_payoff(bb):+.2f}/"
                      f"{perfect_payoff(bb):+.2f}"
                      for bb in (0.10, 0.20, 1 / 3, 0.47)))
print(f"  walk-forward   : {N_FOLDS} folds, test={TEST_FRACTION}, "
      f"cross-fit K={CROSSFIT_FOLDS}")
print(f"  decision rules : {DECISION_RULES}   economic cut p* = "
      f"{ECONOMIC_THRESHOLD:.4f}")
print(f"  payoff         : exit/breach {PAYOFF_BREACH_CORRECT}, "
      f"exit/no-breach {PAYOFF_BREACH_WRONG}, "
      f"hold/no-breach {PAYOFF_NOBREACH_CORRECT}, "
      f"hold/breach {PAYOFF_NOBREACH_WRONG}")
print(f"  run tag        : {RUN_TAG}   (in every output filename)")
print("  NOT in the tag : metric, beta, gate and payoff. A SCORING change "
      "overwrites C6's")
print("                   results for the same data — re-run C6 only, C5's "
      "features are unchanged.")
print("                   C7 records those four in 00_run_index.xlsx and "
      "refuses to pool runs")
print("                   that disagree, which is what the filename cannot do.")
print(f"  shared folder  : {ROOT}   (01/02/03 — every configuration reads these)")
print(f"  results folder : {RESULTS_DIR}   (04/05/06/07/00 — one run's evidence)")
if CYCLE_LENGTHS == "full_only" and IMPUTE_DAILY:
    print("  note: CYCLE_LENGTHS='full_only' has little left to do under "
          "IMPUTE_DAILY=True —")
    print("        filling the calendar already makes almost every cycle 4 days.")

print("\n  self-tests")
self_test(verbose=True)

print("\n" + "=" * 78)
print("  C1_Config loaded — every invariant asserted")
print("  NEXT: C2_Fetch")
print("=" * 78)

  C1_Config — configuration and shared library
  Not in Colab, continuing without Drive (ValueError)
  economic cut   : p* = 0.4000   from exit/breach -1.0, exit/no-breach -1.0, hold/no-breach 1.0, hold/breach -4.0

  sigmas         : [0.1, 0.25, 0.5]   (each one is a separate study)
  windows        : [4] cycles
  models         : ['LogisticRegression', 'SVM', 'XGBoost', 'NeuralNet', 'RandomForest']
                   top-3 ensemble, weighted by out-of-fold score
  feature set    : compact_plus_macro
  cycle lengths  : full_only   (five-trading-day cycles only)
  daily calendar : FILLED   (holidays carry the previous day; realized_vol ignores them)
  features       : D2=9 D3=10 D4=11
  primary metric : fbeta (F1.225, beta=1.225, recall weighted 1.50063x precision)
  beta setting   : F_BETA=1.225, payoff matrix implies 1.225   OK, they agree
  gate           : skill>=0.10 (SKILL scale)   gap<=0.2 ratio>=0.6 (RAW scale, fbeta units)
  score to beat  : b=0.10->+0.296  b=0.20->+0.446  b=0

In [11]:
# @title
# ============================================================================
# C6_TrainPredict — TRAIN, PREDICT AND SCORE
# ============================================================================
#
#   reads  : 04_features_sigma<k>_win<W>_<align>.xlsx
#   writes : 05_results_sigma<k>_win<W>_<align>.xlsx
#
# WORKBOOK LAYOUT
# ---------------
#   predictions   one row per (cycle, day, side): both bands, both contestants'
#                 probabilities, both decisions, and the outcome
#   scores        one row per (task, contestant, decision rule): F-beta, its
#                 skill against the trivial model, plain F1, Brier, Brier skill,
#                 the Murphy decomposition, mean payoff, base rate, n
#   candidates    every (task, model) that was trained, its in-fold and
#                 out-of-fold scores, whether it passed the gate, and whether it
#                 made the ensemble
#   folds         per-fold detail — in-fold and out-of-fold score, their gap,
#                 the fold's own skill, the threshold, the test block's DATE
#                 RANGE and whether that fold passed. A mean of 0.06 can be five
#                 steady folds or four flat ones plus one lucky one, and only
#                 this sheet can tell them apart.
#   manifest      the configuration
#
# HOW ONE TASK IS FITTED
# ----------------------
# For each walk-forward fold, oldest first:
#
#   1. The fold splits into TRAIN, VAL and a sealed TEST block. Test is touched
#      once, at the end, after everything else is decided.
#   2. Hyperparameters are searched on TRAIN only, by randomised search over a
#      small space with an inner stratified CV. The space is small on purpose: at
#      roughly 60-100 positives an aggressive search finds whatever best fits
#      inner-fold noise.
#   3. The model is refitted on TRAIN, and scored IN-FOLD on TRAIN.
#   4. TRAIN+VAL is cross-fitted: every row there receives a probability from a
#      model that never saw it. Those out-of-fold probabilities give both the
#      OUT-OF-FOLD score and the decision THRESHOLD.
#   5. The consistency gate compares in-fold with out-of-fold. A model that only
#      works on rows it was fitted on is dropped before the test block is opened.
#   6. Survivors are ranked by out-of-fold score, the top k are blended, and only
#      then is TEST scored.
#
# THE GAUSSIAN GETS THE SAME TREATMENT
# ------------------------------------
# It has no parameters to tune, but it does need a threshold to turn a
# probability into a decision — and it fits that threshold on the SAME
# cross-fitted TRAIN+VAL block, with the SAME function and the same tie-break.
# Without that the comparison would be measuring two different procedures.
#
# TWO DECISION RULES, BOTH REPORTED
# ---------------------------------
#   "fitted_cut"  a threshold LEARNED from the cross-fitted TRAIN+VAL block, by
#                 maximising whatever PRIMARY_METRIC names. It never sees the
#                 test block. Named "f1_optimal" until the objective became
#                 configurable — the old name promised F1 and, under
#                 PRIMARY_METRIC="payoff", delivered a payoff-optimal cut.
#
#   "economic"    the break-even cut implied by the payoff matrix (C1 prints
#                 p*; it follows the four constants and is not fixed here).
#                 NOTHING IS FITTED — it comes from the four payoff constants
#                 alone, so it is identical for both contestants and for every
#                 task. That is the point: it is the cut a decision-maker would
#                 use, not one chosen to make a score look good.
#
# Both are reported for both contestants, with the mean payoff per cycle, which
# is the number a trader would actually care about.
#
# READ `reliability` BESIDE EVERY ECONOMIC RESULT. p* compares expected values,
# and an expected value built from a systematically biased probability is wrong by
# that same bias. A model tuned for F1 has no reason to be calibrated, so a poor
# economic score can mean mis-calibration rather than poor ranking — and the
# remedy is to calibrate the probability, not to move the cut.
#
# WHY TUNING ML AND NOT THE BASELINE IS FAIR
# ------------------------------------------
# It is not symmetric, and it is deliberately tilted TOWARDS ML: the learned side
# gets a hyperparameter search, the closed form gets nothing because it has
# nothing to search. A null result under that asymmetry is a real null result.
# ============================================================================

try:
    _ = (f_features, f_results, SIGMA_GRID, SIGMA_WINDOW_GRID, CAL_TAG,
         TASKS, MODELS, TOP_K, HYPERPARAMS, TUNE_ITERS, TUNE_INNER_FOLDS,
         make_pipeline, walk_forward_splits, stratified_folds, best_f1_threshold,
         all_scores, model_passes, pick_top_k, ensemble, PRIMARY_METRIC,
         F_BETA, FBETA_LABEL, fbeta_at, f1_at, trivial_fbeta, effective_floor,
         fbeta_skill, MIN_SKILL, skill_of, trivial_score, perfect_score,
         payoff_at, trivial_payoff, perfect_payoff, implied_beta,
         best_primary_threshold,
         DECISION_RULES, ECONOMIC_THRESHOLD, payoff_vector,
         FEATURE_SET, CYCLE_LENGTHS, RUN_TAG,
         RANDOM_STATE, read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import datetime
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV

print("\n" + "=" * 78)
print(f"  C6_TrainPredict — sigmas {SIGMA_GRID}, windows {SIGMA_WINDOW_GRID}, "
      f"tag={RUN_TAG}")
print("=" * 78)

IDENT = ["cycle_id", "expiry_date", "cycle_days", "is_padded"]
NON_FEATURE = IDENT + ["gauss_prob", "breach"]


def score_of(y, prob, thr):
    """The primary metric for one set of predictions at one threshold."""
    return all_scores(y, prob, thr)["primary"]


def contest_scores(y, prob, decision, thr_reported, rule):
    """Score a contestant using the decisions IT ACTUALLY MADE.

    Parameters
    ----------
    y : ndarray of int
        Outcomes.
    prob : ndarray of float
        Reported probabilities.
    decision : ndarray of int
        The 0/1 calls that were made.
    thr_reported : float
        Mean of the per-row thresholds, for the report only.
    rule : str
        Which decision rule produced `decision`.

    Returns
    -------
    dict
        As all_scores, but with F1 recomputed from `decision`.

    Notes
    -----
    F1 cannot be recovered by re-thresholding the probabilities at one cut,
    because the ensemble's effective threshold is a weighted mean of its members'
    own cuts and those were fitted per FOLD. Every row sits in exactly one fold's
    test block, so its threshold is the one belonging to that fold — meaningful,
    but not constant across the task.
    #
    Scoring the probabilities at some average cut would report a number that no
    decision ever produced. The threshold-free metrics (Brier and its
    decomposition) are unaffected and come straight from the probabilities.
    """
    s = all_scores(y, prob, 0.5)
    yt = np.asarray(y, int)
    yp = np.asarray(decision, int)
    tp = int(((yp == 1) & (yt == 1)).sum())
    fp = int(((yp == 1) & (yt == 0)).sum())
    fn = int(((yp == 0) & (yt == 1)).sum())
    # Both measures, from the SAME decisions. F-beta is the headline; plain F1
    # travels alongside it in every sheet so a beta-driven gain can never be
    # mistaken for a genuine one.
    b2 = float(F_BETA) ** 2
    den_b = (1.0 + b2) * tp + b2 * fn + fp
    den_1 = 2 * tp + fp + fn
    s["fbeta"] = 0.0 if den_b <= 0 else (1.0 + b2) * tp / den_b
    s["f1"] = 0.0 if den_1 == 0 else 2.0 * tp / den_1
    _br = float(yt.mean()) if len(yt) else np.nan
    s["fbeta_trivial"] = trivial_fbeta(_br)
    # The sigma-comparable figure. Raw F-beta rises with the base rate on its
    # own, so only this column may be compared across sigmas.
    s["fbeta_skill"] = fbeta_skill(s["fbeta"], _br)
    # The payoff is recomputed from the decisions ACTUALLY MADE, exactly like
    # F-beta above, so under the "economic" rule it reflects the real calls and
    # under "fitted_cut" it reflects that rule's calls. all_scores' own
    # payoff_at_pstar re-cuts the probabilities at p* and is kept as a reference
    # column, but it is not what the headline uses.
    _pay = payoff_vector(yt, yp)
    s["payoff_at_rule"] = float(_pay.mean())
    s["primary"] = {"fbeta": s["fbeta"],
                    "payoff": s["payoff_at_rule"]}.get(PRIMARY_METRIC,
                                                       s["brier_skill"])
    s["primary_skill"] = skill_of(s["primary"], _br)
    s["threshold"] = float(thr_reported)
    s["tp"], s["fp"], s["fn"] = tp, fp, fn
    s["tn"] = int(((yp == 0) & (yt == 0)).sum())
    s["rule"] = rule
    # The number a decision-maker would care about: what acting on these calls
    # would have been worth per cycle, in the payoff matrix's own units.
    s["mean_payoff"] = float(_pay.mean())
    s["total_payoff"] = float(_pay.sum())
    # The floor and ceiling of what was available on these same cycles, so a
    # payoff can be read as a fraction of what perfect foresight was worth rather
    # than as a bare number.
    s["payoff_if_perfect"] = float(payoff_vector(yt, yt).mean())
    s["payoff_if_always_hold"] = float(payoff_vector(yt, np.zeros_like(yt)).mean())
    return s


def crossfit_probs(pipe, X, y, seed=RANDOM_STATE):
    """Out-of-fold probabilities for every row of a block.

    Parameters
    ----------
    pipe : sklearn estimator
        Unfitted; cloned per inner fold so no state carries over.
    X : ndarray
    y : ndarray of int
    seed : int

    Returns
    -------
    ndarray of float
        One probability per row, each from a model that did not see that row.
        NaN where a fold could not be fitted.

    Notes
    -----
    This is what makes the threshold honest. Fitting a cut on probabilities the
    model has already seen picks the cut that flatters the training rows, and it
    does not survive contact with the test block.
    """
    from sklearn.base import clone
    out = np.full(len(y), np.nan)
    for fit_idx, out_idx in stratified_folds(y, seed=seed):
        if len(np.unique(y[fit_idx])) < 2:
            continue
        m = clone(pipe)
        m.fit(X[fit_idx], y[fit_idx])
        out[out_idx] = m.predict_proba(X[out_idx])[:, 1]
    return out


def _payoff_scorer_fn(y_true, y_proba):
    """Mean payoff at p*, for use inside RandomizedSearchCV.

    Parameters
    ----------
    y_true : ndarray of int
    y_proba : ndarray
        predict_proba output. sklearn hands over the full 2-column array for a
        binary problem, so column 1 is taken when that is what arrives.

    Notes
    -----
    THE CUT IS p*, FIXED, NOT FITTED. That is the whole reason this is a sound
    tuning target: the payoff is defined against p*, the economic decision rule
    uses p*, and nothing in the inner loop is free to move it. Fitting a cut in
    here as well would hand the search a second degree of freedom and it would
    spend it on inner-fold noise.

    Consequence worth knowing: at a fixed cut this rewards CALIBRATION as well as
    ranking. A model whose probabilities sit on the wrong side of p* scores badly
    even if it orders cycles perfectly. For a study that will decide at p* that
    is the correct preference, but it does mean class_weight="balanced" is
    penalised — it shifts probabilities toward a 50/50 prior by design.
    """
    pr = np.asarray(y_proba, float)
    if pr.ndim == 2:
        pr = pr[:, 1]
    return payoff_at(y_true, pr)


def _inner_scorer():
    """The scorer RandomizedSearchCV optimises — always the REPORTED metric.

    Returns
    -------
    str or sklearn scorer

    Notes
    -----
    Tuning on one metric and reporting another is not a cosmetic mismatch. The
    grids contain class_weight ["balanced", None], and "balanced" maximises a
    recall-heavy F-measure by re-weighting toward a 50/50 prior — which
    deliberately mis-calibrates the probabilities. Scoring those same
    probabilities with Brier afterwards cost 0.035 (0.0922 -> 0.1267) on a
    measured run, which was most of the apparent ML-versus-Gaussian Brier gap.

    So: fit for whatever PRIMARY_METRIC names, and let the threshold handle the
    operating point. sklearn ships "f1" and "neg_brier_score" as strings; F-beta
    at an arbitrary beta and the payoff both need a built scorer.
    """
    from sklearn.metrics import make_scorer
    if PRIMARY_METRIC == "payoff":
        # response_method is the current spelling; needs_proba is the old one.
        # Supporting both keeps this working across sklearn versions rather than
        # failing deep inside a fold with an unhelpful TypeError.
        try:
            return make_scorer(_payoff_scorer_fn, greater_is_better=True,
                               response_method="predict_proba")
        except TypeError:
            return make_scorer(_payoff_scorer_fn, greater_is_better=True,
                               needs_proba=True)
    if PRIMARY_METRIC == "fbeta":
        from sklearn.metrics import fbeta_score
        return make_scorer(fbeta_score, beta=float(F_BETA), zero_division=0.0)
    return "neg_brier_score"


def tuned_fit(model_name, X, y, seed=RANDOM_STATE):
    """Randomised search on the training block, then the best estimator.

    Parameters
    ----------
    model_name : str
    X, y : ndarray
    seed : int

    Returns
    -------
    (estimator, dict)
        The fitted pipeline and the chosen parameters.

    Notes
    -----
    Falls back to an untuned fit when the minority class is too small to stratify
    an inner CV — which happens on the earliest folds and should not silently
    become an error.
    """
    pipe = make_pipeline(model_name)
    space = HYPERPARAMS[model_name]
    n_pos = int(y.sum())
    inner = min(TUNE_INNER_FOLDS, n_pos, int((1 - y).sum()))
    if inner < 2:
        pipe.fit(X, y)
        return pipe, {"tuned": False}
    # TUNE ON THE METRIC THAT GETS REPORTED.
    #
    # This was hard-coded to "f1" regardless of PRIMARY_METRIC, and that is not a
    # cosmetic mismatch. The grid contains class_weight ["balanced", None], and
    # "balanced" maximises F1 for a rare class by re-weighting toward a 50/50
    # prior — which deliberately mis-calibrates the probabilities. C7 then scored
    # those same probabilities with Brier. Measured on Case 2, the mis-calibration
    # costs 0.035 Brier (0.0922 -> 0.1267), which was most of the apparent
    # ML-versus-Gaussian gap in the reported brier_skill column.
    #
    # There is no single fit that is best for both. F1 is a metric of DECISIONS at
    # a chosen threshold; Brier judges the probability itself. The pipeline already
    # has a cross-fitted threshold and the economic threshold p*, so shifting the
    # operating point is the threshold's job — doing it again inside the fit only
    # corrupts the probability on the way. So: fit for the reported metric, and
    # let the threshold handle the decision.
    inner_scoring = _inner_scorer()
    search = RandomizedSearchCV(
        pipe, space, n_iter=TUNE_ITERS, cv=inner, scoring=inner_scoring,
        random_state=seed, n_jobs=-1, refit=True, error_score=0.0)
    search.fit(X, y)
    return search.best_estimator_, {"tuned": True, **search.best_params_}


# ============================================================================
# ONE WORKBOOK PER CONFIGURATION
# ============================================================================
written = []
for sigma in SIGMA_GRID:
    for window in SIGMA_WINDOW_GRID:
        t0 = time.time()
        print("\n" + "=" * 78)
        print(f"  sigma={sigma}  window={window}")
        print("=" * 78)

        src = f_features(sigma, window)
        bands = read_xlsx(src, sheet="bands")
        band_by_cycle = bands.set_index("cycle_id")[["band_upper", "band_lower"]]

        predictions, scores, candidates, fold_rows = [], [], [], []

        for task in TASKS:
            day, side, name = task["day"], task["side"], f"D{task['day']}_{task['side']}"
            sheet = read_xlsx(src, sheet=name).sort_values("expiry_date")
            sheet = sheet.reset_index(drop=True)

            feats = [c for c in sheet.columns if c not in NON_FEATURE]
            X = sheet[feats].to_numpy(float)
            y = sheet["breach"].to_numpy(int)
            gp = sheet["gauss_prob"].to_numpy(float)

            # Base rate for THIS task. The F-beta gate floor depends on it,
            # because always predicting breach scores higher the commoner the
            # event is. Taken over the whole task rather than per fold so the
            # bar does not drift between folds of the same model.
            _task_base = float(y.mean()) if len(y) else float("nan")

            splits = walk_forward_splits(len(y))
            print(f"\n  {name}: {len(y)} rows, {int(y.sum())} breaches, "
                  f"{len(feats)} features, {len(splits)} fold(s)")
            print(f"    features: {', '.join(feats)}")
            _triv = trivial_score(_task_base)
            _perf = perfect_score(_task_base)
            print(f"    base rate {_task_base:.3f} -> no-model "
                  f"{PRIMARY_METRIC} = {_triv:+.3f}, perfect {_perf:+.3f}, "
                  f"room {_perf - _triv:.3f}, gate floor "
                  f"{effective_floor(_task_base):+.3f} (= {MIN_SKILL:.0%} skill)")
            if PRIMARY_METRIC == "payoff":
                print(f"    EPV: {int(y.sum())} breaches / {len(feats)} features "
                      f"= {y.sum() / max(len(feats), 1):.1f} events per variable"
                      + ("   (below 10 — read any win with suspicion)"
                         if y.sum() / max(len(feats), 1) < 10 else ""))
            if not splits:
                print("    too few rows for a walk-forward split, skipped")
                continue

            # ---- train every model across every fold ------------------------
            per_model = {}
            for model_name in MODELS:
                pairs, test_bits = [], []
                for fi, (tr, va, te) in enumerate(splits, 1):
                    trva = np.concatenate([tr, va])
                    if len(np.unique(y[tr])) < 2 or len(np.unique(y[trva])) < 2:
                        continue

                    est, params = tuned_fit(model_name, X[tr], y[tr])

                    # In-fold: the rows the model was fitted on.
                    p_in = est.predict_proba(X[tr])[:, 1]

                    # Out-of-fold across train+val, which also sets the threshold.
                    p_oof = crossfit_probs(make_pipeline(model_name), X[trva], y[trva])
                    ok = np.isfinite(p_oof)
                    # Fit the cut on the PRIMARY metric, so the rule that uses
                    # it is scored on the same thing it was optimised for.
                    thr, _ = best_primary_threshold(y[trva][ok], p_oof[ok],
                                                    fallback=0.5)
                    if thr is None:
                        continue

                    s_in = score_of(y[tr], p_in, thr)
                    s_oof = score_of(y[trva][ok], p_oof[ok], thr)
                    pairs.append((s_in, s_oof))

                    # The sealed block, opened only now.
                    p_te = est.predict_proba(X[te])[:, 1]
                    test_bits.append({"fold": fi, "idx": te, "prob": p_te,
                                      "thr": thr})

                    # The gate floor uses THIS TASK's base rate, not the fold's.
                    # An earlier version used y[trva].mean() here while the
                    # model-level gate used the task rate, so the per-fold
                    # fold_passed column and the overall PASS/fail verdict were
                    # answering slightly different questions. One bar per task.
                    _te_dates = pd.to_datetime(sheet["expiry_date"].iloc[te])
                    fold_rows.append({
                        "task": name, "model": model_name, "fold": fi,
                        "n_train": len(tr), "n_trainval": len(trva), "n_test": len(te),
                        "infold_score": round(s_in, 4),
                        "oof_score": round(s_oof, 4),
                        "gap": round(abs(s_in - s_oof), 4),
                        # Skill, so a fold can be compared with a fold of a
                        # different task or a different sigma.
                        "infold_skill": round(skill_of(s_in, _task_base), 4),
                        "oof_skill": round(skill_of(s_oof, _task_base), 4),
                        "threshold": round(thr, 4),
                        "fold_passed": bool(model_passes(
                            [(s_in, s_oof)], base_rate=_task_base)[0]),
                        "base_rate": round(_task_base, 4),
                        "fold_base_rate": round(float(y[trva].mean()), 4),
                        "floor_applied": round(effective_floor(_task_base), 4),
                        # WHICH CYCLES this fold actually tested. Without the
                        # dates a weak fold cannot be traced to a market period,
                        # and "the model broke in 2025" is unanswerable.
                        "test_from": _te_dates.min().date(),
                        "test_to": _te_dates.max().date(),
                        "test_breaches": int(y[te].sum()),
                        "params": str({k: v for k, v in params.items()
                                       if k != "tuned"})[:180],
                    })

                if not pairs:
                    print(f"    {model_name:<20} no usable fold")
                    continue

                # The floor must know this task's base rate: at F_BETA > 1 the
                # always-predict-breach score rises with the base rate, so a
                # single static floor would pass a constant predictor on the
                # common tasks and fail a real model on the rare ones.
                passed, mean_oof = model_passes(pairs, base_rate=_task_base)
                per_model[model_name] = {
                    "pairs": pairs, "test": test_bits,
                    "oof_score": mean_oof,
                    "infold_score": float(np.mean([a for a, _ in pairs])),
                    "passed": passed,
                }
                # Print the SKILL beside the raw score. Without it a line
                # reading "oof=0.905 fail gate" looks like the gate is broken,
                # when 0.905 at a base rate of 0.469 is 7 per cent of the
                # available headroom — the number the gate actually judged.
                _inf = per_model[model_name]["infold_score"]
                print(f"    {model_name:<20} oof={mean_oof:+.3f} "
                      f"infold={_inf:+.3f}   skill "
                      f"oof={skill_of(mean_oof, _task_base):+.3f} "
                      f"infold={skill_of(_inf, _task_base):+.3f}   "
                      f"{'PASS' if passed else 'fail gate'}")

                # PER-FOLD, every model. The mean above hides the one thing that
                # decides whether a result is real: is the edge steady across
                # folds, or is one fold carrying the others? Four numbers on one
                # line is cheap, and it is the difference between "+0.06" and
                # "+0.06, from a single fold in 2025".
                _fr = [r for r in fold_rows
                       if r["task"] == name and r["model"] == model_name]
                if _fr:
                    print("        folds  "
                          + "  ".join(
                              f"[{r['fold']}] {r['oof_score']:+.3f}"
                              f"/{r['oof_skill']:+.2f}"
                              f" {'y' if r['fold_passed'] else 'n'}"
                              for r in _fr)
                          + "   (oof/skill, passed)")

            if not per_model:
                print("    nothing trained, skipped")
                continue

            # The feature list is read from the SHEET, not rebuilt from the
            # config, so FEATURE_SET and any constant column C5 dropped are both
            # honoured automatically. Recording it here means a results file says
            # what it was trained on rather than implying the config's answer.
            cand = pd.DataFrame([
                {"task": name, "n_features": len(feats),
                 "features": ", ".join(feats),
                 "model": m, "oof_score": round(d["oof_score"], 4),
                 "infold_score": round(d["infold_score"], 4),
                 "gap": round(abs(d["infold_score"] - d["oof_score"]), 4),
                 "folds": len(d["pairs"]), "passed_gate": d["passed"]}
                for m, d in per_model.items()])

            # ---- select, then ensemble --------------------------------------
            eligible = cand[cand["passed_gate"]]
            gate_used = True
            if eligible.empty:
                # Reporting "no model" would hide the finding. The gate failing
                # for every model IS the result at this task, and the ensemble is
                # still built so the magnitude of the failure is visible.
                print("    no model passed the gate — reporting ungated, flagged")
                eligible, gate_used = cand, False
            chosen = pick_top_k(eligible, TOP_K)
            cand["in_ensemble"] = cand["model"].isin(chosen["model"])
            cand["gate_applied"] = gate_used
            candidates.append(cand)

            # ---- assemble out-of-sample predictions -------------------------
            # Every fold's test block is disjoint, so a cycle appears at most once
            # per task and cannot be counted twice.
            members = list(chosen["model"])
            weights = [max(per_model[m]["oof_score"], 0.01) for m in members]

            ml_prob = np.full(len(y), np.nan)
            ml_dec = np.full(len(y), np.nan)
            ml_thr_row = np.full(len(y), np.nan)     # per row, see contest_scores

            # Gather each member's prediction per row by SCATTERING its fold test
            # blocks into full-length arrays, then blend row by row.
            #
            # The previous version scanned every fold of every member for every
            # row, asking "is this row in that block?" — correct, but O(rows x
            # members x folds x block) and hard to read. The test blocks are
            # disjoint by construction (walk_forward_splits asserts it), so a
            # single assignment per block puts each member's probability exactly
            # where it belongs and nothing can overwrite anything.
            member_prob, member_thr = {}, {}
            for m in members:
                pr = np.full(len(y), np.nan)
                th = np.full(len(y), np.nan)
                for bit in per_model[m]["test"]:
                    pr[bit["idx"]] = bit["prob"]
                    th[bit["idx"]] = bit["thr"]
                member_prob[m], member_thr[m] = pr, th

            for pos in np.flatnonzero(
                    np.any([np.isfinite(member_prob[m]) for m in members], axis=0)):
                probs, thrs, ws = [], [], []
                for m, w in zip(members, weights):
                    if np.isfinite(member_prob[m][pos]):
                        probs.append(float(member_prob[m][pos]))
                        thrs.append(float(member_thr[m][pos]))
                        ws.append(w)
                d, p, eff = ensemble(probs, thrs, ws)
                ml_prob[pos], ml_dec[pos], ml_thr_row[pos] = p, d, eff

            scored = np.isfinite(ml_prob)
            n_scored = int(scored.sum())
            print(f"    out-of-sample rows: {n_scored}")
            if n_scored == 0:
                continue

            # ---- the Gaussian's threshold, PER FOLD, same blocks as ML ------
            #
            # THIS USED TO BE ONE GLOBAL CUT AND IT WAS NOT A FAIR COMPARISON.
            # The old version fitted a single threshold on ~scored — the rows in
            # no fold's test block — and applied it to all 228 scored rows. On a
            # 382-row task that meant:
            #
            #   ML:       fold 1 cut from rows   0-153 (154 rows) -> test 154-210
            #             fold 2 cut from rows   0-210 (211 rows) -> test 211-267
            #             fold 3 cut from rows   0-267 (268 rows) -> test 268-324
            #             fold 4 cut from rows   0-324 (325 rows) -> test 325-381
            #   Gaussian: ONE cut from rows 0-153 (154 rows) -> ALL 228 test rows
            #
            # So the baseline's cut was learned from the first 40 per cent of the
            # history and never updated, while the ML's cuts saw up to 85 per cent
            # including the weeks immediately before each test block. Since the
            # ML's entire measured advantage is a THRESHOLD advantage, that
            # asymmetry sat exactly on top of the headline result.
            #
            # Now: one Gaussian cut per fold, fitted on the SAME train+val block
            # the ML used for that fold, by the SAME function and objective, and
            # applied only to that fold's test rows.
            #
            # Cross-fitting is still not needed on the Gaussian side, and that is
            # not an inconsistency: its probabilities are not fitted to anything,
            # so there is nothing for them to be out-of-fold from. What has to
            # match is the BLOCK the cut is learned from, and now it does.
            g_thr_row = np.full(len(y), np.nan)
            for _fi, (_tr, _va, _te) in enumerate(splits, 1):
                _trva = np.concatenate([_tr, _va])
                _gt, _ = best_primary_threshold(y[_trva], gp[_trva], fallback=0.5)
                g_thr_row[_te] = 0.5 if _gt is None else float(_gt)

            g_dec = np.where(np.isfinite(g_thr_row) & np.isfinite(gp),
                             (gp >= g_thr_row).astype(float), np.nan)

            ml_thr_mean = float(np.nanmean(ml_thr_row))
            g_thr_mean = float(np.nanmean(g_thr_row))

            # The economic rule needs no fitting: p* comes from the payoff matrix,
            # so it is the SAME cut for both contestants and every task. Any
            # difference in payoff is therefore a difference in the probabilities
            # themselves, not in how each side chose its threshold.
            ml_dec_econ = np.where(np.isfinite(ml_prob),
                                   (ml_prob >= ECONOMIC_THRESHOLD).astype(float),
                                   np.nan)
            g_dec_econ = (gp >= ECONOMIC_THRESHOLD).astype(int)

            for nm, prob, dec, thr, rule in (
                    ("ML", ml_prob, ml_dec, ml_thr_mean, "fitted_cut"),
                    ("Gaussian", gp, g_dec, g_thr_mean, "fitted_cut"),
                    ("ML", ml_prob, ml_dec_econ, ECONOMIC_THRESHOLD, "economic"),
                    ("Gaussian", gp, g_dec_econ, ECONOMIC_THRESHOLD, "economic")):
                s = contest_scores(y[scored], prob[scored], dec[scored], thr, rule)
                scores.append({"task": name, "day": day, "side": side,
                               "contestant": nm, **s})

            b = band_by_cycle.reindex(sheet["cycle_id"])
            predictions.append(pd.DataFrame({
                "cycle_id": sheet["cycle_id"], "expiry_date": sheet["expiry_date"],
                "cycle_days": sheet["cycle_days"], "is_padded": sheet["is_padded"],
                "day": day, "side": side,
                "band_upper": b["band_upper"].values,
                "band_lower": b["band_lower"].values,
                # Derived, NOT read from the sheet. days_left_dec is a
                # FEATURE under some FEATURE_SET values and absent under others
                # (z_only drops it, and C5 drops it anyway wherever it is
                # constant). The predictions sheet needs it as an identifier
                # regardless, and cycle_days + 1 - day is the definition — so
                # taking it from the column made the output depend on a feature
                # choice it has nothing to do with.
                "days_left": (pd.to_numeric(sheet["cycle_days"], errors="coerce")
                              + 1 - day),
                "ml_prob": ml_prob, "ml_decision": ml_dec,
                "ml_threshold": ml_thr_row,
                "ml_decision_econ": ml_dec_econ,
                "gauss_prob": gp, "gauss_decision": g_dec,
                # Per-row now, like ml_threshold, because the cut is per fold.
                "gauss_threshold": g_thr_row,
                "gauss_decision_econ": g_dec_econ,
                "economic_threshold": ECONOMIC_THRESHOLD,
                "ml_payoff_econ": payoff_vector(y, np.nan_to_num(ml_dec_econ)),
                "gauss_payoff_econ": payoff_vector(y, g_dec_econ),
                "actual_breach": y,
                "scored_out_of_sample": scored.astype(int),
            })[lambda d: d["scored_out_of_sample"] == 1])

        # ---- collect -------------------------------------------------------
        # pd.concat([]) raises, so an all-tasks-skipped run would die with a
        # confusing "No objects to concatenate" instead of saying what happened.
        if not predictions or not scores:
            print("\n  NOTHING WAS SCORED for this configuration. Every task was "
                  "skipped,\n  which means too few rows for a walk-forward split "
                  "or no usable fold.\n  Check the row counts printed above "
                  "against MIN_TRAIN and TEST_FRACTION.")
            continue

        predictions = (pd.concat(predictions, ignore_index=True)
                       .sort_values(["expiry_date", "day", "side"])
                       .reset_index(drop=True))
        scores = pd.DataFrame(scores)
        candidates = pd.concat(candidates, ignore_index=True)
        folds = pd.DataFrame(fold_rows)

        f1_view = (scores[scores["rule"] == "fitted_cut"]
                   .pivot_table(index=["day", "side"], columns="contestant",
                                values=["primary", "primary_skill", "fbeta",
                                        "f1", "mean_payoff",
                                        "brier_skill"]).round(4))
        print(f"\n  scores at the FITTED cut, sigma={sigma}"
              f"   (cut learned on train+val by maximising {PRIMARY_METRIC})"
              f"\n  primary_skill is the sigma-comparable figure: 0 = no better "
              f"than the no-model predictor, 1 = perfect.")
        print(f1_view.to_string())

        econ_view = (scores[scores["rule"] == "economic"]
                     .pivot_table(index=["day", "side"], columns="contestant",
                                  values=["mean_payoff", "primary_skill",
                                          "fbeta", "f1", "reliability"])
                     .round(4))
        print(f"\n  at the economic cut p*={ECONOMIC_THRESHOLD:.4f}, sigma={sigma}")
        print(econ_view.to_string())
        ref = scores[scores["rule"] == "economic"].groupby(["day", "side"])[
            ["payoff_if_perfect", "payoff_if_always_hold"]].first().round(4)
        print("\n  for reference, the same cycles under perfect foresight and "
              "under always-hold")
        print(ref.to_string())

        # ---- checks --------------------------------------------------------
        fails = []

        def chk(nm, ok, detail=""):
            if not ok:
                fails.append(f"{nm}: {detail}")
            print(f"    {'OK  ' if ok else 'FAIL'} {nm}"
                  + ("" if ok else f"  -- {detail}"))

        print("\n  checks")
        chk("a cycle appears at most once per (day, side)",
            not predictions.duplicated(["cycle_id", "day", "side"]).any())
        chk("both contestants scored the same rows for every task",
            bool((scores.groupby(["day", "side"])["n"].nunique() == 1).all()))
        chk("both decision rules are reported for both contestants",
            bool((scores.groupby(["day", "side"])
                  .apply(lambda g: len(g) == 4, include_groups=False)).all()))
        chk("the economic cut is identical everywhere",
            scores.loc[scores["rule"] == "economic", "threshold"].nunique() == 1,
            "p* comes from the payoff matrix and must not vary by task")
        chk("every economic decision equals int(prob >= p*)",
            bool((predictions["ml_decision_econ"]
                  == (predictions["ml_prob"]
                      >= ECONOMIC_THRESHOLD).astype(int)).all()
                 and (predictions["gauss_decision_econ"]
                      == (predictions["gauss_prob"]
                          >= ECONOMIC_THRESHOLD).astype(int)).all()))
        chk("no payoff beats perfect foresight",
            bool((scores["mean_payoff"] <= scores["payoff_if_perfect"] + 1e-9).all()),
            "a payoff above the perfect-foresight ceiling means the outcome "
            "leaked into the decision")
        chk("every ML decision equals int(prob >= threshold)",
            bool((predictions["ml_decision"]
                  == (predictions["ml_prob"]
                      >= predictions["ml_threshold"]).astype(int)).all()),
            "the reported probability is not the one that drove the decision")
        chk("every Gaussian decision equals int(prob >= threshold)",
            bool((predictions["gauss_decision"]
                  == (predictions["gauss_prob"]
                      >= predictions["gauss_threshold"]).astype(int)).all()))
        # THE FAIRNESS CHECK. Both sides must have a fitted cut on every scored
        # row, or one contestant is being scored on rows the other could not
        # reach. This is what the old single-block Gaussian cut violated.
        chk("both contestants have a fitted threshold on every scored row",
            bool(predictions[["ml_threshold", "gauss_threshold"]]
                 .notna().all().all()),
            "a scored row has a cut for one side and not the other")
        chk("the Gaussian's cut varies by fold, as the ML's does",
            predictions["gauss_threshold"].nunique() > 1
            or len(predictions["day"].unique()) == 0,
            "a single Gaussian threshold across all folds means it was fitted "
            "on one block while the ML got one per fold — the asymmetry this "
            "check exists to catch")
        chk("every scored row came from a sealed test block",
            bool((predictions["scored_out_of_sample"] == 1).all()),
            "a row was reported that was never in any fold's test block")
        chk("the fold sheet covers every model that was trained",
            (folds.empty
             or set(folds["task"] + "|" + folds["model"])
             >= set(candidates["task"] + "|" + candidates["model"])),
            "a model appears in candidates with no per-fold rows behind it")
        chk("bands are attached to every prediction row",
            bool(predictions[["band_upper", "band_lower"]].notna().all().all()))
        chk("no probability falls outside [0, 1]",
            bool(((predictions["ml_prob"] >= 0) & (predictions["ml_prob"] <= 1)).all()
                 and ((predictions["gauss_prob"] >= 0)
                      & (predictions["gauss_prob"] <= 1)).all()))
        if fails:
            raise RuntimeError("  C6 checks failed:\n    - " + "\n    - ".join(fails))

        # ---- write ---------------------------------------------------------
        mins = (time.time() - t0) / 60.0
        manifest = pd.DataFrame([
            ("script", "C6_TrainPredict"),
            ("source", src),
            ("sigma (k)", sigma),
            ("window (cycles)", window),
            ("daily_calendar", CAL_TAG),
            ("cycle_lengths", CYCLE_LENGTHS),
            ("feature_set", FEATURE_SET),
            ("run tag", RUN_TAG),
            ("models", ", ".join(MODELS)),
            ("ensemble size", TOP_K),
            ("primary metric", PRIMARY_METRIC),
            ("tuning scorer", "payoff at p* (fixed, not fitted)"
             if PRIMARY_METRIC == "payoff" else
             f"fbeta beta={F_BETA}" if PRIMARY_METRIC == "fbeta"
             else "neg_brier_score"),
            ("f_beta", F_BETA),
            ("f_beta implied by payoffs", round(implied_beta(), 4)),
            ("metric label", FBETA_LABEL),
            ("gate floor (fraction of room)", MIN_SKILL),
            ("decision rules", ", ".join(DECISION_RULES)),
            ("fitted cut objective",
             f"maximise {PRIMARY_METRIC} on cross-fitted train+val"
             + ("  (brier_skill is threshold-free, so F-beta is used)"
                if PRIMARY_METRIC == "brier_skill" else "")),
            ("gate: floor and ratio on", "SKILL scale"),
            ("gate: gap on", f"RAW {PRIMARY_METRIC} units"),
            ("threshold fitting",
             "BOTH contestants: one cut per fold, fitted on that fold's "
             "train+val by maximising the primary metric. The Gaussian used to "
             "get a single cut from the earliest 40% of rows, which flattered "
             "the ML's threshold advantage."),
            ("economic cut p*", round(ECONOMIC_THRESHOLD, 4)),
            ("payoff: exit & breach", PAYOFF_BREACH_CORRECT),
            ("payoff: exit & no breach", PAYOFF_BREACH_WRONG),
            ("payoff: hold & no breach", PAYOFF_NOBREACH_CORRECT),
            ("payoff: hold & breach", PAYOFF_NOBREACH_WRONG),
            # DERIVED from the constants. The old version stated "a CORRECT exit
            # costs more than a false alarm", which stopped being true the moment
            # the exit cells were set equal — a manifest that misdescribes its own
            # configuration is worse than no manifest.
            ("payoff caveat", (
                "exit cost is FLAT: being right about a breach costs the same as "
                "a false alarm, so all the asymmetry is on the hold side. State "
                "this in the write-up."
                if abs(PAYOFF_BREACH_CORRECT - PAYOFF_BREACH_WRONG) < 1e-12 else
                "a CORRECT exit costs MORE than a false alarm, which assumes the "
                "loss is already realised by the time a breach is actionable. "
                "That is an assumption about execution, not a fact — state it."
                if PAYOFF_BREACH_CORRECT < PAYOFF_BREACH_WRONG else
                "a correct exit costs LESS than a false alarm. Check this is "
                "intended; it implies the exit itself recovers value.")),
            ("tasks scored", int(scores["task"].nunique())),
            ("tasks where no model passed the gate",
             int((~candidates.groupby("task")["gate_applied"].first()).sum())),
            ("out-of-sample prediction rows", len(predictions)),
            ("minutes", round(mins, 1)),
            ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
        ], columns=["key", "value"])

        path = f_results(sigma, window)
        write_xlsx(path, {"predictions": predictions, "scores": scores,
                          "candidates": candidates, "folds": folds,
                          "manifest": manifest})
        written.append(path)
        print(f"  {mins:.1f} minutes")


# ============================================================================
# DONE
# ============================================================================
print("\n" + "=" * 78)
print("  C6_TrainPredict complete")
for p in written:
    print(f"    {p.split('/')[-1]}")
print("  Each workbook is ONE study. Base rates differ between sigmas because the")
print("  band defines the label, so compare ML against the Gaussian WITHIN a")
print("  sigma and never rank sigmas against each other.")
print("=" * 78)


  C6_TrainPredict — sigmas [0.1, 0.25, 0.5], windows [4], tag=filled_full_only_compact_plus_macro

  sigma=0.1  window=4

  D2_upper: 382 rows, 179 breaches, 9 features, 4 fold(s)
    features: z_dist_upper_D2, z_chg_upper_upto_D2, realized_vol_D2, gap_cum_upto_D2, range_max_upto_D2, band_width_pct, vix_daily_D2, usdinr_ln1_D2, brent_ln1_D2
    base rate 0.469 -> no-model fbeta = +0.688, perfect +1.000, room 0.312, gate floor +0.719 (= 10% skill)
    LogisticRegression   oof=+0.753 infold=+0.750   skill oof=+0.208 infold=+0.200   PASS
        folds  [1] +0.749/+0.19 y  [2] +0.744/+0.18 y  [3] +0.764/+0.24 y  [4] +0.755/+0.22 y   (oof/skill, passed)
    SVM                  oof=+0.730 infold=+0.729   skill oof=+0.134 infold=+0.130   PASS
        folds  [1] +0.728/+0.13 y  [2] +0.728/+0.13 y  [3] +0.729/+0.13 y  [4] +0.735/+0.15 y   (oof/skill, passed)
    XGBoost              oof=+0.711 infold=+0.690   skill oof=+0.073 infold=+0.007   fail gate
        folds  [1] +0.733/+0.14 y  [2] +0

In [12]:
# @title
# ============================================================================
# C7_Compare — IS THE DIFFERENCE REAL?
# ============================================================================
#
#   reads  : every 05_results_sigma<k>_win<W>_<align>.xlsx the grids imply
#   writes : 06_comparison_<align>.xlsx
#
# WHY THIS SCRIPT EXISTS
# ----------------------
# C6 reports that one contestant scored higher. That is not yet a finding. With
# 172 to 232 scored cycles per task, a gap of 0.05 in F1 is well inside what
# resampling alone produces. This script asks whether each gap survives being
# shaken.
#
# WHAT IT DOES
# ------------
#   1. PAIRED BOOTSTRAP. Resample CYCLES with replacement, B times, recomputing
#      both contestants' scores on the same resample each time. Paired, because
#      the two forecast the same cycles: a hard cycle is hard for both, and
#      pairing removes that shared difficulty instead of counting it as noise.
#
#   2. A TWO-SIDED p-VALUE from the bootstrap distribution of the difference,
#      plus a 95 per cent interval. The interval is the more informative of the
#      two and should be quoted in the write-up.
#
#   3. BENJAMINI-HOCHBERG across tasks. Six tasks means six chances to find
#      something, and the p-values need adjusting for that.
#
#      But six is the wrong count. upper_D2, upper_D3 and upper_D4 share an
#      IDENTICAL label and differ only in features, so they are three views of
#      one question, not three questions. The same holds for the lower side. The
#      correction is therefore applied within each SIDE FAMILY, and the manifest
#      records that choice rather than leaving a reader to assume six independent
#      tests.
#
#   4. SIGMAS ARE NEVER POOLED. Each sigma has its own band, so its own label and
#      its own base rate. A table that ranked sigmas against each other would be
#      comparing answers to different questions.
#
# WHAT A NULL RESULT MEANS HERE
# -----------------------------
# If the intervals straddle zero, the honest conclusion is that the learned
# models did not beat the closed form on these inputs at this sample size —
# which is a result, and a more useful one than a win that does not replicate.
# The Brier decomposition in the scores sheet says which of the two explanations
# applies: high reliability for the Gaussian means exploitable mis-calibration
# was left on the table, while low resolution for both means the inputs
# themselves are near their limit.
# ============================================================================

try:
    _ = (f_results, ROOT, RUN_TAG, SIGMA_GRID, SIGMA_WINDOW_GRID, CAL_TAG,
         PRIMARY_METRIC, RANDOM_STATE, brier_skill, read_xlsx, write_xlsx,
         F_BETA, FBETA_LABEL, trivial_fbeta,
         payoff_vector, trivial_payoff, perfect_payoff, skill_of,
         ECONOMIC_THRESHOLD)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import os
import datetime

import numpy as np
import pandas as pd

print("\n" + "=" * 78)
print(f"  C7_Compare — sigmas {SIGMA_GRID}, windows {SIGMA_WINDOW_GRID}, "
      f"calendar={CAL_TAG}")
print("=" * 78)

N_BOOT = 5000
ALPHA = 0.05
F_COMPARE = os.path.join(ROOT, f"06_comparison_{RUN_TAG}.xlsx")


def fbeta_from_counts(y, d, beta=None):
    """F-beta computed from decisions, not from re-thresholded probabilities.

    The bootstrap resamples CYCLES, so the decisions must be resampled with
    them; re-fitting a threshold inside each replicate would bootstrap the
    threshold search as well and widen every interval for the wrong reason.
    """
    b2 = float(F_BETA if beta is None else beta) ** 2
    y, d = np.asarray(y, int), np.asarray(d, int)
    tp = int(((d == 1) & (y == 1)).sum())
    fp = int(((d == 1) & (y == 0)).sum())
    fn = int(((d == 0) & (y == 1)).sum())
    den = (1.0 + b2) * tp + b2 * fn + fp
    return 0.0 if den <= 0 else (1.0 + b2) * tp / den


def f1_from_counts(y, d):
    """Plain F1 from decisions, reported next to F-beta."""
    return fbeta_from_counts(y, d, beta=1.0)


def payoff_from_counts(y, d):
    """Mean payoff per cycle from the decisions actually made.

    Parameters
    ----------
    y : array-like of int
    d : array-like of int
        The 0/1 calls. NOT re-derived from probabilities — a bootstrap resamples
        CYCLES, so the decisions must travel with them. Re-cutting inside each
        replicate would bootstrap the threshold search as well and widen every
        interval for the wrong reason.

    Returns
    -------
    float
        In the payoff matrix's own units.

    Notes
    -----
    This is the statistic the study is actually about, and until now it had no
    significance test: the bootstrap knew only fbeta, f1 and brier_skill. A
    5-of-6 split on 228 rows means nothing without an interval.
    """
    y, d = np.asarray(y, int), np.asarray(d, int)
    return float(payoff_vector(y, d).mean())


def payoff_skill_from_counts(y, d):
    """Payoff as a fraction of the room above the no-model predictor.

    Notes
    -----
    Reported ALONGSIDE raw payoff, not instead of it, because the two answer
    different questions. Raw payoff is in risk units and moves with the base
    rate on its own, so it cannot be compared across sigmas. Skill divides that
    movement out and is the figure to quote when comparing bands.

    IMPORTANT for a paired bootstrap: the base rate is recomputed INSIDE each
    replicate, from the resampled rows. Holding it at the full-sample value
    would treat a random quantity as fixed and understate the interval.
    """
    y, d = np.asarray(y, int), np.asarray(d, int)
    if len(y) == 0:
        return float("nan")
    b = float(y.mean())
    t, c = trivial_payoff(b), perfect_payoff(b)
    room = c - t
    if not np.isfinite(room) or room <= 1e-12:
        return float("nan")
    return float((payoff_vector(y, d).mean() - t) / room)


def paired_bootstrap(y, a_prob, a_dec, b_prob, b_dec, metric, n_boot=N_BOOT,
                     seed=RANDOM_STATE):
    """Bootstrap the difference between two contestants on the same cycles.

    Parameters
    ----------
    y : ndarray of int
        Outcomes.
    a_prob, a_dec : ndarray
        First contestant's probabilities and decisions.
    b_prob, b_dec : ndarray
        Second contestant's.
    metric : {"payoff", "payoff_skill", "fbeta", "f1", "brier_skill"}
    n_boot : int
    seed : int

    Returns
    -------
    dict
        observed difference, bootstrap mean, 2.5th and 97.5th percentiles, a
        two-sided p-value, and how many resamples were usable.

    Notes
    -----
    The SAME resampled index is used for both contestants, which is what makes
    the test paired. Resampling them independently would compare two different
    sets of cycles and inflate the spread.

    A resample whose labels are all one class cannot produce the metric and is
    skipped rather than scored as zero — counting it would drag the distribution
    toward a difference of nothing.

    The p-value is the share of resampled differences on the opposite side of
    zero from the observed one, doubled. It is approximate, and the interval
    below is the more honest thing to quote.
    """
    y = np.asarray(y, int)
    n = len(y)
    rng = np.random.default_rng(seed)

    def score(idx, prob, dec):
        if metric == "payoff":
            return payoff_from_counts(y[idx], dec[idx])
        if metric == "payoff_skill":
            return payoff_skill_from_counts(y[idx], dec[idx])
        if metric == "fbeta":
            return fbeta_from_counts(y[idx], dec[idx])
        if metric == "f1":
            return f1_from_counts(y[idx], dec[idx])
        return brier_skill(y[idx], prob[idx])

    obs = score(np.arange(n), a_prob, a_dec) - score(np.arange(n), b_prob, b_dec)

    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        d = score(idx, a_prob, a_dec) - score(idx, b_prob, b_dec)
        if np.isfinite(d):
            diffs.append(d)
    diffs = np.asarray(diffs)
    if len(diffs) < 100:
        return {"observed": obs, "boot_mean": np.nan, "ci_low": np.nan,
                "ci_high": np.nan, "p_value": np.nan, "n_boot_used": len(diffs)}

    share_other_side = (float((diffs <= 0).mean()) if obs > 0
                        else float((diffs >= 0).mean()))
    return {"observed": obs, "boot_mean": float(diffs.mean()),
            "ci_low": float(np.percentile(diffs, 2.5)),
            "ci_high": float(np.percentile(diffs, 97.5)),
            "p_value": float(min(1.0, 2.0 * share_other_side)),
            "n_boot_used": len(diffs)}


def benjamini_hochberg(p, alpha=ALPHA):
    """Step-up FDR adjustment.

    Parameters
    ----------
    p : sequence of float
        Raw p-values; NaNs are passed through.
    alpha : float

    Returns
    -------
    (ndarray, ndarray)
        Adjusted p-values and a boolean reject flag.

    Notes
    -----
    Controls the false discovery RATE, not the family-wise error rate, so it is
    the right tool when the question is "how many of these findings are likely
    spurious" rather than "is any single one certain".
    """
    p = np.asarray(p, float)
    ok = np.isfinite(p)
    adj = np.full(len(p), np.nan)
    if not ok.any():
        return adj, np.zeros(len(p), bool)
    idx = np.flatnonzero(ok)
    order = idx[np.argsort(p[idx])]
    m = len(order)
    running = 1.0
    for rank in range(m - 1, -1, -1):
        val = p[order[rank]] * m / (rank + 1)
        running = min(running, val)
        adj[order[rank]] = running
    return adj, (adj <= alpha) & ok


# ============================================================================
# COMPARE, ONE CONFIGURATION AT A TIME
# ============================================================================
rows = []
for sigma in SIGMA_GRID:
    for window in SIGMA_WINDOW_GRID:
        path = f_results(sigma, window)
        if not os.path.exists(path):
            print(f"  skipping sigma={sigma} window={window}: no results file")
            continue
        pred = read_xlsx(path, sheet="predictions")
        print(f"\n  sigma={sigma} window={window}: {len(pred)} prediction rows")

        for (day, side), g in pred.groupby(["day", "side"]):
            y = g["actual_breach"].to_numpy(int)
            if len(np.unique(y)) < 2:
                print(f"    D{day}_{side}: one class only, skipped")
                continue
            gs_p = g["gauss_prob"].to_numpy(float)

            # BOTH DECISION RULES ARE TESTED, because they answer different
            # questions and can disagree. "fitted_cut" uses each side's learned
            # threshold; "economic" uses the fixed p* for both, so any difference
            # there is a difference in the PROBABILITIES rather than in how each
            # side chose its cut. Reporting only one would hide that.
            _rules = [("fitted_cut", "ml_decision", "gauss_decision")]
            if {"ml_decision_econ", "gauss_decision_econ"} <= set(g.columns):
                _rules.append(("economic", "ml_decision_econ",
                               "gauss_decision_econ"))

            for rule, mlcol, gscol in _rules:
                ml_d = pd.to_numeric(g[mlcol], errors="coerce").fillna(0).to_numpy(int)
                gs_d = pd.to_numeric(g[gscol], errors="coerce").fillna(0).to_numpy(int)
                ml_p = g["ml_prob"].to_numpy(float)

                # payoff is the headline, payoff_skill makes it comparable across
                # sigmas, F-beta and F1 are kept for reference, and Brier is the
                # proper score no threshold choice can game. All five are tested so
                # the BH correction below covers everything that gets reported.
                for metric in ("payoff", "payoff_skill", "fbeta", "f1",
                                 "brier_skill"):
                    r = paired_bootstrap(y, ml_p, ml_d, gs_p, gs_d, metric)
                    rows.append({
                        "sigma": sigma, "window": window,
                        "task": f"D{day}_{side}", "day": day, "side": side,
                        "rule": rule,
                        "metric": metric, "n_cycles": len(y),
                        "base_rate": round(float(y.mean()), 4),
                        "fbeta_trivial": round(trivial_fbeta(float(y.mean())), 4),
                        "payoff_trivial": round(trivial_payoff(float(y.mean())), 4),
                        "payoff_perfect": round(perfect_payoff(float(y.mean())), 4),
                        "ml_minus_gaussian": round(r["observed"], 4),
                        "ci_low": round(r["ci_low"], 4) if np.isfinite(r["ci_low"]) else np.nan,
                        "ci_high": round(r["ci_high"], 4) if np.isfinite(r["ci_high"]) else np.nan,
                        "p_raw": r["p_value"], "n_boot_used": r["n_boot_used"],
                    })

if not rows:
    raise RuntimeError("  no results files found — run C6_TrainPredict first.")

res = pd.DataFrame(rows)


# ============================================================================
# MULTIPLE TESTING, WITHIN SIDE FAMILIES
# ============================================================================
res["p_adjusted"] = np.nan
res["significant"] = False
# The RULE is part of the family. Adding the economic rule doubled the number of
# tests; correcting across rules would pool two different questions into one
# family and make both look more significant than they are.
for (sigma, window, side, metric, rule), g in res.groupby(
        ["sigma", "window", "side", "metric", "rule"]):
    adj, rej = benjamini_hochberg(g["p_raw"].to_numpy())
    res.loc[g.index, "p_adjusted"] = np.round(adj, 4)
    res.loc[g.index, "significant"] = rej

res["verdict"] = np.where(
    ~res["significant"], "no difference beyond noise",
    np.where(res["ml_minus_gaussian"] > 0, "ML better", "Gaussian better"))


# ============================================================================
# REPORT
# ============================================================================
for metric in ("payoff", "payoff_skill", "fbeta", "f1", "brier_skill"):
    for rule in res["rule"].unique():
        m = res[(res["metric"] == metric) & (res["rule"] == rule)]
        if m.empty:
            continue
        print(f"\n  {metric} under '{rule}': ML minus Gaussian, 95% interval")
        print(m[["sigma", "task", "n_cycles", "base_rate", "ml_minus_gaussian",
                 "ci_low", "ci_high", "p_raw", "p_adjusted", "verdict"]]
              .to_string(index=False))

# The headline, per rule. A win under "fitted_cut" and not under "economic"
# means the advantage is in the THRESHOLD; a win under both means it is in the
# probabilities. Printing them separately is what makes that readable.
for rule in res["rule"].unique():
    headline = res[(res["metric"] == PRIMARY_METRIC) & (res["rule"] == rule)]
    if headline.empty:
        continue
    counts = headline["verdict"].value_counts().to_dict()
    print(f"\n  {PRIMARY_METRIC} under '{rule}', across {len(headline)} "
          f"task-configurations: {counts}")

# A win on a THRESHOLD metric with no win on Brier says the edge is in the cut,
# not in the forecast. That is a legitimate and interesting result, but it must
# be stated that way rather than as "the model predicts better".
_thr_win = res[(res["metric"].isin(("payoff", "payoff_skill", "fbeta", "f1")))
               & (res["verdict"] == "ML better")]
_brier_win = res[(res["metric"] == "brier_skill")
                 & (res["verdict"] == "ML better")]
if len(_thr_win) and not len(_brier_win):
    print("\n  READ THIS INTO THE WRITE-UP: ML wins on threshold metrics in "
          f"{len(_thr_win)} cell(s) and on Brier in none.")
    print("  The edge is in WHERE THE CUT SITS, not in the quality of the "
          "probability. Say so.")


# ============================================================================
# CHECKS
# ============================================================================
fails = []


def chk(nm, ok, detail=""):
    if not ok:
        fails.append(f"{nm}: {detail}")
    print(f"    {'OK  ' if ok else 'FAIL'} {nm}" + ("" if ok else f"  -- {detail}"))


print("\n  checks")
chk("every p-value lies in [0, 1]",
    bool(res["p_raw"].dropna().between(0, 1).all()))
chk("adjusted p-values are never below raw ones",
    bool((res["p_adjusted"].fillna(1) >= res["p_raw"].fillna(0) - 1e-9).all()),
    "BH can only make a p-value larger")
chk("the observed difference lies inside its own interval",
    bool(((res["ml_minus_gaussian"] >= res["ci_low"] - 1e-6)
          & (res["ml_minus_gaussian"] <= res["ci_high"] + 1e-6)).all()),
    "a point estimate outside its bootstrap interval means the resample is not "
    "sampling the same quantity")
chk("nothing is called significant while its interval straddles zero",
    not bool((res["significant"]
              & (res["ci_low"] < 0) & (res["ci_high"] > 0)).any()),
    "an interval containing zero cannot support a directional claim")
chk("every resample budget was usable",
    bool((res["n_boot_used"] > N_BOOT * 0.9).all()),
    "too many resamples were discarded for want of both classes")
if fails:
    raise RuntimeError("  C7 checks failed:\n    - " + "\n    - ".join(fails))


# ============================================================================
# WRITE
# ============================================================================
# One row per task, one pair of columns per sigma. The MultiIndex a pivot_table
# produces cannot be written to Excel without an index column, so it is flattened
# into plain names — which also makes the sheet readable without knowing pandas.
pivot = (res[res["metric"] == PRIMARY_METRIC]
         .pivot_table(index="task", columns="sigma",
                      values=["ml_minus_gaussian", "p_adjusted"]).round(4))
pivot.columns = [f"{stat}_sigma{sig}" for stat, sig in pivot.columns]
pivot = pivot.reset_index()

manifest = pd.DataFrame([
    ("script", "C7_Compare"),
    ("daily_calendar", CAL_TAG),
    ("sigmas compared", ", ".join(str(s) for s in SIGMA_GRID)),
    ("bootstrap resamples", N_BOOT),
    ("pairing", "the same resampled cycles for both contestants"),
    ("multiple testing", "Benjamini-Hochberg within each side family"),
    ("why side families",
     "upper_D2, upper_D3 and upper_D4 share an identical label and differ only "
     "in features, so they are three views of one question rather than three "
     "questions. Correcting as if six were independent would be too "
     "conservative and would misdescribe the design."),
    ("sigmas pooled", "NO — each sigma has its own band, label and base rate"),
    ("alpha", ALPHA),
    ("primary metric", PRIMARY_METRIC),
    ("f_beta", F_BETA),
    ("metric label", FBETA_LABEL),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

write_xlsx(F_COMPARE, {"comparison": res, "headline": pivot,
                       "manifest": manifest})

print("\n" + "=" * 78)
print("  C7_Compare complete")
print(f"    {F_COMPARE.split('/')[-1]}")
print("  Quote the interval, not the p-value. An interval that straddles zero at")
print("  this sample size is a genuine null result, not a failed experiment.")
print("=" * 78)


  C7_Compare — sigmas [0.1, 0.25, 0.5], windows [4], calendar=filled

  sigma=0.1 window=4: 1368 prediction rows

  sigma=0.25 window=4: 1368 prediction rows

  sigma=0.5 window=4: 1368 prediction rows

  payoff under 'fitted_cut': ML minus Gaussian, 95% interval
 sigma     task  n_cycles  base_rate  ml_minus_gaussian  ci_low  ci_high  p_raw  p_adjusted                    verdict
  0.10 D2_lower       228     0.4298            -0.2982 -0.4518  -0.1404 0.0012      0.0036            Gaussian better
  0.10 D2_upper       228     0.4781            -0.0570 -0.1886   0.0746 0.3872      0.6006 no difference beyond noise
  0.10 D3_lower       228     0.4298            -0.1053 -0.2105  -0.0044 0.0440      0.0660 no difference beyond noise
  0.10 D3_upper       228     0.4781            -0.0132 -0.1009   0.0702 0.7692      0.7692 no difference beyond noise
  0.10 D4_lower       228     0.4298             0.0658 -0.0351   0.1712 0.2368      0.2368 no difference beyond noise
  0.10 D4_upper      

In [2]:
# @title
!pip uninstall -y sympy
!pip install sympy==1.13.3 catboost lightgbm yfinance openpyxl nselib -q
# @title
# ============================================================================
# C2_Fetch — DOWNLOAD NIFTY 50 DAILY OHLC
# ============================================================================
#
#   reads  : NSE, via nselib's capital_market.index_data("Nifty 50")
#   writes : 01_daily_raw.xlsx     sheets "daily" and "manifest"
#
# Nothing else is fetched. No options chain, no bhavcopy, no VIX, no macro
# series — the study uses index prices only.
#
# TWO RULES THIS SCRIPT WILL NOT BREAK
# ------------------------------------
# 1. IT NEVER WRITES AN INCOMPLETE SERIES. A chunk that fails, or that comes
#    back EMPTY, aborts the run before anything is written. The empty case
#    matters: nselib returns an empty frame instead of raising when NSE serves
#    nothing, so treating "no rows" as success silently shortens history — and
#    an empty chunk at either END of the range leaves no calendar hole for a gap
#    check to find.
#
# 2. IT NEVER DESTROYS THE PREVIOUS FILE ON A FAILED RUN. The existing workbook
#    is read, the new rows are merged in memory, the merged series is checked
#    for continuity, and only then is anything saved.
#
# MODES
# -----
#   "incremental"  keep what is on disk, fetch only the missing recent dates.
#                  The default, and what you want almost always.
#   "full"         refetch everything from START_DATE. Needs CONFIRM_FULL=True
#                  as a second deliberate action, so history cannot be replaced
#                  by leaving a flag where you found it.
# ============================================================================

try:
    _ = (F_RAW, DATE, OPEN, HIGH, LOW, CLOSE, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import time
import datetime

import numpy as np
import pandas as pd
from nselib import capital_market

print("\n" + "=" * 78)
print("  C2_Fetch — NIFTY 50 daily OHLC")
print("=" * 78)


# ============================================================================
# SETTINGS
# ============================================================================
MODE = "incremental"          # "incremental" | "full"
CONFIRM_FULL = False          # must also be True for MODE="full"

START_DATE = datetime.date(2019, 1, 1)
END_DATE = datetime.date.today()
CHUNK_MONTHS = 3              # NSE rejects spans much longer than a quarter
REFETCH_LAST_N = 1            # re-pull the last N stored rows, in case of revisions

MAX_GAP_DAYS = 10             # calendar days between consecutive trading days;
                              # a long weekend plus holidays is about 5
RETRIES = 3
PAUSE = 0.5                   # seconds between chunks; NSE throttles

if MODE == "full" and not CONFIRM_FULL:
    raise RuntimeError(
        'MODE="full" also needs CONFIRM_FULL=True. A full refetch replaces the '
        'stored history, so it takes two deliberate edits, not one.')

print(f"  mode   : {MODE}")
print(f"  output : {F_RAW}")


# ============================================================================
# HELPERS
# ============================================================================
def month_chunks(start, end, months=CHUNK_MONTHS):
    """Split a date range into windows the NSE endpoint accepts.

    Parameters
    ----------
    start, end : datetime.date
        Inclusive range.
    months : int
        Window size in months.

    Returns
    -------
    list of (str, str)
        (from, to) pairs in the endpoint's dd-mm-YYYY format, oldest first.
    """
    out, cur = [], start
    while cur <= end:
        m = cur.month - 1 + months
        y, m = cur.year + m // 12, m % 12 + 1
        stop = min(datetime.date(y, m, 1) - datetime.timedelta(days=1), end)
        out.append((cur.strftime("%d-%m-%Y"), stop.strftime("%d-%m-%Y")))
        cur = stop + datetime.timedelta(days=1)
    return out


def first_present(df, *names):
    """First of several candidate column names present in a frame.

    Notes
    -----
    NSE has renamed its OHLC columns more than once. Trying a list keeps the
    parser tolerant without guessing.
    """
    for n in names:
        if n in df.columns:
            return n
    return None


def to_number(s):
    """Parse a numeric-looking column, stripping thousands separators."""
    return pd.to_numeric(pd.Series(s).astype(str).str.replace(",", "").str.strip(),
                         errors="coerce")


def fetch(chunks):
    """Download every chunk, reporting failures instead of hiding them.

    Parameters
    ----------
    chunks : list of (str, str)

    Returns
    -------
    (pandas.DataFrame, list)
        Concatenated rows, and (from, to, reason) for each chunk that failed.

    Notes
    -----
    An EMPTY response counts as a failure and is retried. NSE answers with an
    empty frame rather than an error when it is throttling or when a date is past
    its archive depth, and a slower retry often succeeds. Treating empty as
    success is how a truncated history gets written over a good one.
    """
    frames, failed = [], []
    for i, (a, b) in enumerate(chunks, 1):
        ok, why = False, "unknown"
        for attempt in range(1, RETRIES + 1):
            try:
                d = capital_market.index_data(index="Nifty 50", from_date=a, to_date=b)
                if d is None or len(d) == 0:
                    why = "empty response (0 rows)"
                    if attempt < RETRIES:
                        time.sleep(2 * attempt)
                        continue
                    break
                frames.append(pd.DataFrame(d))
                ok = True
                break
            except Exception as e:
                why = f"{type(e).__name__}: {str(e)[:100]}"
                if attempt < RETRIES:
                    time.sleep(2 * attempt)
        if not ok:
            failed.append((a, b, why))
            print(f"\n  FAILED {a} -> {b}: {why}")
        print(f"  [{i:3d}/{len(chunks)}] up to {b}", end="\r")
        time.sleep(PAUSE)
    print()
    got = (pd.concat(frames, ignore_index=True).drop_duplicates()
           if frames else pd.DataFrame())
    return got, failed


def check_continuity(dates):
    """Report calendar gaps between consecutive trading days.

    Returns
    -------
    (bool, pandas.Series)
        (no suspicious gaps, the offending gaps)

    Notes
    -----
    A dropped chunk shows up as a large gap and nothing downstream would notice:
    the hole becomes one enormous daily return and poisons every rolling window
    spanning the seam.
    """
    d = pd.to_datetime(pd.Series(dates)).sort_values().reset_index(drop=True)
    gaps = d.diff().dt.days.fillna(0)
    bad = gaps[gaps > MAX_GAP_DAYS]
    if len(bad):
        print(f"  {len(bad)} gap(s) longer than {MAX_GAP_DAYS} calendar days:")
        for i in bad.index[:10]:
            print(f"    {d[i-1].date()} -> {d[i].date()}  ({int(gaps[i])} days)")
        return False, bad
    print(f"  continuity OK, largest gap {int(gaps.max())} calendar days")
    return True, bad


# ============================================================================
# 1. DECIDE THE WINDOW
# ============================================================================
COLS = [DATE, OPEN, HIGH, LOW, CLOSE]

stored = pd.DataFrame(columns=COLS)
if MODE == "incremental":
    try:
        stored = read_xlsx(F_RAW, sheet="daily")
        stored[DATE] = pd.to_datetime(stored[DATE], errors="coerce").dt.normalize()
        stored = (stored.dropna(subset=[DATE]).sort_values(DATE)
                        .drop_duplicates(DATE).reset_index(drop=True))
        if REFETCH_LAST_N > 0 and len(stored) > REFETCH_LAST_N:
            stored = stored.iloc[:-REFETCH_LAST_N].reset_index(drop=True)
        print(f"  stored : {len(stored)} rows up to {stored[DATE].max().date()}")
    except FileNotFoundError:
        print("  stored : none, building from scratch")

fetch_from = (stored[DATE].max().date() if len(stored) else START_DATE)
if fetch_from > END_DATE:
    raise SystemExit("  already current, nothing to fetch")

chunks = month_chunks(fetch_from, END_DATE)
print(f"  fetch  : {fetch_from} -> {END_DATE}  ({len(chunks)} chunk(s))\n")


# ============================================================================
# 2. DOWNLOAD
# ============================================================================
raw, failed = fetch(chunks)

if failed:
    raise SystemExit(
        f"\n  {len(failed)} chunk(s) failed — NOT writing the file.\n"
        + "\n".join(f"    {a} -> {b}   {why}" for a, b, why in failed)
        + "\n  'empty response' usually means throttling or archive depth, not a\n"
          "  holiday. Raise PAUSE and try again before concluding data is gone.")

if raw.empty:
    raise SystemExit("  no rows returned, cannot proceed")


# ============================================================================
# 3. PARSE
# ============================================================================
dc = first_present(raw, "TIMESTAMP", "HistoricalDate", "Date", "date")
cc = first_present(raw, "CLOSE_INDEX_VAL", "CLOSE", "Close")
oc = first_present(raw, "OPEN_INDEX_VAL", "OPEN", "Open")
hc = first_present(raw, "HIGH_INDEX_VAL", "HIGH", "High")
lc = first_present(raw, "LOW_INDEX_VAL", "LOW", "Low")
if dc is None or cc is None:
    raise SystemExit(f"  unexpected NSE schema, columns were: {list(raw.columns)[:12]}")

new = pd.DataFrame({
    DATE: pd.to_datetime(raw[dc], dayfirst=True, errors="coerce").dt.normalize(),
    OPEN: to_number(raw[oc]) if oc else np.nan,
    HIGH: to_number(raw[hc]) if hc else np.nan,
    LOW: to_number(raw[lc]) if lc else np.nan,
    CLOSE: to_number(raw[cc]),
})
new = (new.dropna(subset=[DATE, CLOSE]).sort_values(DATE)
          .drop_duplicates(DATE).reset_index(drop=True))
print(f"  parsed : {len(new)} trading day(s)")


# ============================================================================
# 4. MERGE AND CHECK
# ----------------------------------------------------------------------------
# Continuity is checked on the MERGED series, not on the new rows alone. A gap at
# the seam between old and new is exactly the failure this catches, and it is
# invisible if each piece is checked on its own.
# ============================================================================
merged = pd.concat([stored[COLS], new[COLS]], ignore_index=True)
# Re-coerce the date column. Concatenating against an EMPTY stored frame gives
# object dtype, and .dt then fails several lines later with a message that points
# nowhere near the cause.
merged[DATE] = pd.to_datetime(merged[DATE], errors="coerce").dt.normalize()
merged = (merged.dropna(subset=[DATE]).sort_values(DATE)
                .drop_duplicates(DATE, keep="last").reset_index(drop=True))
print(f"  merged : {len(stored)} + {len(new)} -> {len(merged)}")

ok, _ = check_continuity(merged[DATE])
if not ok:
    raise SystemExit("  merged series has a gap — NOT writing the file. Re-run.")

weekend = merged[DATE].dt.weekday >= 5
if weekend.any():
    print(f"  dropping {int(weekend.sum())} weekend row(s)")
    merged = merged[~weekend].reset_index(drop=True)


# ============================================================================
# 5. WRITE
# ============================================================================
manifest = pd.DataFrame([
    ("script", "C2_Fetch"),
    ("mode", MODE),
    ("rows stored before", len(stored)),
    ("rows fetched", len(new)),
    ("rows total", len(merged)),
    ("period", f"{merged[DATE].min().date()} -> {merged[DATE].max().date()}"),
    ("largest gap (calendar days)",
     int(pd.to_datetime(merged[DATE]).diff().dt.days.max())),
    ("failed chunks", 0),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

write_xlsx(F_RAW, {"daily": merged, "manifest": manifest})

print(f"\n  {len(merged)} rows, {merged[DATE].min().date()} -> "
      f"{merged[DATE].max().date()}")
print("  NEXT: C3_Clean")
print("=" * 78)

Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.

  C2_Fetch — NIFTY 50 daily OHLC
  mode   : incremental
  output :

In [3]:
# @title
# ============================================================================
# C2b_FillCalendar — GIVE THE DAILY FILE A COMPLETE WEEKDAY CALENDAR
# ============================================================================
#
#   reads  : 01_daily_raw.xlsx
#   writes : 01b_daily_filled_<raw|filled>.xlsx   sheets "daily", "filled",
#                                                 "manifest"
#
# COLUMNS WRITTEN
# ---------------
#   Date, Open, High, Low, Close   as in the raw file
#   is_original                    1 = the exchange traded that day
#                                  0 = the market was shut and this row is a
#                                      copy of the previous day
#
# WHAT THIS DOES
# --------------
# It walks the weekday calendar from the first raw trading day to the last, one
# day at a time. A day that traded is copied through untouched. A day that did
# not trade gets a new row carrying the PREVIOUS ROW's Open, High, Low and Close,
# so the close does not move and the daily log return across that day is exactly
# zero. Consecutive holidays chain: the second copies the first, which copied the
# last real day, so a whole shut week carries that day's prices throughout.
#
# WHY, AND WHAT IT BUYS
# ---------------------
# A holiday week has four trading days rather than five, which used to leave the
# weekly cycle with three decision days instead of four and cost the D4 task a
# quarter of its rows. With the calendar complete, every cycle has four decision
# days and the whole family of alignment tricks that used to paper over this —
# front-padding D1, filling inside C4 — becomes unnecessary and has been deleted.
#
# THREE DAYS ARE NEVER FILLED
# ---------------------------
# 1. SATURDAY AND SUNDAY. The market is not expected to trade, so nothing is
#    missing. (This file contains no weekend trading rows at all — occasional
#    Diwali Muhurat sessions do not appear in it. If one ever does it is passed
#    through as an ordinary original row; only weekdays are ever ADDED.)
#
# 2. THE REGIME'S NOMINAL EXPIRY WEEKDAY. This is the important one, and it is
#    not a detail. When the normal settlement weekday is a holiday the exchange
#    settles a day EARLY, so the missing day is the expiry itself — the source
#    of the label. Filling it would fabricate the answer the models are asked to
#    predict. Those cycles are dropped in C4 instead.
#
#    The nominal weekday is taken from C1's EXPIRY_CUTOFF, the same constant
#    is_expiry uses, so the two can never disagree: Thursday before the cutoff,
#    Tuesday from it onwards. Reading the regime from anywhere else — a trailing
#    mode over recent expiries, say — would be a second definition of "expiry
#    day" in a pipeline that already has one.
#
#    This exclusion is also what keeps is_expiry working. Its holiday fallback
#    promotes the day before to expiry when the nominal day is ABSENT from the
#    data. Fill the nominal day and that promotion silently stops firing, and 17
#    expiries in this file would be mis-flagged.
#
# 3. ANYTHING OUTSIDE THE RAW FILE'S OWN RANGE. No row is added before the first
#    real day or after the last. A day that has not happened yet is not a
#    holiday, and inventing rows at the end would hand live inference a cycle
#    that does not exist.
#
# THE RAW FILE IS NEVER MODIFIED
# ------------------------------
# C2_Fetch reads 01_daily_raw.xlsx to merge each incremental fetch and runs its
# continuity check on the merged series. Writing fabricated rows back into it
# would mean every future fetch merges against fiction. So this script writes a
# separate numbered file and leaves the raw exactly as the exchange returned it.
#
# WHAT is_original IS FOR — IT IS NOT JUST AN AUDIT TRAIL
# -------------------------------------------------------
# An imputed day has a log return of exactly zero, and a zero is not a quiet
# neighbour inside a standard deviation. Measured on this file, letting those
# zeros into the 16-day realized_vol window deflates it by more than 2 per cent
# on 30 per cent of rows and by more than 5 per cent on 18 per cent, with a worst
# case of times 0.48. Since z_dist DIVIDES by realized_vol, that would roughly
# double z on the worst cycle and drive the Gaussian's breach probability toward
# zero — quietly corrupting the baseline the entire study is measured against.
#
# So realized_vol is computed on is_original == 1 rows only, in C1's
# daily_realized_vol, and the resulting series is then read on every date. This
# column is what makes that possible.
#
# IMPUTE_DAILY = False
# --------------------
# The script still runs and still writes its file, but fills nothing: every row
# comes through with is_original = 1. Downstream code therefore never branches on
# the switch — it always reads this file, and the switch changes only what is in
# it. That is the whole point of a single pipeline with one place per decision.
# ============================================================================

try:
    _ = (F_RAW, F_FILLED, DATE, OPEN, HIGH, LOW, CLOSE, IMPUTE_DAILY,
         EXPIRY_CUTOFF, is_expiry, read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import datetime

import numpy as np
import pandas as pd

print("\n" + "=" * 78)
print(f"  C2b_FillCalendar — impute_daily={IMPUTE_DAILY}")
print("=" * 78)

WEEKDAY_NAME = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri",
                5: "Sat", 6: "Sun"}
PRICES = [OPEN, HIGH, LOW, CLOSE]


def nominal_expiry_weekday(when):
    """The weekday settlement normally falls on, for the regime containing `when`.

    Parameters
    ----------
    when : pandas.Timestamp

    Returns
    -------
    int
        Monday is 0. Thursday (3) before EXPIRY_CUTOFF, Tuesday (1) from it on.

    Notes
    -----
    This mirrors the first line of is_expiry deliberately. Both read
    EXPIRY_CUTOFF, so a change to the regime moves both at once and the two
    cannot drift apart.
    """
    return 1 if when >= EXPIRY_CUTOFF else 3


# ============================================================================
# 1. LOAD THE RAW FILE
# ============================================================================
raw = read_xlsx(F_RAW, sheet="daily")
need = [DATE] + PRICES
absent = [c for c in need if c not in raw.columns]
if absent:
    raise RuntimeError(
        f"  missing column(s) {absent}. The raw file has {list(raw.columns)}.")

raw = raw[need].copy()
raw[DATE] = pd.to_datetime(raw[DATE], errors="coerce").dt.normalize()
raw = (raw.dropna(subset=[DATE])
          .drop_duplicates(subset=[DATE], keep="last")
          .sort_values(DATE)
          .reset_index(drop=True))
for c in PRICES:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

first_day, last_day = raw[DATE].iloc[0], raw[DATE].iloc[-1]
original_dates = set(raw[DATE])
print(f"  {len(raw)} raw trading days, {first_day.date()} -> {last_day.date()}")
print("  raw weekdays : "
      + ", ".join(f"{WEEKDAY_NAME[k]} {v}" for k, v in
                  raw[DATE].dt.weekday.value_counts().sort_index().items()))


# ============================================================================
# 2. WALK THE CALENDAR, ONE DAY AT A TIME
# ----------------------------------------------------------------------------
# bdate_range gives Monday to Friday inclusive, so weekends never enter the loop
# and need no separate test. An imputed row copies whatever row precedes it in
# the OUTPUT, which is what makes consecutive holidays chain correctly.
# ============================================================================
raw_by_date = {d: row for d, row in zip(raw[DATE], raw[PRICES].to_numpy(float))}

out_rows, filled_log, skipped_expiry = [], [], []
prev_prices = None

for day in pd.bdate_range(first_day, last_day):
    if day in raw_by_date:
        prev_prices = raw_by_date[day]
        out_rows.append((day, *prev_prices, 1))
        continue

    if not IMPUTE_DAILY:
        continue                       # switch off: the gap is simply left open

    if day.weekday() == nominal_expiry_weekday(day):
        # The missing day IS the expiry. See the header: filling it would
        # fabricate the label, and it would stop is_expiry promoting the early
        # settlement that actually happened.
        skipped_expiry.append({
            "date": day,
            "weekday": WEEKDAY_NAME[day.weekday()],
            "reason": "nominal expiry weekday — settlement moved earlier "
                      "instead; C4 drops this cycle",
        })
        continue

    if prev_prices is None:
        continue                       # nothing before it to copy; cannot happen
                                       # after the first real row, kept for safety

    out_rows.append((day, *prev_prices, 0))
    filled_log.append({
        "date": day,
        "weekday": WEEKDAY_NAME[day.weekday()],
        "copied_from": out_rows[-2][0],
        "close": float(prev_prices[3]),
    })

daily = pd.DataFrame(out_rows, columns=[DATE] + PRICES + ["is_original"])
daily[DATE] = pd.to_datetime(daily[DATE])
daily["is_original"] = daily["is_original"].astype(int)

n_fill = int((daily["is_original"] == 0).sum())
print(f"\n  filled  : {n_fill} weekday(s) the market was shut")
if n_fill:
    by_wd = (daily.loc[daily["is_original"] == 0, DATE].dt.weekday
             .map(WEEKDAY_NAME).value_counts().to_dict())
    print(f"            by weekday : {by_wd}")
print(f"  skipped : {len(skipped_expiry)} missing day(s) that were the regime's "
      f"expiry weekday")
if skipped_expiry:
    by_wd = pd.Series([r["weekday"] for r in skipped_expiry]).value_counts().to_dict()
    print(f"            by weekday : {by_wd}  — C4 drops those cycles")
print(f"  rows    : {len(raw)} -> {len(daily)}")


# ============================================================================
# 3. CHECKS
# ----------------------------------------------------------------------------
# The last check is the one that matters most. Everything downstream decides what
# a cycle IS from is_expiry, so if filling the calendar changed a single expiry
# flag, every cycle boundary would move and the change would be invisible in the
# output. It is verified rather than argued.
# ============================================================================
checks = []


def chk(name, ok, detail=""):
    checks.append((name, bool(ok), detail))


chk("at least one row was written", len(daily) > 0)
chk("dates strictly increase, no duplicates",
    bool(daily[DATE].is_monotonic_increasing
         and not daily[DATE].duplicated().any()))
chk("no row falls on a weekend",
    bool((daily[DATE].dt.weekday < 5).all()))
chk("every original row survived",
    int((daily["is_original"] == 1).sum()) == len(raw),
    f"{int((daily['is_original'] == 1).sum())} vs {len(raw)}")
chk("no row was added outside the raw file's own range",
    bool(daily[DATE].min() == first_day and daily[DATE].max() == last_day))

# Original prices must be byte-for-byte what the raw file held.
kept = daily[daily["is_original"] == 1].reset_index(drop=True)
chk("original prices are unchanged",
    bool(np.allclose(kept[PRICES].to_numpy(float),
                     raw[PRICES].to_numpy(float), equal_nan=True)))

# An imputed row must equal the row immediately before it, which is what
# "copy the previous day" means, and which forces its log return to zero.
prev_block = daily[PRICES].shift(1).to_numpy(float)
this_block = daily[PRICES].to_numpy(float)
imp = (daily["is_original"] == 0).to_numpy()
chk("every imputed row copies the row before it",
    bool(imp.sum() == 0
         or np.allclose(this_block[imp], prev_block[imp], equal_nan=True)))
chk("every imputed row has a log return of exactly zero",
    bool(imp.sum() == 0
         or np.all(this_block[imp, 3] == prev_block[imp, 3])))

# No imputed row may sit on a nominal expiry weekday.
bad_wd = [d for d in daily.loc[imp, DATE]
          if d.weekday() == nominal_expiry_weekday(d)]
chk("no imputed row falls on the regime's expiry weekday", not bad_wd,
    str([str(d.date()) for d in bad_wd[:5]]))

# THE EXPIRY FLAGS MUST NOT MOVE.
filled_dates = set(daily[DATE])
exp_raw = {d for d in raw[DATE] if is_expiry(d, original_dates, last_day)}
exp_fil = {d for d in daily[DATE] if is_expiry(d, filled_dates, last_day)}
chk("filling did not change which days are expiries",
    exp_raw == exp_fil,
    f"raw {len(exp_raw)} vs filled {len(exp_fil)}; "
    f"gained {sorted(str(d.date()) for d in (exp_fil - exp_raw))[:4]}, "
    f"lost {sorted(str(d.date()) for d in (exp_raw - exp_fil))[:4]}")

# No expiry may land on an imputed row: an expiry is a settlement that happened.
chk("no expiry falls on an imputed row",
    not (exp_fil & set(daily.loc[imp, DATE])))

print("\n  checks")
for name, ok, detail in checks:
    print(f"    {'OK  ' if ok else 'FAIL'} {name}" + ("" if ok else f"  -- {detail}"))
bad = [n for n, ok, _ in checks if not ok]
if bad:
    raise RuntimeError(f"  C2b checks failed: {bad}")

print(f"  expiries flagged : {len(exp_fil)} — unchanged by filling")


# ============================================================================
# 4. WRITE
# ============================================================================
filled_sheet = (pd.DataFrame(filled_log) if filled_log
                else pd.DataFrame(columns=["date", "weekday", "copied_from",
                                           "close"]))
skipped_sheet = (pd.DataFrame(skipped_expiry) if skipped_expiry
                 else pd.DataFrame(columns=["date", "weekday", "reason"]))

manifest = pd.DataFrame([
    ("script", "C2b_FillCalendar"),
    ("source", F_RAW),
    ("impute_daily", IMPUTE_DAILY),
    ("expiry regime", f"Thu before {EXPIRY_CUTOFF.date()}, Tue from it onwards"),
    ("raw trading days", len(raw)),
    ("rows written", len(daily)),
    ("rows imputed", n_fill),
    ("missing days skipped as expiry", len(skipped_expiry)),
    ("expiries flagged", len(exp_fil)),
    ("period", f"{first_day.date()} -> {last_day.date()}"),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

write_xlsx(F_FILLED, {"daily": daily, "filled": filled_sheet,
                      "skipped_expiry": skipped_sheet, "manifest": manifest})

print(f"\n  {len(daily)} rows ({n_fill} imputed), "
      f"{first_day.date()} -> {last_day.date()}")
print("  NEXT: C3_Clean")
print("=" * 78)


  C2b_FillCalendar — impute_daily=True
  1915 raw trading days, 2019-01-01 -> 2026-10-05
  raw weekdays : Mon 381, Tue 386, Wed 383, Thu 386, Fri 379

  filled  : 93 weekday(s) the market was shut
            by weekday : {'Fri': 26, 'Mon': 24, 'Wed': 22, 'Tue': 16, 'Thu': 5}
  skipped : 17 missing day(s) that were the regime's expiry weekday
            by weekday : {'Thu': 14, 'Tue': 3}  — C4 drops those cycles
  rows    : 1915 -> 2008

  checks
    OK   at least one row was written
    OK   dates strictly increase, no duplicates
    OK   no row falls on a weekend
    OK   every original row survived
    OK   no row was added outside the raw file's own range
    OK   original prices are unchanged
    OK   every imputed row copies the row before it
    OK   every imputed row has a log return of exactly zero
    OK   no imputed row falls on the regime's expiry weekday
    OK   filling did not change which days are expiries
    OK   no expiry falls on an imputed row
  expiries flagged 

In [4]:
# @title
# ============================================================================
# C2c_FetchMacro — INDIA VIX, USD/INR AND OIL, ALIGNED TO THE NIFTY CALENDAR
# ============================================================================
#
#   reads  : 01b_daily_filled_<raw|filled>.xlsx   (for the trading calendar)
#            00_macro_manual.xlsx                 (optional; see MANUAL FALLBACK)
#   writes : 01c_macro_daily_<raw|filled>.xlsx
#            sheets "macro", "overrides", "coverage", "manifest"
#
# WHY THIS IS A SEPARATE SCRIPT
# ----------------------------
# It fetches from sources that are not NSE, on calendars that are not NSE's, and
# any of them can be down without that being a reason to lose the index data. Kept
# apart, a failure here costs you the macro features and nothing else.
#
# ----------------------------------------------------------------------------
# WHEN EACH SERIES CLOSES, AND WHY NOTHING IS LAGGED
# ----------------------------------------------------------------------------
# The decision is taken at the NIFTY close, 15:30 IST. A feature is legitimate
# only if its value was OBSERVABLE by then.
#
#   India VIX   closes 15:30 IST with NSE       -> observable, final
#   USD/INR     still trading at 15:30          -> observable, not yet final
#   Brent       still trading at 15:30          -> observable, not yet final
#
# The intended use of this file is:
#
#   1. The fetch runs and covers history UP TO YESTERDAY. Today is not published
#      yet, so it is not here.
#   2. At 15:30 you read all four numbers off a screen and enter today's row by
#      hand. Oil and USD/INR are their 15:30 levels, used as your estimate of
#      where they will settle.
#   3. A later run replaces your estimate with the published settlement.
#
# Under that workflow NOTHING NEEDS LAGGING. Every number you act on at 15:30 was
# visible at 15:30. LAG_DAYS is therefore 0 for all three.
#
# THE ONE THING THAT IS STILL WORTH KNOWING, because it belongs in the write-up.
# There is no look-ahead in the live call, but there is a TRAIN/SERVE MISMATCH.
# The historical rows hold each day's PUBLISHED settlement — for Brent, roughly
# 23:30 IST, which is after the 15:30 decision it sits against. So the model is
# trained on a number that includes the US session, and served a 15:30 snapshot
# that does not. The backtest is therefore slightly optimistic about what the live
# call can achieve, by however much the 15:30-to-settlement move matters.
#
# That is not a reason to lag. It is a reason to MEASURE it: run once with
# LAG_DAYS 0 and once with usdinr and brent set to 1, and report the difference.
# The gap bounds the optimism, and quoting it is stronger than a caveat. India VIX
# is unaffected either way, since it closes with NSE.
#
# <series>_asof records which date each number came from, so whichever setting you
# use is visible in the sheet rather than taken on trust.
#
# ----------------------------------------------------------------------------
# TODAY'S ANTICIPATED VALUE — THE OVERRIDES SHEET
# ----------------------------------------------------------------------------
# For a live call you may want to enter a value the source has not published yet
# (an anticipated close, or a level read off a screen at 15:25). Put it in
# 00_macro_manual.xlsx, sheet "overrides":
#
#       date        series      value
#       2026-09-30  india_vix   12.85
#       2026-09-30  usdinr      88.40
#
# An override wins over everything, is used EXACTLY as given with no lag applied,
# and is marked source="override" in the output. Overrides are meant for the open
# cycle only. Putting one on a historical date silently changes a training feature,
# so the checks below refuse to run if an override lands more than
# MAX_OVERRIDE_AGE_DAYS before the last trading day.
#
# ----------------------------------------------------------------------------
# HOLIDAY ALIGNMENT
# ----------------------------------------------------------------------------
# These three calendars disagree with NSE's — Brent trades on Diwali, NSE does
# not, and FRED has US holidays NSE lacks. Every series is therefore reindexed to
# the NIFTY trading calendar and carried FORWARD from its last available value.
# <series>_stale_days says how far it was carried; 0 means the value is that day's
# own. A large number means the source has gone quiet and the feature is a
# constant, which the coverage sheet reports per year.
#
# ----------------------------------------------------------------------------
# WHAT IS AND IS NOT ENGINEERED HERE
# ----------------------------------------------------------------------------
# Written here: the aligned LEVELS, plus the log changes over 1 and 5 trading days,
# because those need nothing but the series itself.
#
# NOT written here: vix_over_rv, the ratio of VIX to realized volatility. That
# needs realized_vol, which belongs to C5, and it is the single most useful form of
# VIX — a dimensionless variance risk premium rather than a level that drifts with
# the decade. Build it in C5 as:
#
#       vix_daily   = india_vix / 100 / sqrt(252)      # annual % -> daily sigma
#       vix_over_rv = vix_daily / realized_vol_Dn
#
# Never feed a raw level to these models. USD/INR went from 69 to 88 across this
# sample; a model fitted on the level learns the trend, not the relationship.
#
# ----------------------------------------------------------------------------
# SOURCES, AND WHAT I COULD NOT VERIFY
# ----------------------------------------------------------------------------
# India VIX : NSE, through jugaad_data, the same library C2_Fetch already uses.
# USD/INR   : FRED series DEXINUS (public CSV, no API key).
# Brent     : FRED series DCOILBRENTEU, falling back to DCOILWTICO for WTI.
#
# I could not reach either host from the machine this was written on, so the
# request shapes are unverified. The script therefore tries several candidate
# spellings per series, reports which one answered, and on total failure says
# exactly what to do rather than writing a half-empty file. If a source is blocked
# on your machine too, download the CSV by hand and use the MANUAL FALLBACK.
#
# MANUAL FALLBACK: put the history in 00_macro_manual.xlsx with one sheet per
# series, named india_vix / usdinr / brent, each holding columns date and value.
# A series present there is used instead of fetching, and marked source="manual".
# ============================================================================

try:
    _ = (ROOT, F_FILLED, DATE, CAL_TAG, read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import datetime
import io as _io
import os
import time
import warnings

import numpy as np
import pandas as pd

print("\n" + "=" * 78)
print(f"  C2c_FetchMacro — India VIX, USD/INR, oil   calendar={CAL_TAG}")
print("=" * 78)

F_MACRO = os.path.join(ROOT, f"01c_macro_daily_{CAL_TAG}.xlsx")
F_MANUAL = os.path.join(ROOT, "00_macro_manual.xlsx")

SERIES = ["india_vix", "usdinr", "brent"]

# Trading days to shift each series back.
#
# ZERO for all three, because the intended workflow enters today's 15:30 levels by
# hand and never acts on a number published later. Set usdinr and brent to 1 to run
# the train/serve sensitivity test described in the header; the checks adapt.
LAG_DAYS = {"india_vix": 0, "usdinr": 0, "brent": 0}

# FRED series ids, tried in order until one answers.
FRED_IDS = {"usdinr": ["DEXINUS"],
            "brent": ["DCOILBRENTEU", "DCOILWTICO"]}

# Index names jugaad_data may accept for the volatility index.
VIX_NAMES = ["INDIA VIX", "India VIX", "India Vix", "NIFTY VIX",
             "INDIAVIX", "Nifty VIX"]

MAX_STALE_DAYS = 10          # louder than a warning past this
MAX_OVERRIDE_AGE_DAYS = 10   # an override older than this is refused
MAX_LIVE_GAP_DAYS = 6        # how far past the fetch 'today' may sit

# NSE request limits, mirroring C2_Fetch. The endpoint refuses spans longer than
# roughly a quarter, so the history has to be asked for in windows.
NSE_CHUNK_MONTHS = 3
NSE_RETRIES = 3
NSE_PAUSE = 0.5              # seconds between chunks; NSE throttles


# ============================================================================
# 1. THE TRADING CALENDAR WE MUST MATCH
# ============================================================================
daily = read_xlsx(F_FILLED, sheet="daily")
daily[DATE] = pd.to_datetime(daily[DATE]).dt.normalize()
daily = daily.sort_values(DATE).reset_index(drop=True)
cal = daily[DATE]
first_day, last_day = cal.iloc[0], cal.iloc[-1]
print(f"  target calendar : {len(cal)} rows, {first_day.date()} -> {last_day.date()}")


# ============================================================================
# 2. FETCHERS
# ----------------------------------------------------------------------------
# Each returns a Series indexed by date, or None. None is not an error here — the
# caller decides what to do, and a manual file may cover it.
# ============================================================================
def fetch_fred(series_id, start, end):
    """Daily observations for one FRED series, as a date-indexed Series.

    Parameters
    ----------
    series_id : str
        e.g. "DEXINUS".
    start, end : datetime.date

    Returns
    -------
    pandas.Series or None
        None when the request fails or the payload is unusable. FRED writes "."
        for a non-trading day, which is why the values are coerced rather than
        parsed strictly.
    """
    url = ("https://fred.stlouisfed.org/graph/fredgraph.csv"
           f"?id={series_id}&cosd={start:%Y-%m-%d}&coed={end:%Y-%m-%d}")
    try:
        import urllib.request
        with urllib.request.urlopen(url, timeout=60) as r:
            raw = r.read().decode("utf-8", "replace")
        df = pd.read_csv(_io.StringIO(raw))
        if df.shape[1] < 2 or len(df) == 0:
            return None
        df.columns = ["date", "value"] + list(df.columns[2:])
        out = pd.Series(pd.to_numeric(df["value"], errors="coerce").values,
                        index=pd.to_datetime(df["date"], errors="coerce")).dropna()
        return out[out > 0] if len(out) else None
    except Exception as e:
        print(f"      FRED {series_id}: {type(e).__name__}: {str(e)[:70]}")
        return None


def _parse_dates(col):
    """Parse an NSE date column without pandas' per-element fallback warning.

    Parameters
    ----------
    col : pandas.Series
        Raw date strings, e.g. '11-SEP-2026' or '11-09-2026'.

    Returns
    -------
    pandas.DatetimeIndex

    Notes
    -----
    nselib hands back '11-SEP-2026'. pandas cannot infer that format, so a plain
    to_datetime(..., dayfirst=True) parses every element individually with
    dateutil and emits one UserWarning per call. Across 32 chunks that is 32
    warnings in the middle of the progress line, so the known formats are tried
    explicitly first and the slow path is kept only as a genuine fallback.
    """
    s = col.astype(str).str.strip()
    for fmt in ("%d-%b-%Y", "%d-%m-%Y", "%Y-%m-%d", "%d-%B-%Y"):
        got = pd.to_datetime(s, format=fmt, errors="coerce")
        if got.notna().sum() == s.notna().sum() and got.notna().any():
            return pd.DatetimeIndex(got)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        return pd.DatetimeIndex(
            pd.to_datetime(s, errors="coerce", dayfirst=True))


def _vix_from_frame(d):
    """Pull (date, close) out of an nselib India VIX frame.

    Parameters
    ----------
    d : DataFrame-like

    Returns
    -------
    pandas.Series or None
        Date-indexed closing level, or None when the frame has no usable pair.

    Notes
    -----
    Confirmed against nselib 2.5.1. india_vix_data returns:

        TIMESTAMP        '11-SEP-2026'          dd-MMM-yyyy
        INDEX_NAME       'INDIA VIX'
        OPEN_INDEX_VAL   CLOSE_INDEX_VAL   HIGH_INDEX_VAL   LOW_INDEX_VAL
        PREV_CLOSE       VIX_PTS_CHG       VIX_PERC_CHG

    index_data(index="INDIA VIX") returns the same shape as Nifty 50, with
    TURN_OVER and TRADED_QTY both zero and no PREV_CLOSE. Either works; the close
    is CLOSE_INDEX_VAL in both.

    The close search excludes PREV, or PREV_CLOSE would be a candidate and the
    series would silently be yesterday's number throughout.
    """
    if d is None:
        return None
    d = pd.DataFrame(d)
    if len(d) == 0:
        return None
    up = {c: str(c).strip().upper() for c in d.columns}
    dcol = next((c for c, u in up.items()
                 if u in ("TIMESTAMP", "DATE", "HISTORICALDATE", "TRADE_DATE")
                 or u.endswith("DATE")), None)
    ccol = next((c for c, u in up.items()
                 if "CLOS" in u and "PREV" not in u), None)
    if dcol is None or ccol is None:
        return None
    v = pd.to_numeric(
        d[ccol].astype(str).str.replace(",", "", regex=False), errors="coerce")
    out = pd.Series(v.values, index=_parse_dates(d[dcol])).dropna()
    out = out[(out > 0) & (out < 200)]        # VIX is a percentage, not a level
    out = out[~out.index.duplicated(keep="last")]
    return out.sort_index() if len(out) else None


def _nse_chunks(start, end, months=NSE_CHUNK_MONTHS):
    """Split a range into windows the NSE endpoint will accept.

    Returns
    -------
    list of (str, str)
        dd-mm-YYYY pairs, oldest first.

    Notes
    -----
    Deliberately a copy of C2_Fetch's month_chunks rather than an import: these
    are separate notebook cells with no shared module, and a quietly diverging
    copy is less dangerous than a cell that cannot run on its own.

    This matters more than it looks. NSE refuses spans longer than about a
    quarter, and the first version of this script asked for the whole seven-year
    history in ONE call — which is why a 20-day diagnostic succeeded while the
    real run returned nothing.
    """
    out, cur = [], start
    while cur <= end:
        m = cur.month - 1 + months
        y, m = cur.year + m // 12, m % 12 + 1
        stop = min(datetime.date(y, m, 1) - datetime.timedelta(days=1), end)
        out.append((cur.strftime("%d-%m-%Y"), stop.strftime("%d-%m-%Y")))
        cur = stop + datetime.timedelta(days=1)
    return out


def fetch_india_vix(start, end):
    """India VIX closing level from NSE, chunked and retried.

    Parameters
    ----------
    start, end : datetime.date

    Returns
    -------
    (pandas.Series or None, str)
        The series and a description of the call that produced it.

    Notes
    -----
    Two routes, in order, both confirmed working on nselib 2.5.1:

        capital_market.india_vix_data(from_date=..., to_date=...)
        capital_market.index_data(index="INDIA VIX", from_date=..., to_date=...)

    Dates MUST be dd-mm-YYYY STRINGS. Passing datetime.date objects raises
    "strptime() argument 1 must be str, not datetime.date" inside nselib.

    An empty chunk is a retryable failure, not a success — the same rule C2_Fetch
    uses, and for the same reason: NSE answers with an empty frame when it is
    throttling, and accepting that silently is how a truncated history gets
    written over a good one. A run that loses chunks says so and names them.
    """
    try:
        from nselib import capital_market as _cm
    except Exception as e:
        print(f"      India VIX: cannot import nselib.capital_market "
              f"({type(e).__name__}: {e})")
        return None, ""

    routes = []
    if getattr(_cm, "india_vix_data", None) is not None:
        routes.append(("india_vix_data",
                       lambda a, b: _cm.india_vix_data(from_date=a, to_date=b)))
    routes.append(("index_data('INDIA VIX')",
                   lambda a, b: _cm.index_data(index="INDIA VIX",
                                               from_date=a, to_date=b)))

    chunks = _nse_chunks(start, end)
    for label, call in routes:
        frames, failed = [], []
        for i, (a, b) in enumerate(chunks, 1):
            got, why = None, "unknown"
            for attempt in range(1, NSE_RETRIES + 1):
                try:
                    got = _vix_from_frame(call(a, b))
                    if got is None or len(got) == 0:
                        why, got = "empty response", None
                        if attempt < NSE_RETRIES:
                            time.sleep(2 * attempt)
                            continue
                        break
                    break
                except Exception as e:
                    why = f"{type(e).__name__}: {str(e)[:70]}"
                    got = None
                    if attempt < NSE_RETRIES:
                        time.sleep(2 * attempt)
            if got is None:
                failed.append((a, b, why))
            else:
                frames.append(got)
            print(f"      India VIX via {label}: [{i}/{len(chunks)}] {b}   ",
                  end="\r")
            time.sleep(NSE_PAUSE)
        print()
        if frames:
            s = pd.concat(frames)
            s = s[~s.index.duplicated(keep="last")].sort_index()
            if failed:
                print(f"      {len(failed)} of {len(chunks)} chunk(s) came back "
                      f"empty; first {failed[0][0]} -> {failed[0][1]} "
                      f"({failed[0][2]})")
            return s, (f"nselib capital_market.{label}, "
                       f"{len(chunks) - len(failed)}/{len(chunks)} chunks")
        print(f"      India VIX via {label}: every chunk failed "
              f"(first: {failed[0][2] if failed else 'n/a'})")

    print("      India VIX: no route returned data. "
          f"capital_market offers: "
          f"{[a for a in dir(_cm) if 'vix' in a.lower()] or 'nothing with vix'}")
    return None, ""


def load_manual(sheet):
    """One series from 00_macro_manual.xlsx, or None when absent."""
    if not os.path.exists(F_MANUAL):
        return None
    try:
        d = pd.read_excel(F_MANUAL, sheet_name=sheet)
    except Exception:
        return None
    cols = {str(c).strip().lower(): c for c in d.columns}
    if "date" not in cols or "value" not in cols:
        return None
    out = pd.Series(pd.to_numeric(d[cols["value"]], errors="coerce").values,
                    index=pd.to_datetime(d[cols["date"]], errors="coerce")).dropna()
    return out.sort_index() if len(out) else None


# ============================================================================
# 3. COLLECT EACH SERIES
# ============================================================================
raw_series, source_of, detail_of = {}, {}, {}
start = (first_day - pd.Timedelta(days=30)).date()   # margin for the lag and ffill
end = (last_day + pd.Timedelta(days=1)).date()

print("\n  collecting")
for name in SERIES:
    man = load_manual(name)
    if man is not None:
        raw_series[name] = man
        source_of[name] = "manual"
        detail_of[name] = f"00_macro_manual.xlsx sheet {name!r}"
        print(f"    {name:10} manual file, {len(man)} rows")
        continue
    if name == "india_vix":
        s, how = fetch_india_vix(start, end)
    else:
        s, how = None, ""
        for sid in FRED_IDS[name]:
            s = fetch_fred(sid, start, end)
            if s is not None:
                how = f"FRED {sid}"
                break
    if s is None:
        raw_series[name] = None
        source_of[name] = "MISSING"
        detail_of[name] = ""
        print(f"    {name:10} NOT AVAILABLE")
        continue
    raw_series[name] = s
    source_of[name] = "fetch"
    detail_of[name] = how
    print(f"    {name:10} {how}, {len(s)} rows, "
          f"{s.index.min().date()} -> {s.index.max().date()}")

missing = [n for n in SERIES if raw_series[n] is None]
if len(missing) == len(SERIES):
    raise RuntimeError(
        "  no macro series could be obtained.\n"
        f"  Tried nselib (then jugaad_data) for India VIX, and FRED for "
        f"{FRED_IDS}.\n"
        "  If your network blocks them, download each history as CSV and build\n"
        f"  {F_MANUAL} with one sheet per series named "
        f"{SERIES}, each holding columns 'date' and 'value'.\n"
        "  FRED: https://fred.stlouisfed.org/series/DEXINUS  (Download > CSV)\n"
        "        https://fred.stlouisfed.org/series/DCOILBRENTEU\n"
        "  India VIX: NSE > Products > Equity Derivatives > India VIX > Historical")
if missing:
    print(f"\n  WARNING: {missing} unavailable — those columns will be all-NaN, and\n"
          f"  C5 will drop any feature built from them. Supply them via {F_MANUAL}\n"
          f"  if you need them.")


# ============================================================================
# 4. LAG, ALIGN, CARRY FORWARD
# ----------------------------------------------------------------------------
# Order matters. The lag is applied on the SERIES' OWN trading days, before
# reindexing, so "one trading day back" means one of ITS sessions, not one NSE
# session. Only then is it mapped onto the NIFTY calendar and carried forward.
# ============================================================================
macro = pd.DataFrame({DATE: cal})

for name in SERIES:
    s = raw_series[name]
    if s is None:
        for suf in ("", "_asof", "_stale_days", "_source"):
            macro[f"{name}{suf}"] = np.nan if suf != "_source" else "missing"
        continue

    s = s[~s.index.duplicated(keep="last")].sort_index()
    lag = int(LAG_DAYS.get(name, 0))
    shifted = s.shift(lag)                      # lag in the SERIES' own sessions
    asof = pd.Series(s.index, index=s.index).shift(lag)

    # Reindex onto the NIFTY calendar, carrying the last known value forward.
    val = shifted.reindex(shifted.index.union(cal)).ffill().reindex(cal)
    src_date = asof.reindex(asof.index.union(cal)).ffill().reindex(cal)

    macro[name] = val.to_numpy()
    macro[f"{name}_asof"] = pd.to_datetime(src_date.to_numpy())
    macro[f"{name}_stale_days"] = (
        (cal.to_numpy() - pd.to_datetime(src_date.to_numpy())) /
        np.timedelta64(1, "D"))
    macro[f"{name}_source"] = source_of[name]


# ============================================================================
# 5. OVERRIDES — TODAY'S 15:30 VALUES, ENTERED BY HAND
# ----------------------------------------------------------------------------
# An override may fall on a date the fetch has not reached. That is the normal
# case, not an edge case: the fetch covers history up to YESTERDAY, so at 15:30
# today's row does not exist yet and the override has to CREATE it.
#
# An earlier version only overwrote rows that already existed and printed "not a
# trading day, ignored" otherwise — which would have thrown away every live value
# silently. So an override dated after the last fetched day appends a row.
#
# PUT OVERRIDES IN 00_macro_manual.xlsx, NOT IN THIS SCRIPT'S OUTPUT.
# 01c_macro_daily_*.xlsx is rewritten from scratch on every run, so anything typed
# into it is lost the next time the fetch runs. 00_macro_manual.xlsx is only ever
# READ, never written, so what you put there survives.
# ============================================================================
override_rows, appended_dates = [], []
if os.path.exists(F_MANUAL):
    try:
        odf = pd.read_excel(F_MANUAL, sheet_name="overrides")
        cols = {str(c).strip().lower(): c for c in odf.columns}
        if {"date", "series", "value"} <= set(cols):
            for _, r in odf.iterrows():
                d = pd.to_datetime(r[cols["date"]], errors="coerce")
                nm = str(r[cols["series"]]).strip()
                v = pd.to_numeric(r[cols["value"]], errors="coerce")
                if pd.isna(d) or pd.isna(v) or nm not in SERIES:
                    continue
                d = d.normalize()

                # A row beyond the fetched history is the live row: create it.
                if d > macro[DATE].max():
                    if d.weekday() >= 5:
                        print(f"    override {nm} {d.date()}: a weekend, ignored")
                        continue
                    if (d - macro[DATE].max()).days > MAX_LIVE_GAP_DAYS:
                        raise RuntimeError(
                            f"  override for {nm} is dated {d.date()}, which is "
                            f"{(d - macro[DATE].max()).days} days after the last\n"
                            f"  fetched day ({macro[DATE].max().date()}). That is too "
                            f"far to be 'today'.\n"
                            f"  Re-run C2_Fetch and C2b so the history is current, or "
                            f"fix the date.")
                    if d not in set(macro[DATE]):
                        blank = {DATE: d}
                        for nm2 in SERIES:
                            blank[nm2] = np.nan
                            blank[f"{nm2}_asof"] = pd.NaT
                            blank[f"{nm2}_stale_days"] = np.nan
                            blank[f"{nm2}_source"] = "awaiting override"
                        macro = pd.concat([macro, pd.DataFrame([blank])],
                                          ignore_index=True).sort_values(DATE)
                        macro = macro.reset_index(drop=True)
                        appended_dates.append(d)
                        print(f"    appended live row {d.date()} "
                              f"(beyond the fetched history)")

                age = (macro[DATE].max() - d).days
                if age > MAX_OVERRIDE_AGE_DAYS:
                    raise RuntimeError(
                        f"  override for {nm} on {d.date()} is {age} days before the\n"
                        f"  last row. Overrides exist for the OPEN cycle; one this old\n"
                        f"  silently rewrites a training feature. Remove it, or raise\n"
                        f"  MAX_OVERRIDE_AGE_DAYS deliberately and say so.")

                hit = macro[DATE] == d
                if not hit.any():
                    print(f"    override {nm} {d.date()}: not a trading day, ignored")
                    continue
                macro.loc[hit, nm] = float(v)
                macro.loc[hit, f"{nm}_asof"] = d
                macro.loc[hit, f"{nm}_stale_days"] = 0.0
                macro.loc[hit, f"{nm}_source"] = "override"
                override_rows.append({"date": d, "series": nm, "value": float(v)})
    except RuntimeError:
        raise
    except Exception as e:
        print(f"    overrides sheet unreadable ({type(e).__name__}), ignored")

# A live row is only usable if EVERY series on it was supplied. A half-filled row
# would reach C5 as a NaN feature and the whole decision day would be dropped
# without a word, which at 15:30 is the worst possible moment to find out.
for d in appended_dates:
    hit = macro[DATE] == d
    absent = [n for n in SERIES
              if pd.isna(pd.to_numeric(macro.loc[hit, n], errors="coerce")).all()]
    if absent:
        raise RuntimeError(
            f"  the live row {d.date()} is missing {absent}.\n"
            f"  Every series needs a value on the live row, or C5 drops that\n"
            f"  decision day. Add the missing one(s) to the overrides sheet of\n"
            f"  {F_MANUAL} — the 15:30 screen level is what this expects.")

macro["is_live_row"] = macro[DATE].isin(appended_dates).astype(int)

if override_rows:
    print(f"\n  applied {len(override_rows)} manual override(s)"
          + (f", {len(appended_dates)} on a newly created live row"
             if appended_dates else ""))
    for r in override_rows:
        print(f"    {r['series']:10} {r['date'].date()}  {r['value']}")


# ============================================================================
# 6. THE DERIVED COLUMNS THAT NEED ONLY THE SERIES ITSELF
# ----------------------------------------------------------------------------
# Log changes over 1 and 5 trading days. Stationary, scale-free, and safe to feed
# a model — unlike the levels, which trend hard across this sample.
#
# The ratio of VIX to realized volatility is NOT here: it needs realized_vol, so
# it belongs in C5. See the header.
# ============================================================================
for name in SERIES:
    v = pd.to_numeric(macro[name], errors="coerce")
    with np.errstate(divide="ignore", invalid="ignore"):
        macro[f"{name}_ln1"] = np.log(v / v.shift(1))
        macro[f"{name}_ln5"] = np.log(v / v.shift(5))

# VIX in the same units as a daily standard deviation, so C5 can divide directly.
if raw_series["india_vix"] is not None:
    macro["vix_daily"] = pd.to_numeric(macro["india_vix"],
                                       errors="coerce") / 100.0 / np.sqrt(252.0)
else:
    macro["vix_daily"] = np.nan


# ============================================================================
# 7. COVERAGE AND CHECKS
# ============================================================================
cov_rows = []
for name in SERIES:
    v = pd.to_numeric(macro[name], errors="coerce")
    st = pd.to_numeric(macro[f"{name}_stale_days"], errors="coerce")
    cov_rows.append({
        "series": name, "source": source_of[name], "detail": detail_of[name],
        "lag_trading_days": LAG_DAYS.get(name, 0),
        "rows_present": int(v.notna().sum()), "rows_missing": int(v.isna().sum()),
        "pct_present": round(100.0 * v.notna().mean(), 2),
        "median_stale_days": round(float(st.median()), 2) if st.notna().any() else np.nan,
        "max_stale_days": round(float(st.max()), 1) if st.notna().any() else np.nan,
        "first": v.first_valid_index(), "last": v.last_valid_index(),
    })
coverage = pd.DataFrame(cov_rows)
print("\n  coverage")
print(coverage[["series", "source", "lag_trading_days", "pct_present",
                "median_stale_days", "max_stale_days"]].to_string(index=False))

checks = []


def chk(name, ok, detail=""):
    checks.append((name, bool(ok), detail))


n_live = int(macro["is_live_row"].sum())
chk("rows are ordered and unique",
    bool(macro[DATE].is_monotonic_increasing and not macro[DATE].duplicated().any()))
chk("the fetched rows match the daily calendar exactly",
    bool((macro.loc[macro["is_live_row"] == 0, DATE].to_numpy()
          == cal.to_numpy()).all()),
    "a fetched row drifted from 01b's calendar")
chk("at most one live row, and it is last",
    n_live <= 1 and (n_live == 0 or bool(macro["is_live_row"].iloc[-1] == 1)),
    f"{n_live} live row(s)")

for name in SERIES:
    if raw_series[name] is None:
        continue
    v = pd.to_numeric(macro[name], errors="coerce")
    chk(f"{name}: every value positive", bool((v.dropna() > 0).all()))

    # THE LEAKAGE CHECK. A lagged series must never carry a value dated on or
    # after the row it sits on. This is the check that would catch a lag silently
    # reverting to zero.
    asof = pd.to_datetime(macro[f"{name}_asof"])
    if LAG_DAYS.get(name, 0) > 0:
        bad = macro.loc[(asof >= macro[DATE])
                        & (macro[f"{name}_source"] != "override"), DATE]
        chk(f"{name}: no value dated on or after its own row (lag "
            f"{LAG_DAYS[name]})", len(bad) == 0,
            f"{len(bad)} row(s), first {bad.iloc[0].date() if len(bad) else ''}")
    else:
        bad = macro.loc[asof > macro[DATE], DATE]
        chk(f"{name}: no value from the future", len(bad) == 0, f"{len(bad)} row(s)")

    st = pd.to_numeric(macro[f"{name}_stale_days"], errors="coerce")
    n_stale = int((st > MAX_STALE_DAYS).sum())
    chk(f"{name}: carried forward no more than {MAX_STALE_DAYS} days",
        n_stale == 0,
        f"{n_stale} row(s) exceed it — the feature is a constant there")

    chk(f"{name}: log changes are finite where the level is",
        bool(np.isfinite(pd.to_numeric(macro[f"{name}_ln1"],
                                       errors="coerce").dropna()).all()))

print("\n  checks")
for nm, ok, det in checks:
    print(f"    {'OK  ' if ok else 'FAIL'} {nm}" + ("" if ok else f"  -- {det}"))
bad = [n for n, ok, _ in checks if not ok]
if bad:
    raise RuntimeError(f"  C2c checks failed: {bad}")


# ============================================================================
# 8. WRITE
# ============================================================================
ov_template = (pd.DataFrame(override_rows) if override_rows else
               pd.DataFrame({"date": pd.Series(dtype="datetime64[ns]"),
                             "series": pd.Series(dtype=object),
                             "value": pd.Series(dtype=float)}))

manifest = pd.DataFrame([
    ("script", "C2c_FetchMacro"),
    ("calendar source", F_FILLED),
    ("daily_calendar", CAL_TAG),
    ("trading days", len(macro)),
    ("period", f"{first_day.date()} -> {last_day.date()}"),
    ("india_vix source", f"{source_of['india_vix']} {detail_of['india_vix']}"),
    ("usdinr source", f"{source_of['usdinr']} {detail_of['usdinr']}"),
    ("brent source", f"{source_of['brent']} {detail_of['brent']}"),
    ("lag, trading days", str(LAG_DAYS)),
    ("lag rationale", "India VIX closes 15:30 IST with NSE so it is same-day; "
                      "FRED USD/INR is NY noon and Brent settles late evening "
                      "IST, so both are shifted back one of their own sessions"),
    ("overrides applied", len(override_rows)),
    ("live row appended", ", ".join(str(d.date()) for d in appended_dates)
                          if appended_dates else "none"),
    ("series unavailable", ", ".join(missing) if missing else "none"),
    ("NOT built here", "vix_over_rv — needs realized_vol, build it in C5"),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

write_xlsx(F_MACRO, {"macro": macro, "overrides": ov_template,
                     "coverage": coverage, "manifest": manifest})

print(f"\n  {len(macro)} rows written, {first_day.date()} -> {last_day.date()}")
print(f"  levels + ln1 + ln5 for {SERIES}, plus vix_daily")
print("  NEXT: C3_Clean  (C5 joins this file by date when FEATURE_SET uses macro)")
print("=" * 78)


  C2c_FetchMacro — India VIX, USD/INR, oil   calendar=filled
  target calendar : 2008 rows, 2019-01-01 -> 2026-10-05

  collecting

    india_vix  nselib capital_market.india_vix_data, 32/32 chunks, 1943 rows, 2018-12-03 -> 2026-10-05
    usdinr     FRED DEXINUS, 1956 rows, 2018-12-03 -> 2026-10-02
    brent      FRED DCOILBRENTEU, 1980 rows, 2018-12-03 -> 2026-09-29

  coverage
   series source  lag_trading_days  pct_present  median_stale_days  max_stale_days
india_vix  fetch                 0        100.0                0.0             3.0
   usdinr  fetch                 0        100.0                0.0             3.0
    brent  fetch                 0        100.0                0.0             6.0

  checks
    OK   rows are ordered and unique
    OK   the fetched rows match the daily calendar exactly
    OK   at most one live row, and it is last
    OK   india_vix: every value positive
    OK   india_vix: no value from the future
    OK   india_vix: carried forward no more tha

In [5]:
# @title
# ============================================================================
# C3_Clean — VALIDATE THE DAILY FILE
# ============================================================================
#
#   reads  : 01b_daily_filled_<raw|filled>.xlsx   (C2b's output, never the raw)
#   writes : 02_daily_clean_<raw|filled>.xlsx     sheets "daily" and "manifest"
#
# One row per trading day: Date, Open, High, Low, Close — and then a few
# audit_* columns that reproduce, day by day, the quantities C5 builds its
# features from.
#
# WHY THE audit_ PREFIX, AND WHY THIS IS NOT A RETURN TO THE OLD DESIGN
# --------------------------------------------------------------------
# The previous pipeline stored daily_log_return, realized_vol and Volatility in
# this file, and they caused real confusion: `realized_vol` here meant a 10-day
# lagged window while `realized_vol_Dn` in the training sheets meant a 16-day
# same-day window. Same name, different number.
#
# These columns are for CHECKING ONLY. C5 reads Date, Open, High, Low and Close
# and recomputes everything itself, so nothing downstream can consume a stale
# audit column by accident. The prefix says so in every spreadsheet that opens
# the file, and each column's window length is in its own name.
#
# WHAT WAS DROPPED FROM THE OLD CLEANUP, AND WHY
# ----------------------------------------------
#   daily_log_return   C5 recomputes it; storing it invites two versions
#   realized_vol       a 10-day reference column no feature ever used, whose
#                      name collided with the MODEL's realized_vol_Dn and
#                      meant something different
#   Volatility         an alias of the above, named as if it were implied VIX
#   intraday_range     computed, never used
#
# THE GOVERNING RULE: NEVER SILENTLY IMPUTE
# -----------------------------------------
# A fabricated price is worse than a missing one because it looks like data.
# BAD_PRICE controls what happens to a missing or non-positive price:
#
#   "stop"  print the offending rows and halt. The default — you want to know,
#           and the fix belongs upstream in C2.
#   "drop"  remove them and report how many. The log return then correctly spans
#           the gap instead of pretending the day traded flat.
#
# There is deliberately no "ffill" option. Forward-filling manufactures a
# zero-return day, which deflates volatility for as long as the rolling window
# is wide — and z_dist DIVIDES by that volatility.
# ============================================================================

try:
    _ = (F_FILLED, F_CLEAN, DATE, OPEN, HIGH, LOW, CLOSE, SIGMA_WINDOW_GRID,
         rv_window_days,
         read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import datetime

import numpy as np
import pandas as pd

print("\n" + "=" * 78)
print("  C3_Clean — validate the daily file")
print("=" * 78)

BAD_PRICE = "stop"          # "stop" | "drop"
MAX_GAP_DAYS = 10


# ============================================================================
# 1. LOAD
# ============================================================================
df = read_xlsx(F_FILLED, sheet="daily")
n_in = len(df)
print(f"  read {n_in} rows from {F_FILLED.split('/')[-1]}")

need = [DATE, OPEN, HIGH, LOW, CLOSE]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(
        f"  missing column(s) {missing}. The file has {list(df.columns)}.\n"
        f"  Open is REQUIRED, not optional: the gap feature is\n"
        f"  ln(Open / previous Close) and cannot be built without it.\n"
        f"  Run C2_Fetch then C2b_FillCalendar first.")

# is_original comes from C2b and is carried through untouched. It is not
# decoration: C1's daily_realized_vol reads it to keep imputed zero-return days
# out of the volatility window, and without it those zeros would deflate every
# window they touch. A file without the column is treated as all-original, which
# is correct for an unfilled calendar.
if "is_original" not in df.columns:
    print("  no is_original column — treating every row as original")
    df["is_original"] = 1
df = df[need + ["is_original"]].copy()
df["is_original"] = pd.to_numeric(df["is_original"], errors="coerce").fillna(1).astype(int)
n_imputed_in = int((df["is_original"] == 0).sum())
print(f"  of these, {n_imputed_in} row(s) are imputed market holidays")


# ============================================================================
# 2. DATES, DUPLICATES, WEEKENDS
# ============================================================================
df[DATE] = pd.to_datetime(df[DATE], errors="coerce").dt.normalize()
n_bad_date = int(df[DATE].isna().sum())
if n_bad_date:
    print(f"  dropped {n_bad_date} row(s) with an unreadable date")
df = df.dropna(subset=[DATE]).sort_values(DATE).reset_index(drop=True)

dup = df.duplicated(subset=[DATE], keep="last")
n_dup = int(dup.sum())
if n_dup:
    print(f"  removed {n_dup} duplicate date(s): "
          f"{df.loc[dup, DATE].dt.date.tolist()[:8]}")
    df = df[~dup].reset_index(drop=True)

wk = df[DATE].dt.weekday >= 5
n_weekend = int(wk.sum())
if n_weekend:
    print(f"  removed {n_weekend} weekend row(s)")
    df = df[~wk].reset_index(drop=True)


# ============================================================================
# 3. PRICE INTEGRITY — VALIDATE FIRST, THEN ACT
# ----------------------------------------------------------------------------
# Order matters. Repairing before validating makes the validation vacuous: it
# then confirms the repair rather than the data.
# ============================================================================
PRICES = [OPEN, HIGH, LOW, CLOSE]
for c in PRICES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

nonpos = (df[PRICES] <= 0).any(axis=1)
absent = df[PRICES].isna().any(axis=1)
inverted = df[HIGH] < df[LOW]
unbracketed = ((df[HIGH] < df[[OPEN, CLOSE]].max(axis=1))
               | (df[LOW] > df[[OPEN, CLOSE]].min(axis=1)))

print(f"  as loaded: non-positive={int(nonpos.sum())} missing={int(absent.sum())} "
      f"High<Low={int(inverted.sum())} High/Low not bracketing={int(unbracketed.sum())}")

# High < Low is never repairable. Swapping the two makes the row LOOK valid while
# Open and Close on it stay wrong, which hides a parsing fault instead of fixing
# one.
if inverted.any():
    raise RuntimeError(
        f"  {int(inverted.sum())} row(s) have High < Low:\n"
        f"{df.loc[inverted, [DATE] + PRICES].head(10)}\n"
        f"  The source was mis-parsed, so Open and Close there are suspect too.\n"
        f"  Fix the raw file or re-run C2; do not swap High and Low.")

bad = nonpos | absent
n_bad = int(bad.sum())
if n_bad:
    if BAD_PRICE == "stop":
        raise RuntimeError(
            f"  {n_bad} row(s) have a missing or non-positive price:\n"
            f"{df.loc[bad, [DATE] + PRICES].head(10)}\n"
            f"  Investigate the raw file. If these rows are genuinely bad, set\n"
            f"  BAD_PRICE='drop' — then the log return correctly spans the gap\n"
            f"  instead of recording a day that traded flat.")
    print(f"  dropping {n_bad} bad row(s): {df.loc[bad, DATE].dt.date.tolist()[:8]}")
    df = df[~bad].reset_index(drop=True)

if unbracketed.any():
    print(f"  note: {int(unbracketed.sum())} row(s) where High/Low do not bracket "
          f"Open/Close — worth a look, not fatal")


# ============================================================================
# 4. CONTINUITY
# ============================================================================
gaps = df[DATE].diff().dt.days.fillna(0)
big = gaps[gaps > MAX_GAP_DAYS]
if len(big):
    print(f"  {len(big)} gap(s) longer than {MAX_GAP_DAYS} calendar days:")
    for i in big.index[:5]:
        print(f"    {df[DATE][i-1].date()} -> {df[DATE][i].date()} ({int(gaps[i])}d)")
    raise RuntimeError("  continuity check failed — re-run C2_Fetch.")
print(f"  continuity OK, largest gap {int(gaps.max())} calendar days")


# ============================================================================
# 5. REPORT WHAT IS IN THE DATA
# ----------------------------------------------------------------------------
# Exact-zero returns are reported, not asserted away. Two consecutive identical
# closes come from NSE, not from this script — nothing here fabricates a price.
# They matter because they pull the rolling volatility down, and z_dist divides
# by that volatility on the very day the decision is taken. Disclose them in the
# write-up rather than discovering them later.
# ============================================================================
ret = np.log(df[CLOSE] / df[CLOSE].shift(1))
zero_dates = df.loc[ret == 0, DATE].dt.date.tolist()
n_zero = len(zero_dates)
if n_zero:
    print(f"  {n_zero} day(s) with an exact-zero return (identical consecutive "
          f"closes): {zero_dates[:10]}" + (" ..." if n_zero > 10 else ""))
else:
    print("  no exact-zero returns")

print(f"  daily return std: {ret.std():.5f}   "
      f"(a DAILY figure; if it looks like a VIX level something is wrong)")


# ============================================================================
# 5b. AUDIT COLUMNS
# ----------------------------------------------------------------------------
# Each one is the day-level input to a feature, written out so any row of a task
# sheet can be traced back by hand.
#
#   audit_log_return      ln(Close / previous Close)
#   audit_gap             ln(Open / previous Close) — the overnight jump, which
#                         is what gap_Dn is on the day it is read
#   audit_rv_<N>d         std of the last N daily log returns INCLUDING this
#                         day's, ddof=1. One column per window in the grid,
#                         because the feature's window is band_window x 4.
#
# The three traps that make a hand check disagree, all of them settled here:
# N returns need N+1 closes; the window ENDS on this row rather than the one
# before; and the standard deviation is the sample form, ddof=1 — Excel's
# STDEV.S, not STDEV.P.
# ============================================================================
print("  📌 STEP 5b: audit columns")
df["audit_log_return"] = np.log(df[CLOSE] / df[CLOSE].shift(1))
df["audit_gap"] = np.log(df[OPEN] / df[CLOSE].shift(1))

# audit_rv MUST EXCLUDE IMPUTED DAYS, for the same reason the model's
# realized_vol does: an imputed day repeats the previous close, so its log return
# is exactly zero, and zeros shrink a standard deviation. On this file, letting
# them in deflates the 16-day number by more than 5 per cent on roughly a fifth
# of rows. An audit column computed a different way from the thing it audits is
# worse than no audit column, because a hand check against it would "confirm" a
# number the model never used.
#
# So the window is taken over CONSECUTIVE TRADING DAYS and the result is carried
# forward onto imputed dates — exactly what C1's daily_realized_vol does. C5's
# realized_vol_Dn read on a date should equal audit_rv_<n>d on that same row.
_real = df["is_original"] == 1
_obs_ret = np.log(df.loc[_real, CLOSE] / df.loc[_real, CLOSE].shift(1))

# Ask C1 for the window rather than multiplying by four here, so this audit
# column can never describe a different number from the feature it audits.
rv_windows = sorted({rv_window_days(w) for w in SIGMA_WINDOW_GRID})
for n in rv_windows:
    _rv_real = _obs_ret.rolling(n, min_periods=n).std(ddof=1)
    df[f"audit_rv_{n}d"] = _rv_real.reindex(df.index).ffill()
print(f"     audit_log_return, audit_gap, "
      + ", ".join(f"audit_rv_{n}d" for n in rv_windows))
print(f"     audit_rv skips imputed rows and carries forward — same rule as the")
print(f"     model's realized_vol, so the two agree cell for cell")
print(f"     these are CHECKING columns — C5 recomputes from Open/High/Low/Close")


# ============================================================================
# 6. CHECKS ON WHAT IS ABOUT TO BE WRITTEN
# ============================================================================
# Verify one audit_rv cell against a slice computed independently, far enough in
# to be past the warm-up.
#
# The hand computation walks the ORIGINAL rows only, which is the point of it. A
# check that used every row would disagree with the column by exactly the
# deflation the column exists to avoid, and the disagreement would read as a bug
# in the column rather than in the check.
_n0 = rv_windows[0]
_orig_close = df.loc[df["is_original"] == 1, CLOSE].to_numpy(float)
_orig_pos = np.flatnonzero((df["is_original"] == 1).to_numpy())
_k = min(len(_orig_pos) - 1, _n0 + 50)
_i = int(_orig_pos[_k])
_slice = _orig_close[_k - _n0:_k + 1]            # n0 + 1 closes -> n0 returns
_hand = float(pd.Series(np.log(_slice[1:] / _slice[:-1])).std(ddof=1))
_audit_rv_ok = np.isclose(float(df[f"audit_rv_{_n0}d"].iloc[_i]), _hand)

checks = [
    ("all prices positive", bool((df[PRICES] > 0).all().all())),
    ("High >= Low everywhere", bool((df[HIGH] >= df[LOW]).all())),
    ("dates strictly increasing",
     bool(df[DATE].is_monotonic_increasing and not df[DATE].duplicated().any())),
    ("no weekend rows", bool((df[DATE].dt.weekday < 5).all())),
    ("returns finite apart from the first row",
     bool(np.isfinite(ret.dropna()).all())),
    ("dates are timezone-naive, so Excel keeps them", df[DATE].dt.tz is None),
    ("the five price columns come first, in order",
     list(df.columns)[:5] == need),
    # is_original is the one non-audit extra: C1's realized_vol reads it, so it
    # is a real input, not a checking column, and it is named without the prefix
    # on purpose.
    ("every extra column is audit_ or is_original",
     all(c.startswith("audit_") or c == "is_original"
         for c in list(df.columns)[5:]),),
    ("is_original is present and 0/1 only",
     bool("is_original" in df.columns
          and df["is_original"].isin([0, 1]).all())),
    # The audit column must equal what C5 will compute, or it is worse than
    # useless: it would send anyone checking a feature down the wrong path.
    ("audit_rv reproduces a hand-computed window",
     bool(_audit_rv_ok)),
]
for name, ok in checks:
    print(f"    {'OK  ' if ok else 'FAIL'} {name}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise RuntimeError(f"  validation failed: {failed}")


# ============================================================================
# 7. WRITE
# ============================================================================
manifest = pd.DataFrame([
    ("script", "C3_Clean"),
    ("source", F_FILLED),
    ("rows in", n_in),
    ("of which imputed holidays", n_imputed_in),
    ("rows out", len(df)),
    ("bad-price policy", BAD_PRICE),
    ("rows dropped, bad price", n_bad if BAD_PRICE == "drop" else 0),
    ("rows dropped, unreadable date", n_bad_date),
    ("duplicate dates removed", n_dup),
    ("weekend rows removed", n_weekend),
    ("exact-zero returns", n_zero),
    ("exact-zero return dates", ", ".join(str(d) for d in zero_dates[:20]) or "none"),
    ("daily return std", round(float(ret.std()), 6)),
    ("audit columns",
     "audit_log_return, audit_gap, "
     + ", ".join(f"audit_rv_{n}d" for n in rv_windows)
     + " — for checking only; C5 recomputes from the price columns"),
    ("period", f"{df[DATE].min().date()} -> {df[DATE].max().date()}"),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

write_xlsx(F_CLEAN, {"daily": df, "manifest": manifest})

print(f"\n  {len(df)} rows, {df[DATE].min().date()} -> {df[DATE].max().date()}")
print("  NEXT: C4_Cycles")
print("=" * 78)


  C3_Clean — validate the daily file
  read 2008 rows from 01b_daily_filled_filled.xlsx
  of these, 93 row(s) are imputed market holidays
  as loaded: non-positive=0 missing=0 High<Low=0 High/Low not bracketing=0
  continuity OK, largest gap 3 calendar days
  94 day(s) with an exact-zero return (identical consecutive closes): [datetime.date(2019, 3, 4), datetime.date(2019, 4, 17), datetime.date(2019, 4, 19), datetime.date(2019, 4, 29), datetime.date(2019, 5, 1), datetime.date(2019, 6, 5), datetime.date(2019, 8, 12), datetime.date(2019, 9, 2), datetime.date(2019, 9, 10), datetime.date(2019, 10, 2)] ...
  daily return std: 0.01072   (a DAILY figure; if it looks like a VIX level something is wrong)
  📌 STEP 5b: audit columns
     audit_log_return, audit_gap, audit_rv_16d
     audit_rv skips imputed rows and carries forward — same rule as the
     model's realized_vol, so the two agree cell for cell
     these are CHECKING columns — C5 recomputes from Open/High/Low/Close
    OK   all pric

In [6]:
# @title
# ============================================================================
# C4_Cycles — BUILD ONE ROW PER WEEKLY CYCLE
# ============================================================================
#
#   reads  : 02_daily_clean_<raw|filled>.xlsx
#   writes : 03_cycles_<raw|filled>_<lengths>.xlsx
#            sheets "cycles", "lengths", "manifest", and
#            "dropped_expiry_moved" when any cycle was dropped
#
# COLUMNS WRITTEN, AND NOTHING ELSE
# ---------------------------------
#   cycle_id       sequential index, oldest first
#   cycle_days     DECISION days in this cycle: 4 normally
#   is_padded      1 if any decision day in this cycle is an imputed holiday
#   n_imputed      how many of them
#   imputed_slots  which ones, e.g. "D3"
#   d1..d4_date    the decision dates; NaT where a short cycle has no such day
#   d1..d4_close   their closes
#   expiry_date    settlement date — ALWAYS a day that really traded
#   expiry_close   settlement close — THE LABEL'S SOURCE
#
# HOW A CYCLE IS FOUND
# --------------------
# A cycle is the run of days AFTER one expiry up to and including the next. Its
# last day is the expiry; the rest are decision days D1, D2, ...
#
# EXPIRIES ARE FLAGGED ON THE ORIGINAL TRADING DAYS ONLY
# ------------------------------------------------------
# This is the one subtlety in the script and it is worth reading twice.
#
# is_expiry promotes a day to expiry when the NOMINAL expiry weekday is ABSENT
# from the data, reasoning that a missing trading day was a holiday. C2b is
# careful never to fill a nominal expiry weekday, so that reasoning survives —
# but is_expiry also has a two-days-early branch that additionally requires the
# FALLBACK weekday to have been absent, and C2b does fill those. On the current
# file that branch never fires, so nothing changes either way; if it ever fired,
# passing the filled calendar would silently stop an expiry being recognised and
# every cycle boundary after it would move.
#
# So the date set handed to is_expiry is built from is_original == 1 rows. The
# cycles themselves are then cut from the FULL calendar, imputed days included.
# Flags from the real calendar, structure from the complete one.
#
# WHAT IMPUTE_DAILY CHANGED HERE
# ------------------------------
# It deleted code. Three alignment modes used to live in this script — keep the
# short cycle as it is, fabricate a D1 at the front, or fill the missing day
# inside the cycle. All three existed because the daily calendar had holes, and
# C2b now closes them before this script runs, so all three collapsed into one
# path and have been removed. Nothing is fabricated here any more; this script
# only reads what the calendar says.
#
# THE ONE CYCLE THAT STILL CANNOT BE USED
# ---------------------------------------
# When the normal expiry weekday is a holiday the exchange settles a day EARLY.
# The day that is missing is then the expiry itself — the source of the label —
# and C2b deliberately did not fill it. Such a cycle has one fewer step than
# every other, and no imputation can repair it without fabricating the answer,
# so it is DROPPED and listed in its own sheet with the holiday that caused it.
#
# On the current file that is 15 cycles, and after they are gone every remaining
# short cycle has a stated cause: two holidays inside one week. A short cycle with
# no explanation means the calendar model has a hole, so this script counts them
# and the checks fail if any is found.
# ============================================================================

try:
    _ = (F_CLEAN, F_CYCLES, DATE, CLOSE, CAL_TAG, CYCLE_LENGTHS, IMPUTE_DAILY,
         MIN_CYCLE_DAYS, EXPIRY_CUTOFF, is_expiry, read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import datetime

import numpy as np
import pandas as pd

print("\n" + "=" * 78)
print(f"  C4_Cycles — calendar={CAL_TAG}, lengths={CYCLE_LENGTHS}")
print("=" * 78)

MAX_DECISION_DAYS = 4       # D1..D4; a longer run means a missed expiry
WEEKDAY_NAME = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri"}

# Under "full_only" a cycle must have the full four decision days. Under "all"
# anything from MIN_CYCLE_DAYS upwards is kept.
MIN_DAYS_REQUIRED = (MAX_DECISION_DAYS if CYCLE_LENGTHS == "full_only"
                     else MIN_CYCLE_DAYS)


# ============================================================================
# 1. LOAD, AND FLAG EXPIRIES ON THE REAL CALENDAR
# ============================================================================
daily = read_xlsx(F_CLEAN, sheet="daily")
daily[DATE] = pd.to_datetime(daily[DATE]).dt.normalize()
daily = daily.sort_values(DATE).reset_index(drop=True)
if "is_original" not in daily.columns:
    daily["is_original"] = 1
daily["is_original"] = pd.to_numeric(
    daily["is_original"], errors="coerce").fillna(1).astype(int)

n_imputed_days = int((daily["is_original"] == 0).sum())
print(f"  {len(daily)} calendar days, {daily[DATE].min().date()} -> "
      f"{daily[DATE].max().date()}")
print(f"  of these, {n_imputed_days} are imputed holidays "
      f"({len(daily) - n_imputed_days} really traded)")

# Flags from the ORIGINAL calendar — see the header.
original_dates = set(daily.loc[daily["is_original"] == 1, DATE])
last_original = daily.loc[daily["is_original"] == 1, DATE].max()
flag = np.array([is_expiry(d, original_dates, last_original)
                 and d in original_dates
                 for d in daily[DATE]], int)
expiry_pos = np.flatnonzero(flag == 1)
print(f"  {len(expiry_pos)} expiries, {daily[DATE][expiry_pos[0]].date()} -> "
      f"{daily[DATE][expiry_pos[-1]].date()}")
print("  expiry weekdays : "
      + ", ".join(f"{WEEKDAY_NAME[k]} {v}" for k, v in
                  daily[DATE].iloc[expiry_pos].dt.weekday
                  .value_counts().sort_index().items()))


# ============================================================================
# 1b. WHICH EXPIRIES SETTLED EARLY BECAUSE THE NORMAL DAY WAS A HOLIDAY
# ----------------------------------------------------------------------------
# The test is simply whether the expiry landed on its regime's nominal weekday.
# C2b uses the same EXPIRY_CUTOFF for the same purpose, so the two cannot drift:
# any day C2b refused to fill as "the expiry weekday" is the day this section
# looks for.
#
# No trailing-mode estimate is needed. An earlier version of this logic learned
# the prevailing weekday from recent expiries, which worked but introduced a
# second, independent definition of "expiry weekday" into a pipeline that already
# had one in is_expiry. One definition is worth more than a clever one.
# ============================================================================
def nominal_expiry_weekday(when):
    """The weekday settlement normally falls on for the regime containing `when`.

    Parameters
    ----------
    when : pandas.Timestamp

    Returns
    -------
    int
        Monday is 0. Thursday (3) before EXPIRY_CUTOFF, Tuesday (1) from it on.
    """
    return 1 if when >= EXPIRY_CUTOFF else 3


def settled_early(expiry_date):
    """Did this expiry move off its normal weekday because that day was shut?

    Parameters
    ----------
    expiry_date : pandas.Timestamp

    Returns
    -------
    (bool, str)
        Whether it moved, and a short reason naming the holiday.
    """
    nominal = nominal_expiry_weekday(expiry_date)
    if expiry_date.weekday() == nominal:
        return False, ""
    candidate = expiry_date + pd.Timedelta(days=(nominal - expiry_date.weekday()) % 7)
    return True, (f"settled {WEEKDAY_NAME[expiry_date.weekday()]} because "
                  f"{WEEKDAY_NAME[nominal]} {candidate.date()} was a holiday")


# ============================================================================
# 2. BUILD THE CYCLES
# ----------------------------------------------------------------------------
# Each cycle is the slice AFTER the previous expiry through the next one. The
# first expiry starts no cycle, because the days before it belong to a cycle whose
# own start is off the front of the file.
# ============================================================================
rows, length_counts, dropped_shift = [], [], []
n_short_unexplained = 0

for prev_end, end in zip(expiry_pos[:-1], expiry_pos[1:]):
    block = daily.iloc[prev_end + 1:end + 1].reset_index(drop=True)
    if len(block) < 2:
        continue

    decision = block.iloc[:-1]           # everything except the expiry itself
    expiry = block.iloc[-1]
    n_dec = len(decision)
    length_counts.append(n_dec)

    # An early settlement cannot be repaired: the missing day IS the label.
    moved, why = settled_early(expiry[DATE])
    if moved:
        dropped_shift.append({
            "expiry_date": expiry[DATE],
            "expiry_weekday": WEEKDAY_NAME[expiry[DATE].weekday()],
            "normal_weekday": WEEKDAY_NAME[nominal_expiry_weekday(expiry[DATE])],
            "decision_days": n_dec,
            "reason": why,
        })
        continue

    if n_dec < MIN_DAYS_REQUIRED or n_dec > MAX_DECISION_DAYS:
        continue                         # too short to use, or a missed expiry

    # Any cycle still short of four days, on a filled calendar and with early
    # settlements already removed, had two or more holidays in the same week.
    if IMPUTE_DAILY and n_dec < MAX_DECISION_DAYS:
        n_short_unexplained += 1

    dates = list(decision[DATE])
    closes = list(pd.to_numeric(decision[CLOSE], errors="coerce"))
    orig = list(pd.to_numeric(decision["is_original"], errors="coerce").astype(int))

    slots = [f"D{i+1}" for i, o in enumerate(orig) if o == 0]
    row = {
        "cycle_days": len(dates),
        "is_padded": 1 if slots else 0,
        "n_imputed": len(slots),
        "imputed_slots": ", ".join(slots),
    }
    for i in range(MAX_DECISION_DAYS):
        row[f"d{i+1}_date"] = dates[i] if i < len(dates) else pd.NaT
        row[f"d{i+1}_close"] = closes[i] if i < len(closes) else np.nan
    row["expiry_date"] = expiry[DATE]
    row["expiry_close"] = float(expiry[CLOSE])
    rows.append(row)

cycles = pd.DataFrame(rows).sort_values("expiry_date").reset_index(drop=True)
cycles.insert(0, "cycle_id", np.arange(1, len(cycles) + 1))

ORDER = (["cycle_id", "cycle_days", "is_padded", "n_imputed", "imputed_slots"]
         + [f"d{i}_{w}" for i in (1, 2, 3, 4) for w in ("date", "close")]
         + ["expiry_date", "expiry_close"])
cycles = cycles[ORDER]


# ============================================================================
# 3. REPORT
# ============================================================================
lengths = (pd.Series(length_counts).value_counts().sort_index()
           .rename_axis("decision_days").reset_index(name="cycles_seen"))
# This table counts what was SEEN, before any cycle was dropped. On a filled
# calendar a 3-day cycle is almost always an early settlement, which is dropped in
# section 2 — so saying "kept" here would contradict the usable count below it.
_n_short_seen = int(sum(1 for k in length_counts if k == 3))
lengths["meaning"] = lengths["decision_days"].map(
    lambda k: "standard, D1..D4" if k == 4
    else ("short — settled early, dropped" if IMPUTE_DAILY
          else ("short — DROPPED (CYCLE_LENGTHS='full_only')"
                if CYCLE_LENGTHS == "full_only" else "short holiday week — kept")) if k == 3
    else "too short, dropped" if k < 3
    else "LONG — a missed expiry, dropped")
print("\n  cycle lengths found in the data")
print(lengths.to_string(index=False))

n_touched = int((cycles["n_imputed"] > 0).sum())
print(f"\n  usable cycles : {len(cycles)}")
print(f"  by cycle_days : "
      f"{cycles['cycle_days'].value_counts().sort_index().to_dict()}")
print(f"  dropped       : {len(dropped_shift)} cycle(s) that settled early "
      f"because the normal")
print(f"                  expiry weekday was a holiday. The missing day there IS")
print(f"                  the expiry, and the expiry is the label — no imputation")
print(f"                  can repair it. Listed in 'dropped_expiry_moved'.")
if IMPUTE_DAILY:
    print(f"  with holidays : {n_touched} cycle(s) contain an imputed decision day")
    if n_touched:
        print(f"                  slot(s) : "
              f"{cycles.loc[cycles['n_imputed'] > 0, 'imputed_slots'].value_counts().to_dict()}")
    print(f"  still short   : {n_short_unexplained} cycle(s) below 4 days — these")
    print(f"                  had two or more holidays in one week")
else:
    print(f"  IMPUTE_DAILY is off, so holiday weeks stay 3 decision days long")

for day in (2, 3, 4):
    n = int((cycles["cycle_days"] >= day).sum())
    print(f"  rows for the D{day} tasks : {n}")


# ============================================================================
# 4. CHECKS
# ============================================================================
checks = []


def chk(name, ok, detail=""):
    checks.append((name, bool(ok), detail))


chk("at least one cycle was built", len(cycles) > 0)
chk("cycle_id is unique", cycles["cycle_id"].is_unique)
chk("expiry_date is unique", cycles["expiry_date"].is_unique)
chk("expiry dates strictly increase", cycles["expiry_date"].is_monotonic_increasing)
chk("every expiry_close is positive", bool((cycles["expiry_close"] > 0).all()))
if CYCLE_LENGTHS == "full_only":
    chk("every cycle has the full four decision days",
        bool((cycles["cycle_days"] == 4).all()),
        str(sorted(cycles["cycle_days"].unique())))
else:
    chk("cycle_days is 3 or 4 everywhere",
        bool(cycles["cycle_days"].isin([3, 4]).all()),
        str(sorted(cycles["cycle_days"].unique())))

# d1..d{cycle_days} must be present, and later days absent. This is what keeps a
# short cycle out of the D4 task instead of letting a blank through as a zero.
present_ok = True
for _, r in cycles.iterrows():
    k = int(r["cycle_days"])
    if any(pd.isna(r[f"d{i}_date"]) for i in range(1, k + 1)):
        present_ok = False
        break
    if any(not pd.isna(r[f"d{i}_date"]) for i in range(k + 1, 5)):
        present_ok = False
        break
chk("exactly cycle_days decision days are filled", present_ok)

# Decision days must be CONSECUTIVE calendar rows ending just before expiry. On a
# filled calendar this is now exact — there are no repeated dates to allow for,
# which is precisely what the old is_padded exemption used to hide.
pos_of = {d: i for i, d in enumerate(daily[DATE])}
consecutive_ok = True
for _, r in cycles.iterrows():
    k = int(r["cycle_days"])
    seq = [pos_of.get(r[f"d{i}_date"]) for i in range(1, k + 1)]
    seq += [pos_of.get(r["expiry_date"])]
    if any(s is None for s in seq) or not (np.diff(seq) == 1).all():
        consecutive_ok = False
        break
chk("decision days are consecutive rows ending at expiry", consecutive_ok)

# Every date written must exist in the daily file, or C5's lookups return NaN and
# the cycle disappears from every task sheet without a word.
dates_exist = all(
    r[f"d{i}_date"] in pos_of
    for _, r in cycles.iterrows()
    for i in range(1, int(r["cycle_days"]) + 1))
chk("every decision date exists in the daily file", dates_exist)

# THE EXPIRY IS NEVER IMPUTED. It is the label's source; if one were fabricated
# the study would be predicting a number nobody observed.
imputed_dates = set(daily.loc[daily["is_original"] == 0, DATE])
chk("no expiry_date is an imputed day",
    not (set(cycles["expiry_date"]) & imputed_dates))

# Every expiry landed on its regime's nominal weekday, because the ones that did
# not were dropped.
off_weekday = [d for d in cycles["expiry_date"]
               if d.weekday() != nominal_expiry_weekday(d)]
chk("every surviving expiry is on its regime's normal weekday", not off_weekday,
    str([str(d.date()) for d in off_weekday[:5]]))

dropped_dates = {r["expiry_date"] for r in dropped_shift}
chk("no surviving cycle ends on an early settlement",
    not (set(cycles["expiry_date"]) & dropped_dates))

if IMPUTE_DAILY and CYCLE_LENGTHS == "all":
    # On a filled calendar, with early settlements removed, a cycle can only be
    # short because two holidays fell in one week. If the count is non-zero the
    # cycles are still usable for D2 and D3, but the number belongs on screen
    # rather than buried, because it is the only remaining irregularity.
    chk("every cycle is 4 days, or short for a stated reason",
        int((cycles["cycle_days"] < 4).sum()) == n_short_unexplained,
        f"{int((cycles['cycle_days'] < 4).sum())} short vs "
        f"{n_short_unexplained} accounted for")

    # The point of filling the calendar: the D4 task should now see nearly every
    # cycle. If it does not, C2b did not do its job and this run is not the study
    # it claims to be.
    share = (int((cycles["cycle_days"] >= 4).sum()) / len(cycles)
             if len(cycles) else 0.0)
    chk("the D4 task sees at least 95% of cycles", share >= 0.95,
        f"{share:.1%}")

print("\n  checks")
for name, ok, detail in checks:
    print(f"    {'OK  ' if ok else 'FAIL'} {name}" + ("" if ok else f"  -- {detail}"))
bad = [n for n, ok, _ in checks if not ok]
if bad:
    raise RuntimeError(f"  C4 checks failed: {bad}")


# ============================================================================
# 5. WRITE
# ============================================================================
manifest = pd.DataFrame([
    ("script", "C4_Cycles"),
    ("source", F_CLEAN),
    ("daily_calendar", CAL_TAG),
    ("impute_daily", IMPUTE_DAILY),
    ("cycle_lengths", CYCLE_LENGTHS),
    ("calendar days read", len(daily)),
    ("of which imputed", n_imputed_days),
    ("expiries flagged", len(expiry_pos)),
    ("cycles usable", len(cycles)),
    ("cycles containing an imputed day", n_touched),
    ("cycles dropped — settled early", len(dropped_shift)),
    ("cycles still short — 2+ holidays in a week", n_short_unexplained),
    ("cycles with 4 decision days", int((cycles["cycle_days"] == 4).sum())),
    ("cycles with 3 decision days", int((cycles["cycle_days"] == 3).sum())),
    ("rows for D2 / D3 / D4 tasks",
     " / ".join(str(int((cycles["cycle_days"] >= d).sum())) for d in (2, 3, 4))),
    ("period", f"{cycles['expiry_date'].min().date()} -> "
               f"{cycles['expiry_date'].max().date()}"),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

# The dropped cycles get their own sheet rather than vanishing. A row set that
# quietly excludes 15 weeks is not auditable; one that names them, with the
# holiday behind each, is.
sheets_out = {"cycles": cycles, "lengths": lengths, "manifest": manifest}
if dropped_shift:
    sheets_out["dropped_expiry_moved"] = pd.DataFrame(dropped_shift)

write_xlsx(F_CYCLES, sheets_out)

print(f"\n  {len(cycles)} cycles, {cycles['expiry_date'].min().date()} -> "
      f"{cycles['expiry_date'].max().date()}")
print("  NEXT: C4b_BandAudit, then C5_Features")
print("=" * 78)


  C4_Cycles — calendar=filled, lengths=full_only
  2008 calendar days, 2019-01-01 -> 2026-10-05
  of these, 93 are imputed holidays (1915 really traded)
  405 expiries, 2019-01-03 -> 2026-09-29
  expiry weekdays : Mon 3, Tue 54, Wed 14, Thu 334

  cycle lengths found in the data
 decision_days  cycles_seen                        meaning
             2            1             too short, dropped
             3           17 short — settled early, dropped
             4          386               standard, D1..D4

  usable cycles : 386
  by cycle_days : {4: 386}
  dropped       : 17 cycle(s) that settled early because the normal
                  expiry weekday was a holiday. The missing day there IS
                  the expiry, and the expiry is the label — no imputation
                  can repair it. Listed in 'dropped_expiry_moved'.
  with holidays : 87 cycle(s) contain an imputed decision day
                  slot(s) : {'D2': 24, 'D1': 22, 'D4': 20, 'D3': 18, 'D2, D4': 1, 'D1, D3

In [8]:
# @title
# ============================================================================
# C5_Features — BANDS, LABELS AND THE SIX TASK SHEETS
# ============================================================================
#
#   reads  : 02_daily_clean.xlsx, 03_cycles_<align>.xlsx
#   writes : 04_features_sigma<k>_win<W>_<align>.xlsx
#            ONE WORKBOOK PER (sigma, window) IN THE GRIDS
#
# WHY ONE FILE PER SIGMA
# ----------------------
# The band defines the label. A different k means different breach outcomes and
# a different base rate, so σ=0.5 and σ=1.0 are not two settings of one study —
# they are two studies. Separate files stop them being pooled into a single
# leaderboard by accident, and the configuration is in the filename so nothing
# can be mistaken for something else.
#
# WORKBOOK LAYOUT
# ---------------
#   D2_upper ... D4_lower   six model-ready sheets, one row per usable cycle:
#                           identifiers | features | breach
#   bands                   every cycle with mu, sigma, both edges and both
#                           labels — the audit trail for the target
#   summary                 rows, features used, breaches, base rate, EPV
#   formulas                every derived column with its exact formula
#   manifest                the configuration that produced the workbook
#
# READ THIS ABOUT THE TARGET
# --------------------------
# `breach` is ONE-VS-REST for that side. On D3_upper, breach = 0 covers both
# "settled inside the band" AND "breached the LOWER band" — different events with
# the same label. It also means upper_D2, upper_D3 and upper_D4 share an
# IDENTICAL breach column and differ only in features, because a breach is a
# property of the cycle, not of the decision day. For multiple-testing
# correction there are about two independent families here, not six.
#
# NOTHING IS TRAINED HERE. This script builds what the models will see and writes
# it down, so the inputs can be inspected in Excel before any fitting happens.
# ============================================================================

try:
    _ = (F_CLEAN, F_CYCLES, f_features, SIGMA_GRID, SIGMA_WINDOW_GRID,
         CAL_TAG, CYCLE_LENGTHS, FEATURE_SET, DROP_CONSTANT_FEATURES, RUN_TAG,
         rv_window_days, F_MACRO, MACRO_SOURCE_COLS, MACRO_FEATURE_SETS,
         TASKS, add_bands_and_labels, drop_warmup, add_features,
         task_features, task_rows, gaussian_breach_prob, read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import datetime

import os

import numpy as np
import pandas as pd

print("\n" + "=" * 78)
print(f"  C5_Features — sigmas {SIGMA_GRID}, windows {SIGMA_WINDOW_GRID}, "
      f"tag={RUN_TAG}")
print("=" * 78)

IDENT = ["cycle_id", "expiry_date", "cycle_days", "is_padded"]

daily = read_xlsx(F_CLEAN, sheet="daily")


_macro_cols, _macro_report = [], []
if os.path.exists(F_MACRO):
    _mac = read_xlsx(F_MACRO, sheet="macro")
    _mac[DATE] = pd.to_datetime(_mac[DATE]).dt.normalize()

    # (1) THE DANGEROUS ONE. A left join on a non-unique key does not add
    # columns, it MULTIPLIES ROWS. Keep the LAST row per date, which is the
    # later-published value if C2c ever appended a revision.
    _n_dupe = int(_mac[DATE].duplicated().sum())
    if _n_dupe:
        _mac = _mac.sort_values(DATE).drop_duplicates(subset=[DATE], keep="last")
        _macro_report.append(f"{_n_dupe} duplicate date(s) collapsed")
    _mac = _mac[_mac[DATE].notna()].sort_values(DATE)

    _macro_cols = [c for c in MACRO_SOURCE_COLS if c in _mac.columns]
    if _macro_cols:
        for c in _macro_cols:
            _mac[c] = pd.to_numeric(_mac[c], errors="coerce")

        # (2) A LEVEL of zero or less is a feed sentinel, not a price. Only VIX
        # is a level here; the ln columns are changes and may be negative.
        for c in [c for c in ("india_vix", "vix_daily") if c in _macro_cols]:
            _bad = int((_mac[c] <= 0).sum())
            if _bad:
                _mac.loc[_mac[c] <= 0, c] = np.nan
                _macro_report.append(f"{_bad} non-positive {c} -> NaN")

        # (3) REPORTED, NEVER CLIPPED. A 25 per cent one-day move in the rupee
        # or in Brent is what a crisis week looks like. Winsorising it away
        # would delete the only signal a macro feature set could plausibly
        # carry. March 2020 is data, not an outlier.
        for c in [c for c in _macro_cols if c.endswith(("_ln1", "_ln5"))]:
            _ext = _mac.loc[_mac[c].abs() > 0.25, [DATE, c]]
            if len(_ext):
                _macro_report.append(
                    f"{len(_ext)} extreme {c} (|ln|>0.25) KEPT — inspect: "
                    + ", ".join(f"{d.date()}:{v:+.3f}"
                                for d, v in _ext.head(5).itertuples(index=False)))

        daily[DATE] = pd.to_datetime(daily[DATE]).dt.normalize()
        _n_before = len(daily)
        daily = daily.merge(_mac[[DATE] + _macro_cols], on=DATE, how="left")
        if len(daily) != _n_before:
            raise RuntimeError(
                f"  the macro join changed the row count: {_n_before} -> "
                f"{len(daily)}.\n"
                f"  {os.path.basename(F_MACRO)} still has duplicate dates, "
                f"which corrupts every\n"
                f"  cycle count downstream. Re-run C2c_FetchMacro.")

        _cov = {c: f"{100.0 * daily[c].notna().mean():.0f}%" for c in _macro_cols}
        print(f"  macro joined from {os.path.basename(F_MACRO)}: {_cov}")
        for _m in _macro_report:
            print(f"    macro cleaning: {_m}")

        # (4) COVERAGE IS A COMPARABILITY PROBLEM, not only a quality one. Each
        # gap becomes a dropped cycle, so a macro feature set gets scored on a
        # SMALLER sample than `full` and the two columns of the results table
        # stop being answers about the same cycles.
        _worst = min(daily[c].notna().mean() for c in _macro_cols)
        if FEATURE_SET in MACRO_FEATURE_SETS:
            if _worst < 0.95:
                raise RuntimeError(
                    f"  macro coverage is only {100 * _worst:.1f}% of the "
                    f"trading calendar.\n"
                    f"  {FEATURE_SET!r} would be scored on a materially "
                    f"smaller sample than the\n"
                    f"  non-macro sets, so the two could not be compared. "
                    f"Forward-fill the gaps\n"
                    f"  across holidays in C2c first.")
            if _worst < 0.99:
                print(f"    WARN macro coverage {100 * _worst:.1f}% — compare "
                      f"the `rows` column in the summary sheet below against "
                      f"a non-macro run before pooling the two.")
    else:
        print(f"  macro file found but none of {MACRO_SOURCE_COLS} are in it")
else:
    print(f"  no macro file at {os.path.basename(F_MACRO)} — "
          f"vix/usdinr/brent features will NOT be built")

if FEATURE_SET in MACRO_FEATURE_SETS and not _macro_cols:
    raise RuntimeError(
        f"  FEATURE_SET={FEATURE_SET!r} needs the macro file, and\n"
        f"  {F_MACRO}\n"
        f"  is missing or empty. Run C2c_FetchMacro first.\n"
        f"  Refusing rather than quietly falling back to the non-macro\n"
        f"  parent feature set — that is exactly the failure this check\n"
        f"  exists to stop.")

cycles_in = read_xlsx(F_CYCLES, sheet="cycles")
print(f"  {len(daily)} trading days, {len(cycles_in)} cycles")


def formula_sheet(sigma, window):
    """Every derived column, its formula, and whether (sigma, window) moves it.

    Parameters
    ----------
    sigma : float
        Band multiplier k.
    window : int
        Rolling window in cycles.

    Returns
    -------
    pandas.DataFrame
        Columns: column, role, formula, notes, config_dependent.

    Notes
    -----
    Written per workbook rather than kept in a separate document, because a
    formula sheet that lives apart from the numbers goes stale without anyone
    noticing. Here it is regenerated from the same constants that built the data.
    """
    w4 = rv_window_days(window)
    rows = [
        ("cycle_id", "identifier", "—",
         "sequential index of the weekly cycle", "no"),
        ("expiry_date", "identifier", "—", "settlement date", "no"),
        ("cycle_days", "identifier", "trading days in the cycle minus the expiry",
         "4 normally, 3 in a holiday week. Drives days_left", "no"),
        ("is_padded", "identifier",
         "1 if any decision day in this cycle was an imputed market holiday",
         f"0 everywhere when IMPUTE_DAILY is off. This run: {CAL_TAG}", "no"),
        ("mu_step", "intermediate",
         f"rolling mean over {window} cycles of ln(expiry/d1) / cycle_days, "
         f"lagged 1 cycle",
         "drift PER STEP, so cycles of different lengths can be pooled", "YES"),
        ("sigma_step", "intermediate",
         f"rolling std over {window} cycles of ln(expiry/d1) / sqrt(cycle_days), "
         f"lagged 1 cycle",
         "volatility PER STEP. The lag is what stops a cycle's own outcome from "
         "setting its own band", "YES"),
        ("mu", "intermediate", "mu_step * cycle_days",
         "the per-step drift scaled onto this cycle's own horizon", "YES"),
        ("sigma", "intermediate", "sigma_step * sqrt(cycle_days)",
         "the per-step volatility scaled onto this cycle's horizon", "YES"),
        ("band_upper", "band", f"d1_close * exp(mu + {sigma} * sigma)",
         "the upper edge, fixed at D1's close for the whole cycle", "YES"),
        ("band_lower", "band", f"d1_close * exp(mu - {sigma} * sigma)",
         "the lower edge", "YES"),
        ("band_width_pct", "FEATURE", "ln(band_upper / band_lower)",
         f"identically 2 * {sigma} * sigma. Legal as a feature only because each "
         f"sheet carries ONE side: ln(U/C) - ln(L/C) = ln(U/L) exactly, so a "
         f"sheet holding both distances plus this column would be "
         f"rank-deficient", "YES"),
        ("days_left_dec", "FEATURE", "cycle_days + 1 - decision_day",
         "steps from the decision day's close to expiry. Varies across rows only "
         "because short cycles are kept; under pad_d1 it is constant and carries "
         "no information", "no"),
        ("realized_vol_Dn", "FEATURE",
         f"std of DAILY ln returns over the last {w4} trading days "
         f"(= window x 4), read on d{{n}}_date and INCLUDING that day's return",
         "short-horizon DAILY volatility. A different quantity from sigma, which "
         "is a per-step volatility estimated ACROSS cycles. No lag: the decision "
         "is taken minutes before that day's close, so the day's own return is "
         "already known", "YES"),
        ("gap_Dn", "FEATURE", "ln(Open on d{n}_date / d{n-1}_close)",
         "the overnight jump into that day. For D1 there is no D0, so the "
         "previous TRADING day is used — the prior cycle's expiry close", "no"),
        ("z_dist_upper_Dn", "FEATURE (upper sheets)",
         "ln(band_upper / d{n}_close) / (realized_vol_Dn * sqrt(days_left_n))",
         "how many standard deviations of the REMAINING move the upper edge sits "
         "away. 1 - Phi(z) is exactly the Gaussian baseline's breach "
         "probability, so this is the baseline's own statistic", "YES"),
        ("z_dist_lower_Dn", "FEATURE (lower sheets)",
         "ln(band_lower / d{n}_close) / (realized_vol_Dn * sqrt(days_left_n))",
         "the same measured down to the lower edge. Negative whenever price sits "
         "above the lower edge, which is the usual case", "YES"),
        ("gauss_prob", "BASELINE", "1 - Phi(z) for upper, Phi(z) for lower, "
         "with z = ln(band / d{n}_close) / (sigma_step * sqrt(days_left))",
         "the analytical contestant's answer. No free parameters, so it is "
         "computed here rather than trained. NOT a feature — never hand it to a "
         "model", "YES"),
        ("breach", "TARGET", "1 if the expiry close is outside THIS side's edge",
         "one-vs-rest: 0 includes cycles that breached the OTHER side", "YES"),
    ]
    return pd.DataFrame(rows, columns=["column", "role", "formula", "notes",
                                       "config_dependent"])


# ============================================================================
# ONE WORKBOOK PER CONFIGURATION
# ============================================================================
written = []
for sigma in SIGMA_GRID:
    for window in SIGMA_WINDOW_GRID:
        print("\n" + "-" * 78)
        print(f"  sigma={sigma}  window={window}")
        print("-" * 78)

        banded = add_bands_and_labels(cycles_in, sigma, window)
        banded = drop_warmup(banded, verbose=True)
        feat = add_features(banded, daily, window)

        # --- the six task sheets ---------------------------------------------
        sheets, summary = {}, []
        for task in TASKS:
            day, side, label = task["day"], task["side"], task["label"]
            cols = task_features(day, side)
            rows = task_rows(feat, day)

            # Drop any feature with no variation on THIS task's rows. Such a
            # column is all-zeros after scaling: absorbed by the intercept in a
            # linear model, unsplittable by a tree. Removing it changes no
            # prediction and makes the reported feature count honest.
            # days_left_dec is the usual one — constant at D4 because only full
            # cycles reach D4, and constant everywhere under pad_d1.
            if DROP_CONSTANT_FEATURES:
                flat = [c for c in cols
                        if pd.to_numeric(rows[c], errors="coerce").nunique(
                            dropna=True) <= 1]
                if flat:
                    print(f"    D{day}_{side}: dropping constant feature(s) {flat}")
                    cols = [c for c in cols if c not in flat]

            # The Gaussian's probability is computed HERE, not in C6.
            #
            # It has no free parameters, so there is nothing to fit and nothing
            # to leak: it reads this side's band, the decision day's close,
            # sigma_step and days_left, all known at the moment of decision. Two
            # things are gained by writing it into the sheet. The baseline becomes
            # visible in Excel beside the features, and a row survives only when
            # BOTH contestants can score it, so they are compared on provably
            # identical rows rather than on two sets that happen to overlap.
            gauss = gaussian_breach_prob(rows, day, side)

            usable = (rows[label].notna()
                      & np.isfinite(rows[cols]).all(axis=1)
                      & np.isfinite(pd.Series(gauss, index=rows.index)))
            n_drop = int((~usable).sum())
            gauss = np.asarray(gauss)[usable.to_numpy()]
            rows = rows[usable].reset_index(drop=True)

            sheet = rows[[c for c in IDENT if c in rows.columns]].copy()
            for c in cols:
                sheet[c] = rows[c].values
            sheet["gauss_prob"] = gauss
            sheet["breach"] = rows[label].astype(int).values
            name = f"D{day}_{side}"
            sheets[name] = sheet

            n_breach = int(sheet["breach"].sum())
            summary.append({
                "sheet": name, "day": day, "side": side,
                "rows": len(sheet), "dropped_nonfinite": n_drop,
                "features": len(cols),
                "feature_names": ", ".join(cols),
                "breaches": n_breach,
                "base_rate": round(n_breach / len(sheet), 4) if len(sheet) else np.nan,
                # Events per variable. Below about 10 a fitted coefficient is not
                # an estimate of anything, so this number belongs beside every
                # result rather than in a footnote.
                "EPV": round(n_breach / len(cols), 2) if len(cols) else np.nan,
                "days_left_values":
                    ", ".join(str(int(v)) for v in
                              sorted(sheet["days_left_dec"].unique()))
                    if "days_left_dec" in sheet.columns else "",
            })

        summary = pd.DataFrame(summary)
        print(summary.to_string(index=False))

        # --- the bands sheet: the audit trail for the label -------------------
        band_cols = (IDENT + ["d1_close", "expiry_close", "mu_step", "sigma_step",
                              "mu", "sigma", "band_upper", "band_lower",
                              "band_width_pct", "upper_breach", "lower_breach"])
        bands = feat[[c for c in band_cols if c in feat.columns]].copy()

        # --- checks -----------------------------------------------------------
        fails = []

        def chk(name, ok, detail=""):
            if not ok:
                fails.append(f"{name}: {detail}")
            print(f"    {'OK  ' if ok else 'FAIL'} {name}"
                  + ("" if ok else f"  -- {detail}"))

        print("\n  checks")
        ok_band = bands["sigma"].notna()
        chk("band_width_pct == 2*k*sigma",
            np.allclose(bands.loc[ok_band, "band_width_pct"],
                        2.0 * sigma * bands.loc[ok_band, "sigma"]))
        chk("band_upper is above band_lower everywhere",
            bool((bands["band_upper"] > bands["band_lower"]).all()))
        chk("no label is missing on a usable row",
            bool(bands[["upper_breach", "lower_breach"]].notna().all().all()))
        chk("upper and lower breaches never coincide",
            int(((bands["upper_breach"] == 1) & (bands["lower_breach"] == 1)).sum()) == 0,
            "a cycle cannot settle above the upper edge and below the lower one")

        # The three upper sheets must carry the SAME label for the same cycle. If
        # they do not, the day sheets have drifted apart and any cross-day
        # comparison is comparing different targets.
        for side in ("upper", "lower"):
            ref = sheets[f"D4_{side}"].set_index("cycle_id")["breach"]
            same = True
            for day in (2, 3):
                s = sheets[f"D{day}_{side}"].set_index("cycle_id")["breach"]
                shared = ref.index.intersection(s.index)
                same &= bool((ref.loc[shared] == s.loc[shared]).all())
            chk(f"the {side} label agrees across D2, D3 and D4", same)

        NON_FEATURE = IDENT + ["breach", "gauss_prob"]
        chk("every feature value is finite",
            all(bool(np.isfinite(s[[c for c in s.columns
                                    if c not in NON_FEATURE]]).all().all())
                for s in sheets.values()))
        chk("the Gaussian scored every row that survived",
            all(bool(np.isfinite(s["gauss_prob"]).all()) for s in sheets.values()),
            "otherwise a row is scored for ML and not for the baseline")
        # [0, 1] INCLUSIVE, deliberately. Phi saturates in double precision once
        # |z| passes about 8, so a cycle sitting very far from its band returns
        # exactly 0.0 or 1.0. That is arithmetic, not a bug — but it is worth
        # counting, because a saturated probability makes log-loss infinite and
        # means the baseline is expressing certainty.
        chk("the Gaussian's probabilities lie within [0, 1]",
            all(bool(((s["gauss_prob"] >= 0) & (s["gauss_prob"] <= 1)).all())
                for s in sheets.values()))
        n_sat = sum(int(((s["gauss_prob"] <= 0) | (s["gauss_prob"] >= 1)).sum())
                    for s in sheets.values())
        print(f"    note  {n_sat} row(s) across all sheets where the Gaussian "
              f"saturates at 0 or 1")
        chk("no sheet leaks the opposite side",
            all(not any(("lower" if "upper" in n else "upper") in c
                        for c in s.columns)
                for n, s in sheets.items()))
        chk("no sheet carries a later day than its own",
            all(not any(f"_D{later}" in c
                        for c in s.columns
                        for later in range(int(n.split('_')[0][1:]) + 1, 5))
                for n, s in sheets.items()))
        chk("expiry_close is absent from every task sheet",
            all("expiry_close" not in s.columns for s in sheets.values()),
            "that column IS the answer")

        if fails:
            raise RuntimeError(f"  C5 checks failed at sigma={sigma}, "
                               f"window={window}:\n    - " + "\n    - ".join(fails))

        # --- write ------------------------------------------------------------
        manifest = pd.DataFrame([
            ("script", "C5_Features"),
            ("sigma (k)", sigma),
            ("window (cycles)", window),
            ("realized_vol window (days)", rv_window_days(window)),
            ("daily_calendar", CAL_TAG),
            ("cycle_lengths", CYCLE_LENGTHS),
            ("feature_set", FEATURE_SET),
            ("constant features dropped", DROP_CONSTANT_FEATURES),
            ("run tag", RUN_TAG),
            ("cycles in", len(cycles_in)),
            ("cycles after warm-up", len(feat)),
            ("cycles padded", int(feat["is_padded"].sum())),
            ("features D2 / D3 / D4 (after dropping constants)",
             " / ".join(str(int(summary.loc[summary["day"] == d, "features"].iloc[0]))
                        for d in (2, 3, 4))),
            ("base rate upper D4",
             round(float(sheets["D4_upper"]["breach"].mean()), 4)),
            ("base rate lower D4",
             round(float(sheets["D4_lower"]["breach"].mean()), 4)),
            ("Gaussian saturated rows", n_sat),
            ("period", f"{feat['expiry_date'].min().date()} -> "
                       f"{feat['expiry_date'].max().date()}"),
            ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
        ], columns=["key", "value"])

        out = {**{f"D{d}_{s}": sheets[f"D{d}_{s}"]
                  for d in (2, 3, 4) for s in ("upper", "lower")},
               "bands": bands, "summary": summary,
               "formulas": formula_sheet(sigma, window), "manifest": manifest}
        path = f_features(sigma, window)
        write_xlsx(path, out)
        written.append(path)


# ============================================================================
# DONE
# ============================================================================
print("\n" + "=" * 78)
print("  C5_Features complete")
for p in written:
    print(f"    {p.split('/')[-1]}")
print("  Each workbook is ONE study: a different sigma means a different label")
print("  and a different base rate, so report them separately.")
print("  NEXT: C6_TrainPredict")
print("=" * 78)


  C5_Features — sigmas [0.1, 0.25, 0.5], windows [4], tag=filled_full_only_compact_plus_macro
  macro joined from 01c_macro_daily_filled.xlsx: {'india_vix': '100%', 'vix_daily': '100%', 'usdinr_ln1': '100%', 'usdinr_ln5': '100%', 'brent_ln1': '100%', 'brent_ln5': '100%'}
    macro cleaning: 5 extreme brent_ln1 (|ln|>0.25) KEPT — inspect: 2020-03-09:-0.255, 2020-03-31:-0.256, 2020-04-03:+0.486, 2020-04-21:-0.644, 2020-04-22:+0.412
    macro cleaning: 25 extreme brent_ln5 (|ln|>0.25) KEPT — inspect: 2020-03-09:-0.396, 2020-03-10:-0.384, 2020-03-11:-0.409, 2020-03-12:-0.503, 2020-03-13:-0.346
  2008 trading days, 386 cycles

------------------------------------------------------------------------------
  sigma=0.1  window=4
------------------------------------------------------------------------------
  warm-up: dropped 4 cycle(s) with undefined mu/sigma
   sheet  day  side  rows  dropped_nonfinite  features                                                                                 

In [9]:
# @title
# ============================================================================
# C8_Live_MultiConfig — ONE RUN, EVERY SIGMA, EACH WITH ITS OWN CONFIG
# ============================================================================
#
#   reads  : 02_daily_clean_<cal>.xlsx, 03_cycles_<cal>_<lengths>.xlsx,
#            01c_macro_daily_<cal>.xlsx   (only for macro feature sets)
#            05_results_sigma<k>_win<W>_<tag>.xlsx  (optional, picks the window)
#   writes : RESULTS_DIR/09_live_multiconfig_<cal>_<lengths>.xlsx
#
# WHY THIS EXISTS, AND WHAT IT REPLACES
# -------------------------------------
# C8_Live carries ONE FEATURE_SET and ONE PRIMARY_METRIC, taken from C1. That is
# correct for a single study. It cannot, however, score three sigmas whose chosen
# configurations differ — which is what a per-sigma selection produces — because
# every sigma would need its own C1 edit, its own run and its own output file,
# and the three could not be read side by side.
#
# This script does it in one pass. Each sigma names its own feature set and its
# own training objective in LIVE_CONFIGS below; the script rebinds the handful of
# C1 globals those two settings control, scores that sigma, and restores them
# before moving on. Everything else — the daily calendar, the open-cycle
# geometry, the expiry projection, the band statistics, the Gaussian — is shared
# and computed once, so the three sigmas are genuinely comparable.
#
# WHAT IT DOES NOT CHANGE
# -----------------------
# The band, the labels, the walk-forward machinery and both contestants are
# identical to C8_Live. This is a driver, not a new method. If a number here
# disagrees with a single-config C8_Live run for the same sigma, that is a bug in
# this file, not a modelling difference.
#
# READ THIS BEFORE ACTING ON THE OUTPUT
# -------------------------------------
# A per-sigma configuration was chosen by ranking many candidates at each sigma.
# When those rankings rest on differences that no significance test can separate
# from zero — which was the case on the 2,520-test consolidation — the choice is
# fitted to noise, and the live prediction inherits that. SINGLE_CONFIG below
# exists for exactly this reason: set it and every sigma uses one pre-registered
# configuration instead, which is what belongs in a write-up. The per-sigma map
# is for inspection.
#
# THE TIMING ASSUMPTION, unchanged from C8_Live
# --------------------------------------------
# Built for a decision taken minutes before the close, with the live index level
# standing in for today's official close. Two consequences:
#
#   * the number fed in is a live tick, not a settled close. A few minutes of
#     drift matters most when price sits near the band — exactly when the
#     decision is hardest.
#   * days_left for an OPEN cycle is a PROJECTION from the weekday rule. An
#     unscheduled holiday shortens the true horizon and makes every z_dist
#     slightly too small.
# ============================================================================

try:
    _ = (F_CLEAN, F_CYCLES, f_results, ROOT, RESULTS_DIR,
         SIGMA_GRID, SIGMA_WINDOW_GRID,
         CAL_TAG, TASKS, MODELS, TOP_K, DATE, CLOSE, OPEN, HIGH, LOW,
         EXPIRY_CUTOFF, is_expiry, add_bands_and_labels, drop_warmup,
         add_features, task_features, task_rows, gaussian_breach_prob,
         make_pipeline, stratified_folds, best_primary_threshold, pick_top_k,
         F_BETA, FBETA_LABEL, fbeta_at, PRIMARY_METRIC, GATE,
         HYPERPARAMS, TUNE_ITERS, RANDOM_STATE,
         ensemble, ECONOMIC_THRESHOLD, model_passes, available_features,
         FEATURE_SET, _FEATURE_SETS, CYCLE_LENGTHS, RV_TAG, IMPUTE_DAILY,
         rv_window_days, F_MACRO, MACRO_SOURCE_COLS, MACRO_FEATURE_SETS,
         read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import os
import datetime
import contextlib

import numpy as np
import pandas as pd


# ============================================================================
# 0. THE CONTROL PANEL
# ============================================================================
# One entry per sigma. Each names the feature set and the training objective to
# use for THAT band width. Keys must match SIGMA_GRID in C1 exactly — a sigma in
# the grid with no entry here, or an entry with no matching grid value, is
# refused rather than silently skipped.
#
# The defaults below are the per-sigma leaders from the 14-run, 2,520-test
# consolidation, each judged on its own primary metric. Every one of their
# eighteen cells was NULL (adjusted p from 0.18 to 1.00), and the sigma-0.10
# leader ranked 18th of 23 at sigma 0.25. Treat them as a sensitivity check, not
# as three findings.
LIVE_CONFIGS = {
    0.10: dict(feature_set="compact",          primary="payoff"),
    0.25: dict(feature_set="no_rv_plus_vix",   primary="payoff"),
    0.50: dict(feature_set="full_plus_macro",  primary="fbeta"),
}

# Set this to override every sigma with ONE configuration. This is what a
# write-up should use: a single pre-registered choice, applied uniformly, with no
# per-sigma selection to defend. None means use LIVE_CONFIGS above.
#
#   SINGLE_CONFIG = dict(feature_set="compact", primary="brier_skill")
#
# `compact` is the most parsimonious set (6/7/8 features, so the highest events
# per variable) and `brier_skill` is the only proper scoring rule of the three,
# so neither half of that choice depends on having seen the results.
SINGLE_CONFIG = None

# The live index level, for deciding before the close. Leave both None to use the
# last close already in the clean file.
LIVE_CLOSE = None
LIVE_OPEN = None          # today's open, needed when today is not yet in the file

F_LIVE = os.path.join(
    RESULTS_DIR, f"09_live_multiconfig_{CAL_TAG}_{CYCLE_LENGTHS}.xlsx")


# ---- resolve and validate the plan ------------------------------------------
if SINGLE_CONFIG is not None:
    PLAN = {float(s): dict(SINGLE_CONFIG) for s in SIGMA_GRID}
    PLAN_SOURCE = "SINGLE_CONFIG — one configuration for every sigma"
else:
    PLAN = {float(k): dict(v) for k, v in LIVE_CONFIGS.items()}
    PLAN_SOURCE = "LIVE_CONFIGS — a separate configuration per sigma"

_grid = {float(s) for s in SIGMA_GRID}
_plan = set(PLAN)
if _grid != _plan:
    raise RuntimeError(
        f"  the plan and SIGMA_GRID disagree.\n"
        f"    SIGMA_GRID has : {sorted(_grid)}\n"
        f"    the plan has   : {sorted(_plan)}\n"
        f"    missing from the plan : {sorted(_grid - _plan)}\n"
        f"    not in the grid       : {sorted(_plan - _grid)}\n"
        f"  Refusing rather than scoring a subset, because a table with a sigma "
        f"silently absent reads as a complete one.")

for _s, _c in PLAN.items():
    if _c["feature_set"] not in _FEATURE_SETS:
        raise ValueError(f"  sigma {_s}: feature_set {_c['feature_set']!r} is not "
                         f"one of {_FEATURE_SETS}")
    if _c["primary"] not in GATE:
        raise ValueError(f"  sigma {_s}: primary {_c['primary']!r} has no GATE "
                         f"entry; GATE knows {tuple(GATE)}")

print("\n" + "=" * 78)
print(f"  C8_Live_MultiConfig — the open cycle, calendar={CAL_TAG}")
print("=" * 78)
print(f"  plan: {PLAN_SOURCE}")
for _s in sorted(PLAN):
    _c = PLAN[_s]
    print(f"    sigma {_s:<5} feature_set={_c['feature_set']:<17} "
          f"primary={_c['primary']}")
print(f"  F_BETA = {F_BETA} ({FBETA_LABEL}) for every sigma; "
      f"economic cut p* = {ECONOMIC_THRESHOLD:.4f}")


# ============================================================================
# 0b. SWITCHING CONFIGURATION WITHOUT RE-RUNNING C1
# ----------------------------------------------------------------------------
# FEATURE_SET and PRIMARY_METRIC are module globals that C1's own functions read
# at call time: task_features reads FEATURE_SET, best_primary_threshold reads
# PRIMARY_METRIC, and fold_passes reads MAX_GAP / MIN_OOF / MIN_RATIO, which C1
# derives from GATE[PRIMARY_METRIC] when it loads.
#
# So changing the configuration means rebinding FIVE names, not two. Rebinding
# only the first two would leave the consistency gate set to the metric from the
# previous sigma — a model would be judged on one scale and gated on another, and
# nothing in the output would show it. That is the whole reason this is a context
# manager with an explicit list rather than two assignments inline.
#
# RUN_TAG is rebound too, because f_results() builds the 05 filename from it and
# the window lookup in section 3 would otherwise read another configuration's
# results.
_GLOBAL_NAMES = ("FEATURE_SET", "PRIMARY_METRIC", "MAX_GAP", "MIN_OOF",
                 "MIN_RATIO", "RUN_TAG")


@contextlib.contextmanager
def use_config(feature_set, primary):
    """Temporarily make C1's helpers behave as if C1 had been run with these.

    Parameters
    ----------
    feature_set : str
        One of C1's _FEATURE_SETS.
    primary : str
        A key of C1's GATE dict.

    Yields
    ------
    None

    Notes
    -----
    Restores every name on the way out, including on an exception, so a failure
    part-way through one sigma cannot leave the next one running under a mixture
    of two configurations.
    """
    g = globals()
    saved = {n: g.get(n) for n in _GLOBAL_NAMES}
    try:
        g["FEATURE_SET"] = feature_set
        g["PRIMARY_METRIC"] = primary
        g["MAX_GAP"] = GATE[primary]["gap"]
        g["MIN_OOF"] = GATE[primary]["floor"]
        g["MIN_RATIO"] = GATE[primary]["ratio"]
        # Built exactly as C1 builds it, so f_results() finds the same file C6
        # wrote for this configuration.
        g["RUN_TAG"] = f"{CAL_TAG}_{CYCLE_LENGTHS}_{feature_set}{RV_TAG}"
        yield
    finally:
        for n, v in saved.items():
            g[n] = v


def verdict(decision, side):
    """Turn a 0/1 call into words.

    A bare 1 or 0 is easy to misread with two contestants, two rules and two
    sides on the same row, and the sides mean opposite directions of travel.
    """
    if decision is None or (isinstance(decision, float) and np.isnan(decision)):
        return "no prediction"
    if int(decision) == 1:
        return ("BREACH expected above the upper band — exit" if side == "upper"
                else "BREACH expected below the lower band — exit")
    return "no breach expected — hold"


# ============================================================================
# 1. WHERE ARE WE IN THE CYCLE?
# ============================================================================
daily = read_xlsx(F_CLEAN, sheet="daily")
daily[DATE] = pd.to_datetime(daily[DATE]).dt.normalize()
daily = daily.sort_values(DATE).reset_index(drop=True)

# is_original comes from C2b and does two load-bearing jobs here:
#   - realized_vol must be measured on days that TRADED, so an imputed holiday's
#     zero return cannot deflate it. daily_realized_vol reads this column.
#   - is_expiry must see the REAL calendar. Its holiday fallback fires when the
#     nominal expiry weekday is ABSENT; an imputed row would make that day look
#     present and silently stop the promotion, moving every later cycle boundary.
if "is_original" not in daily.columns:
    daily["is_original"] = 1
daily["is_original"] = pd.to_numeric(
    daily["is_original"], errors="coerce").fillna(1).astype(int)
n_imputed_days = int((daily["is_original"] == 0).sum())
print(f"\n  {len(daily)} calendar rows, {n_imputed_days} imputed holidays")

if LIVE_CLOSE is not None:
    today = pd.Timestamp(datetime.date.today()).normalize()
    if today in set(daily[DATE]):
        daily.loc[daily[DATE] == today, CLOSE] = float(LIVE_CLOSE)
        if LIVE_OPEN is not None:
            daily.loc[daily[DATE] == today, OPEN] = float(LIVE_OPEN)
        print(f"  overwrote {today.date()} with the live level {LIVE_CLOSE}")
    else:
        if LIVE_OPEN is None:
            raise RuntimeError(
                "  LIVE_CLOSE was given for a date not in the clean file, so "
                "LIVE_OPEN is needed too — gap_Dn is ln(Open / previous Close) "
                "and cannot be built without today's open.")
        # is_original = 1: today is a session that is actually happening, so it
        # must enter the volatility window and be visible to is_expiry.
        daily = pd.concat([daily, pd.DataFrame([{
            DATE: today, OPEN: float(LIVE_OPEN), HIGH: float(LIVE_CLOSE),
            LOW: float(LIVE_CLOSE), CLOSE: float(LIVE_CLOSE),
            "is_original": 1}])],
            ignore_index=True).sort_values(DATE).reset_index(drop=True)
        daily["is_original"] = daily["is_original"].fillna(1).astype(int)
        print(f"  appended {today.date()} at the live level {LIVE_CLOSE}")

real_dates = set(daily.loc[daily["is_original"] == 1, DATE])
last_real = daily.loc[daily["is_original"] == 1, DATE].max()
last_date = daily[DATE].max()
flags = np.array([is_expiry(d, real_dates, last_real) and d in real_dates
                  for d in daily[DATE]], int)
expiry_pos = np.flatnonzero(flags == 1)

if len(expiry_pos) == 0:
    raise RuntimeError("  no expiry found in the daily file.")
if expiry_pos[-1] == len(daily) - 1:
    raise SystemExit(
        f"  {last_date.date()} IS an expiry, so no cycle is open. The next cycle "
        f"begins on the following trading day.")

open_block = daily.iloc[expiry_pos[-1] + 1:].reset_index(drop=True)
n_so_far = len(open_block)          # decision days completed, today included
print(f"  open cycle began {open_block[DATE].iloc[0].date()}, "
      f"today is {last_date.date()}, decision day D{n_so_far}")

if n_so_far < 2:
    raise SystemExit(
        "  today is D1 of the open cycle and there is no D1 task — the band is "
        "drawn at D1's close, so the first decision is taken on D2.")
if n_so_far > 4:
    raise SystemExit(
        f"  {n_so_far} decision days have passed without an expiry. Either the "
        f"daily file is missing an expiry or the expiry rule needs checking.")


def project_expiry(after):
    """The next scheduled expiry strictly after a date, from the weekday rule.

    A PROJECTION. It applies the Tuesday or Thursday rule and assumes that day
    trades. An unscheduled holiday pulls the real expiry earlier, making the true
    horizon shorter than assumed and every z_dist slightly too small, because
    z_dist divides by sqrt(days_left).
    """
    d = after + pd.Timedelta(days=1)
    for _ in range(30):
        target = 1 if d >= EXPIRY_CUTOFF else 3
        if d.weekday() == target:
            return d.normalize()
        d += pd.Timedelta(days=1)
    raise RuntimeError("  could not project the next expiry.")


expiry_date = project_expiry(last_date)
future_days = pd.bdate_range(last_date + pd.Timedelta(days=1), expiry_date)
days_left_now = len(future_days)
cycle_days = n_so_far + max(days_left_now - 1, 0)
print(f"  projected expiry {expiry_date.date()}, days_left={days_left_now}, "
      f"cycle_days={cycle_days}")
if cycle_days != 4:
    print(f"  note: this cycle projects to {cycle_days} decision days, not 4")


# ============================================================================
# 1b. THE MACRO JOIN, CHECKED AGAINST EVERY CONFIGURATION IN THE PLAN
# ----------------------------------------------------------------------------
# The join is a LEFT join on the daily calendar, so it adds columns and never
# rows. The requirement check is per-configuration, not global: with three
# sigmas it is entirely possible for one to need the macro columns and the others
# not, and a single global check would either pass when it should fail or fail
# when it should pass.
# ============================================================================
_macro_cols = []
if os.path.exists(F_MACRO):
    _mac = read_xlsx(F_MACRO, sheet="macro")
    _mac[DATE] = pd.to_datetime(_mac[DATE]).dt.normalize()
    _macro_cols = [c for c in MACRO_SOURCE_COLS if c in _mac.columns]
    if _macro_cols:
        _n_before = len(daily)
        daily = daily.merge(_mac[[DATE] + _macro_cols], on=DATE, how="left")
        if len(daily) != _n_before:
            raise RuntimeError(
                f"  the macro join changed the row count: {_n_before} -> "
                f"{len(daily)}. {os.path.basename(F_MACRO)} has duplicate dates, "
                f"which would corrupt every cycle boundary. Re-run C2c.")
        _cov = {c: f"{100.0 * daily[c].notna().mean():.0f}%" for c in _macro_cols}
        print(f"  macro joined from {os.path.basename(F_MACRO)}: {_cov}")
    else:
        print(f"  macro file found but none of {MACRO_SOURCE_COLS} are in it")
else:
    print(f"  no macro file at {os.path.basename(F_MACRO)} — "
          f"vix/usdinr/brent features will NOT be built")

_need_macro = sorted({s for s, c in PLAN.items()
                      if c["feature_set"] in MACRO_FEATURE_SETS})
if _need_macro and not _macro_cols:
    raise RuntimeError(
        f"  these sigmas need the macro file and it is missing or empty:\n"
        f"    " + ", ".join(f"sigma {s} ({PLAN[s]['feature_set']})"
                            for s in _need_macro) + "\n"
        f"  {F_MACRO}\n"
        f"  Run C2c_FetchMacro first. Refusing rather than quietly falling back "
        f"to the\n"
        f"  non-macro parent feature set — that silent fallback is exactly what "
        f"this\n"
        f"  check exists to stop.")


# ============================================================================
# 2. THE LIVE ROW, BUILT LIKE ANY COMPLETED CYCLE
# ----------------------------------------------------------------------------
# expiry_close is unknown, so the row carries NaN there and gets no label. It is
# appended to the completed cycles ONLY so the band statistics roll forward onto
# it; it contributes nothing to those statistics, because they are lagged by one
# cycle.
# ============================================================================
cycles = read_xlsx(F_CYCLES, sheet="cycles")
for c in ("expiry_date",) + tuple(f"d{i}_date" for i in (1, 2, 3, 4)):
    cycles[c] = pd.to_datetime(cycles[c], errors="coerce")

live = {"cycle_id": int(cycles["cycle_id"].max()) + 1,
        "cycle_days": cycle_days, "is_padded": 0,
        "expiry_date": expiry_date, "expiry_close": np.nan}
for i in range(1, 5):
    live[f"d{i}_date"] = (open_block[DATE].iloc[i - 1] if i <= n_so_far else pd.NaT)
    live[f"d{i}_close"] = (float(open_block[CLOSE].iloc[i - 1]) if i <= n_so_far
                           else np.nan)

combined = pd.concat([cycles, pd.DataFrame([live])], ignore_index=True)
print(f"  {len(cycles)} completed cycles + 1 open")


# ============================================================================
# 3. REFIT ON EVERYTHING, THEN PREDICT
# ============================================================================
def _inner_scoring():
    """Tuning score for the live refit — must match C6's choice exactly.

    Reads the CURRENT PRIMARY_METRIC, so it changes with use_config. A payoff
    scorer is not offered here deliberately: C6 tunes payoff through a custom
    scorer cutting at p*, and reproducing it in two places is how the two stop
    agreeing. Under payoff the live refit therefore tunes on Brier, which is the
    proper score and the conservative choice; the payoff decision still comes
    from the fixed p* cut, which is where payoff is actually realised.
    """
    if PRIMARY_METRIC == "fbeta":
        from sklearn.metrics import fbeta_score, make_scorer
        return make_scorer(fbeta_score, beta=float(F_BETA), zero_division=0.0)
    return "neg_brier_score"


def fit_and_predict(feat_all, live_pos, day, side, sigma, window):
    """Refit the ensemble on all completed cycles and score the open one.

    Parameters
    ----------
    feat_all : pandas.DataFrame
        Completed cycles plus the live row, bands and features attached.
    live_pos : int
        Index of the live row.
    day, side : int, str
        The task.
    sigma, window : float, int
        The configuration.

    Returns
    -------
    dict or None
        Probabilities, decisions and the members used. None when the task cannot
        be fitted.

    Notes
    -----
    Every C1 helper called here reads the globals use_config has rebound, so this
    function is configuration-agnostic by construction rather than by having the
    feature set passed down through it.

    The threshold is cross-fitted over the whole training set, exactly as in C6,
    and fitted on the PRIMARY metric rather than always on F-beta — fitting a cut
    to maximise one quantity and then reporting another is a mismatch that made
    one task read 0.108 when its own members scored about 0.35.
    """
    # available_features, not task_features: C5 may have dropped a constant
    # column, and asking for one that is absent should fail here rather than in
    # the place the decision is taken.
    rows = task_rows(feat_all, day)
    cols = available_features(rows, day, side)
    if live_pos not in rows.index or not cols:
        return None
    live_row = rows.loc[[live_pos]]
    train = rows.drop(index=live_pos)
    label = f"{side}_breach"

    ok = train[label].notna() & np.isfinite(train[cols]).all(axis=1)
    train = train[ok]
    if len(train) < 40 or not np.isfinite(live_row[cols].to_numpy(float)).all():
        return None

    X = train[cols].to_numpy(float)
    y = train[label].to_numpy(int)
    x_live = live_row[cols].to_numpy(float)

    from sklearn.base import clone
    from sklearn.model_selection import RandomizedSearchCV

    members = []
    for name in MODELS:
        pipe = make_pipeline(name)
        n_pos = int(y.sum())
        inner = min(3, n_pos, int((1 - y).sum()))
        if inner >= 2:
            search = RandomizedSearchCV(
                pipe, HYPERPARAMS[name], n_iter=TUNE_ITERS, cv=inner,
                scoring=_inner_scoring(), n_jobs=-1,
                random_state=RANDOM_STATE, error_score=0.0)
            search.fit(X, y)
            est = search.best_estimator_
        else:
            est = pipe.fit(X, y)

        oof = np.full(len(y), np.nan)
        for fit_idx, out_idx in stratified_folds(y):
            if len(np.unique(y[fit_idx])) < 2:
                continue
            m = clone(make_pipeline(name)).fit(X[fit_idx], y[fit_idx])
            oof[out_idx] = m.predict_proba(X[out_idx])[:, 1]
        good = np.isfinite(oof)
        thr, _ = best_primary_threshold(y[good], oof[good], fallback=0.5)
        if thr is None:
            continue
        from_in = est.predict_proba(X)[:, 1]
        # The same measure C6 gates on, or the live gate would be a different
        # test from the one that selected these models in the first place.
        s_in = fbeta_at(y, from_in, thr)
        s_oof = fbeta_at(y[good], oof[good], thr)
        members.append({"model": name, "infold_score": s_in, "oof_score": s_oof,
                        "threshold": thr,
                        "prob": float(est.predict_proba(x_live)[0, 1]),
                        "passed_gate": model_passes(
                            [(s_in, s_oof)], base_rate=float(y.mean()))[0]})

    if not members:
        return None
    cand = pd.DataFrame(members)
    eligible = cand[cand["passed_gate"]]
    gated = not eligible.empty
    chosen = pick_top_k(eligible if gated else cand, TOP_K)

    dec, prob, eff = ensemble(list(chosen["prob"]), list(chosen["threshold"]),
                              list(chosen["oof_score"]))
    return {"ml_prob": prob, "ml_decision": dec, "ml_threshold": eff,
            "ml_gated": gated, "n_train": len(train), "n_features": len(cols),
            "features": ", ".join(cols),
            "members": ", ".join(f"{r.model}({r.oof_score:.2f})"
                                 for r in chosen.itertuples()),
            "candidates": cand.assign(task=f"D{day}_{side}", sigma=sigma,
                                      window=window)}


# Every decision day from D2 to today. On D2 that is one day, on D4 it is three,
# so the run gets longer later in the week — the cost of seeing the trajectory.
DAYS_TO_SCORE = list(range(2, n_so_far + 1))
print(f"  scoring decision day(s): "
      f"{', '.join('D' + str(d) for d in DAYS_TO_SCORE)}")
print(f"  {len(PLAN)} sigma(s) x {len(DAYS_TO_SCORE)} day(s) x 2 side(s) = "
      f"{2 * len(PLAN) * len(DAYS_TO_SCORE)} prediction(s)")


# ============================================================================
# 4. THE MAIN LOOP — ONE CONFIGURATION PER SIGMA
# ============================================================================
live_rows, all_cands, input_rows = [], [], []

for sigma in sorted(PLAN):
    cfg = PLAN[sigma]
    print("\n" + "-" * 78)
    print(f"  sigma {sigma}   feature_set={cfg['feature_set']}   "
          f"primary={cfg['primary']}")
    print("-" * 78)

    with use_config(cfg["feature_set"], cfg["primary"]):
        # ---- which window won, for THIS configuration --------------------
        # Per (sigma, task), the window whose mean OUT-OF-FOLD score was highest
        # in C6. Out-of-fold and never test: choosing a window on test
        # performance is selection on the outcome and would make the C6 numbers
        # an optimistic description of this script. With one window in the grid
        # it reduces to that window, and the choice is still recorded so it stays
        # auditable when the grid grows.
        best_window = {}
        for window in SIGMA_WINDOW_GRID:
            path = f_results(sigma, window)
            if not os.path.exists(path):
                continue
            cand_sheet = read_xlsx(path, sheet="candidates")
            for task, g in cand_sheet.groupby("task"):
                score = float(g["oof_score"].max())
                key = (sigma, task)
                if key not in best_window or score > best_window[key][1]:
                    best_window[key] = (window, score)
        if not best_window:
            print(f"    no C6 results for {RUN_TAG} at sigma {sigma} — using "
                  f"window {SIGMA_WINDOW_GRID[0]} (the only one in the grid)"
                  if len(SIGMA_WINDOW_GRID) == 1 else
                  f"    no C6 results for {RUN_TAG} at sigma {sigma} — falling "
                  f"back to window {SIGMA_WINDOW_GRID[0]} WITHOUT evidence")

        # ---- bands and features, built ONCE per (sigma, window) ----------
        # add_features does not consult FEATURE_SET: it builds every candidate
        # column and task_features selects from them. So the frame can be shared
        # across all six tasks of this sigma instead of rebuilt twelve times,
        # which is the same numbers and a sixth of the work.
        feat_cache = {}

        def _frame(window):
            if window not in feat_cache:
                banded = add_bands_and_labels(combined, sigma, window)
                feat_cache[window] = add_features(banded, daily, window)
            return feat_cache[window]

        for day in DAYS_TO_SCORE:
            for side in ("upper", "lower"):
                task = f"D{day}_{side}"
                window, won_with = best_window.get(
                    (sigma, task), (SIGMA_WINDOW_GRID[0], np.nan))

                feat_all = _frame(window)
                live_pos = feat_all.index[-1]
                row = feat_all.loc[live_pos]

                gp = float(gaussian_breach_prob(
                    feat_all.loc[[live_pos]], day, side)[0])
                out = fit_and_predict(feat_all, live_pos, day, side, sigma,
                                      window)

                # The Gaussian needs a fitted cut too, on the completed cycles of
                # this task, with the SAME function and the same objective C6
                # uses. Without it the two contestants would be compared under
                # different rules — ML at its own learned cut and the baseline at
                # p* — which is not a comparison of forecasts at all.
                _rows = task_rows(feat_all, day).drop(index=live_pos)
                _lab = _rows[f"{side}_breach"]
                _ok = _lab.notna()
                _gtr = gaussian_breach_prob(_rows[_ok], day, side)
                g_thr, _ = best_primary_threshold(
                    _lab[_ok].astype(int).to_numpy(), _gtr, fallback=0.5)
                if g_thr is None:
                    g_thr = 0.5

                # days_left for a PAST day of this same cycle is known exactly,
                # because the projected expiry is fixed. Only today's depends on
                # the projection.
                dl = int(cycle_days + 1 - day)
                entry = {
                    "sigma": sigma, "feature_set": cfg["feature_set"],
                    "primary_metric": cfg["primary"], "window": window,
                    "task": task, "decision_day": day,
                    "is_today": int(day == n_so_far),
                    "date": open_block[DATE].iloc[day - 1].date(), "side": side,
                    "asof": last_date.date(),
                    "expiry_projected": expiry_date.date(),
                    "days_left": dl, "cycle_days": cycle_days,
                    "close_used": float(row[f"d{day}_close"]),
                    "band_upper": float(row["band_upper"]),
                    "band_lower": float(row["band_lower"]),
                    "distance_to_band_pct": round(
                        100.0 * (float(row[f"band_{side}"])
                                 / float(row[f"d{day}_close"]) - 1.0), 3),
                    "gauss_prob": gp,
                    "gauss_threshold_fit": round(float(g_thr), 4),
                    "gauss_decision_fit": int(gp >= g_thr),
                    "gauss_verdict_fit": verdict(int(gp >= g_thr), side),
                    "gauss_decision_econ": int(gp >= ECONOMIC_THRESHOLD),
                    "gauss_verdict_econ": verdict(
                        int(gp >= ECONOMIC_THRESHOLD), side),
                    "window_chosen_on_oof": (round(won_with, 4)
                                             if np.isfinite(won_with)
                                             else "no C6 results")}

                if out is None:
                    entry.update({
                        "ml_prob": np.nan, "ml_threshold_fit": np.nan,
                        "ml_decision_fit": np.nan,
                        "ml_verdict_fit": "no prediction",
                        "ml_decision_econ": np.nan,
                        "ml_verdict_econ": "no prediction",
                        "ml_gated": False, "n_features": 0,
                        "ensemble_members": "could not fit", "n_train": 0})
                else:
                    _flag = "" if out["ml_gated"] else "  [failed the gate]"
                    _d_econ = int(out["ml_prob"] >= ECONOMIC_THRESHOLD)
                    entry.update({
                        "ml_prob": out["ml_prob"],
                        "ml_threshold_fit": round(float(out["ml_threshold"]), 4),
                        "ml_decision_fit": out["ml_decision"],
                        "ml_verdict_fit": verdict(
                            out["ml_decision"], side) + _flag,
                        "ml_decision_econ": _d_econ,
                        "ml_verdict_econ": verdict(_d_econ, side) + _flag,
                        "ml_gated": out["ml_gated"],
                        "n_features": out["n_features"],
                        "ensemble_members": out["members"],
                        "n_train": out["n_train"]})
                    all_cands.append(out["candidates"].assign(
                        feature_set=cfg["feature_set"],
                        primary_metric=cfg["primary"]))

                # Agreement is reported PER RULE. Comparing one contestant's
                # fitted-cut call with the other's economic call is not a
                # disagreement about the forecast, only about the cut — and
                # mixing them is how a table like this misleads.
                for _r, _mk, _gk in (("fit", "ml_decision_fit",
                                      "gauss_decision_fit"),
                                     ("econ", "ml_decision_econ",
                                      "gauss_decision_econ")):
                    entry[f"agreement_{_r}"] = (
                        "ML absent" if pd.isna(entry[_mk])
                        else "agree" if int(entry[_mk]) == int(entry[_gk])
                        else "DISAGREE")
                live_rows.append(entry)

                input_rows.append({
                    "sigma": sigma, "feature_set": cfg["feature_set"],
                    "window": window, "decision_day": day, "side": side,
                    **{c: float(row[c]) for c in task_features(day, side)
                       if c in row.index and pd.notna(row[c])}})

                _mlp = (f"{out['ml_prob']:.3f}" if out else "  —  ")
                print(f"    D{day} {side:<5}  gauss {gp:.3f}  ml {_mlp}  "
                      f"{'' if out is None else ('gated' if out['ml_gated'] else 'UNGATED')}")

live_df = pd.DataFrame(live_rows)


# ============================================================================
# 5. OUTPUT — EVERY SIGMA, ML BESIDE THE GAUSSIAN
# ============================================================================
print("\n" + "=" * 78)
print("  BANDS AND PROBABILITIES")
print("=" * 78)
print(live_df[["sigma", "feature_set", "decision_day", "date", "side",
               "close_used", "band_upper", "band_lower",
               "distance_to_band_pct", "days_left", "gauss_prob", "ml_prob"]]
      .to_string(index=False))

print("\n" + "=" * 78)
print("  AT EACH CONTESTANT'S OWN FITTED CUT")
print("=" * 78)
print(live_df[["sigma", "primary_metric", "decision_day", "side",
               "gauss_threshold_fit", "gauss_decision_fit",
               "ml_threshold_fit", "ml_decision_fit", "agreement_fit",
               "ml_gated"]].to_string(index=False))

print("\n" + "=" * 78)
print(f"  AT THE ECONOMIC CUT p* = {ECONOMIC_THRESHOLD:.4f} "
      f"(identical for both contestants, nothing fitted)")
print("=" * 78)
print(live_df[["sigma", "decision_day", "side", "gauss_prob", "ml_prob",
               "gauss_decision_econ", "ml_decision_econ", "agreement_econ"]]
      .to_string(index=False))

# TODAY is the row that matters for acting, so it is printed on its own rather
# than left among the earlier days of the same cycle.
print("\n" + "=" * 78)
print(f"  TODAY — D{n_so_far}, {last_date.date()}, "
      f"{days_left_now} day(s) to projected expiry {expiry_date.date()}")
print("=" * 78)
for sigma in sorted(PLAN):
    cfg = PLAN[sigma]
    t = live_df[(live_df["sigma"] == sigma) & (live_df["is_today"] == 1)]
    if t.empty:
        continue
    print(f"\n  sigma {sigma}   {cfg['feature_set']} / {cfg['primary']}")
    for _, r in t.iterrows():
        band = r["band_upper"] if r["side"] == "upper" else r["band_lower"]
        print(f"    {r['side']:<5} band {band:,.1f}  "
              f"({r['distance_to_band_pct']:+.2f}% from "
              f"{r['close_used']:,.1f})")
        print(f"      Gaussian  p={r['gauss_prob']:.3f}   "
              f"fitted cut {r['gauss_threshold_fit']:.3f} -> "
              f"{r['gauss_verdict_fit']}")
        print(f"      {'':>10}{'':>6}        "
              f"economic p* -> {r['gauss_verdict_econ']}")
        if pd.notna(r["ml_prob"]):
            print(f"      ML        p={r['ml_prob']:.3f}   "
                  f"fitted cut {r['ml_threshold_fit']:.3f} -> "
                  f"{r['ml_verdict_fit']}")
            print(f"      {'':>10}{'':>6}        "
                  f"economic p* -> {r['ml_verdict_econ']}")
            print(f"      {'':>10}{'':>6}        "
                  f"{r['n_features']} features, {r['n_train']} training "
                  f"cycles, members {r['ensemble_members']}")
        else:
            print("      ML        no prediction")
        print(f"      agreement: fitted cut {r['agreement_fit']}, "
              f"economic {r['agreement_econ']}")

_ungated = live_df[(live_df["is_today"] == 1) & (~live_df["ml_gated"])]
if len(_ungated):
    print(f"\n  WARNING: {len(_ungated)} of today's ML predictions come from "
          f"models that FAILED the")
    print("  consistency gate. They are reported so the failure is visible, not "
          "because they")
    print("  are usable. Read the Gaussian on those rows.")

_dis = live_df[(live_df["is_today"] == 1)
               & (live_df["agreement_econ"] == "DISAGREE")]
print(f"\n  today, at the economic cut, the two contestants disagree on "
      f"{len(_dis)} of {len(live_df[live_df['is_today'] == 1])} rows.")


# ============================================================================
# 6. CHECKS
# ============================================================================
fails = []


def chk(nm, ok, detail=""):
    if not ok:
        fails.append(f"{nm}: {detail}")
    print(f"    {'OK  ' if ok else 'FAIL'} {nm}"
          + ("" if ok else f"  -- {detail}"))


print("\n  checks")
chk("the open cycle has no label",
    bool(pd.isna(combined["expiry_close"].iloc[-1])),
    "expiry_close must be unknown for a cycle still in progress")
chk("the live row was not used to build its own band",
    len(combined) == len(cycles) + 1)
chk("every sigma produced both sides for every day scored",
    len(live_df) == 2 * len(PLAN) * len(DAYS_TO_SCORE),
    f"{len(live_df)} rows against {2 * len(PLAN) * len(DAYS_TO_SCORE)} expected")
chk("exactly one row per (sigma, day, side)",
    not live_df.duplicated(["sigma", "decision_day", "side"]).any())
# The point of this whole file: each sigma must have used the configuration the
# plan names for it, and nothing else.
chk("every row used the configuration its sigma was assigned",
    bool(all((live_df.loc[live_df["sigma"] == s, "feature_set"]
              == PLAN[s]["feature_set"]).all()
             and (live_df.loc[live_df["sigma"] == s, "primary_metric"]
                  == PLAN[s]["primary"]).all() for s in PLAN)),
    "a sigma was scored under another sigma's configuration — the rebinding in "
    "use_config did not hold")
# Rebinding globals is the one genuinely fragile thing here, so prove it was
# undone. If these have drifted, every later cell in the notebook is affected.
chk("C1's globals were restored after the last sigma",
    FEATURE_SET == globals().get("FEATURE_SET")
    and MAX_GAP == GATE[PRIMARY_METRIC]["gap"]
    and MIN_OOF == GATE[PRIMARY_METRIC]["floor"]
    and MIN_RATIO == GATE[PRIMARY_METRIC]["ratio"],
    "the gate constants no longer match PRIMARY_METRIC, so a later cell would "
    "silently use the wrong gate")
chk("the feature count differs across sigmas only where the plan says it should",
    bool(live_df.groupby("sigma")["feature_set"].nunique().eq(1).all()))
chk("days_left counts down as the day index rises",
    bool((live_df.sort_values("decision_day")
          .groupby(["sigma", "side"])["days_left"]
          .apply(lambda s: s.is_monotonic_decreasing)).all()))
chk("every row carries a verdict in words under both rules",
    bool(live_df[["gauss_verdict_fit", "gauss_verdict_econ",
                  "ml_verdict_fit", "ml_verdict_econ"]].notna().all().all()))
chk("each Gaussian verdict matches its own decision",
    bool(all((("BREACH" in r["gauss_verdict_econ"])
              == (int(r["gauss_decision_econ"]) == 1))
             and (("BREACH" in r["gauss_verdict_fit"])
                  == (int(r["gauss_decision_fit"]) == 1))
             for _, r in live_df.iterrows())),
    "a verdict and its 0/1 decision have drifted apart")
chk("agreement is computed within a rule, not across rules",
    bool(all((r[f"agreement_{k}"] == "ML absent")
             or ((r[f"agreement_{k}"] == "agree")
                 == (int(r[f"ml_decision_{m}"])
                     == int(r[f"gauss_decision_{m}"])))
             for _, r in live_df.iterrows()
             for k, m in (("fit", "fit"), ("econ", "econ")))))
chk("every economic decision equals int(prob >= p*)",
    bool((live_df["gauss_decision_econ"]
          == (live_df["gauss_prob"] >= ECONOMIC_THRESHOLD).astype(int)).all()
         and all(int(r["ml_decision_econ"])
                 == int(r["ml_prob"] >= ECONOMIC_THRESHOLD)
                 for _, r in live_df.iterrows() if pd.notna(r["ml_prob"]))),
    "p* comes from the payoff matrix and nothing may move it")
chk("the upper band sits above the lower one",
    bool((live_df["band_upper"] > live_df["band_lower"]).all()))
chk("each row used its own day's close",
    bool(all(np.isclose(r["close_used"],
                        float(open_block[CLOSE].iloc[int(r["decision_day"]) - 1]))
             for _, r in live_df.iterrows())))
chk("the band widens with sigma on the same day",
    bool(all(live_df[live_df["decision_day"] == d]
             .drop_duplicates("sigma").sort_values("sigma")
             .assign(w=lambda x: x["band_upper"] - x["band_lower"])["w"]
             .is_monotonic_increasing for d in DAYS_TO_SCORE)),
    "a larger k must give a wider band, or the bands are not from one formula")
chk("days_left is at least 1", days_left_now >= 1)
chk("probabilities are within [0, 1]",
    bool(live_df[["gauss_prob", "ml_prob"]].stack().between(0, 1).all()))
if fails:
    raise RuntimeError("  checks failed:\n    - " + "\n    - ".join(fails))


# ============================================================================
# 7. WRITE
# ============================================================================
plan_sheet = pd.DataFrame([
    {"sigma": s, "feature_set": PLAN[s]["feature_set"],
     "primary_metric": PLAN[s]["primary"],
     "n_features_used": int(live_df.loc[live_df["sigma"] == s,
                                        "n_features"].max()),
     "needs_macro": PLAN[s]["feature_set"] in MACRO_FEATURE_SETS,
     "gate_gap": GATE[PLAN[s]["primary"]]["gap"],
     "gate_floor": GATE[PLAN[s]["primary"]]["floor"],
     "gate_ratio": GATE[PLAN[s]["primary"]]["ratio"]}
    for s in sorted(PLAN)])

manifest = pd.DataFrame([
    ("script", "C8_Live_MultiConfig"),
    ("plan source", PLAN_SOURCE),
    ("as of", str(last_date.date())),
    ("decision day today", f"D{n_so_far}"),
    ("days scored", ", ".join("D" + str(d) for d in DAYS_TO_SCORE)),
    ("projected expiry", str(expiry_date.date())),
    ("days_left", days_left_now),
    ("cycle_days projected", cycle_days),
    ("close used", "live tick" if LIVE_CLOSE is not None
     else "last stored close"),
    ("daily_calendar", CAL_TAG),
    ("cycle_lengths", CYCLE_LENGTHS),
    ("impute_daily", IMPUTE_DAILY),
    ("realized_vol window (days)", rv_window_days(SIGMA_WINDOW_GRID[0])),
    ("imputed holidays in the daily file", n_imputed_days),
    ("sigmas", ", ".join(str(s) for s in sorted(PLAN))),
    ("configuration per sigma",
     "; ".join(f"{s}: {PLAN[s]['feature_set']}/{PLAN[s]['primary']}"
               for s in sorted(PLAN))),
    ("f_beta", F_BETA),
    ("metric label", FBETA_LABEL),
    ("economic cut p*", round(ECONOMIC_THRESHOLD, 4)),
    ("window choice",
     "highest mean OUT-OF-FOLD score in C6, per (sigma, task)"),
    ("models refitted on", "every completed cycle, no sealed block"),
    ("completed cycles trained on", len(cycles)),
    ("selection caveat",
     "A per-sigma configuration was chosen by ranking candidates within each "
     "sigma. On the 2,520-test consolidation every one of those leaders' cells "
     "was statistically null, so the choice is fitted to noise. Set "
     "SINGLE_CONFIG for anything published or acted on."),
    ("timing caveat",
     "built for a decision minutes before the close. days_left assumes the "
     "projected expiry trades; an unscheduled holiday shortens the real horizon "
     "and makes every z_dist slightly too small."),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

sheets = {"live": live_df, "plan": plan_sheet,
          "inputs": pd.DataFrame(input_rows), "manifest": manifest}
if all_cands:
    sheets["candidates"] = pd.concat(all_cands, ignore_index=True)
write_xlsx(F_LIVE, sheets)

print("\n" + "=" * 78)
print(f"  C8_Live_MultiConfig complete — {os.path.basename(F_LIVE)}")
print(f"    {len(live_df)} prediction(s): {len(PLAN)} sigma(s) x "
      f"{len(DAYS_TO_SCORE)} day(s) x 2 side(s)")
print("  Re-run each day of the cycle. A row with ml_gated=False came from "
      "models that")
print("  failed the consistency gate — read the Gaussian on those.")
print("=" * 78)


  C8_Live_MultiConfig — the open cycle, calendar=filled
  plan: LIVE_CONFIGS — a separate configuration per sigma
    sigma 0.1   feature_set=compact           primary=payoff
    sigma 0.25  feature_set=no_rv_plus_vix    primary=payoff
    sigma 0.5   feature_set=full_plus_macro   primary=fbeta
  F_BETA = 1.225 (F1.225) for every sigma; economic cut p* = 0.4000

  2008 calendar rows, 93 imputed holidays
  open cycle began 2026-09-30, today is 2026-10-05, decision day D4
  projected expiry 2026-10-06, days_left=1, cycle_days=4
  macro joined from 01c_macro_daily_filled.xlsx: {'india_vix': '100%', 'vix_daily': '100%', 'usdinr_ln1': '100%', 'usdinr_ln5': '100%', 'brent_ln1': '100%', 'brent_ln5': '100%'}
  386 completed cycles + 1 open
  scoring decision day(s): D2, D3, D4
  3 sigma(s) x 3 day(s) x 2 side(s) = 18 prediction(s)

------------------------------------------------------------------------------
  sigma 0.1   feature_set=compact   primary=payoff
---------------------------------

In [13]:
# @title
# ============================================================================
# C8_Liveold — PREDICT THE CYCLE THAT IS STILL OPEN
# ============================================================================
#
#   reads  : 02_daily_clean.xlsx, 03_cycles_<align>.xlsx,
#            05_results_sigma<k>_win<W>_<align>.xlsx  (to pick the window)
#   writes : 07_live_<align>.xlsx
#
# WHAT THIS DOES, AND HOW IT DIFFERS FROM C6
# ------------------------------------------
# C6 measures. This decides. It takes the cycle currently in progress, works out
# which decision day today is, and produces a prediction for EVERY decision day
# from D2 up to and including today, per sigma, for each band edge — the
# Gaussian's and the ensemble's, under both decision rules.
#
# Why every day and not only today: on D4 you want to see whether the call has
# been stable. A probability that read 0.2 on D2, 0.4 on D3 and 0.8 on D4 is a
# developing move; one that jumps to 0.8 only on the last day is a late signal
# worth less. The earlier days are recomputed from the same stored closes, so they
# reproduce exactly what you would have seen on those days — nothing is
# back-filled with information from later in the week.
#
# Each row carries a plain-language verdict as well as the 0/1 decision, because
# "1" alone is easy to misread when two contestants and two rules are side by
# side.
#
# The models are REFITTED HERE on every completed cycle. C6 deliberately keeps a
# sealed test block, so its models have never seen the most recent cycles; for a
# live decision you want the opposite — every observation available. Same
# pipeline, same tuning, same cross-fitted threshold, all of the data.
#
# WHICH WINDOW IS USED
# --------------------
# Per (sigma, task), the window whose mean OUT-OF-FOLD score was highest in C6.
# Out-of-fold, never test: choosing a window by test performance is selection on
# the outcome, and it would make the C6 numbers an optimistic description of this
# script's behaviour. With a single window in the grid this reduces to that
# window, and the choice is recorded anyway so it stays auditable when the grid
# grows.
#
# THE TIMING ASSUMPTION
# ---------------------
# This is built for a decision taken a few minutes before the close, using the
# live index level in place of today's official close. Every feature it needs is
# then available: z_dist uses today's close, realized_vol includes today's return,
# and the gap used this morning's open. Two consequences worth stating:
#
#   * the number you feed in is a live tick, not a settled close. Five minutes of
#     drift is roughly 18 index points, about 0.115 of a daily move — largest
#     relative to signal when the price sits close to the band, which is exactly
#     when the decision is hardest.
#   * days_left for an OPEN cycle is a projection. The expiry is taken from the
#     weekday rule; if NSE declares an unexpected holiday the real horizon is
#     shorter than assumed and every z_dist is correspondingly too small.
#
# WHAT IT REFUSES TO DO
# ---------------------
# If today is the cycle's first decision day there is no D1 task, and if today is
# the expiry there is nothing left to decide. In both cases it says so and stops
# rather than inventing a prediction.
# ============================================================================

try:
    _ = (F_CLEAN, F_CYCLES, f_results, ROOT, SIGMA_GRID, SIGMA_WINDOW_GRID,
         CAL_TAG, TASKS, MODELS, TOP_K, DATE, CLOSE, OPEN,
         EXPIRY_CUTOFF, is_expiry, add_bands_and_labels, drop_warmup,
         add_features, task_features, task_rows, gaussian_breach_prob,
         make_pipeline, stratified_folds, best_f1_threshold, pick_top_k,
         ensemble, ECONOMIC_THRESHOLD, model_passes, available_features,
         FEATURE_SET, CYCLE_LENGTHS, RUN_TAG, IMPUTE_DAILY,
         rv_window_days, F_MACRO, MACRO_SOURCE_COLS, read_xlsx, write_xlsx)
except NameError as e:
    raise RuntimeError(f"Run C1_Config first — missing {e}") from None

import os
import datetime

import numpy as np
import pandas as pd

print("\n" + "=" * 78)
print(f"  C8_Live — the open cycle, calendar={CAL_TAG}")
print("=" * 78)

F_LIVE = os.path.join(ROOT, f"07_live_{RUN_TAG}.xlsx")

# Set this to the live index level to decide before the close. Leave None to use
# the last close already in the clean file.
LIVE_CLOSE = None
LIVE_OPEN = None          # today's open, if today is not yet in the clean file


def verdict(decision, side):
    """Turn a 0/1 call into words.

    Parameters
    ----------
    decision : int, float or None
        1 to exit, 0 to hold. NaN when the model could not be fitted.
    side : {"upper", "lower"}

    Returns
    -------
    str

    Notes
    -----
    A bare 1 or 0 is easy to misread with two contestants and two decision rules
    on the same row, and the two sides mean opposite directions of travel. Writing
    it out removes both ambiguities.
    """
    if decision is None or (isinstance(decision, float) and np.isnan(decision)):
        return "no prediction"
    if int(decision) == 1:
        return ("BREACH expected above the upper band — exit" if side == "upper"
                else "BREACH expected below the lower band — exit")
    return "no breach expected — hold"


# ============================================================================
# 1. WHERE ARE WE IN THE CYCLE?
# ============================================================================
daily = read_xlsx(F_CLEAN, sheet="daily")
daily[DATE] = pd.to_datetime(daily[DATE]).dt.normalize()
daily = daily.sort_values(DATE).reset_index(drop=True)

# is_original comes from C2b. It does two jobs here, both load-bearing:
#   - realized_vol must be measured on days that TRADED, so the zero returns of an
#     imputed holiday cannot deflate it. daily_realized_vol reads this column.
#   - is_expiry must be evaluated on the REAL calendar. Its holiday fallback fires
#     when the nominal expiry weekday is ABSENT, and an imputed day would make that
#     day look present, silently stopping the promotion. See section 1b of C4.
if "is_original" not in daily.columns:
    daily["is_original"] = 1
daily["is_original"] = pd.to_numeric(
    daily["is_original"], errors="coerce").fillna(1).astype(int)
n_imputed_days = int((daily["is_original"] == 0).sum())
print(f"  {len(daily)} calendar rows, {n_imputed_days} imputed holidays, "
      f"calendar={CAL_TAG}")

if LIVE_CLOSE is not None:
    # Append today as a row so every feature builder sees it like any other day.
    today = pd.Timestamp(datetime.date.today()).normalize()
    if today in set(daily[DATE]):
        daily.loc[daily[DATE] == today, CLOSE] = float(LIVE_CLOSE)
        if LIVE_OPEN is not None:
            daily.loc[daily[DATE] == today, OPEN] = float(LIVE_OPEN)
        print(f"  overwrote {today.date()} with the live level {LIVE_CLOSE}")
    else:
        if LIVE_OPEN is None:
            raise RuntimeError(
                "  LIVE_CLOSE was given for a date not in the clean file, so "
                "LIVE_OPEN is needed too — gap_Dn is ln(Open / previous Close) "
                "and cannot be built without today's open.")
        # is_original = 1: today is a session that is actually happening, so it
        # must enter the volatility window and must be visible to is_expiry.
        daily = pd.concat([daily, pd.DataFrame([{
            DATE: today, OPEN: float(LIVE_OPEN), HIGH: float(LIVE_CLOSE),
            LOW: float(LIVE_CLOSE), CLOSE: float(LIVE_CLOSE),
            "is_original": 1}])],
            ignore_index=True).sort_values(DATE).reset_index(drop=True)
        daily["is_original"] = daily["is_original"].fillna(1).astype(int)
        print(f"  appended {today.date()} at the live level {LIVE_CLOSE}")

# Flags from the REAL trading days; the cycle's structure from the full calendar.
# Passing the filled calendar here would hand is_expiry a set in which no day is
# ever missing, so its holiday fallback could never fire and an early settlement
# would go unrecognised — moving every cycle boundary after it.
real_dates = set(daily.loc[daily["is_original"] == 1, DATE])
last_real = daily.loc[daily["is_original"] == 1, DATE].max()
all_dates = real_dates
last_date = daily[DATE].max()
flags = np.array([is_expiry(d, real_dates, last_real) and d in real_dates
                  for d in daily[DATE]], int)
expiry_pos = np.flatnonzero(flags == 1)

if len(expiry_pos) == 0:
    raise RuntimeError("  no expiry found in the daily file.")
if expiry_pos[-1] == len(daily) - 1:
    raise SystemExit(
        f"  {last_date.date()} IS an expiry, so no cycle is open. The next cycle "
        f"begins on the following trading day.")

open_block = daily.iloc[expiry_pos[-1] + 1:].reset_index(drop=True)
n_so_far = len(open_block)          # decision days completed, today included
print(f"  open cycle began {open_block[DATE].iloc[0].date()}, "
      f"today is {last_date.date()}, decision day D{n_so_far}")

if n_so_far < 2:
    raise SystemExit(
        "  today is D1 of the open cycle and there is no D1 task — the band is "
        "drawn at D1's close, so the first decision is taken on D2.")
if n_so_far > 4:
    raise SystemExit(
        f"  {n_so_far} decision days have passed without an expiry. Either the "
        f"daily file is missing an expiry or the expiry rule needs checking.")


# ---- project the expiry, and with it days_left --------------------------------
def project_expiry(after):
    """The next scheduled expiry strictly after a date, from the weekday rule.

    Parameters
    ----------
    after : pandas.Timestamp

    Returns
    -------
    pandas.Timestamp

    Notes
    -----
    A PROJECTION. It applies the Tuesday or Thursday rule and assumes that day
    trades. An unscheduled holiday would pull the real expiry earlier, making the
    true horizon shorter than assumed — and every z_dist slightly too small,
    because it divides by sqrt(days_left).
    """
    d = after + pd.Timedelta(days=1)
    for _ in range(30):
        target = 1 if d >= EXPIRY_CUTOFF else 3
        if d.weekday() == target:
            return d.normalize()
        d += pd.Timedelta(days=1)
    raise RuntimeError("  could not project the next expiry.")


expiry_date = project_expiry(last_date)
future_days = pd.bdate_range(last_date + pd.Timedelta(days=1), expiry_date)
days_left_now = len(future_days)
cycle_days = n_so_far + max(days_left_now - 1, 0)

print(f"  projected expiry {expiry_date.date()}, days_left={days_left_now}, "
      f"cycle_days={cycle_days}")
if cycle_days != 4:
    print(f"  note: this cycle projects to {cycle_days} decision days, not 4")


# ============================================================================
# 2. THE LIVE ROW, BUILT LIKE ANY COMPLETED CYCLE
# ----------------------------------------------------------------------------
# expiry_close is unknown, so the row carries NaN there and gets no label. It is
# appended to the completed cycles ONLY so the band statistics roll forward onto
# it; it contributes nothing to those statistics, because they are lagged by one
# cycle.
# ============================================================================
# ============================================================================
# 1b. JOIN THE MACRO FILE, IF C2c PRODUCED ONE
# ----------------------------------------------------------------------------
# This is the step whose absence made a macro FEATURE_SET silently train on
# z_dist alone. C2c writes 01c_macro_daily_<cal>.xlsx; nothing read it. Now the
# columns are merged onto `daily` by date, and add_features maps them onto each
# decision day exactly as it does realized_vol.
#
# The join is a LEFT join on the daily calendar, so it can add columns but never
# rows — a macro file with extra dates cannot change which days exist.
#
# The print is the point. A short feature list is otherwise indistinguishable
# from a working one, which is how the first version of this went unnoticed.
# ============================================================================
_macro_cols = []
if os.path.exists(F_MACRO):
    _mac = read_xlsx(F_MACRO, sheet="macro")
    _mac[DATE] = pd.to_datetime(_mac[DATE]).dt.normalize()
    _macro_cols = [c for c in MACRO_SOURCE_COLS if c in _mac.columns]
    if _macro_cols:
        daily[DATE] = pd.to_datetime(daily[DATE]).dt.normalize()
        daily = daily.merge(_mac[[DATE] + _macro_cols], on=DATE, how="left")
        _cov = {c: f"{100.0 * daily[c].notna().mean():.0f}%" for c in _macro_cols}
        print(f"  macro joined from {os.path.basename(F_MACRO)}: {_cov}")
    else:
        print(f"  macro file found but none of {MACRO_SOURCE_COLS} are in it")
else:
    print(f"  no macro file at {os.path.basename(F_MACRO)} — "
          f"vix/usdinr/brent features will NOT be built")

if FEATURE_SET in ("z_plus_macro", "z_only_plus_vix") and not _macro_cols:
    raise RuntimeError(
        f"  FEATURE_SET={FEATURE_SET!r} needs the macro file, and\n"
        f"  {F_MACRO}\n"
        f"  is missing or empty. Run C2c_FetchMacro first.\n"
        f"  Refusing rather than quietly training on z_dist alone — that is\n"
        f"  exactly the failure this check exists to stop.")

cycles = read_xlsx(F_CYCLES, sheet="cycles")
for c in ("expiry_date",) + tuple(f"d{i}_date" for i in (1, 2, 3, 4)):
    cycles[c] = pd.to_datetime(cycles[c], errors="coerce")

live = {"cycle_id": int(cycles["cycle_id"].max()) + 1,
        "cycle_days": cycle_days, "is_padded": 0,
        "expiry_date": expiry_date, "expiry_close": np.nan}
for i in range(1, 5):
    live[f"d{i}_date"] = (open_block[DATE].iloc[i - 1] if i <= n_so_far else pd.NaT)
    live[f"d{i}_close"] = (float(open_block[CLOSE].iloc[i - 1]) if i <= n_so_far
                           else np.nan)

combined = pd.concat([cycles, pd.DataFrame([live])], ignore_index=True)
print(f"  {len(cycles)} completed cycles + 1 open")


# ============================================================================
# 3. WHICH WINDOW WON, PER (SIGMA, TASK)
# ============================================================================
best_window = {}
for sigma in SIGMA_GRID:
    for window in SIGMA_WINDOW_GRID:
        path = f_results(sigma, window)
        if not os.path.exists(path):
            continue
        cand = read_xlsx(path, sheet="candidates")
        for task, g in cand.groupby("task"):
            score = float(g["oof_score"].max())
            key = (sigma, task)
            if key not in best_window or score > best_window[key][1]:
                best_window[key] = (window, score)

if not best_window:
    print("  no C6 results found, falling back to the first window in the grid")
    for sigma in SIGMA_GRID:
        for t in TASKS:
            best_window[(sigma, f"D{t['day']}_{t['side']}")] = (
                SIGMA_WINDOW_GRID[0], np.nan)


# ============================================================================
# 4. REFIT ON EVERYTHING, THEN PREDICT
# ============================================================================
def fit_and_predict(feat_all, live_pos, day, side, sigma, window):
    """Refit the ensemble on all completed cycles and score the open one.

    Parameters
    ----------
    feat_all : pandas.DataFrame
        Completed cycles plus the live row, with bands and features attached.
    live_pos : int
        Index of the live row.
    day, side : int, str
        The task.
    sigma, window : float, int
        The configuration.

    Returns
    -------
    dict or None
        Probabilities, decisions and the members used. None when the task cannot
        be fitted.

    Notes
    -----
    The threshold is cross-fitted over the whole training set, exactly as in C6.
    Members are ranked by their own out-of-fold score and the gate is applied; if
    nothing passes, the ensemble is still built from the top k and `gated` is
    False, so a weak live prediction is labelled rather than hidden.
    """
    # available_features, not task_features: C5 may have dropped a constant
    # column, and asking for one that is not there would fail here rather than in
    # the place the decision was made.
    rows = task_rows(feat_all, day)
    cols = available_features(rows, day, side)
    if live_pos not in rows.index:
        return None
    live_row = rows.loc[[live_pos]]
    train = rows.drop(index=live_pos)
    label = f"{side}_breach"

    ok = train[label].notna() & np.isfinite(train[cols]).all(axis=1)
    train = train[ok]
    if len(train) < 40 or not np.isfinite(live_row[cols].to_numpy(float)).all():
        return None

    X = train[cols].to_numpy(float)
    y = train[label].to_numpy(int)
    x_live = live_row[cols].to_numpy(float)

    from sklearn.base import clone
    from sklearn.model_selection import RandomizedSearchCV

    members = []
    for name in MODELS:
        pipe = make_pipeline(name)
        n_pos = int(y.sum())
        inner = min(3, n_pos, int((1 - y).sum()))
        if inner >= 2:
            search = RandomizedSearchCV(pipe, HYPERPARAMS[name], n_iter=TUNE_ITERS,
                                        cv=inner, scoring="f1", n_jobs=-1,
                                        random_state=RANDOM_STATE, error_score=0.0)
            search.fit(X, y)
            est = search.best_estimator_
        else:
            est = pipe.fit(X, y)

        oof = np.full(len(y), np.nan)
        for fit_idx, out_idx in stratified_folds(y):
            if len(np.unique(y[fit_idx])) < 2:
                continue
            m = clone(make_pipeline(name)).fit(X[fit_idx], y[fit_idx])
            oof[out_idx] = m.predict_proba(X[out_idx])[:, 1]
        good = np.isfinite(oof)
        thr, _ = best_f1_threshold(y[good], oof[good], fallback=0.5)
        if thr is None:
            continue
        from_in = est.predict_proba(X)[:, 1]
        s_in = f1_at(y, from_in, thr)
        s_oof = f1_at(y[good], oof[good], thr)
        members.append({"model": name, "infold_score": s_in, "oof_score": s_oof,
                        "threshold": thr,
                        "prob": float(est.predict_proba(x_live)[0, 1]),
                        "passed_gate": model_passes([(s_in, s_oof)])[0]})

    if not members:
        return None
    cand = pd.DataFrame(members)
    eligible = cand[cand["passed_gate"]]
    gated = not eligible.empty
    chosen = pick_top_k(eligible if gated else cand, TOP_K)

    dec, prob, eff = ensemble(list(chosen["prob"]), list(chosen["threshold"]),
                              list(chosen["oof_score"]))
    return {"ml_prob": prob, "ml_decision": dec, "ml_threshold": eff,
            "ml_gated": gated, "n_train": len(train),
            "members": ", ".join(f"{r.model}({r.oof_score:.2f})"
                                 for r in chosen.itertuples()),
            "candidates": cand.assign(task=f"D{day}_{side}", sigma=sigma,
                                      window=window)}


# Every decision day from D2 to today. On D2 that is one day, on D4 it is three,
# so the run gets longer later in the week — the cost of seeing the trajectory.
DAYS_TO_SCORE = list(range(2, n_so_far + 1))
print(f"  scoring decision day(s): {', '.join('D' + str(d) for d in DAYS_TO_SCORE)}")

live_rows, all_cands, input_rows = [], [], []
for sigma in SIGMA_GRID:
  for day in DAYS_TO_SCORE:
    for side in ("upper", "lower"):
        task = f"D{day}_{side}"
        window, won_with = best_window.get(
            (sigma, task), (SIGMA_WINDOW_GRID[0], np.nan))

        banded = add_bands_and_labels(combined, sigma, window)
        feat_all = add_features(banded, daily, window)
        live_pos = feat_all.index[-1]
        row = feat_all.loc[live_pos]

        gp = float(gaussian_breach_prob(feat_all.loc[[live_pos]], day, side)[0])
        out = fit_and_predict(feat_all, live_pos, day, side, sigma, window)

        # The Gaussian needs an F1-optimal cut too, fitted on the completed cycles
        # of this task exactly as C6 fits it. Without this the two contestants
        # would be compared under different rules: ML at its own fitted cut and
        # the baseline at p*, which is not a comparison of forecasts at all.
        _rows = task_rows(feat_all, day).drop(index=live_pos)
        _lab = _rows[f"{side}_breach"]
        _ok = _lab.notna()
        _gtr = gaussian_breach_prob(_rows[_ok], day, side)
        g_thr_f1, _ = best_f1_threshold(_lab[_ok].astype(int).to_numpy(),
                                        _gtr, fallback=0.5)
        if g_thr_f1 is None:
            g_thr_f1 = 0.5

        # days_left for a PAST day of this same cycle is known exactly, because
        # the projected expiry is fixed. Only today's depends on the projection.
        dl = int(cycle_days + 1 - day)
        entry = {"sigma": sigma, "window": window, "task": task,
                 "decision_day": day, "is_today": int(day == n_so_far),
                 "date": open_block[DATE].iloc[day - 1].date(), "side": side,
                 "asof": last_date.date(), "expiry_projected": expiry_date.date(),
                 "days_left": dl, "cycle_days": cycle_days,
                 "close_used": float(row[f"d{day}_close"]),
                 "band_upper": float(row["band_upper"]),
                 "band_lower": float(row["band_lower"]),
                 "distance_to_band_pct": round(
                     100.0 * (float(row[f"band_{side}"])
                              / float(row[f"d{day}_close"]) - 1.0), 3),
                 "gauss_prob": gp,
                 "gauss_threshold_f1": round(float(g_thr_f1), 4),
                 "gauss_decision_f1": int(gp >= g_thr_f1),
                 "gauss_verdict_f1": verdict(int(gp >= g_thr_f1), side),
                 "gauss_decision_econ": int(gp >= ECONOMIC_THRESHOLD),
                 "gauss_verdict_econ": verdict(int(gp >= ECONOMIC_THRESHOLD), side),
                 "window_chosen_on_oof": round(won_with, 4)
                 if np.isfinite(won_with) else "no C6 results"}
        if out is None:
            entry.update({"ml_prob": np.nan, "ml_threshold_f1": np.nan,
                          "ml_decision_f1": np.nan, "ml_verdict_f1": "no prediction",
                          "ml_decision_econ": np.nan,
                          "ml_verdict_econ": "no prediction",
                          "ml_gated": False,
                          "ensemble_members": "could not fit", "n_train": 0})
        else:
            _flag = "" if out["ml_gated"] else "  [models failed the gate]"
            _d_econ = int(out["ml_prob"] >= ECONOMIC_THRESHOLD)
            entry.update({
                "ml_prob": out["ml_prob"],
                "ml_threshold_f1": round(float(out["ml_threshold"]), 4),
                "ml_decision_f1": out["ml_decision"],
                "ml_verdict_f1": verdict(out["ml_decision"], side) + _flag,
                "ml_decision_econ": _d_econ,
                "ml_verdict_econ": verdict(_d_econ, side) + _flag,
                "ml_gated": out["ml_gated"],
                "ensemble_members": out["members"], "n_train": out["n_train"]})
            all_cands.append(out["candidates"])

        # Agreement is reported PER RULE. Comparing one contestant's F1 call with
        # the other's economic call is not a disagreement about the forecast, only
        # about the cut — and mixing them is how a table like this misleads.
        for _r, _mk, _gk in (("f1", "ml_decision_f1", "gauss_decision_f1"),
                             ("econ", "ml_decision_econ", "gauss_decision_econ")):
            entry[f"agreement_{_r}"] = (
                "ML absent" if pd.isna(entry[_mk])
                else "agree" if int(entry[_mk]) == int(entry[_gk])
                else "DISAGREE")
        live_rows.append(entry)

        input_rows.append({"sigma": sigma, "window": window,
                           "decision_day": day, "side": side,
                           **{c: float(row[c]) for c in task_features(day, side)
                              if c in row.index and pd.notna(row[c])}})

live_df = pd.DataFrame(live_rows)

print("\n  bands and probabilities")
print(live_df[["sigma", "decision_day", "date", "side", "close_used",
               "band_upper", "band_lower", "distance_to_band_pct", "days_left",
               "gauss_prob", "ml_prob"]].to_string(index=False))

print("\n  at the F1-optimal cut (each contestant's own fitted threshold)")
print(live_df[["sigma", "decision_day", "side", "gauss_threshold_f1",
               "gauss_verdict_f1", "ml_threshold_f1", "ml_verdict_f1",
               "agreement_f1"]].to_string(index=False))

print(f"\n  at the economic cut p*={ECONOMIC_THRESHOLD:.4f} (the same for both)")
print(live_df[["sigma", "decision_day", "side", "gauss_verdict_econ",
               "ml_verdict_econ", "agreement_econ"]].to_string(index=False))

# TODAY is the row that matters for acting. Printed on its own so it is not lost
# among the earlier days of the same cycle.
today_rows = live_df[live_df["is_today"] == 1]
print(f"\n  TODAY — D{n_so_far}, {last_date.date()}, "
      f"{days_left_now} day(s) to expiry")
for _, r in today_rows.iterrows():
    band = r["band_upper"] if r["side"] == "upper" else r["band_lower"]
    print(f"    sigma {r['sigma']}  {r['side']:<5}  band {band:.1f}  "
          f"({r['distance_to_band_pct']:+.2f}% away)")
    print(f"{'':>10}Gaussian p={r['gauss_prob']:.3f}  "
          f"F1 cut -> {r['gauss_verdict_f1']}")
    print(f"{'':>10}{'':>18}economic -> {r['gauss_verdict_econ']}")
    if pd.notna(r["ml_prob"]):
        print(f"{'':>10}ML       p={r['ml_prob']:.3f}  "
              f"F1 cut -> {r['ml_verdict_f1']}")
        print(f"{'':>10}{'':>18}economic -> {r['ml_verdict_econ']}")
    else:
        print(f"{'':>10}ML       no prediction")


# ============================================================================
# 5. CHECKS
# ============================================================================
fails = []


def chk(nm, ok, detail=""):
    if not ok:
        fails.append(f"{nm}: {detail}")
    print(f"    {'OK  ' if ok else 'FAIL'} {nm}" + ("" if ok else f"  -- {detail}"))


print("\n  checks")
chk("the open cycle has no label",
    bool(pd.isna(combined["expiry_close"].iloc[-1])),
    "expiry_close must be unknown for a cycle still in progress")
chk("the live row was not used to build its own band",
    len(combined) == len(cycles) + 1)
chk("every sigma produced both sides for every day scored",
    len(live_df) == 2 * len(SIGMA_GRID) * len(DAYS_TO_SCORE),
    f"{len(live_df)} rows against "
    f"{2 * len(SIGMA_GRID) * len(DAYS_TO_SCORE)} expected")
chk("exactly one row per (sigma, day, side)",
    not live_df.duplicated(["sigma", "decision_day", "side"]).any())
chk("days_left counts down as the day index rises",
    bool((live_df.sort_values("decision_day")
          .groupby(["sigma", "side"])["days_left"]
          .apply(lambda s: s.is_monotonic_decreasing)).all()))
chk("every row carries a verdict in words under both rules",
    bool(live_df[["gauss_verdict_f1", "gauss_verdict_econ",
                  "ml_verdict_f1", "ml_verdict_econ"]].notna().all().all()))
chk("each verdict matches its own decision",
    bool(all((("BREACH" in r["gauss_verdict_econ"])
              == (int(r["gauss_decision_econ"]) == 1))
             and (("BREACH" in r["gauss_verdict_f1"])
                  == (int(r["gauss_decision_f1"]) == 1))
             for _, r in live_df.iterrows())),
    "a verdict and its 0/1 decision have drifted apart")
chk("agreement is computed within a rule, not across rules",
    bool(all((r[f"agreement_{k}"] == "ML absent")
             or ((r[f"agreement_{k}"] == "agree")
                 == (int(r[f"ml_decision_{m}"]) == int(r[f"gauss_decision_{m}"])))
             for _, r in live_df.iterrows()
             for k, m in (("f1", "f1"), ("econ", "econ")))))
chk("the upper band sits above the lower one",
    bool((live_df["band_upper"] > live_df["band_lower"]).all()))
chk("each row used its own day's close",
    bool(all(np.isclose(r["close_used"],
                        float(open_block[CLOSE].iloc[int(r["decision_day"]) - 1]))
             for _, r in live_df.iterrows())))
chk("days_left is at least 1", days_left_now >= 1)
chk("probabilities are within [0, 1]",
    bool(live_df[["gauss_prob", "ml_prob"]].stack().between(0, 1).all()))
if fails:
    raise RuntimeError("  C8 checks failed:\n    - " + "\n    - ".join(fails))


# ============================================================================
# 6. WRITE
# ============================================================================
manifest = pd.DataFrame([
    ("script", "C8_Live"),
    ("as of", str(last_date.date())),
    ("decision day today", f"D{n_so_far}"),
    ("days scored", ", ".join("D" + str(d) for d in DAYS_TO_SCORE)),
    ("projected expiry", str(expiry_date.date())),
    ("days_left", days_left_now),
    ("cycle_days projected", cycle_days),
    ("close used", "live tick" if LIVE_CLOSE is not None else "last stored close"),
    ("daily_calendar", CAL_TAG),
    ("cycle_lengths", CYCLE_LENGTHS),
    ("impute_daily", IMPUTE_DAILY),
    ("realized_vol window (days)", rv_window_days(SIGMA_WINDOW_GRID[0])),
    ("imputed holidays in the daily file", n_imputed_days),
    ("feature_set", FEATURE_SET),
    ("sigmas", ", ".join(str(s) for s in SIGMA_GRID)),
    ("window choice", "highest mean OUT-OF-FOLD score in C6, per (sigma, task)"),
    ("models refitted on", "every completed cycle, no sealed block"),
    ("economic cut p*", round(ECONOMIC_THRESHOLD, 4)),
    ("timing caveat",
     "built for a decision minutes before the close. days_left assumes the "
     "projected expiry trades; an unscheduled holiday shortens the real horizon "
     "and makes every z_dist slightly too small."),
    ("generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
], columns=["key", "value"])

sheets = {"live": live_df, "inputs": pd.DataFrame(input_rows),
          "manifest": manifest}
if all_cands:
    sheets["candidates"] = pd.concat(all_cands, ignore_index=True)
write_xlsx(F_LIVE, sheets)

print("\n" + "=" * 78)
print(f"  C8_Live complete — {F_LIVE.split('/')[-1]}")
print("  Re-run it each day of the cycle. A prediction marked ml_gated=False came")
print("  from models that failed the consistency gate: read the Gaussian instead.")
print("=" * 78)


  C8_Live — the open cycle, calendar=filled
  2006 calendar rows, 92 imputed holidays, calendar=filled
  open cycle began 2026-09-30, today is 2026-10-01, decision day D2
  projected expiry 2026-10-06, days_left=3, cycle_days=4
  macro joined from 01c_macro_daily_filled.xlsx: {'india_vix': '100%', 'vix_daily': '100%', 'usdinr_ln1': '100%', 'usdinr_ln5': '100%', 'brent_ln1': '100%', 'brent_ln5': '100%'}
  386 completed cycles + 1 open
  scoring decision day(s): D2


KeyboardInterrupt: 